# Clean replacement — seven-policy pilot with heterogeneous-κ fallback

Revision identifier: `2026-08-02-clean-v4-seven-policy-pilot`. All stored outputs and execution counts were removed before delivery. The companion files are `benchmark_indices_rkappa_v4.jl` and `seven_policy_pilot_rkappa_v1.jl`.


In [1]:
# Setup cell
using Pkg

Pkg.instantiate()
@show VERSION
@show Base.active_project()
Pkg.status()

VERSION = v"1.12.6"
Base.active_project() = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\Project.toml"
Status `C:\Users\JNINO\Documents\GitHub\paperPCLsensingErrors\Project.toml`
  [336ed68f] CSV v0.10.16
  [a93c6f00] DataFrames v1.8.2
  [31c24e10] Distributions v0.25.130
  [7073ff75] IJulia v1.34.4
  [b964fa9f] LaTeXStrings v1.4.0
  [8314cec4] PGFPlotsX v1.6.2
  [2913bbd2] StatsBase v0.34.12


In [2]:
# pwd()

In [3]:
# CPU/thread cell

@show Sys.CPU_THREADS # logical CPU threads
@show Threads.nthreads(:default) # threads available for computations
@show Threads.nthreads(:interactive) # separate interactive threads

Sys.CPU_THREADS = 12
Threads.nthreads(:default) = 12
Threads.nthreads(:interactive) = 1


1

In [4]:
# import cell

using PGFPlotsX, LaTeXStrings
using CSV, DataFrames
using StatsBase, Distributions
using Statistics, Printf

In [5]:
# --- belief updates (no z) ---

"""
    φ0(x; p01, ρ)

Passive belief update: φ⁰(x)=p01+ρx.
"""
φ0(x; p01, ρ) = p01 + ρ*x

"""
    φ1(x; p01, ρ, κ)

Active no-ACK belief update: φ¹(x)= p01 + ρ * ((1-κ)x)/(1-κx).
"""
φ1(x; p01, ρ, κ) = p01 + ρ * ((1 - κ) * x) / (1 - κ * x)


# --- helpers for τ_{[0,z]}^1 ---

"""
    φ1_fixed_points(; p01, ρ, κ) -> (x_lo, x_hi)
"""
function φ1_fixed_points(; p01, ρ, κ)
    p11 = p01 + ρ
    B = 1 - ρ + κ * p11
    Δ = B^2 - 4 * κ * p01
    @assert Δ > 0 "Discriminant must be positive (check parameters)."
    sΔ = sqrt(Δ)
    xlo = (B - sΔ) / (2 * κ)
    xhi = (B + sΔ) / (2 * κ)
    return xlo, xhi
end

"""
    fixed_points(; p01, ρ, κ) -> (x0, x1, xhi)

Return the passive fixed point x⁰ and the two fixed points of
the active no-ACK map φ¹.
"""
function fixed_points(; p01, ρ, κ)
    x0 = p01 / (1 - ρ)
    x1, xhi = φ1_fixed_points(; p01=p01, ρ=ρ, κ=κ)
    return x0, x1, xhi
end


# --- crossing times (depend on z) ---

"""
    τ0(x, z; p01, ρ)

Passive threshold up-crossing time: min{t≥0 : φ_t⁰(x) > z}.
Returns Inf if no up-crossing occurs.
"""
function τ0(x, z; p01, ρ)
    x0 = p01 / (1 - ρ)

    if x > z
        return 0
    elseif z >= x0
        return Inf
    else
        val = log((x0 - z) / (x0 - x)) / log(ρ)

        # Closed-form candidate.
        t = max(1, 1 + floor(Int, val))

        # Directly evaluate φ_t⁰(x), correcting possible rounding
        # of the logarithmic expression near an integer boundary.
        φ0_iterate(t) = muladd(x - x0, ρ^t, x0)

        # Ensure that the strict crossing has occurred.
        while φ0_iterate(t) <= z
            t += 1
        end

        # Ensure that t is the first strict crossing time.
        while t > 0 && φ0_iterate(t - 1) > z
            t -= 1
        end

        return t
    end
end

"""
    τ1_0z(x, z; p01, ρ, κ)

Active no-ACK down-crossing time into [0,z]:
min{t≥0 : φ_t¹(x) ≤ z}.

Returns 0 if x≤z, Inf if x>z and z≤x¹, else the closed form.
"""
function τ1_0z(x, z; p01, ρ, κ)
    if x <= z
        return 0
    end

    x1, xhi = φ1_fixed_points(; p01, ρ, κ)          # x1 = x_lo^1
    if z <= x1
        return Inf
    end

    μ  = ρ * (1 - κ) / (1 - κ * x1)^2             # (φ¹)'(x¹) ∈ (0,1)
    Ψ(u) = (u - x1) / (u - xhi)

    rhs = Ψ(z) / Ψ(x)                              # > 0 when x>z>x1
    val = log(rhs) / log(μ)                        # log(μ) < 0
    t = ceil(Int, val - 1e-12)                     # numerical guard
    return max(t, 0)
end


# Optional ASCII aliases:
phi0(x; p01, rho) = φ0(x; p01=p01, ρ=rho)
phi1(x; p01, rho, kappa) = φ1(x; p01=p01, ρ=rho, κ=kappa)
tau0(x, z; p01, rho) = τ0(x, z; p01=p01, ρ=rho)
tau1_0z(x, z; p01, rho, kappa) = τ1_0z(x, z; p01=p01, ρ=rho, κ=kappa)

tau1_0z (generic function with 1 method)

In [6]:
# Skeleton map-with-a-gap ϕ
ϕ(x, z; p01, ρ, κ) = (x > z) ? φ1(x; p01=p01, ρ=ρ, κ=κ) : φ0(x; p01=p01, ρ=ρ)

"""
    varphi_t(x, z, t; p01, ρ, κ)

Iterates of the map-with-a-gap ϕ:
    ϕ₀(x,z)=x,
    ϕ_t(x,z)=ϕ(ϕ_{t-1}(x,z), z),  t≥1.

Assumes ϕ(x,z; p01,ρ,κ) is already defined.
"""
function varphi_t(x, z, t::Integer; p01, ρ, κ)
    @assert t ≥ 0 "t must be nonnegative"
    y = x
    for _ in 1:t
        y = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
    end
    return y
end

"""
    skeleton_orbit(x, z, T; p01, ρ, κ)

Returns the skeleton orbit X̃₀..X̃_T, where X̃_t = ϕ_t(x,z).
"""
function skeleton_orbit(x, z, T::Integer; p01, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    X = Vector{Float64}(undef, T + 1)
    y = x
    for t in 0:T
        X[t+1] = y
        if t < T
            y = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
        end
    end
    return X
end

"""
    skeleton_itinerary(x, z, T; p01, ρ, κ)

Returns the skeleton itinerary Ã₀..Ã_T as Ints, where Ã_t = 1{X̃_t > z}.
"""
function skeleton_itinerary(x, z, T::Integer; p01, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    A = Vector{Int}(undef, T + 1)
    y = x
    for t in 0:T
        A[t+1] = (y > z) ? 1 : 0
        if t < T
            y = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
        end
    end
    return A
end

"""
    skeleton_orbit_itinerary(x, z, T; p01, ρ, κ)

One-pass computation of both (often handy):
Returns (X, A) with X[t+1]=X̃_t and A[t+1]=Ã_t for t=0..T.
"""
function skeleton_orbit_itinerary(x, z, T::Integer; p01, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    X = Vector{Float64}(undef, T + 1)
    A = Vector{Int}(undef, T + 1)

    y = x
    for t in 0:T
        X[t+1] = y
        A[t+1] = (y > z) ? 1 : 0
        if t < T
            y = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
        end
    end
    return X, A
end

"""
    skeleton_itinerary_string(x, z, T; p01, ρ, κ)

String encoding of the skeleton itinerary (Ã₀..Ã_T), where Ã_t = 1{ϕ_t(x,z) > z}.
Returns a String of length T+1 over {'0','1'}.
"""
function skeleton_itinerary_string(x, z, T::Integer; p01, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"

    chars = Vector{Char}(undef, T + 1)
    y = x

    for t in 0:T
        chars[t+1] = (y > z) ? '1' : '0'
        if t < T
            y = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
        end
    end

    return String(chars)
end

# Optional ASCII aliases
Xtilde(x, z, T; p01, rho, kappa) = skeleton_orbit(x, z, T; p01=p01, ρ=rho, κ=kappa)
Atilde(x, z, T; p01, rho, kappa) = skeleton_itinerary(x, z, T; p01=p01, ρ=rho, κ=kappa)

Atilde (generic function with 1 method)

In [7]:
"""
    Gamma_t(x, z, t; p01, ρ, κ)

No-ACK survival probability:
    Γ_t(x,z) = P_x^z{T ≥ t},  t ≥ 0

Recursion/product (paper):
    Γ_0 = 1
    Γ_t = ∏_{j=0}^{t-1} (1 - κ * X̃_j(x,z) * Ã_j(x,z))
        = ∏_{0 ≤ j < t : X̃_j(x,z) > z} (1 - κ * X̃_j(x,z))

Computed in one pass without allocating orbit or itinerary vectors.
"""
function Gamma_t(x, z, t::Integer; p01, ρ, κ)
    @assert t ≥ 0 "t must be nonnegative"
    Γ = 1.0
    y = x
    for j in 0:(t-1)
        if y > z
            Γ *= (1 - κ * y)
        end
        y = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
    end
    return Γ
end


"""
    Gamma_path(x, z, T; p01, ρ, κ)

Returns Γ[1]=Γ_0, ..., Γ[T+1]=Γ_T as a Vector{Float64}.
Computed in one pass using the recursion.
"""
function Gamma_path(x, z, T::Integer; p01, ρ, κ)
    @assert T ≥ 0 "T must be nonnegative"
    Γ = Vector{Float64}(undef, T + 1)
    Γ[1] = 1.0
    y = x
    for t in 1:T
        # update from Γ_{t-1} to Γ_t using X̃_{t-1}=y and Ã_{t-1}=1{y>z}
        Γ[t+1] = Γ[t] * ((y > z) ? (1 - κ * y) : 1.0)
        y = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
    end
    return Γ
end


"""
    Gamma_t_from_skeleton(x, z, t; p01, ρ, κ)

Same Γ_t(x,z) but uses your (X̃, Ã) vectors for clarity/debugging.
This allocates X and A up to t-1.
"""
function Gamma_t_from_skeleton(x, z, t::Integer; p01, ρ, κ)
    @assert t ≥ 0 "t must be nonnegative"
    t == 0 && return 1.0
    X, A = skeleton_orbit_itinerary(x, z, t-1; p01=p01, ρ=ρ, κ=κ)
    Γ = 1.0
    for j in 0:(t-1-0)  # j=0..t-1
        if A[j+1] == 1
            Γ *= (1 - κ * X[j+1])
        end
    end
    return Γ
end


"""
    Gamma_infty(x, z; p01, ρ, κ)

Infinite-horizon no-ACK probability Γ∞(x,z).

Under the model assumptions p01>0 and κ>0:

- If z < x⁰, activations occur infinitely often and Γ∞=0.
- If z ≥ x⁰ and x≤z, the skeleton remains passive and Γ∞=1.
- If z ≥ x⁰ and x>z, there is a finite initial active phase,
  followed by permanent passivity.
"""
function Gamma_infty(x, z; p01, ρ, κ)
    @assert p01 > 0
    @assert 0 < ρ < 1
    @assert 0 < κ <= 1

    x0 = p01 / (1 - ρ)

    if z < x0
        return 0.0
    elseif x <= z
        return 1.0
    else
        t1 = τ1_0z(x, z; p01=p01, ρ=ρ, κ=κ)
        @assert t1 isa Integer "A finite down-crossing is expected when z ≥ x⁰."
        return Gamma_t(x, z, t1; p01=p01, ρ=ρ, κ=κ)
    end
end

Gamma_infty

In [8]:
"""
    plot_Gamma_path_axis(x, z, T; p01, ρ, κ, ...)

Axis for Γ_t(x,z) vs t=0..T with NO grid and NO title by default.
"""
function plot_Gamma_path_axis(x, z, T::Integer; p01, ρ, κ,
                              xlabel=L"$t$",
                              ylabel=L"$\Gamma_t(x,z)$",
                              ymin=0, ymax=1, xmin=0,
                              width="11cm", height="7cm",
                              mark="*")
    @assert T ≥ 0 "T must be nonnegative"

    Γ = Gamma_path(x, z, T; p01=p01, ρ=ρ, κ=κ)
    coords = collect(zip(0:T, Γ))

    # Robust ticks for arbitrary T, including T=0.
    xticks = if T == 0
        "{0}"
    else
        values = unique(vcat(collect(0:5:T), T))
        "{" * join(values, ",") * "}"
    end

    # Avoid a degenerate x-axis when only Γ₀ is plotted.
    xmax_plot = (T == 0) ? 1 : T

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,
            ymin   => ymin,
            ymax   => ymax,
            xmin => xmin,
            xmax    => xmax_plot,
            "xtick" => xticks,
            "ytick" => "{0,1}",
            "tick align" => "outside",
            "scaled y ticks" => "false",
            # no grid, no title
        },
        Plot(
            {
                "mark" => mark,
                "mark size" => "1.6pt",
                "line width" => "0.5pt",
            },
            Coordinates(coords)
        )
    )
end

"""
    save_Gamma_path_plot_pdf(outpdf, x, z, T; p01, ρ, κ, engine="pdflatex", quiet=true)

Writes standalone .tex and compiles to PDF.
If quiet=true, suppresses LaTeX console output; prints log tail only on failure.
"""
function save_Gamma_path_plot_pdf(outpdf::AbstractString, x, z, T::Integer;
                                  p01, ρ, κ,
                                  axisfun::Function = plot_Gamma_path_axis,   # <--- add
                                  engine::AbstractString="pdflatex",
                                  quiet::Bool=true,
                                  keep_tex::Bool=true,
                                  kwargs...)
    ax  = axisfun(x, z, T; p01=p01, ρ=ρ, κ=κ, kwargs...)  # <--- change
    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"             # also normalizes .PDF to .pdf
    else
        throw(ArgumentError("outpdf must have extension .pdf or no extension"))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs 

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    engine_path = Sys.which(engine)
    engine_path === nothing &&
        throw(ArgumentError("LaTeX engine '$engine' was not found"))
    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    isfile(pdffile) ||
    error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_Gamma_path_plot_pdf

In [9]:
# parameters (example)
p01  = 0.25
ρ  = 0.6
κ  = 0.8

x0, x1, xhi = fixed_points(; p01=p01, ρ=ρ, κ=κ)
z = 0.5 * (x1 + x0)
x  = z - 0.3
T  = 25

save_Gamma_path_plot_pdf("Gamma_path.pdf", x, z, T; p01=p01, ρ=ρ, κ=κ)  # quiet by default

"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\Gamma_path.pdf"

In [10]:
###############################################################################
# Pre-ACK metrics with guaranteed truncation error (X̃_t ∈ [0,1])
#
# Uses a common truncation horizon T based on the geometric tail bound for G̃:
#   tail_geom(T) = β^(T+1)/(1-β).
#
# Guarantees, assuming X̃_t ∈ [0,1], 0 ≤ κ ≤ 1, and r ≥ 0:
#   δ_T = β^(T+1)/(1-β),
#   |G̃ - G̃_T| ≤ δ_T,
#   |F̃ - F̃_T| ≤ rκ * δ_T,
#   |Θ̃ - Θ̃_T| ≤ κ * δ_T ≤ δ_T.
#
# The selected horizon makes δ_T ≤ ε. Therefore it guarantees errors
# at most ε for G̃ and Θ̃. For F̃, the guarantee is rκ ε, which is at
# most ε only when rκ ≤ 1.
###############################################################################

"""
    trunc_horizon_common(β, ε)

Smallest integer T ≥ 0 such that the geometric tail bound satisfies
    β^(T+1)/(1-β) ≤ ε.
"""
# --- Unicode versions (canonical) ---

function trunc_horizon_common(β::Real, ε::Real)
    @assert 0 < β < 1
    @assert ε > 0
    rhs = ε * (1 - β)
    rhs >= 1 && return 0
    T = ceil(Int, log(rhs) / log(β) - 1)
    return max(T, 0)
end

function tildeG(x, z; p01, ρ, κ, β, ε::Real=1e-8)
    T = trunc_horizon_common(β, ε)
    y, Γ, βt = x, 1.0, 1.0
    sG = 0.0
    for t in 0:T
        if y > z
            sG += βt * Γ
            Γ *= (1 - κ * y)
        end
        y  = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
        βt *= β
    end
    tailG = β^(T+1) / (1 - β)
    return (sG, T, tailG)
end

function tildeF(x, z; p01, ρ, κ, r, β, ε::Real=1e-8)
    T = trunc_horizon_common(β, ε)
    y, Γ, βt = x, 1.0, 1.0
    sF = 0.0
    for t in 0:T
        if y > z
            sF += βt * Γ * y
            Γ *= (1 - κ * y)
        end
        y  = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
        βt *= β
    end
    Fapprox = r * κ * sF
    tailF   = (r * κ) * (β^(T+1) / (1 - β))
    return (Fapprox, T, tailF)
end

function tildeFG(x, z; p01, ρ, κ, r, β, ε::Real=1e-8)
    T = trunc_horizon_common(β, ε)
    y, Γ, βt = x, 1.0, 1.0
    sF, sG = 0.0, 0.0
    for t in 0:T
        if y > z
            sF += βt * Γ * y
            sG += βt * Γ
            Γ *= (1 - κ * y)
        end
        y  = ϕ(y, z; p01=p01, ρ=ρ, κ=κ)
        βt *= β
    end
    tailG = β^(T+1) / (1 - β)
    tailF = (r * κ) * tailG
    return (r * κ * sF, sG, T, tailF, tailG)
end

"""
    tildeTheta(x, z; p01, ρ, κ, β, ε=1e-8)

Compute Θ̃(x,z) by evaluating F̃(x,z) at r=1, since F̃ = r Θ̃.

Returns (Θ̃_approx, T, tailΘ_bound), where
  |Θ̃ - Θ̃_approx| ≤ tailΘ_bound.

Note: tail bound is taken directly from tildeF(...; r=1).
"""
function tildeTheta(x, z; p01, ρ, κ, β, ε::Real=1e-8)
    F̂1, T, tailF1 = tildeF(x, z; p01=p01, ρ=ρ, κ=κ, r=1.0, β=β, ε=ε)
    Θ̂   = F̂1
    tailΘ = tailF1
    return (Θ̂, T, tailΘ)
end

"""
    tildeTheta_from_Ftilde(F̃; r)

Utility: Θ̃ = F̃/r given a computed F̃ (requires r>0).
"""
tildeTheta_from_Ftilde(F̃; r) = ( @assert r > 0; F̃ / r )


# --- Optional ASCII wrappers (different names, so no collision) ---

tildeF_ascii(x, z; p01, rho, kappa, r, beta, eps=1e-8) =
    tildeF(x, z; p01=p01, ρ=rho, κ=kappa, r=r, β=beta, ε=eps)

tildeG_ascii(x, z; p01, rho, kappa, beta, eps=1e-8) =
    tildeG(x, z; p01=p01, ρ=rho, κ=kappa, β=beta, ε=eps)

tildeFG_ascii(x, z; p01, rho, kappa, r, beta, eps=1e-8) =
    tildeFG(x, z; p01=p01, ρ=rho, κ=kappa, r=r, β=beta, ε=eps)

tildeFG_ascii (generic function with 1 method)

In [11]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 2.0
ε   = 1e-10

x0, x1, xhi = fixed_points(; p01=p01, ρ=ρ, κ=κ)

z = 0.5 * (x1 + x0)
x = z + 0.1

@assert 0 ≤ x ≤ 1
@assert x1 < z < x0
@assert x > z

# Compute (F̃, G̃) jointly in one pass.
F̂, Ĝ, T, tailF, tailG = tildeFG(
    x, z;
    p01=p01, ρ=ρ, κ=κ,
    r=r, β=β, ε=ε
)

# Compute Θ̃ independently and compare it with F̃/r.
Θ̂, TΘ, tailΘ = tildeTheta(
    x, z;
    p01=p01, ρ=ρ, κ=κ,
    β=β, ε=ε
)

Fr = F̂ / r

println("Parameters: p01=$p01, ρ=$ρ, κ=$κ, β=$β, r=$r")
println("Point: x=$x, z=$z, ε=$ε")
println()

println("Horizons and truncation-error bounds:")
println("  T(F̃,G̃) = $T")
println("    |F̃-F̃_T| ≤ $tailF")
println("    |G̃-G̃_T| ≤ $tailG")
println("  T(Θ̃)    = $TΘ")
println("    |Θ̃-Θ̃_T| ≤ $tailΘ")
println()

println("Values:")
println("  F̃(x,z)   = $F̂")
println("  G̃(x,z)   = $Ĝ")
println("  Θ̃(x,z)   = $Θ̂")
println("  F̃(x,z)/r = $Fr")
println()

# Sanity checks
@assert T == TΘ
@assert tailG ≤ ε
@assert tailF == r * κ * tailG
@assert tailΘ == κ * tailG
@assert isapprox(Θ̂, Fr; rtol=0, atol=1e-12)

Parameters: p01=0.25, ρ=0.6, κ=0.8, β=0.95, r=2.0
Point: x=0.5608406756197305, z=0.4608406756197305, ε=1.0e-10

Horizons and truncation-error bounds:
  T(F̃,G̃) = 507
    |F̃-F̃_T| ≤ 1.5443348253115318e-10
    |G̃-G̃_T| ≤ 9.652092658197074e-11
  T(Θ̃)    = 507
    |Θ̃-Θ̃_T| ≤ 7.721674126557659e-11

Values:
  F̃(x,z)   = 1.7275555470777328
  G̃(x,z)   = 2.0613774205333235
  Θ̃(x,z)   = 0.8637777735388664
  F̃(x,z)/r = 0.8637777735388664



In [12]:
"""
    make_xticks_and_labels(x; p01, ρ, κ, tol=1e-12)

Return `(xticks, xtick_str, xticklab_str)` for PGFPlotsX. The ticks are at
`0`, `p01`, `x`, `x^1`, `x^0`, `p01+ρ`, and `1`, sorted and merged within
`tol`. A uniform vertical strut aligns the labels.
"""
function make_xticks_and_labels(point; p01, ρ, κ,
                                tol=1e-12,
                                point_label::AbstractString="x")
    x0_fp, x1_fp, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

    items = [
        (0.0, "\\ensuremath{0}"),
        (float(p01), "\\ensuremath{p_{01}}"),
        (float(point), "\\ensuremath{$point_label}"),
        (float(x1_fp), "\\ensuremath{x^{1}}"),
        (float(x0_fp), "\\ensuremath{x^{0}}"),
        (float(p01 + ρ), "\\ensuremath{p_{11}}"),
        (1.0, "\\ensuremath{1}"),
    ]

    sort!(items, by=first)

    # Keep the first label when locations coincide within tol.
    merged = Tuple{Float64,String}[]
    for (z, lab) in items
        if isempty(merged) || abs(z - merged[end][1]) > tol
            push!(merged, (z, lab))
        end
    end

    xticks = [z for (z, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"

    # Uniform strut so that all labels have consistent height and depth.
    phantom = "\\vphantom{\\ensuremath{p_{01}+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    return xticks, xtick_str, xticklab_str
end

# Compress a sampled right-continuous step function into horizontal segments,
# vertical jumps, and open/filled endpoint markers.
function _stepdata_right_continuous(zgrid::AbstractVector{<:Real},
                                    ygrid::AbstractVector{<:Real};
                                    atol::Real=1e-12)
    @assert length(zgrid) == length(ygrid) ≥ 2
    @assert issorted(zgrid) "zgrid must be sorted in nondecreasing order"

    breaks = Int[]
    for i in 1:length(ygrid)-1
        if !isapprox(ygrid[i+1], ygrid[i]; atol=atol, rtol=0)
            push!(breaks, i)
        end
    end

    line_coords = Tuple{Float64,Float64}[]
    vline_coords = Tuple{Float64,Float64}[]
    open_marks = Tuple{Float64,Float64}[]
    fill_marks = Tuple{Float64,Float64}[]

    push!(fill_marks, (float(zgrid[1]), float(ygrid[1])))

    seg_start = 1
    for i in breaks
        zL = float(zgrid[seg_start])
        zB = float(zgrid[i+1])
        yL = float(ygrid[i])
        yR = float(ygrid[i+1])

        push!(line_coords, (zL, yL))
        push!(line_coords, (zB, yL))
        push!(line_coords, (NaN, NaN))

        push!(vline_coords, (zB, yL))
        push!(vline_coords, (zB, yR))
        push!(vline_coords, (NaN, NaN))

        push!(open_marks, (zB, yL))
        push!(fill_marks, (zB, yR))

        seg_start = i + 1
    end

    # Last horizontal segment, ending at the final grid point.
    zL = float(zgrid[seg_start])
    zE = float(zgrid[end])
    yE = float(ygrid[end])

    push!(line_coords, (zL, yE))
    push!(line_coords, (zE, yE))
    push!(line_coords, (NaN, NaN))

    push!(fill_marks, (zE, yE))

    return line_coords, vline_coords, open_marks, fill_marks
end

# Build horizontal segments on which the two sampled step functions coincide.
function _coincidence_hsegments(zgrid::AbstractVector{<:Real},
                                Fvals::AbstractVector{<:Real},
                                Gvals::AbstractVector{<:Real};
                                atol::Real=1e-12,
                                eqtol::Real=1e-12)
    @assert length(zgrid) == length(Fvals) == length(Gvals) ≥ 2
    @assert issorted(zgrid) "zgrid must be sorted in nondecreasing order"

    breaks = Int[]
    for i in 1:length(zgrid)-1
        if !isapprox(Fvals[i+1], Fvals[i]; atol=atol, rtol=0) ||
           !isapprox(Gvals[i+1], Gvals[i]; atol=atol, rtol=0)
            push!(breaks, i)
        end
    end

    coords = Tuple{Float64,Float64}[]
    seg_start = 1

    for i in breaks
        zL = float(zgrid[seg_start])
        zB = float(zgrid[i+1])
        yF = float(Fvals[i])
        yG = float(Gvals[i])

        if isfinite(yF) && isfinite(yG) && isapprox(yF, yG; atol=eqtol, rtol=0)
            push!(coords, (zL, yF))
            push!(coords, (zB, yF))
            push!(coords, (NaN, NaN))
        end

        seg_start = i + 1
    end

    zL = float(zgrid[seg_start])
    zE = float(zgrid[end])
    yF = float(Fvals[end])
    yG = float(Gvals[end])

    if isfinite(yF) && isfinite(yG) && isapprox(yF, yG; atol=eqtol, rtol=0)
        push!(coords, (zL, yF))
        push!(coords, (zE, yF))
        push!(coords, (NaN, NaN))
    end

    return coords
end

function plot_tildeFG_vs_z_axis(x; p01, ρ, κ, r, β, ε=1e-10,
                               zgrid=collect(range(0.0, 1.0; length=2001)),
                               xlabel=L"$z$",
                               ylabel=L"$\widetilde F(x,z),\ \widetilde G(x,z)$",
                               width="11cm", height="7cm",
                               xpad=0.03,
                               ypad=0.05,
                               atol=1e-12,
                               eqtol=1e-12)

    @assert length(zgrid) ≥ 2 "zgrid must contain at least two points"
    @assert issorted(zgrid) "zgrid must be sorted in nondecreasing order"

    _, xtick_str, xticklab_str = make_xticks_and_labels(x; p01=p01, ρ=ρ, κ=κ)

    Fvals = similar(zgrid, Float64)
    Gvals = similar(zgrid, Float64)
    for (i, z) in enumerate(zgrid)
        F̂, Ĝ, _, _, _ = tildeFG(x, z; p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        Fvals[i] = float(F̂)
        Gvals[i] = float(Ĝ)
    end

    # Keep the horizontal segments and use open markers only to localize jumps.
    F_line, _, F_open, _ = _stepdata_right_continuous(zgrid, Fvals; atol=atol)
    G_line, _, G_open, _ = _stepdata_right_continuous(zgrid, Gvals; atol=atol)

    FG_eq_line = _coincidence_hsegments(zgrid, Fvals, Gvals; atol=atol, eqtol=eqtol)

    finite_vals = filter(isfinite, vcat(Fvals, Gvals))
    isempty(finite_vals) &&
        error("No finite values were produced on zgrid")

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # Rug marks at sampled detections of jumps. The grid localizes, but does
    # not generally identify, the exact discontinuity locations.
    jump_zs = sort!(unique!(vcat(first.(F_open), first.(G_open))))
    extra_xtick_str    = "{" * join(string.(jump_zs), ",") * "}"
    extra_xticklab_str = "{" * join(fill("", length(jump_zs)), ",") * "}"

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" => "{tick align=inside, tick style={black!55}}",

            "ytick" => "{0}",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
            "legend pos" => "north east",
        },

        Plot({ "line width" => "0.30pt", "solid" => nothing, "black" => nothing },
             Coordinates(F_line)),
        LegendEntry(L"$\widetilde F(x,z)$"),

        Plot({ "line width" => "0.30pt", "solid" => nothing, "color" => "black!40" },
             Coordinates(G_line)),
        LegendEntry(L"$\widetilde G(x,z)$"),

        # Overlay coincident segments without adding another legend entry.
        Plot({ "line width" => "0.30pt", "solid" => nothing,
               "color" => "black!55", "forget plot" => nothing },
             Coordinates(FG_eq_line)),
    )
end

"""
    save_tildeFG_vs_z_plot_pdf(outpdf, x; p01, ρ, κ, r, β,
                               engine="pdflatex", quiet=true, keep_tex=true, kwargs...)

Build the axis, write a standalone `.tex` file, and compile it to PDF.
"""
function save_tildeFG_vs_z_plot_pdf(outpdf::AbstractString, x;
                                    p01, ρ, κ, r, β,
                                    engine::AbstractString="pdflatex",
                                    quiet::Bool=true,
                                    keep_tex::Bool=true,
                                    kwargs...)
    ax  = plot_tildeFG_vs_z_axis(x; p01=p01, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError("outpdf must have extension .pdf or no extension"))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    engine_path = Sys.which(engine)
    engine_path === nothing &&
        throw(ArgumentError("LaTeX engine '$engine' was not found"))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_tildeFG_vs_z_plot_pdf

In [13]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0

x0, _, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)
p11 = p01 + ρ
x   = 0.5 * (x0 + p11)

@assert 0 ≤ x0 < x < p11 ≤ 1

save_tildeFG_vs_z_plot_pdf(
    "tildeFG_vs_z.pdf", x;
    p01=p01, ρ=ρ, κ=κ, r=r, β=β,
    ε=1e-14,
    zgrid=collect(range(0.0, 1.0; length=10001)),
    atol=1e-12,
    eqtol=1e-12,
    keep_tex=false
)

"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\tildeFG_vs_z.pdf"

In [14]:
"""
    F_metric(x, z; p01, ρ, κ, r, β, ε=1e-10)

Compute the full discounted reward metric from the renewal equations

    F(x,z) = F̃(x,z) + β Θ̃(x,z) F(p11,z),

    F(p11,z) =
        F̃(p11,z) / (1 - β Θ̃(p11,z)),

where p11 = p01 + ρ and F̃ = r Θ̃.

The extra factor β appears because an ACK in the current slot resets
the belief to p11 at the beginning of the next slot.
"""
function F_metric(x, z; p01, ρ, κ, r, β, ε=1e-10)
    Θ̃,  _, _ = tildeTheta(
        x, z;
        p01=p01, ρ=ρ, κ=κ, β=β, ε=ε
    )

    p11 = p01 + ρ
    Θ̃R, _, _ = tildeTheta(
        p11, z;
        p01=p01, ρ=ρ, κ=κ, β=β, ε=ε
    )

    denom = 1.0 - β * Θ̃R
    denom > 0 ||
        throw(DomainError(denom, "renewal denominator must be positive"))

    F̃  = r * Θ̃
    F̃R = r * Θ̃R
    FR  = F̃R / denom

    return F̃ + β * Θ̃ * FR
end

"""
    G_metric(x, z; p01, ρ, κ, β, ε=1e-10)

Compute the full discounted work metric from the renewal equations

    G(x,z) = G̃(x,z) + β Θ̃(x,z) G(p11,z),

    G(p11,z) =
        G̃(p11,z) / (1 - β Θ̃(p11,z)),

where p11 = p01 + ρ.

The extra factor β appears because an ACK in the current slot resets
the belief to p11 at the beginning of the next slot.
"""
function G_metric(x, z; p01, ρ, κ, β, ε=1e-10)
    # With r=1, the first output of tildeFG is Θ̃.
    Θ̃, G̃, _, _, _ = tildeFG(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )

    p11 = p01 + ρ
    Θ̃R, G̃R, _, _, _ = tildeFG(
        p11, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )

    denom = 1.0 - β * Θ̃R
    denom > 0 ||
        throw(DomainError(denom, "renewal denominator must be positive"))

    GR = G̃R / denom

    return G̃ + β * Θ̃ * GR
end

G_metric

In [15]:
"""
    show_FG(x, z; p01, ρ, κ, r, β, ε=1e-10, digits=12)

Compute and display numerical approximations to the full discounted
metrics F(x,z) and G(x,z).

The tolerance ε controls the truncated pre-ACK quantities used by
F_metric and G_metric; it is not necessarily an absolute-error bound
for the resulting renewal quotients.
"""
function show_FG(x, z; p01, ρ, κ, r, β, ε=1e-10, digits=12)
    Fxz = F_metric(x, z; p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    Gxz = G_metric(x, z; p01=p01, ρ=ρ, κ=κ, β=β, ε=ε)

    println("--------------------------------------------------")
    println("Inputs: x = $(round(x, digits=digits)), z = $(round(z, digits=digits))")
    println("Params: p01 = $p01, ρ = $ρ, κ = $κ, r = $r, β = $β, ε = $ε")
    println("--------------------------------------------------")
    println("F(x,z) = $(round(Fxz, digits=digits))")
    println("G(x,z) = $(round(Gxz, digits=digits))")
    println("--------------------------------------------------")

    return Fxz, Gxz
end

# Example usage:
p01  = 0.25
ρ  = 0.60
κ  = 0.80
r  = 1.0
β  = 0.95
ε  = 1e-10

x = 0.73
z = 0.55

Fxz, Gxz = show_FG(x, z; p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

--------------------------------------------------
Inputs: x = 0.73, z = 0.55
Params: p01 = 0.25, ρ = 0.6, κ = 0.8, r = 1.0, β = 0.95, ε = 1.0e-10
--------------------------------------------------
F(x,z) = 7.449433492972
G(x,z) = 12.582785079591
--------------------------------------------------


(7.449433492972471, 12.582785079590664)

In [16]:
"""
    FG_metric(x, z; p01, ρ, κ, r, β, ε=1e-10)

Compute the full discounted reward and work metrics jointly.

The tolerance ε controls the pre-ACK truncations. It is not necessarily
an absolute-error bound for the final renewal metrics.
"""
function FG_metric(x, z; p01, ρ, κ, r, β, ε=1e-10)
    # With r=1, the first output of tildeFG is Θ̃.
    theta_x, Gtilde_x, _, _, _ = tildeFG(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )

    p11 = p01 + ρ
    theta_R, Gtilde_R, _, _, _ = tildeFG(
        p11, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )

    denom = 1.0 - β * theta_R
    denom > 0 ||
        throw(DomainError(denom, "renewal denominator must be positive"))

    # Since F̃ = rΘ̃:
    F = r * theta_x / denom
    G = Gtilde_x + β * theta_x * Gtilde_R / denom

    return F, G
end

function plot_FG_vs_z_axis(x; p01, ρ, κ, r, β, ε=1e-10,
                           zgrid=collect(range(0.0, 1.0; length=2001)),
                           xlabel=L"$z$",
                           ylabel=L"$F(x,z),\ G(x,z)$",
                           width="11cm", height="7cm",
                           xpad=0.03,
                           ypad=0.05,
                           atol=1e-12,
                           eqtol=1e-12)

    @assert length(zgrid) ≥ 2 "zgrid must contain at least two points"
    @assert all(isfinite, zgrid) "zgrid must contain only finite values"
    @assert all(diff(zgrid) .> 0) "zgrid must be strictly increasing"

    _, xtick_str, xticklab_str =
        make_xticks_and_labels(x; p01=p01, ρ=ρ, κ=κ)

    # Sample F and G jointly.
    Fvals = similar(zgrid, Float64)
    Gvals = similar(zgrid, Float64)

    for (i, z) in enumerate(zgrid)
        Fi, Gi = FG_metric(
            x, z;
            p01=p01, ρ=ρ, κ=κ,
            r=r, β=β, ε=ε
        )
        Fvals[i] = float(Fi)
        Gvals[i] = float(Gi)
    end

    # Horizontal-only step polylines; open marks localize sampled jumps.
    F_line, _, F_open, _ =
        _stepdata_right_continuous(zgrid, Fvals; atol=atol)
    G_line, _, G_open, _ =
        _stepdata_right_continuous(zgrid, Gvals; atol=atol)

    FG_eq_line =
        _coincidence_hsegments(
            zgrid, Fvals, Gvals;
            atol=atol, eqtol=eqtol
        )

    finite_vals = filter(isfinite, vcat(Fvals, Gvals))
    isempty(finite_vals) &&
        error("No finite F or G values were produced on zgrid")

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)

    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # Rug marks at sampled detections of jumps. They generally localize,
    # rather than identify exactly, the true discontinuity locations.
    jump_zs = sort!(unique!(vcat(first.(F_open), first.(G_open))))
    extra_xtick_str =
        "{" * join(string.(jump_zs), ",") * "}"
    extra_xticklab_str =
        "{" * join(fill("", length(jump_zs)), ",") * "}"

    # Base curves are 0.40pt. A wider gray line beneath the black
    # coincidence overlay produces visible gray shoulders.
    lw_base  = "0.40pt"
    lw_shldr = "0.85pt"

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin => ymin,
            ymax => ymax,

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,
            "xticklabel style" =>
                "{text height=1.8ex, text depth=0.25ex}",

            "ytick" => "{0}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
            "legend pos" => "north east",

            "extra x ticks" => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" =>
                "{tick align=inside, tick style={black!55}}",
        },

        # F: solid black.
        Plot(
            {
                "line width" => lw_base,
                "solid" => nothing,
                "black" => nothing
            },
            Coordinates(F_line)
        ),
        LegendEntry(L"$F(x,z)$"),

        # G: solid gray.
        Plot(
            {
                "line width" => lw_base,
                "solid" => nothing,
                "color" => "black!50"
            },
            Coordinates(G_line)
        ),
        LegendEntry(L"$G(x,z)$"),

        # Coincident segments: wide gray underlay plus narrow black center.
        Plot(
            {
                "line width" => lw_shldr,
                "solid" => nothing,
                "color" => "black!55",
                "forget plot" => nothing
            },
            Coordinates(FG_eq_line)
        ),
        Plot(
            {
                "line width" => lw_base,
                "solid" => nothing,
                "black" => nothing,
                "forget plot" => nothing
            },
            Coordinates(FG_eq_line)
        ),
    )
end

function save_FG_vs_z_plot_pdf(outpdf::AbstractString, x;
                               p01, ρ, κ, r, β,
                               engine::AbstractString="pdflatex",
                               quiet::Bool=true,
                               keep_tex::Bool=true,
                               kwargs...)
    ax = plot_FG_vs_z_axis(
        x;
        p01=p01, ρ=ρ, κ=κ, r=r, β=β,
        kwargs...
    )
    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError(
            "outpdf must have extension .pdf or no extension"
        ))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    engine_path = Sys.which(engine)
    engine_path === nothing &&
        throw(ArgumentError("LaTeX engine '$engine' was not found"))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_FG_vs_z_plot_pdf (generic function with 1 method)

In [17]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0

x0, _, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)
p11 = p01 + ρ
x   = 0.5 * (x0 + p11)

@assert 0 ≤ x0 < x < p11 ≤ 1

save_FG_vs_z_plot_pdf(
    "FG_vs_z.pdf", x;
    p01=p01, ρ=ρ, κ=κ, r=r, β=β,
    ε=1e-14,
    zgrid=collect(range(0.0, 1.0; length=10001)),
    atol=1e-12,
    eqtol=1e-12,
    keep_tex=false
)

"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\FG_vs_z.pdf"

In [18]:
# ------------------------------------------------------------------
# FG vs x for fixed z:
#   - draw both curves SOLID
#   - F in black, G in gray (black!50)
#   - if F and G coincide on a segment: show gray shoulders + black
#   - optional NaN break at x = z to avoid a spurious vertical join
# ------------------------------------------------------------------

# Build a sampled polyline, inserting NaN breaks at prescribed x-locations.
function _curve_with_breaks(xgrid::AbstractVector{<:Real},
                            yvals::AbstractVector{<:Real};
                            breaks::AbstractVector{<:Real}=Float64[],
                            atol::Real=1e-12)
    @assert length(xgrid) == length(yvals)

    bks = sort(float.(collect(breaks)))
    coords = Tuple{Float64,Float64}[]

    for i in eachindex(xgrid)
        x = float(xgrid[i])
        y = float(yvals[i])

        push!(coords, isfinite(y) ? (x, y) : (NaN, NaN))

        if i < length(xgrid)
            xnext = float(xgrid[i+1])
            crossed = any(b -> ((x < b && b <= xnext) ||
                                isapprox(x, b; atol=atol, rtol=0)), bks)
            if crossed
                push!(coords, (NaN, NaN))
            end
        end
    end

    return coords
end

# Build curve pieces where Y1 ≈ Y2, again allowing NaN breaks at prescribed x-locations.
function _coincidence_segments(xgrid::AbstractVector{<:Real},
                               Y1::AbstractVector{<:Real},
                               Y2::AbstractVector{<:Real};
                               eqtol::Real=1e-12,
                               breaks::AbstractVector{<:Real}=Float64[],
                               atol::Real=1e-12)
    @assert length(xgrid) == length(Y1) == length(Y2)

    bks = sort(float.(collect(breaks)))
    coords = Tuple{Float64,Float64}[]
    inrun = false

    for i in eachindex(xgrid)
        same = isfinite(Y1[i]) && isfinite(Y2[i]) &&
               isapprox(Y1[i], Y2[i]; atol=eqtol, rtol=0)

        if same
            push!(coords, (float(xgrid[i]), float(Y1[i])))
            inrun = true
        elseif inrun
            push!(coords, (NaN, NaN))
            inrun = false
        end

        if i < length(xgrid)
            x     = float(xgrid[i])
            xnext = float(xgrid[i+1])
            crossed = any(b -> ((x < b && b <= xnext) ||
                                isapprox(x, b; atol=atol, rtol=0)), bks)
            if crossed && inrun
                push!(coords, (NaN, NaN))
                inrun = false
            end
        end
    end

    if inrun
        push!(coords, (NaN, NaN))
    end

    return coords
end

function plot_FG_vs_x_axis(z; p01, ρ, κ, r, β, ε=1e-10,
                           xgrid=collect(range(0.0, 1.0; length=2001)),
                           xlabel=L"$x$",
                           ylabel=L"$F(x,z),\ G(x,z)$",
                           width="11cm", height="7cm",
                           xpad=0.03,
                           ypad=0.05,
                           atol=1e-12,
                           eqtol=1e-12,
                           break_at_z::Bool=true,
                           breaks::AbstractVector{<:Real}=Float64[])

    length(xgrid) ≥ 2 ||
        throw(ArgumentError("xgrid must contain at least two points"))
    all(isfinite, xgrid) ||
        throw(ArgumentError("xgrid must contain only finite values"))
    all(diff(xgrid) .> 0) ||
        throw(ArgumentError("xgrid must be strictly increasing"))
    0 ≤ z ≤ 1 ||
        throw(ArgumentError("z must belong to [0,1]"))
    xpad ≥ 0 ||
        throw(ArgumentError("xpad must be nonnegative"))
    ypad ≥ 0 ||
        throw(ArgumentError("ypad must be nonnegative"))

    _, xtick_str, xticklab_str =
        make_xticks_and_labels(
            z;
            p01=p01, ρ=ρ, κ=κ,
            point_label="z"
        )

    # The reset quantities depend on z but not on the varying initial state x,
    # so compute them only once.
    p11 = p01 + ρ
    theta_R, Gtilde_R, _, _, _ = tildeFG(
        p11, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )

    denom = 1.0 - β * theta_R
    denom > 0 ||
        throw(DomainError(denom, "renewal denominator must be positive"))

    Fvals = Vector{Float64}(undef, length(xgrid))
    Gvals = Vector{Float64}(undef, length(xgrid))

    for (i, x) in enumerate(xgrid)
        theta_x, Gtilde_x, _, _, _ = tildeFG(
            x, z;
            p01=p01, ρ=ρ, κ=κ,
            r=1.0, β=β, ε=ε
        )

        Fvals[i] = float(r * theta_x / denom)
        Gvals[i] = float(
            Gtilde_x + β * theta_x * Gtilde_R / denom
        )
    end

    # Break the sampled polylines at x=z by default. Any other analytically
    # known discontinuity locations can be supplied through `breaks`.
    curve_breaks = float.(collect(breaks))
    break_at_z && push!(curve_breaks, float(z))
    sort!(unique!(curve_breaks))

    F_line = _curve_with_breaks(
        xgrid, Fvals;
        breaks=curve_breaks, atol=atol
    )
    G_line = _curve_with_breaks(
        xgrid, Gvals;
        breaks=curve_breaks, atol=atol
    )

    FG_eq_line = _coincidence_segments(
        xgrid, Fvals, Gvals;
        eqtol=eqtol,
        breaks=curve_breaks,
        atol=atol
    )

    finite_vals = filter(isfinite, vcat(Fvals, Gvals))
    isempty(finite_vals) &&
        error("No finite F or G values were produced on xgrid")

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)

    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    lw_base  = "0.40pt"
    lw_shldr = "0.85pt"

    ax = @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin => ymin,
            ymax => ymax,

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,
            "xticklabel style" =>
                "{text height=1.8ex, text depth=0.25ex}",

            "ytick" => "{0}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
            "legend pos" => "north east",
        },

        Plot(
            {
                "line width" => lw_base,
                "solid" => nothing,
                "black" => nothing
            },
            Coordinates(F_line)
        ),
        LegendEntry(L"$F(x,z)$"),

        Plot(
            {
                "line width" => lw_base,
                "solid" => nothing,
                "color" => "black!50"
            },
            Coordinates(G_line)
        ),
        LegendEntry(L"$G(x,z)$"),
    )

    if !isempty(FG_eq_line)
        # Wide gray underlay.
        push!(
            ax,
            @pgf Plot(
                {
                    "line width" => lw_shldr,
                    "solid" => nothing,
                    "color" => "black!50",
                    "forget plot" => nothing
                },
                Coordinates(FG_eq_line)
            )
        )

        # Narrow black centerline.
        push!(
            ax,
            @pgf Plot(
                {
                    "line width" => lw_base,
                    "solid" => nothing,
                    "black" => nothing,
                    "forget plot" => nothing
                },
                Coordinates(FG_eq_line)
            )
        )
    end

    return ax
end

function save_FG_vs_x_plot_pdf(outpdf::AbstractString, z;
                               p01, ρ, κ, r, β,
                               engine::AbstractString="pdflatex",
                               quiet::Bool=true,
                               keep_tex::Bool=true,
                               kwargs...)
    ax = plot_FG_vs_x_axis(
        z;
        p01=p01, ρ=ρ, κ=κ, r=r, β=β,
        kwargs...
    )
    pic = TikzPicture(ax)

    # Accept either "figure" or "figure.pdf", but reject other extensions.
    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError(
            "outpdf must have extension .pdf or no extension"
        ))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    engine_path = Sys.which(engine)
    engine_path === nothing &&
        throw(ArgumentError("LaTeX engine '$engine' was not found"))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            tail = lines[max(1, length(lines) - 80):end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_FG_vs_x_plot_pdf (generic function with 1 method)

In [19]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0

x0, x1, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

# Choose z in the intermediate regime x¹ < z < x⁰.
z = 0.5 * (x1 + x0)

@assert 0 ≤ x1 < z < x0 ≤ 1

save_FG_vs_x_plot_pdf(
    "FG_vs_x.pdf", z;
    p01=p01, ρ=ρ, κ=κ, r=r, β=β,
    ε=1e-14,
    xgrid=range(0.0, 1.0; length=1001),
    atol=1e-12,
    eqtol=1e-12,
    keep_tex=false
)

"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\FG_vs_x.pdf"

In [20]:
raw"""
    tilde_marginal_metrics(x, z; p01, ρ, κ, r, β, ε=1e-10)

Compute the pre-renewal marginal metrics
    (\tilde f(x,z), \tilde g(x,z), \tilde θ(x,z)).

The identity \tilde f(x,z) = r\tilde θ(x,z) is used.
"""
function tilde_marginal_metrics(x, z;
                                p01, ρ, κ, r, β,
                                ε=1e-10)
    φ0x = φ0(x; p01=p01, ρ=ρ)
    φ1x = φ1(x; p01=p01, ρ=ρ, κ=κ)

    θ̃1, G̃1, _, _, _ = tildeFG(
        φ1x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )

    θ̃0, G̃0, _, _, _ = tildeFG(
        φ0x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )

    tθ = κ*x + β*((1 - κ*x)*θ̃1 - θ̃0)
    tf = r*tθ
    tg = 1 + β*((1 - κ*x)*G̃1 - G̃0)

    return tf, tg, tθ
end


raw"""
    tildef_metric(x, z; p01, ρ, κ, r, β, ε=1e-10)

Compute the pre-renewal marginal reward metric \tilde f(x,z).
"""
function tildef_metric(x, z;
                       p01, ρ, κ, r, β,
                       ε=1e-10)
    tf, _, _ = tilde_marginal_metrics(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )
    return tf
end


raw"""
    tildeg_metric(x, z; p01, ρ, κ, β, ε=1e-10)

Compute the pre-renewal marginal work metric \tilde g(x,z).
"""
function tildeg_metric(x, z;
                       p01, ρ, κ, β,
                       ε=1e-10)
    _, tg, _ = tilde_marginal_metrics(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )
    return tg
end


raw"""
    tildetheta_metric(x, z; p01, ρ, κ, β, ε=1e-10)

Compute the pre-renewal marginal factor \tilde θ(x,z).
"""
function tildetheta_metric(x, z;
                           p01, ρ, κ, β,
                           ε=1e-10)
    _, _, tθ = tilde_marginal_metrics(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )
    return tθ
end


raw"""
    fg_metric2(x, z; p01, ρ, κ, r, β, ε=1e-10)

Compute the full marginal reward and work metrics:
    f = \tilde f + β\tilde θ F(p11,z),
    g = \tilde g + β\tilde θ G(p11,z).
"""
function fg_metric2(x, z;
                    p01, ρ, κ, r, β,
                    ε=1e-10)
    tf, tg, tθ = tilde_marginal_metrics(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )

    FR, GR = FG_metric(
        p01 + ρ, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )

    f = tf + β*tθ*FR
    g = tg + β*tθ*GR

    return f, g
end


function f_metric2(x, z;
                   p01, ρ, κ, r, β,
                   ε=1e-10)
    f, _ = fg_metric2(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )
    return f
end


function g_metric2(x, z;
                   p01, ρ, κ, β,
                   ε=1e-10)
    _, g = fg_metric2(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=1.0, β=β, ε=ε
    )
    return g
end


raw"""
    tildef_from_tildetheta(x, z; p01, ρ, κ, r, β, ε=1e-10)

Compute \tilde f(x,z) from the identity
    \tilde f(x,z) = r\tilde θ(x,z).
"""
function tildef_from_tildetheta(x, z;
                                p01, ρ, κ, r, β,
                                ε=1e-10)
    return r * tildetheta_metric(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    )
end

tildef_from_tildetheta (generic function with 1 method)

In [21]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0
ε   = 1e-14

x0, x1, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

x = 0.73
z = 0.55

@assert 0 ≤ x ≤ 1
@assert x1 < z < x0

@show x1 x0 x z

# Consolidated pre-renewal marginal metrics.
tf, tg, tθ = tilde_marginal_metrics(
    x, z;
    p01=p01, ρ=ρ, κ=κ,
    r=r, β=β, ε=ε
)

# Independent direct calculation of tilde f.
φ0x = φ0(x; p01=p01, ρ=ρ)
φ1x = φ1(x; p01=p01, ρ=ρ, κ=κ)

F̃1, _, _ = tildeF(
    φ1x, z;
    p01=p01, ρ=ρ, κ=κ,
    r=r, β=β, ε=ε
)

F̃0, _, _ = tildeF(
    φ0x, z;
    p01=p01, ρ=ρ, κ=κ,
    r=r, β=β, ε=ε
)

tf_direct =
    r*κ*x + β*((1 - κ*x)*F̃1 - F̃0)

# Compute the two full marginal metrics jointly.
f, g = fg_metric2(
    x, z;
    p01=p01, ρ=ρ, κ=κ,
    r=r, β=β, ε=ε
)

identity_error = tf_direct - r*tθ

@show tf_direct tf tg tθ identity_error
@show f g

@assert isapprox(
    tf_direct, r*tθ;
    atol=1e-11,
    rtol=1e-11
)

x1 = 0.29668135123946104
x0 = 0.625
x = 0.73
z = 0.55
tf_direct = 0.054305567707403246
tf = 0.054305567707403246
tg = 0.02852427746230002
tθ = 0.054305567707403246
identity_error = 0.0
f = 0.4619583975422029
g = 0.7069989566634313


In [22]:
# Extract the scalar value from functions returning
# either value or (value, diagnostics...).
_metric_value(u) = u isa Tuple ? first(u) : u


function diagnose_z_lt_p01(x, z;
                           p01, ρ, κ, r, β,
                           ε=1e-14)

    0.0 ≤ x ≤ 1.0 ||
        throw(ArgumentError("x must belong to [0,1]"))
    0.0 ≤ z < p01 ||
        throw(ArgumentError("this diagnostic requires 0 ≤ z < p01"))
    0.0 ≤ p01 ≤ 1.0 ||
        throw(ArgumentError("p01 must belong to [0,1]"))
    0.0 ≤ ρ ≤ 1.0 - p01 ||
        throw(ArgumentError("ρ must satisfy 0 ≤ ρ ≤ 1-p01"))
    0.0 < κ < 1.0 ||
        throw(ArgumentError("this diagnostic assumes 0 < κ < 1"))
    0.0 ≤ β < 1.0 ||
        throw(ArgumentError("β must belong to [0,1)"))
    r > 0 ||
        throw(ArgumentError("r must be positive"))

    println("\n==============================")
    println("Diagnostic at x=$x, z=$z (z<p01)")
    println("==============================")

    # ----------------------------------------------------------
    # Basic states
    # ----------------------------------------------------------
    φ0x = φ0(x; p01=p01, ρ=ρ)
    φ1x = φ1(x; p01=p01, ρ=ρ, κ=κ)
    p11 = p01 + ρ

    println("\n[Basic states]")
    println("x       = $x")
    println("z       = $z")
    println("φ0(x)   = $φ0x")
    println("φ1(x)   = $φ1x")
    println("p11     = $p11")
    println("x>z?    = $(x > z)")
    println("φ0(x)>z? = $(φ0x > z)")
    println("φ1(x)>z? = $(φ1x > z)")
    println("p11>z?   = $(p11 > z)")

    # Under 0 ≤ z < p01 and ρ ≥ 0, all three successor/reset
    # states must lie strictly above the threshold.
    @assert φ0x > z
    @assert φ1x > z
    @assert p11 > z

    # ----------------------------------------------------------
    # Primitive pre-ACK quantities
    # ----------------------------------------------------------
    tFx = _metric_value(tildeF(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    ))
    tGx = _metric_value(tildeG(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))
    tΘx = _metric_value(tildeTheta(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))

    tF0 = _metric_value(tildeF(
        φ0x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    ))
    tF1 = _metric_value(tildeF(
        φ1x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    ))

    tG0 = _metric_value(tildeG(
        φ0x, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))
    tG1 = _metric_value(tildeG(
        φ1x, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))
    tGR = _metric_value(tildeG(
        p11, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))

    tΘ0 = _metric_value(tildeTheta(
        φ0x, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))
    tΘ1 = _metric_value(tildeTheta(
        φ1x, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))
    tΘR = _metric_value(tildeTheta(
        p11, z;
        p01=p01, ρ=ρ, κ=κ,
        β=β, ε=ε
    ))

    println("\n[Pre-ACK metrics]")
    println("tildeF(x,z)     = $tFx")
    println("tildeG(x,z)     = $tGx")
    println("tildeTheta(x,z) = $tΘx")
    println("tildeTheta - tildeF/r = $(tΘx - tFx/r)  (should be ~0)")

    # ----------------------------------------------------------
    # Direct pre-ACK marginal calculations
    #
    # These remain independent of tilde_marginal_metrics.
    # ----------------------------------------------------------
    tf_direct =
        r*κ*x + β*((1 - κ*x)*tF1 - tF0)

    tg_direct =
        1 + β*((1 - κ*x)*tG1 - tG0)

    tθ_direct =
        κ*x + β*((1 - κ*x)*tΘ1 - tΘ0)

    # Consolidated implementation.
    tf, tg, tθ = tilde_marginal_metrics(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )

    println("\n[Pre-ACK marginal metrics]")
    println("tilde f, direct       = $tf_direct")
    println("tilde f, consolidated = $tf")
    println("difference            = $(tf - tf_direct)")

    println("tilde g, direct       = $tg_direct")
    println("tilde g, consolidated = $tg")
    println("difference            = $(tg - tg_direct)")

    println("tilde θ, direct       = $tθ_direct")
    println("tilde θ, consolidated = $tθ")
    println("difference            = $(tθ - tθ_direct)")

    println("tilde f_direct - r*tilde θ_direct = ",
            tf_direct - r*tθ_direct, "  (should be ~0)")

    # ----------------------------------------------------------
    # Full performance metrics
    # ----------------------------------------------------------
    Fx, Gx = FG_metric(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )
    F0, G0 = FG_metric(
        φ0x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )
    F1, G1 = FG_metric(
        φ1x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )
    FR, GR = FG_metric(
        p11, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )

    f, g = fg_metric2(
        x, z;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β, ε=ε
    )

    println("\n[Full marginal metrics]")
    println("f(x,z) = $f")
    println("g(x,z) = $g")

    # For z<p01, the full marginals satisfy
    #     f(x,z)=rκx,    g(x,z)=1.
    println("\n[z<p01 full-marginal predictions]")
    println("f(x,z) - rκx = $(f - r*κ*x)  (should be ~0)")
    println("g(x,z) - 1   = $(g - 1.0)    (should be ~0)")

    # ----------------------------------------------------------
    # Posterior-mean identity
    # ----------------------------------------------------------
    posterior_residual =
        φ0x - (κ*x*p11 + (1 - κ*x)*φ1x)

    println("\n[Posterior-mean identity]")
    println(
        "φ0(x) - [κx p11 + (1-κx)φ1(x)] = ",
        posterior_residual,
        "  (should be ~0)"
    )

    # ----------------------------------------------------------
    # Renewal identities
    # ----------------------------------------------------------
    F_renewal_residual =
        Fx - (tFx + β*tΘx*FR)

    G_renewal_residual =
        Gx - (tGx + β*tΘx*GR)

    println("\n[Renewal identities]")
    println(
        "F - [tildeF + β tildeTheta F(p11,z)] = ",
        F_renewal_residual,
        "  (should be ~0)"
    )
    println(
        "G - [tildeG + β tildeTheta G(p11,z)] = ",
        G_renewal_residual,
        "  (should be ~0)"
    )

    # ----------------------------------------------------------
    # z<p01 full-work identities
    # ----------------------------------------------------------
    Gconstant = 1 / (1 - β)

    println("\n[z<p01 full-work checks]")
    println("G(p11,z)   - 1/(1-β) = $(GR - Gconstant)")
    println("G(φ0(x),z) - 1/(1-β) = $(G0 - Gconstant)")
    println("G(φ1(x),z) - 1/(1-β) = $(G1 - Gconstant)")

    # If y>z and G(y,z)=1/(1-β), then
    #     tildeG(y,z) = [1-β tildeTheta(y,z)]/(1-β).
    tG_prediction(tΘ) = (1 - β*tΘ) / (1 - β)

    println("\n[z<p01 pre-ACK work checks]")

    if x > z
        println(
            "tildeG(x,z) - prediction = ",
            tGx - tG_prediction(tΘx),
            "  (should be ~0)"
        )
    else
        println("Skipping the x-level prediction because x≤z.")
    end

    println(
        "tildeG(φ0(x),z) - prediction = ",
        tG0 - tG_prediction(tΘ0),
        "  (should be ~0)"
    )
    println(
        "tildeG(φ1(x),z) - prediction = ",
        tG1 - tG_prediction(tΘ1),
        "  (should be ~0)"
    )
    println(
        "tildeG(p11,z) - prediction = ",
        tGR - tG_prediction(tΘR),
        "  (should be ~0)"
    )

    # ----------------------------------------------------------
    # Direct one-step reconstruction of full marginals
    # ----------------------------------------------------------
    f_one_step =
        r*κ*x +
        β*(κ*x*FR + (1 - κ*x)*F1 - F0)

    g_one_step =
        1 +
        β*(κ*x*GR + (1 - κ*x)*G1 - G0)

    println("\n[One-step reconstruction of full marginals]")
    println("f_one_step - f = $(f_one_step - f)  (should be ~0)")
    println("g_one_step - g = $(g_one_step - g)  (should be ~0)")

    return nothing
end

diagnose_z_lt_p01 (generic function with 1 method)

In [23]:
# Parameters for the z < p01 diagnostic
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0

x = 0.5
z = p01 / 10

@assert 0 ≤ z < p01
@assert 0 ≤ x ≤ 1

diagnose_z_lt_p01(
    x, z;
    p01=p01, ρ=ρ, κ=κ,
    r=r, β=β,
    ε=1e-14
)


Diagnostic at x=0.5, z=0.025 (z<p01)

[Basic states]
x       = 0.5
z       = 0.025
φ0(x)   = 0.55
φ1(x)   = 0.35
p11     = 0.85
x>z?    = true
φ0(x)>z? = true
φ1(x)>z? = true
p11>z?   = true

[Pre-ACK metrics]
tildeF(x,z)     = 0.8962868117797698
tildeG(x,z)     = 2.970550576184375
tildeTheta(x,z) = 0.8962868117797698
tildeTheta - tildeF/r = 0.0  (should be ~0)

[Pre-ACK marginal metrics]
tilde f, direct       = 0.036705078958601145
tilde f, consolidated = 0.036705078958601145
difference            = 0.0
tilde g, direct       = 0.30260349978659873
tilde g, consolidated = 0.30260349978659873
difference            = 0.0
tilde θ, direct       = 0.036705078958601145
tilde θ, consolidated = 0.036705078958601145
difference            = 0.0
tilde f_direct - r*tilde θ_direct = 0.0  (should be ~0)

[Full marginal metrics]
f(x,z) = 0.4000000000000115
g(x,z) = 1.00000000000002

[z<p01 full-marginal predictions]
f(x,z) - rκx = 1.149080830487037e-14  (should be ~0)
g(x,z) - 1   = 1.998401444325281

In [24]:
function plot_tildeg_tildetheta_vs_z_axis(
    x;
    p01, ρ, κ, β,
    ε=1e-10,
    zgrid=collect(range(0.0, 1.0; length=2001)),
    xlabel=L"$z$",
    ylabel=L"$\widetilde g(x,z),\ \widetilde\theta(x,z)$",
    width="11cm",
    height="7cm",
    xpad=0.03,
    ypad=0.05,
    atol=1e-12,
    eqtol=1e-12
)

    @assert length(zgrid) ≥ 2 "zgrid must contain at least two points"
    @assert issorted(zgrid) "zgrid must be sorted in nondecreasing order"

    # Labeled x-ticks.
    _, xtick_str, xticklab_str =
        make_xticks_and_labels(x; p01=p01, ρ=ρ, κ=κ)

    # Sample the two step functions.
    gvals  = Vector{Float64}(undef, length(zgrid))
    thvals = Vector{Float64}(undef, length(zgrid))

    for (i, z) in enumerate(zgrid)
        # With r=1, the first output F̃ equals Θ̃.
        θ̃, g̃, _, _, _ = tildeFG(
            x, z;
            p01=p01, ρ=ρ, κ=κ,
            r=1.0, β=β, ε=ε
        )

        gvals[i]  = float(g̃)
        thvals[i] = float(θ̃)
    end

    # Retain only the horizontal pieces. Open markers are used solely
    # to localize the sampled detections of jumps.
    g_line,  _, g_open,  _ =
        _stepdata_right_continuous(zgrid, gvals; atol=atol)

    th_line, _, th_open, _ =
        _stepdata_right_continuous(zgrid, thvals; atol=atol)

    # Horizontal segments on which g̃ and θ̃ coincide.
    eq_line = _coincidence_hsegments(
        zgrid, gvals, thvals;
        atol=atol,
        eqtol=eqtol
    )

    # Axis limits.
    finite_vals = filter(isfinite, vcat(gvals, thvals))
    isempty(finite_vals) &&
        error("No finite values were produced on zgrid")

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)

    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # Rug marks at sampled detections of jumps. Unless a discontinuity
    # lies on the grid, these only localize its position to grid accuracy.
    jump_zs = sort!(unique!(
        vcat(first.(g_open), first.(th_open))
    ))

    extra_xtick_str =
        "{" * join(string.(jump_zs), ",") * "}"

    extra_xticklab_str =
        "{" * join(fill("", length(jump_zs)), ",") * "}"

    lw_base  = "0.40pt"
    lw_shldr = "0.85pt"

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin => ymin,
            ymax => ymax,

            "xticklabel style" =>
                "{text height=1.8ex, text depth=0.25ex}",

            "tick align"       => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick"       => xtick_str,
            "xticklabels" => xticklab_str,

            "extra x ticks"       => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style" =>
                "{tick align=inside, tick style={black!55}}",

            "ytick" => "{0,1}",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",

            "legend pos" => "north west",
        },

        # g̃: black.
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "black"      => nothing
            },
            Coordinates(g_line)
        ),
        LegendEntry(L"$\widetilde g(x,z)$"),

        # θ̃: 50% gray.
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "color"      => "black!50"
            },
            Coordinates(th_line)
        ),
        LegendEntry(L"$\widetilde\theta(x,z)$"),

        # Gray shoulder on coincident segments.
        Plot(
            {
                "line width"  => lw_shldr,
                "solid"       => nothing,
                "color"       => "black!50",
                "forget plot" => nothing
            },
            Coordinates(eq_line)
        ),

        # Black centerline on coincident segments.
        Plot(
            {
                "line width"  => lw_base,
                "solid"       => nothing,
                "black"       => nothing,
                "forget plot" => nothing
            },
            Coordinates(eq_line)
        ),
    )
end

"""
    save_tildeg_tildetheta_vs_z_plot_pdf(outpdf, x; p01, ρ, κ, β, engine="pdflatex", quiet=true)

    Standalone .tex + compile to PDF (quiet by default).
    Matches your save_tildeFG_vs_z_plot_pdf wrapper.
"""
function save_tildeg_tildetheta_vs_z_plot_pdf(outpdf::AbstractString, x;
                                              p01, ρ, κ, β,
                                              engine::AbstractString="pdflatex",
                                              quiet::Bool=true,
                                              keep_tex::Bool=true,
                                              kwargs...)
    ax  = plot_tildeg_tildetheta_vs_z_axis(x; p01=p01, ρ=ρ, κ=κ, β=β, kwargs...)
    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError("outpdf must have extension .pdf or no extension"))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs
    PGFPlotsX.save(texfile, pic; include_preamble=true)

    engine_path = Sys.which(engine)
    engine_path === nothing &&
        throw(ArgumentError("LaTeX engine '$engine' was not found"))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    isfile(pdffile) ||
            error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_tildeg_tildetheta_vs_z_plot_pdf

In [25]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95

x0, _, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

p11 = p01 + ρ
x   = (x0 + p11) / 2

@assert 0 ≤ x0 < x < p11 ≤ 1

outpdf = save_tildeg_tildetheta_vs_z_plot_pdf(
    "tilde_g_theta_vs_z.pdf",
    x;
    p01=p01,
    ρ=ρ,
    κ=κ,
    β=β,
    ε=1e-14,
    zgrid=collect(range(0.0, 1.0; length=100_001)),
    keep_tex=false
)

@show x0 p11 x outpdf

x0 = 0.625
p11 = 0.85
x = 0.7375
outpdf = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\tilde_g_theta_vs_z.pdf"


"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\tilde_g_theta_vs_z.pdf"

In [26]:
function plot_tildef_tildeg_vs_z_axis(
    x;
    p01, ρ, κ, r, β,
    ε=1e-10,
    zgrid=collect(range(0.0, 1.0; length=2001)),
    xlabel=L"$z$",
    ylabel=L"$\widetilde f(x,z),\ \widetilde g(x,z)$",
    width="11cm",
    height="7cm",
    xpad=0.03,
    ypad=0.03,
    atol=1e-12,
    eqtol=1e-12
)

    length(zgrid) ≥ 2 ||
        throw(ArgumentError("zgrid must contain at least two points"))
    issorted(zgrid) ||
        throw(ArgumentError("zgrid must be sorted"))
    all(z -> isfinite(z) && 0.0 ≤ z ≤ 1.0, zgrid) ||
        throw(ArgumentError("zgrid must contain finite points in [0,1]"))

    # Labeled x-ticks.
    _, xtick_str, xticklab_str =
        make_xticks_and_labels(x; p01=p01, ρ=ρ, κ=κ)

    # Sample both pre-ACK marginal metrics jointly.
    fvals = Vector{Float64}(undef, length(zgrid))
    gvals = Vector{Float64}(undef, length(zgrid))

    for (i, z) in enumerate(zgrid)
        tf, tg, _ = tilde_marginal_metrics(
            x, z;
            p01=p01, ρ=ρ, κ=κ,
            r=r, β=β, ε=ε
        )

        fvals[i] = float(tf)
        gvals[i] = float(tg)
    end

    # Horizontal pieces of the right-continuous step functions.
    f_line, _, f_open, _ =
        _stepdata_right_continuous(zgrid, fvals; atol=atol)

    g_line, _, g_open, _ =
        _stepdata_right_continuous(zgrid, gvals; atol=atol)

    # Segments on which f̃ and g̃ coincide.
    eq_line = _coincidence_hsegments(
        zgrid, fvals, gvals;
        atol=atol,
        eqtol=eqtol
    )

    # Reference level y = 1-β.
    yref = 1 - β
    ref_tbl = PGFPlotsX.Table(
        x=[0.0, 1.0],
        y=[yref, yref]
    )

    # Include 0, 1, and the reference level in the displayed range.
    finite_vals = filter(
        isfinite,
        vcat(fvals, gvals, [0.0, 1.0, yref])
    )

    isempty(finite_vals) &&
        error("No finite values were produced on zgrid")

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)

    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # Rug marks at sampled jump detections.
    jump_zs = sort!(unique!(
        vcat(first.(f_open), first.(g_open))
    ))

    extra_xtick_str =
        "{" * join(string.(jump_zs), ",") * "}"

    extra_xticklab_str =
        "{" * join(fill("", length(jump_zs)), ",") * "}"

    lw_base  = "0.40pt"
    lw_shldr = "0.85pt"

    ax = @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin => ymin,
            ymax => ymax,

            "xticklabel style" =>
                "{text height=1.8ex, text depth=0.25ex}",

            "tick align"         => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick"       => xtick_str,
            "xticklabels" => xticklab_str,

            "extra x ticks"       => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style"  =>
                "{tick align=inside, tick style={black!55}}",

            "ytick"            => "{0,1}",
            "scaled y ticks"   => "false",
            "unbounded coords" => "jump",

            "legend pos" => "north west",
        },

        # f̃: black
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "black"      => nothing
            },
            Coordinates(f_line)
        ),
        LegendEntry(L"$\widetilde f(x,z)$"),

        # g̃: gray
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "color"      => "black!50"
            },
            Coordinates(g_line)
        ),
        LegendEntry(L"$\widetilde g(x,z)$"),
    )

    # Add the coincidence overlay only if coincident segments exist.
    if !isempty(eq_line)
        push!(
            ax,
            @pgf Plot(
                {
                    "line width"  => lw_shldr,
                    "solid"       => nothing,
                    "color"       => "black!50",
                    "forget plot" => nothing
                },
                Coordinates(eq_line)
            )
        )

        push!(
            ax,
            @pgf Plot(
                {
                    "line width"  => lw_base,
                    "solid"       => nothing,
                    "black"       => nothing,
                    "forget plot" => nothing
                },
                Coordinates(eq_line)
            )
        )
    end

    # Reference line, drawn last.
    push!(
        ax,
        @pgf Plot(
            {
                "line width"   => lw_base,
                "dash pattern" => "on 1pt off 1pt",
                "black"        => nothing
            },
            ref_tbl
        )
    )
    push!(ax, LegendEntry(L"$1-\beta$"))

    return ax
end


function save_tildef_tildeg_vs_z_plot_pdf(
    outpdf::AbstractString,
    x;
    p01, ρ, κ, r, β,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)

    ax = plot_tildef_tildeg_vs_z_axis(
        x;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β,
        kwargs...
    )
    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError(
            "outpdf must have extension .pdf or no extension"
        ))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    engine_path = Sys.which(engine)
    engine_path === nothing &&
        throw(ArgumentError("LaTeX engine '$engine' was not found"))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            tail = lines[max(1, length(lines) - 80):end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log", ".out")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end

save_tildef_tildeg_vs_z_plot_pdf (generic function with 1 method)

In [27]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0

x0, _, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

p11 = p01 + ρ
x   = (x0 + p11) / 2

@assert 0 ≤ x0 < x < p11 ≤ 1

outpdf = save_tildef_tildeg_vs_z_plot_pdf(
    "tildef_tildeg_vs_z_plot.pdf",
    x;
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    ε=1e-14,
    zgrid=collect(range(0.0, 1.0; length=10_001)),
    keep_tex=false
)

@show x0 p11 x outpdf

x0 = 0.625
p11 = 0.85
x = 0.7375
outpdf = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\tildef_tildeg_vs_z_plot.pdf"


"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\tildef_tildeg_vs_z_plot.pdf"

In [28]:
function plot_fg_metrics2_vs_z_axis(
    x;
    p01, ρ, κ, r, β,
    ε=1e-10,
    zgrid=collect(range(0.0, 1.0; length=2001)),
    xlabel=L"$z$",
    ylabel=L"$f(x,z),\ g(x,z)$",
    width="11cm",
    height="7cm",
    xpad=0.03,
    ypad=0.03,
    atol=1e-12,
    eqtol=1e-12
)

    length(zgrid) ≥ 2 ||
        throw(ArgumentError("zgrid must contain at least two points"))

    all(isfinite, zgrid) ||
        throw(ArgumentError("zgrid must contain only finite values"))

    all(z -> 0.0 ≤ z ≤ 1.0, zgrid) ||
        throw(ArgumentError("zgrid must be contained in [0,1]"))

    all(diff(zgrid) .> 0) ||
        throw(ArgumentError("zgrid must be strictly increasing"))

    0.0 ≤ β < 1.0 ||
        throw(ArgumentError("β must satisfy 0 ≤ β < 1"))

    # Labeled x-ticks.
    _, xtick_str, xticklab_str =
        make_xticks_and_labels(x; p01=p01, ρ=ρ, κ=κ)

    # Sample both full marginal metrics jointly.
    fvals = Vector{Float64}(undef, length(zgrid))
    gvals = Vector{Float64}(undef, length(zgrid))

    for (i, z) in enumerate(zgrid)
        f, g = fg_metric2(
            x, z;
            p01=p01,
            ρ=ρ,
            κ=κ,
            r=r,
            β=β,
            ε=ε
        )

        fvals[i] = float(f)
        gvals[i] = float(g)
    end

    # Horizontal pieces of the right-continuous step functions.
    f_line, _, f_open, _ =
        _stepdata_right_continuous(zgrid, fvals; atol=atol)

    g_line, _, g_open, _ =
        _stepdata_right_continuous(zgrid, gvals; atol=atol)

    # Segments on which f and g coincide.
    eq_line = _coincidence_hsegments(
        zgrid, fvals, gvals;
        atol=atol,
        eqtol=eqtol
    )

    # Rug marks at sampled detections of jumps.
    jump_zs = sort!(unique!(
        vcat(first.(f_open), first.(g_open))
    ))

    extra_xtick_str =
        "{" * join(string.(jump_zs), ",") * "}"

    extra_xticklab_str =
        "{" * join(fill("", length(jump_zs)), ",") * "}"

    # Reference level y = 1-β.
    yref = 1 - β

    ref_tbl = PGFPlotsX.Table(
        x=[0.0, 1.0],
        y=[yref, yref]
    )

    # Axis limits, including the displayed y-ticks and reference level.
    metric_vals = filter(isfinite, vcat(fvals, gvals))

    isempty(metric_vals) &&
        error("No finite metric values were produced on zgrid")

    finite_vals = vcat(metric_vals, [0.0, 1.0, yref])

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)

    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    lw_base  = "0.40pt"
    lw_shldr = "0.85pt"

    ax = @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin => ymin,
            ymax => ymax,

            "xticklabel style" =>
                "{text height=1.8ex, text depth=0.25ex}",

            "tick align"         => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick"       => xtick_str,
            "xticklabels" => xticklab_str,

            "extra x ticks"       => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style"  =>
                "{tick align=inside, tick style={black!55}}",

            "ytick"            => "{0,1}",
            "scaled y ticks"   => "false",
            "unbounded coords" => "jump",

            "legend pos" => "north west",
        },

        # f: black.
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "black"      => nothing
            },
            Coordinates(f_line)
        ),
        LegendEntry(L"$f(x,z)$"),

        # g: gray.
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "color"      => "black!50"
            },
            Coordinates(g_line)
        ),
        LegendEntry(L"$g(x,z)$"),
    )

    # Add coincidence highlighting only when such segments exist.
    if !isempty(eq_line)
        push!(
            ax,
            @pgf Plot(
                {
                    "line width"  => lw_shldr,
                    "solid"       => nothing,
                    "color"       => "black!50",
                    "forget plot" => nothing
                },
                Coordinates(eq_line)
            )
        )

        push!(
            ax,
            @pgf Plot(
                {
                    "line width"  => lw_base,
                    "solid"       => nothing,
                    "black"       => nothing,
                    "forget plot" => nothing
                },
                Coordinates(eq_line)
            )
        )
    end

    # Draw the reference line last so that it remains visible.
    push!(
        ax,
        @pgf Plot(
            {
                "line width"   => lw_base,
                "dash pattern" => "on 1pt off 1pt",
                "black"        => nothing
            },
            ref_tbl
        )
    )
    push!(ax, LegendEntry(L"$1-\beta$"))

    return ax
end


function save_fg_metrics2_vs_z_plot_pdf(
    outpdf::AbstractString,
    x;
    p01, ρ, κ, r, β,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)

    ax = plot_fg_metrics2_vs_z_axis(
        x;
        p01=p01, ρ=ρ, κ=κ,
        r=r, β=β,
        kwargs...
    )
    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError(
            "outpdf must have extension .pdf or no extension"
        ))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    engine_path = Sys.which(engine)
    engine_path === nothing &&
        throw(ArgumentError("LaTeX engine '$engine' was not found"))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            tail = lines[max(1, length(lines) - 80):end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log", ".out")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end

save_fg_metrics2_vs_z_plot_pdf (generic function with 1 method)

In [29]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0
ε   = 1e-14

x0, x1, xhi = fixed_points(; p01=p01, ρ=ρ, κ=κ)

p11 = p01 + ρ
z   = (x1 + x0) / 2
x   = (x0 + p11) / 2

@assert x1 < z < x0 < x < p11 ≤ 1

# Joint evaluation of the pre-ACK marginal quantities.
tf_direct, tg, ttheta = tilde_marginal_metrics(
    x, z;
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    ε=ε
)

# Independent wrapper based on f̃ = r θ̃.
tf_from_theta = tildef_from_tildetheta(
    x, z;
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    ε=ε
)

difference = tf_direct - tf_from_theta

@show x0 x1 xhi p11 z x
@show tf_direct tf_from_theta difference tg ttheta

@assert isapprox(
    tf_direct,
    tf_from_theta;
    atol=1e-12,
    rtol=1e-12
)

x0 = 0.625
x1 = 0.29668135123946104
xhi = 1.0533186487605388
p11 = 0.85
z = 0.4608406756197305
x = 0.7375
tf_direct = 0.061900764363861493
tf_from_theta = 0.061900764363861493
difference = 0.0
tg = 0.06475480667019795
ttheta = 0.061900764363861493


In [30]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.99
r   = 1.0
ε   = 1e-14

x0, _, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

p11 = p01 + ρ
x   = (x0 + p11) / 2

@assert 0 ≤ x0 < x < p11 ≤ 1

outpdf = save_fg_metrics2_vs_z_plot_pdf(
    "fg_vs_z_plot.pdf",
    x;
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    ε=ε,
    zgrid=collect(range(0.0, 1.0; length=10_001)),
    keep_tex=false
)

@show x0 p11 x outpdf

x0 = 0.625
p11 = 0.85
x = 0.7375
outpdf = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\fg_vs_z_plot.pdf"


"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\fg_vs_z_plot.pdf"

In [31]:
# Build horizontal segments on which A ≈ B.
# The segmentation includes every sampled change in either step function.
function _coincidence_hsegments(
    xgrid::AbstractVector{<:Real},
    Avals::AbstractVector{<:Real},
    Bvals::AbstractVector{<:Real};
    atol::Real=1e-12,
    eqtol::Real=1e-12
)
    n = length(xgrid)

    n ≥ 2 ||
        throw(ArgumentError("xgrid must contain at least two points"))

    length(Avals) == n ||
        throw(DimensionMismatch("Avals and xgrid must have equal lengths"))

    length(Bvals) == n ||
        throw(DimensionMismatch("Bvals and xgrid must have equal lengths"))

    breaks = Int[]

    for i in 1:n-1
        Achanges = !isapprox(
            Avals[i + 1], Avals[i];
            atol=atol,
            rtol=0
        )

        Bchanges = !isapprox(
            Bvals[i + 1], Bvals[i];
            atol=atol,
            rtol=0
        )

        (Achanges || Bchanges) && push!(breaks, i)
    end

    coords = Tuple{Float64,Float64}[]
    seg_start = 1

    for i in breaks
        xL = float(xgrid[seg_start])
        xR = float(xgrid[i + 1])
        yA = float(Avals[i])
        yB = float(Bvals[i])

        if isfinite(yA) &&
           isfinite(yB) &&
           isapprox(yA, yB; atol=eqtol, rtol=0)

            push!(coords, (xL, yA))
            push!(coords, (xR, yA))
            push!(coords, (NaN, NaN))
        end

        seg_start = i + 1
    end

    # Last common segment.
    xL = float(xgrid[seg_start])
    xR = float(xgrid[end])
    yA = float(Avals[end])
    yB = float(Bvals[end])

    if isfinite(yA) &&
       isfinite(yB) &&
       isapprox(yA, yB; atol=eqtol, rtol=0)

        push!(coords, (xL, yA))
        push!(coords, (xR, yA))
        push!(coords, (NaN, NaN))
    end

    return coords
end


function plot_fg_vs_x_axis(
    z;
    p01, ρ, κ, r, β,
    ε=1e-10,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    xlabel=L"$x$",
    ylabel=L"$f(x,z),\ g(x,z)$",
    width="11cm",
    height="7cm",
    xpad=0.03,
    ypad=0.05,
    atol=1e-12,
    eqtol=1e-12
)
    length(xgrid) ≥ 2 ||
        throw(ArgumentError("xgrid must contain at least two points"))

    all(x -> isfinite(x) && 0.0 ≤ x ≤ 1.0, xgrid) ||
        throw(ArgumentError(
            "xgrid must contain finite points in [0,1]"
        ))

    all(diff(xgrid) .> 0) ||
        throw(ArgumentError(
            "xgrid must be strictly increasing"
        ))

    # Structural ticks, with the fixed threshold identified by z.
    _, xtick_str, xticklab_str =
        make_xticks_and_labels(
            z;
            p01=p01,
            ρ=ρ,
            κ=κ,
            point_label=L"z"
        )

    # Compute f and g jointly, avoiding duplicate trajectory calculations.
    fvals = Vector{Float64}(undef, length(xgrid))
    gvals = Vector{Float64}(undef, length(xgrid))

    for (i, x) in enumerate(xgrid)
        f, g = fg_metric2(
            x, z;
            p01=p01,
            ρ=ρ,
            κ=κ,
            r=r,
            β=β,
            ε=ε
        )

        fvals[i] = float(f)
        gvals[i] = float(g)
    end

    # Horizontal-only right-continuous step polylines.
    f_line, _, f_open, _ =
        _stepdata_right_continuous(
            xgrid, fvals;
            atol=atol
        )

    g_line, _, g_open, _ =
        _stepdata_right_continuous(
            xgrid, gvals;
            atol=atol
        )

    # Horizontal segments on which f and g coincide.
    eq_line = _coincidence_hsegments(
        xgrid, fvals, gvals;
        atol=atol,
        eqtol=eqtol
    )

    # Keep all finite values and the zero tick visible.
    finite_vals = filter(
        isfinite,
        vcat(fvals, gvals, [0.0])
    )

    isempty(finite_vals) &&
        error("No finite metric values were produced on xgrid")

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)

    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    # Rug marks at sampled jump detections.
    jump_xs = sort!(unique!(
        vcat(first.(f_open), first.(g_open))
    ))

    extra_xtick_str =
        "{" * join(string.(jump_xs), ",") * "}"

    extra_xticklab_str =
        "{" * join(fill("", length(jump_xs)), ",") * "}"

    lw_base  = "0.40pt"
    lw_shldr = "0.85pt"

    ax = @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin => ymin,
            ymax => ymax,

            "xtick"       => xtick_str,
            "xticklabels" => xticklab_str,

            "xticklabel style" =>
                "{text height=1.8ex, text depth=0.25ex}",

            "extra x ticks"       => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style"  =>
                "{tick align=inside, tick style={black!55}}",

            "ytick"             => "{0}",
            "tick align"        => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",
            "scaled y ticks"    => "false",

            # Each NaN terminates a horizontal piece.
            "unbounded coords" => "jump",

            "legend pos" => "north east",
        },

        # f: solid black.
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "black"      => nothing
            },
            Coordinates(f_line)
        ),
        LegendEntry(L"$f(x,z)$"),

        # g: solid gray.
        Plot(
            {
                "line width" => lw_base,
                "solid"      => nothing,
                "color"      => "black!50"
            },
            Coordinates(g_line)
        ),
        LegendEntry(L"$g(x,z)$"),
    )

    # Add the coincidence highlight only when such segments exist.
    if !isempty(eq_line)
        # Thick gray underlay creates the shoulders.
        push!(
            ax,
            @pgf Plot(
                {
                    "line width"  => lw_shldr,
                    "solid"       => nothing,
                    "color"       => "black!40",
                    "forget plot" => nothing
                },
                Coordinates(eq_line)
            )
        )

        # Restore f's black centerline at the requested base width.
        push!(
            ax,
            @pgf Plot(
                {
                    "line width"  => lw_base,
                    "solid"       => nothing,
                    "black"       => nothing,
                    "forget plot" => nothing
                },
                Coordinates(eq_line)
            )
        )
    end

    return ax
end


function save_fg_vs_x_plot_pdf(
    outpdf::AbstractString,
    z;
    p01, ρ, κ, r, β,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)
    ax = plot_fg_vs_x_axis(
        z;
        p01=p01,
        ρ=ρ,
        κ=κ,
        r=r,
        β=β,
        kwargs...
    )

    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError(
            "outpdf must have extension .pdf or no extension"
        ))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs

    PGFPlotsX.save(
        texfile,
        pic;
        include_preamble=true
    )

    engine_path = Sys.which(engine)

    engine_path === nothing &&
        throw(ArgumentError(
            "LaTeX engine '$engine' was not found"
        ))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            tail = lines[max(1, length(lines) - 80):end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log", ".out")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end

save_fg_vs_x_plot_pdf (generic function with 1 method)

In [32]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0
ε   = 1e-12

x0, x1, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

p11 = p01 + ρ
z   = (x1 + x0) / 2

@assert 0 ≤ p01 < x1 < z < x0 < p11 ≤ 1

# Include the main structural points exactly.
xgrid = sort!(unique!(vcat(
    collect(range(0.0, 1.0; length=1001)),
    [p01, x1, z, x0, p11]
)))

outpdf = save_fg_vs_x_plot_pdf(
    "fg_vs_x.pdf",
    z;
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    ε=ε,
    xgrid=xgrid,
    keep_tex=false
)

@show x1 z x0 p11 outpdf

x1 = 0.29668135123946104
z = 0.4608406756197305
x0 = 0.625
p11 = 0.85
outpdf = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\fg_vs_x.pdf"


"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\fg_vs_x.pdf"

In [33]:
using Statistics


# Snap values to the anchor of the current run.
# Comparing with an anchor prevents tolerance chaining across a gradual trend.
function _snap_consecutive(
    y::AbstractVector{<:Real};
    tol::Real=1e-8
)
    tol ≥ 0 ||
        throw(ArgumentError("tol must be nonnegative"))

    yy = Float64.(y)
    isempty(yy) && return yy

    anchor = yy[1]

    @inbounds for i in 2:length(yy)
        yi = yy[i]

        if isfinite(yi) &&
           isfinite(anchor) &&
           abs(yi - anchor) ≤ tol

            yy[i] = anchor
        else
            anchor = yi
        end
    end

    return yy
end


# Construct a sampled polyline with NaN breaks at apparent jumps.
#
# A large adjacent change is classified as a jump only when its secant
# slope is also a local outlier. This avoids treating an ordinary smooth
# slope as a discontinuity.
function _polyline_with_jump_breaks(
    xgrid::AbstractVector{<:Real},
    ygrid::AbstractVector{<:Real};
    jump_atol::Real=1e-6,
    jump_rtol::Real=0.0,
    jump_factor::Real=20.0
)
    n = length(xgrid)

    n ≥ 2 ||
        throw(ArgumentError("xgrid must contain at least two points"))

    length(ygrid) == n ||
        throw(DimensionMismatch(
            "xgrid and ygrid must have equal lengths"
        ))

    jump_atol ≥ 0 ||
        throw(ArgumentError("jump_atol must be nonnegative"))

    jump_rtol ≥ 0 ||
        throw(ArgumentError("jump_rtol must be nonnegative"))

    jump_factor > 1 ||
        throw(ArgumentError("jump_factor must exceed 1"))

    xx = Float64.(xgrid)
    yy = Float64.(ygrid)

    all(isfinite, xx) ||
        throw(ArgumentError("xgrid must contain only finite values"))

    all(diff(xx) .> 0) ||
        throw(ArgumentError("xgrid must be strictly increasing"))

    increments = fill(NaN, n - 1)
    slopes     = fill(NaN, n - 1)

    for i in 1:n-1
        y1 = yy[i]
        y2 = yy[i + 1]

        if isfinite(y1) && isfinite(y2)
            increments[i] = abs(y2 - y1)
            slopes[i] = increments[i] / (xx[i + 1] - xx[i])
        end
    end

    jump_after = falses(n - 1)

    for i in 1:n-1
        δ = increments[i]
        isfinite(δ) || continue

        cutoff = max(
            jump_atol,
            jump_rtol * max(abs(yy[i]), abs(yy[i + 1]))
        )

        δ > cutoff || continue

        neighbour_slopes = Float64[]

        for j in max(1, i - 2):min(n - 1, i + 2)
            j == i && continue

            isfinite(slopes[j]) &&
                push!(neighbour_slopes, slopes[j])
        end

        if isempty(neighbour_slopes)
            # With only one interval, magnitude is the only available test.
            jump_after[i] = true
        else
            local_slope = median(neighbour_slopes)

            jump_after[i] =
                slopes[i] >
                jump_factor * max(local_slope, eps(Float64))
        end
    end

    coords    = Tuple{Float64,Float64}[]
    jump_locs = Float64[]

    for i in 1:n
        if isfinite(yy[i])
            push!(coords, (xx[i], yy[i]))
        else
            push!(coords, (NaN, NaN))
        end

        if i < n && jump_after[i]
            push!(coords, (NaN, NaN))
            push!(jump_locs, xx[i + 1])
        end
    end

    sort!(unique!(jump_locs))

    return coords, jump_locs
end


# Construct curve pieces on which f and g coincide.
# Breaks are inserted at detected jumps of either curve.
function _coincidence_curve_segments(
    xgrid::AbstractVector{<:Real},
    fvals::AbstractVector{<:Real},
    gvals::AbstractVector{<:Real},
    jump_locs::AbstractVector{<:Real};
    eqtol::Real=1e-12
)
    n = length(xgrid)

    length(fvals) == n ||
        throw(DimensionMismatch(
            "fvals and xgrid must have equal lengths"
        ))

    length(gvals) == n ||
        throw(DimensionMismatch(
            "gvals and xgrid must have equal lengths"
        ))

    xx = Float64.(xgrid)
    ff = Float64.(fvals)
    gg = Float64.(gvals)

    jump_set = Set(Float64.(jump_locs))
    coords   = Tuple{Float64,Float64}[]
    open_run = false

    for i in 1:n-1
        equal_left =
            isfinite(ff[i]) &&
            isfinite(gg[i]) &&
            isapprox(ff[i], gg[i]; atol=eqtol, rtol=0)

        equal_right =
            isfinite(ff[i + 1]) &&
            isfinite(gg[i + 1]) &&
            isapprox(ff[i + 1], gg[i + 1]; atol=eqtol, rtol=0)

        interval_is_continuous =
            !(xx[i + 1] in jump_set)

        if equal_left &&
           equal_right &&
           interval_is_continuous

            if !open_run
                push!(coords, (xx[i], ff[i]))
            end

            push!(coords, (xx[i + 1], ff[i + 1]))
            open_run = true
        elseif open_run
            push!(coords, (NaN, NaN))
            open_run = false
        end
    end

    open_run && push!(coords, (NaN, NaN))

    return coords
end


function plot_fg_metrics2_on_diagonal_vs_x_axis(;
    p01, ρ, κ, r, β,
    ε=1e-10,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    xlabel=L"$x$",
    ylabel=L"$f(x,x),\ g(x,x)$",
    width="11cm",
    height="7cm",
    xpad=0.03,
    ypad=0.03,
    snap_tol_f=0.0,
    snap_tol_g=1e-8,
    jump_atol=1e-6,
    jump_rtol=0.0,
    jump_factor=20.0,
    eqtol=1e-12,
    smooth_curves::Bool=true
)
    length(xgrid) ≥ 2 ||
        throw(ArgumentError(
            "xgrid must contain at least two points"
        ))

    all(x -> isfinite(x) && 0.0 ≤ x ≤ 1.0, xgrid) ||
        throw(ArgumentError(
            "xgrid must contain finite points in [0,1]"
        ))

    all(diff(xgrid) .> 0) ||
        throw(ArgumentError(
            "xgrid must be strictly increasing"
        ))

    0.0 ≤ β < 1.0 ||
        throw(ArgumentError("β must satisfy 0 ≤ β < 1"))

    xpad ≥ 0 ||
        throw(ArgumentError("xpad must be nonnegative"))

    ypad ≥ 0 ||
        throw(ArgumentError("ypad must be nonnegative"))

    # Fixed points and structural x-ticks.
    x0_fp, x1_fp, _ =
        fixed_points(; p01=p01, ρ=ρ, κ=κ)

    p11 = p01 + ρ

    items = [
        (0.0,          "\\ensuremath{0}"),
        (Float64(p01), "\\ensuremath{p_{01}}"),
        (Float64(x1_fp), "\\ensuremath{x^{1}}"),
        (Float64(x0_fp), "\\ensuremath{x^{0}}"),
        (Float64(p11), "\\ensuremath{p_{11}}"),
        (1.0,          "\\ensuremath{1}")
    ]

    sort!(items; by=first)

    merged = Tuple{Float64,String}[]

    for (u, label) in items
        if isempty(merged) ||
           abs(u - merged[end][1]) > 1e-12

            push!(merged, (u, label))
        end
    end

    xticks = first.(merged)
    labels = last.(merged)

    xtick_str =
        "{" * join(string.(xticks), ",") * "}"

    phantom =
        "\\vphantom{\\ensuremath{p_{11}}}"

    xticklab_str =
        "{" *
        join([phantom * label for label in labels], ",") *
        "}"

    # Evaluate f and g jointly once at each diagonal point (x,x).
    fvals = Vector{Float64}(undef, length(xgrid))
    gvals = Vector{Float64}(undef, length(xgrid))

    for (i, x) in enumerate(xgrid)
        f, g = fg_metric2(
            x, x;
            p01=p01,
            ρ=ρ,
            κ=κ,
            r=r,
            β=β,
            ε=ε
        )

        fvals[i] = float(f)
        gvals[i] = float(g)
    end

    snap_tol_f > 0 &&
        (fvals = _snap_consecutive(fvals; tol=snap_tol_f))

    snap_tol_g > 0 &&
        (gvals = _snap_consecutive(gvals; tol=snap_tol_g))

    # Curves with breaks at detected discontinuities.
    f_line, f_jumps =
        _polyline_with_jump_breaks(
            xgrid, fvals;
            jump_atol=jump_atol,
            jump_rtol=jump_rtol,
            jump_factor=jump_factor
        )

    g_line, g_jumps =
        _polyline_with_jump_breaks(
            xgrid, gvals;
            jump_atol=jump_atol,
            jump_rtol=jump_rtol,
            jump_factor=jump_factor
        )

    jump_xs = sort!(unique!(vcat(f_jumps, g_jumps)))

    eq_line = _coincidence_curve_segments(
        xgrid,
        fvals,
        gvals,
        jump_xs;
        eqtol=eqtol
    )

    extra_xtick_str =
        "{" * join(string.(jump_xs), ",") * "}"

    extra_xticklab_str =
        "{" * join(fill("", length(jump_xs)), ",") * "}"

    # Reference line.
    yref = 1 - β

    ref_tbl = PGFPlotsX.Table(
        x=[0.0, 1.0],
        y=[yref, yref]
    )

    # Include the displayed ticks 0 and 1 in the y-range.
    metric_vals = filter(isfinite, vcat(fvals, gvals))

    isempty(metric_vals) &&
        error("No finite metric values were produced on xgrid")

    finite_vals = vcat(metric_vals, [0.0, 1.0, yref])

    ymin = minimum(finite_vals)
    ymax = maximum(finite_vals)

    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    lw_base  = "0.40pt"
    lw_shldr = "0.85pt"

    f_opts = Pair{Any,Any}[
        "line width" => lw_base,
        "black"      => nothing
    ]

    g_opts = Pair{Any,Any}[
        "line width" => lw_base,
        "color"      => "black!50"
    ]

    if smooth_curves
        push!(f_opts, "smooth" => nothing)
        push!(g_opts, "smooth" => nothing)
    else
        push!(f_opts, "solid" => nothing)
        push!(g_opts, "solid" => nothing)
    end

    ax = @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin => -xpad,
            xmax => 1 + xpad,
            ymin => ymin,
            ymax => ymax,

            "xtick"       => xtick_str,
            "xticklabels" => xticklab_str,

            "xticklabel style" =>
                "{text height=1.8ex, text depth=0.25ex}",

            "extra x ticks"       => extra_xtick_str,
            "extra x tick labels" => extra_xticklab_str,
            "extra x tick style"  =>
                "{tick align=inside, tick style={black!55}}",

            "xtick pos" => "bottom",
            "ytick pos" => "left",

            "ytick"          => "{0,1}",
            "scaled y ticks" => "false",

            "tick align"         => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "unbounded coords" => "jump",
            "legend pos"      => "north west"
        },

        Plot(
            PGFPlotsX.Options(f_opts...),
            Coordinates(f_line)
        ),
        LegendEntry(L"$f(x,x)$"),

        Plot(
            PGFPlotsX.Options(g_opts...),
            Coordinates(g_line)
        ),
        LegendEntry(L"$g(x,x)$")
    )

    # Show coincident portions as a black centerline with gray shoulders.
    if !isempty(eq_line)
        eq_shoulder_opts = Pair{Any,Any}[
            "line width"  => lw_shldr,
            "color"       => "black!50",
            "forget plot" => nothing
        ]

        eq_center_opts = Pair{Any,Any}[
            "line width"  => lw_base,
            "black"       => nothing,
            "forget plot" => nothing
        ]

        if smooth_curves
            push!(eq_shoulder_opts, "smooth" => nothing)
            push!(eq_center_opts, "smooth" => nothing)
        else
            push!(eq_shoulder_opts, "solid" => nothing)
            push!(eq_center_opts, "solid" => nothing)
        end

        push!(
            ax,
            Plot(
                PGFPlotsX.Options(eq_shoulder_opts...),
                Coordinates(eq_line)
            )
        )

        push!(
            ax,
            Plot(
                PGFPlotsX.Options(eq_center_opts...),
                Coordinates(eq_line)
            )
        )
    end

    # Draw the reference line last.
    push!(
        ax,
        @pgf Plot(
            {
                "line width"   => lw_base,
                "dash pattern" => "on 1pt off 1pt",
                "black"        => nothing
            },
            ref_tbl
        )
    )

    push!(ax, LegendEntry(L"$1-\beta$"))

    return ax
end


function save_fg_metrics2_on_diagonal_vs_x_plot_pdf(
    outpdf::AbstractString;
    p01, ρ, κ, r, β,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)
    ax = plot_fg_metrics2_on_diagonal_vs_x_axis(
        ;
        p01=p01,
        ρ=ρ,
        κ=κ,
        r=r,
        β=β,
        kwargs...
    )

    pic = TikzPicture(ax)

    rawpath = abspath(outpdf)
    root, ext = splitext(rawpath)

    outpdf_abs = if isempty(ext)
        root * ".pdf"
    elseif lowercase(ext) == ".pdf"
        root * ".pdf"
    else
        throw(ArgumentError(
            "outpdf must have extension .pdf or no extension"
        ))
    end

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    mkpath(outdir)

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = outpdf_abs

    PGFPlotsX.save(
        texfile,
        pic;
        include_preamble=true
    )

    engine_path = Sys.which(engine)

    engine_path === nothing &&
        throw(ArgumentError(
            "LaTeX engine '$engine' was not found"
        ))

    cmd = `$(engine_path) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            tail = lines[max(1, length(lines) - 80):end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX finished without producing $(pdffile)")

    if !keep_tex
        for ext in (".tex", ".aux", ".log", ".out")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end

save_fg_metrics2_on_diagonal_vs_x_plot_pdf (generic function with 1 method)

In [34]:
# Parameters (example)
p01 = 0.25
ρ   = 0.6
κ   = 0.8
β   = 0.95
r   = 1.0
ε   = 1e-12

outpdf = save_fg_metrics2_on_diagonal_vs_x_plot_pdf(
    "fg_diag.pdf";
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    ε=ε,
    xgrid=range(0.0, 1.0; length=2001),
    jump_atol=5e-3,
    keep_tex=false
)

@show outpdf

outpdf = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\fg_diag.pdf"


"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\fg_diag.pdf"

In [35]:
# MP index m(x) = f(x,x) / g(x,x)
function mp_index(
    x;
    p01, ρ, κ, r, β,
    ε=1e-10,
    gtol=0.0
)
    xf = Float64(x)

    0.0 ≤ xf ≤ 1.0 ||
        throw(DomainError(x, "x must belong to [0,1]"))

    # Joint evaluation avoids duplicating the trajectory calculations.
    fxx, gxx = fg_metric2(
        xf, xf;
        p01=p01,
        ρ=ρ,
        κ=κ,
        r=r,
        β=β,
        ε=ε
    )

    fxx = Float64(fxx)
    gxx = Float64(gxx)

    return (
        isfinite(fxx) &&
        isfinite(gxx) &&
        gxx > gtol
    ) ? fxx / gxx : NaN
end


"""
    zstar(λ; p01, ρ, κ, r, β, ...)

Numerically invert the MP index m(x).

The endpoint convention is

- z*(λ) = 0 for λ < 0;
- z*(λ) = 1 for λ > m(1);
- for 0 ≤ λ ≤ m(1), solve m(z) = λ.

For an interior λ, the routine returns the lower generalized inverse

    inf{x ∈ [0,1] : m(x) ≥ λ}.

Hence, if m is strictly increasing, it returns the unique solution.
If m has a flat portion at level λ, it selects its left endpoint.

Assumes that m is continuous and nondecreasing on [0,1], with m(0)=0.
"""
function zstar(
    λ;
    p01, ρ, κ, r, β,
    ε=1e-10,
    gtol=0.0,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200
)
    λf = Float64(λ)

    isfinite(λf) ||
        throw(ArgumentError("λ must be finite"))

    xatol > 0 ||
        throw(ArgumentError("xatol must be positive"))

    λatol ≥ 0 ||
        throw(ArgumentError("λatol must be nonnegative"))

    maxiter ≥ 1 ||
        throw(ArgumentError("maxiter must be positive"))

    # Prescribed lower-boundary convention.
    λf < 0.0 && return 0.0

    m0 = mp_index(
        0.0;
        p01=p01, ρ=ρ, κ=κ, r=r, β=β,
        ε=ε, gtol=gtol
    )

    m1 = mp_index(
        1.0;
        p01=p01, ρ=ρ, κ=κ, r=r, β=β,
        ε=ε, gtol=gtol
    )

    isfinite(m0) ||
        error("m(0) is not finite")

    isfinite(m1) ||
        error("m(1) is not finite")

    abs(m0) ≤ λatol ||
        error("Expected m(0)=0, but obtained m(0)=$m0")

    m1 ≥ m0 - λatol ||
        error("Endpoint values are inconsistent with a nondecreasing index: " *
              "m(0)=$m0 and m(1)=$m1")

    # Prescribed upper-boundary convention.
    λf > m1 && return 1.0

    # Numerically identify the endpoint cases.
    λf ≤ λatol && return 0.0
    m1 - λf ≤ λatol && return 1.0

    # Maintain m(a) < λ ≤ m(b). Returning b yields the
    # lower generalized inverse, including on a flat portion.
    a = 0.0
    b = 1.0

    for _ in 1:maxiter
        0.5 * (b - a) ≤ xatol && return b

        c = 0.5 * (a + b)

        mc = mp_index(
            c;
            p01=p01, ρ=ρ, κ=κ, r=r, β=β,
            ε=ε, gtol=gtol
        )

        isfinite(mc) ||
            error("m($c) is not finite")

        if mc < λf
            a = c
        else
            b = c
        end
    end

    return b
end

zstar

In [36]:
# --- helper: snap consecutive values within tol (optional) ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

"""
Plot z*(λ) versus λ.

By default, plots on [0, m(1)] × [0,1].
If desired, one can extend the λ-range outside [0,m(1)] to show the clipped parts
z*(λ)=0 for λ<0 and z*(λ)=1 for λ>m(1).
"""
function plot_zstar_vs_lambda_axis(; p01, ρ, κ, r, β, ε=1e-10,
                                   lambdagrid=nothing,
                                   lambda_min=nothing,
                                   lambda_max=nothing,
                                   nlambda=2001,
                                   xlabel=L"$\lambda$",
                                   ylabel=L"$z^\star(\lambda)$",
                                   width="8cm", height="8cm",
                                   snap_tol_z=0.0,
                                   smooth_curve::Bool=false)

    # fixed points
    x0_fp, x1_fp, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

    # structural λ-values
    mp01 = mp_index(float(p01);      p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx1  = mp_index(float(x1_fp);  p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx0  = mp_index(float(x0_fp);  p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mp11 = mp_index(float(p01+ρ);    p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    m1   = mp_index(1.0;           p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    if !isfinite(m1)
        error("m(1) is not finite.")
    end

    λmin = isnothing(lambda_min) ? 0.0       : float(lambda_min)
    λmax = isnothing(lambda_max) ? float(m1) : float(lambda_max)

    if isnothing(lambdagrid)
        λgrid = collect(range(λmin, λmax; length=nlambda))
    else
        λgrid = Float64.(collect(lambdagrid))
        λmin = minimum(λgrid)
        λmax = maximum(λgrid)
    end

    items = [
        (0.0,         "\\ensuremath{0}"),
        (float(mp01), "\\makebox[1.5pt][r]{\\ensuremath{m(p_{01})}\\hspace{0.5pt}}"),
        (float(mx1),  "\\makebox[1.5pt][l]{\\ensuremath{m(x^{1})}\\hspace{0.5pt}}"),
        (float(mx0),  "\\makebox[1.5pt][r]{\\ensuremath{m(x^{0})}\\hspace{0.5pt}}"),
        (float(mp11), "\\ensuremath{m(p_{11})}"),
        (float(m1),   "\\ensuremath{m(1)}"),
    ]
    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{m(p_{11})}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    zvals = similar(λgrid, Float64)
    for (i, λ) in enumerate(λgrid)
        zvals[i] = float(zstar(λ; p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    end

    if snap_tol_z > 0
        zvals = _snap_consecutive(zvals; tol=snap_tol_z)
    end

    coords = Tuple{Float64,Float64}[
        (float(λ), isfinite(z) ? float(z) : NaN) for (λ, z) in zip(λgrid, zvals)
    ]

    curve_opts = Pair{Any,Any}[
        "line width" => "0.40pt",
        "black"      => nothing,
    ]
    if smooth_curve
        push!(curve_opts, "smooth" => nothing)
    else
        push!(curve_opts, "solid" => nothing)
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => λmin,
            xmax   => λmax,
            ymin   => 0,
            ymax   => 1,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0,1}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot(PGFPlotsX.Options(curve_opts...), Coordinates(coords)),
    )
end

function save_zstar_vs_lambda_plot_pdf(outpdf::AbstractString;
                                       p01, ρ, κ, r, β,
                                       engine::AbstractString="pdflatex",
                                       quiet::Bool=true,
                                       keep_tex::Bool=true,
                                       kwargs...)
    ax  = plot_zstar_vs_lambda_axis(; p01=p01, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_zstar_vs_lambda_plot_pdf (generic function with 1 method)

In [37]:
p01 = 0.25
ρ   = 0.6
κ   = 0.8
r   = 1.0
β   = 0.95

save_zstar_vs_lambda_plot_pdf(
    "zstar_vs_lambda.pdf";
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    smooth_curve=false,
    keep_tex=false
)

"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\zstar_vs_lambda.pdf"

In [38]:
# Optimal Lagrangian value:
# L_λ(x) = F(x, z*(λ)) - λ G(x, z*(λ))
function L_lambda(
    λ, x;
    p01, ρ, κ, r, β,
    ε=1e-10,
    gtol=0.0,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200
)
    λf = Float64(λ)
    xf = Float64(x)

    isfinite(λf) ||
        throw(ArgumentError("λ must be finite"))

    0.0 ≤ xf ≤ 1.0 ||
        throw(DomainError(x, "x must belong to [0,1]"))

    z = zstar(
        λf;
        p01=p01,
        ρ=ρ,
        κ=κ,
        r=r,
        β=β,
        ε=ε,
        gtol=gtol,
        xatol=xatol,
        λatol=λatol,
        maxiter=maxiter
    )

    Fx = F_metric(
        xf, z;
        p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε
    )

    Gx = G_metric(
        xf, z;
        p01=p01, ρ=ρ, κ=κ, β=β, ε=ε
    )

    return Float64(Fx) - λf * Float64(Gx)
end

L_lambda (generic function with 1 method)

In [39]:
"""
Plot L(x, λ) = F(x, z*(λ)) - λ G(x, z*(λ)) versus λ by default on [0,1], for fixed x.

The x-axis uses the structural λ-ticks
0, m(p01), m(x¹), m(x⁰), m(p11), m(1),
restricted to those that fall inside [0,1].
"""
function plot_L_lambda_vs_lambda_axis(x; p01, ρ, κ, r, β, ε=1e-10,
                                      lambdagrid=collect(range(0.0, 1.0; length=2001)),
                                      xlabel=L"$\lambda$",
                                      ylabel=L"$L(x,\lambda)$",
                                      width="8cm", height="8cm",
                                      ypad=0.05)

    λgrid = Float64.(collect(lambdagrid))

    length(λgrid) ≥ 2 ||
        throw(ArgumentError("lambdagrid must contain at least two points"))

    all(isfinite, λgrid) ||
        throw(ArgumentError("lambdagrid must contain only finite values"))

    issorted(λgrid) ||
        throw(ArgumentError("lambdagrid must be nondecreasing"))

    λmin = minimum(λgrid)
    λmax = maximum(λgrid)

    λmin < λmax ||
        throw(ArgumentError("lambdagrid must span a nonzero interval"))

    # fixed points
    x0_fp, x1_fp, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

    # structural λ-values
    mp01 = mp_index(float(p01);      p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx1  = mp_index(float(x1_fp);  p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mx0  = mp_index(float(x0_fp);  p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    mp11 = mp_index(float(p01+ρ);    p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    m1   = mp_index(1.0;           p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    items = [
        (0.0,        "\\ensuremath{0}"),
        (float(mp01),"\\makebox[1.5pt][r]{\\ensuremath{m(p_{01})}\\hspace{0.5pt}}"),
        (float(mx1), "\\makebox[1.5pt][l]{\\ensuremath{m(x^{1})}\\hspace{0.5pt}}"),
        (float(mx0), "\\makebox[0.5pt][r]{\\ensuremath{m(x^{0})}\\hspace{1.5pt}}"),
        (float(mp11),"\\makebox[20.0pt][l]{\\ensuremath{m(p_{11})}\\hspace{4.0pt}}"),
        (float(m1),  "\\makebox[1.5pt][l]{\\ensuremath{m(1)}\\hspace{0.5pt}}"),
        (1.0,        "\\ensuremath{1}"),
    ]
    sort!(items, by=first)

    # keep only ticks inside the plotting λ-range
    items = [(u, lab) for (u, lab) in items if λmin - 1e-12 <= u <= λmax + 1e-12]

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = isempty(xticks) ? "{}" : "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{m(p_{11})}}"
    xticklab_str = isempty(labels) ? "{}" : "{" * join([phantom * lab for lab in labels], ",") * "}"

    # sample L(x, λ)
    Lvals = similar(λgrid, Float64)
    for (i, λ) in enumerate(λgrid)
        Lvals[i] = float(L_lambda(λ, x; p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
    end

    finitevals = [v for v in Lvals if isfinite(v)]

    isempty(finitevals) &&
        error("No finite values found for L(x, λ) on the supplied lambda grid.")

    # Include zero so that the requested y=0 tick lies inside the plotted range.
    push!(finitevals, 0.0)

    ymin = minimum(finitevals)
    ymax = maximum(finitevals)
    
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    coords = Tuple{Float64,Float64}[
        (float(λ), isfinite(L) ? float(L) : NaN) for (λ, L) in zip(λgrid, Lvals)
    ]

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => λmin,
            xmax   => λmax,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot({
            "line width" => "0.40pt",
            "black"      => nothing,
         },
         Coordinates(coords)),
        )
end

function save_L_lambda_vs_lambda_plot_pdf(outpdf::AbstractString, x;
                                          p01, ρ, κ, r, β,
                                          engine::AbstractString="pdflatex",
                                          quiet::Bool=true,
                                          keep_tex::Bool=true,
                                          kwargs...)
    ax  = plot_L_lambda_vs_lambda_axis(x; p01=p01, ρ=ρ, κ=κ, r=r, β=β, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_L_lambda_vs_lambda_plot_pdf (generic function with 1 method)

In [40]:
p01 = 0.25
ρ   = 0.6
κ   = 0.8
r   = 1.0
β   = 0.95

x0, x1, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)
x = 0.5 * (x1 + x0)

save_L_lambda_vs_lambda_plot_pdf(
    "L_lambda_vs_lambda.pdf",
    x;
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    keep_tex=false
)

"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\L_lambda_vs_lambda.pdf"

In [41]:
"""
Discounted Lagrangian-relaxation value

    ℒ(λ, x) = Σₙ Lₙ(λ, xₙ) + Mλ/(1-β).
"""
function Lvector(
    λ, xvector;
    qvector, ρvector, κvector, rvector,
    β, M,
    ε=1e-10
)
    λf = Float64(λ)
    βf = Float64(β)
    Mf = Float64(M)

    isfinite(λf) ||
        throw(ArgumentError("λ must be finite"))

    isfinite(βf) && 0.0 ≤ βf < 1.0 ||
        throw(ArgumentError("β must satisfy 0 ≤ β < 1"))

    xvec = Float64.(collect(xvector))
    qvec = Float64.(collect(qvector))
    ρvec = Float64.(collect(ρvector))
    κvec = Float64.(collect(κvector))
    rvec = Float64.(collect(rvector))

    N = length(xvec)

    N ≥ 1 ||
        throw(ArgumentError("xvector must be nonempty"))

    length(qvec) == N ||
        throw(ArgumentError("qvector must have length $N"))
    length(ρvec) == N ||
        throw(ArgumentError("ρvector must have length $N"))
    length(κvec) == N ||
        throw(ArgumentError("κvector must have length $N"))
    length(rvec) == N ||
        throw(ArgumentError("rvector must have length $N"))

    all(isfinite, xvec) ||
        throw(ArgumentError("xvector must contain only finite values"))

    all(x -> 0.0 ≤ x ≤ 1.0, xvec) ||
        throw(ArgumentError("every initial belief must belong to [0,1]"))

    isfinite(Mf) && 0.0 ≤ Mf ≤ N ||
        throw(ArgumentError("M must satisfy 0 ≤ M ≤ N"))

    total = 0.0
    @inbounds for n in 1:N
        total += L_lambda(
            λf, xvec[n];
            p01=qvec[n],
            ρ=ρvec[n],
            κ=κvec[n],
            r=rvec[n],
            β=βf,
            ε=ε
        )
    end

    return total + Mf * λf / (1.0 - βf)
end

Lvector

In [42]:
# Example with three projects
qvector = [0.25, 0.10, 0.30]
ρvector = [0.60, 0.80, 0.65]
κvector = [0.80, 0.60, 0.70]
rvector = [1.00, 0.80, 1.20]

β       = 0.95
M       = 2
xvector = [0.50, 0.50, 0.50]

λ = 0.80

L_total = Lvector(
    λ,
    xvector;
    qvector=qvector,
    ρvector=ρvector,
    κvector=κvector,
    rvector=rvector,
    β=β,
    M=M,
    ε=1e-10
)

println("ℒ($λ, xvector) = ", L_total)

ℒ(0.8, xvector) = 31.999999999999975


In [43]:
function plotLvector(xvector; qvector, ρvector, κvector, rvector, β, M, ε=1e-10,
                     lambdagrid=collect(range(0.0, 1.0; length=101)),
                     xlabel=L"$\lambda$",
                     ylabel=L"$\mathcal{L}(\mathbf{x};\lambda)$",
                     width="8cm", height="8cm",
                     ypad=0.05)

    λgrid = Float64.(collect(lambdagrid))

    length(λgrid) ≥ 2 ||
        throw(ArgumentError("lambdagrid must contain at least two points"))

    all(isfinite, λgrid) ||
        throw(ArgumentError("lambdagrid must contain only finite values"))

    issorted(λgrid) ||
        throw(ArgumentError("lambdagrid must be nondecreasing"))

    λmin = minimum(λgrid)
    λmax = maximum(λgrid)

    λmin < λmax ||
        throw(ArgumentError("lambdagrid must span a nonzero interval"))

    # sample Lvector(λ)
    Lvals = similar(λgrid, Float64)
    for (i, λ) in enumerate(λgrid)
        Lvals[i] = float(Lvector(λ, xvector;
                                 qvector=qvector, ρvector=ρvector,
                                 κvector=κvector, rvector=rvector,
                                 β=β, M=M, ε=ε))
    end

    finitevals = [v for v in Lvals if isfinite(v)]

    isempty(finitevals) &&
        error("No finite values found for Lvector on the supplied lambda grid.")

    # Include zero so that the requested y=0 tick lies inside the plot.
    push!(finitevals, 0.0)

    ymin = minimum(finitevals)
    ymax = maximum(finitevals)
    
    if ymax > ymin
        pad = ypad * (ymax - ymin)
        ymin -= pad
        ymax += pad
    else
        ymin -= 1
        ymax += 1
    end

    coords = Tuple{Float64,Float64}[
        (float(λ), isfinite(L) ? float(L) : NaN) for (λ, L) in zip(λgrid, Lvals)
    ]

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => λmin,
            xmax   => λmax,
            ymin   => ymin,
            ymax   => ymax,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            # only ticks 0 and 1 on the lambda axis
            "xtick" => "{0,1}",
            "xticklabels" => "{0,1}",

            "ytick" => "{0}",
            "enlargelimits" => "false",

            "scaled x ticks" => "false",
            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot({
            "line width" => "0.40pt",
            "black"      => nothing,
         },
         Coordinates(coords)),
    )
end

function save_Lvector_plot_pdf(outpdf::AbstractString, xvector;
                               qvector, ρvector, κvector, rvector, β, M,
                               engine::AbstractString="pdflatex",
                               quiet::Bool=true,
                               keep_tex::Bool=true,
                               kwargs...)
    ax  = plotLvector(xvector;
                      qvector=qvector, ρvector=ρvector,
                      κvector=κvector, rvector=rvector,
                      β=β, M=M, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

save_Lvector_plot_pdf (generic function with 1 method)

In [44]:
# Example with three projects
qvector = [0.25, 0.10, 0.30]
ρvector = [0.60, 0.80, 0.65]
κvector = [0.80, 0.60, 0.70]
rvector = [1.00, 0.80, 1.20]

β       = 0.95
M       = 2
xvector = [0.50, 0.50, 0.50]

save_Lvector_plot_pdf(
    "Lvector_vs_lambda.pdf",
    xvector;
    qvector=qvector,
    ρvector=ρvector,
    κvector=κvector,
    rvector=rvector,
    β=β,
    M=M,
    keep_tex=false, lambdagrid=range(0.0, 1.0; length=501),
)

"C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\Lvector_vs_lambda.pdf"

In [45]:
"""
    lambda_star_vector(xvector; qvector, ρvector, κvector, rvector, β, M,
                       ε=1e-10, λlo=0.0, λhi=1.0, tol=1e-8, maxiter=200)

Approximately minimize the convex function

    Lvector(λ, xvector; qvector=qvector, ρvector=ρvector,
            κvector=κvector, rvector=rvector, β=β, M=M, ε=ε)

over λ in [λlo, λhi], and return `(λstar, Lstar)`.

The default search interval is [0,1], but a larger `λhi` may be needed
when the relevant project indices exceed one.

Method:
- golden-section search on [λlo, λhi];
- final comparison among the interior candidates and both endpoints.
"""
function lambda_star_vector(
    xvector;
    qvector, ρvector, κvector, rvector,
    β, M,
    ε=1e-10,
    λlo=0.0,
    λhi=1.0,
    tol=1e-8,
    maxiter=200
)
    λlo_f = Float64(λlo)
    λhi_f = Float64(λhi)
    tol_f = Float64(tol)

    all(isfinite, (λlo_f, λhi_f)) ||
        throw(ArgumentError("λlo and λhi must be finite"))

    0.0 ≤ λlo_f ≤ λhi_f ||
        throw(ArgumentError("Require 0 ≤ λlo ≤ λhi"))

    isfinite(tol_f) && tol_f > 0.0 ||
        throw(ArgumentError("tol must be finite and strictly positive"))

    maxiter isa Integer && maxiter ≥ 1 ||
        throw(ArgumentError("maxiter must be a positive integer"))

    function f(λ)
        value = Float64(Lvector(
            λ,
            xvector;
            qvector=qvector,
            ρvector=ρvector,
            κvector=κvector,
            rvector=rvector,
            β=β,
            M=M,
            ε=ε
        ))

        isfinite(value) ||
            throw(DomainError(λ, "Lvector returned a nonfinite value"))

        return value
    end

    if λlo_f == λhi_f
        value = f(λlo_f)
        return λlo_f, value
    end

    # Golden-section search
    ϕ = (sqrt(5.0) - 1.0) / 2.0

    a = λlo_f
    b = λhi_f
    c = b - ϕ * (b - a)
    d = a + ϕ * (b - a)

    fc = f(c)
    fd = f(d)

    for _ in 1:maxiter
        (b - a) ≤ tol_f && break

        if fc ≤ fd
            b  = d
            d  = c
            fd = fc
            c  = b - ϕ * (b - a)
            fc = f(c)
        else
            a  = c
            c  = d
            fc = fd
            d  = a + ϕ * (b - a)
            fd = f(d)
        end
    end

    λmid = 0.5 * (a + b)
    fmid = f(λmid)

    candidates = [λlo_f, λhi_f, c, d, λmid]
    values     = [f(λlo_f), f(λhi_f), fc, fd, fmid]

    i = argmin(values)
    return candidates[i], values[i]
end

lambda_star_vector

In [46]:
# Example with three projects
qvector = [0.25, 0.10, 0.30]
ρvector = [0.60, 0.80, 0.65]
κvector = [0.80, 0.60, 0.70]
rvector = [1.00, 0.80, 1.20]

β       = 0.95
M       = 2
xvector = [0.50, 0.50, 0.50]

λstar, Lstar = lambda_star_vector(
    xvector;
    qvector=qvector,
    ρvector=ρvector,
    κvector=κvector,
    rvector=rvector,
    β=β,
    M=M
)

println("λ* = ", λstar)
println("ℒ(λ*, xvector) = ", Lstar)

λ* = 0.3360349015344217
ℒ(λ*, xvector) = 23.626427419321224


In [47]:
using Printf

# ----------------------------------------------------------
# Helper: extract scalar value if metric returns a tuple
# ----------------------------------------------------------
val(u) = u isa Tuple ? u[1] : u

# ----------------------------------------------------------
# Compressed representation of repeated project types
# ----------------------------------------------------------
struct CompressedTypeGroup
    p01::Float64
    ρ::Float64
    κ::Float64
    r::Float64
    xs::Vector{Float64}   # distinct initial beliefs for this type
    mult::Vector{Int}     # multiplicities of those beliefs
end

"""
    compress_project_vectors(xvector; qvector, ρvector, κvector, rvector)

Compress projects by identical parameter tuples (p01,ρ,κ,r), and within each type
compress repeated x-values by multiplicity.

This lets us:
- compute zstar(λ) once per type per λ
- evaluate F/G once per distinct x-value per type, multiplying by multiplicity
"""
function compress_project_vectors(xvector;
                                  qvector,
                                  ρvector,
                                  κvector,
                                  rvector)

    N = length(xvector)

    N ≥ 1 ||
        throw(ArgumentError("xvector must be nonempty"))

    length(qvector) == N ||
        throw(ArgumentError("qvector must have length $N"))
    length(ρvector) == N ||
        throw(ArgumentError("ρvector must have length $N"))
    length(κvector) == N ||
        throw(ArgumentError("κvector must have length $N"))
    length(rvector) == N ||
        throw(ArgumentError("rvector must have length $N"))

    groups_dict = Dict{NTuple{4,Float64}, Dict{Float64,Int}}()

    for i in 1:N
        key = (
            Float64(qvector[i]),
            Float64(ρvector[i]),
            Float64(κvector[i]),
            Float64(rvector[i])
        )
        x = Float64(xvector[i])

        xdict = get!(groups_dict, key, Dict{Float64,Int}())
        xdict[x] = get(xdict, x, 0) + 1
    end

    groups = CompressedTypeGroup[]
    for (key, xdict) in groups_dict
        xs = collect(keys(xdict))
        sort!(xs)
        mult = [xdict[x] for x in xs]
        push!(groups, CompressedTypeGroup(key[1], key[2], key[3], key[4], xs, mult))
    end

    sort!(groups; by = g -> (g.p01, g.ρ, g.κ, g.r))
    return groups
end

# ----------------------------------------------------------
# Evaluate Lvector(λ) and one valid subgradient value
# ----------------------------------------------------------
"""
    eval_Lvector_grouped(λ, groups; β, M, ε=1e-10, xatol=1e-10, λatol=1e-10, maxiter=200)

Evaluate
    L(λ) = M*λ/(1-β) + Σ_n [F_n(x_n,z*_n(λ)) - λ G_n(x_n,z*_n(λ))]
and one valid subgradient
    s(λ) = M/(1-β) - Σ_n G_n(x_n,z*_n(λ))

where z*_n(λ) is common within each compressed type group.
"""
function eval_Lvector_grouped(λ::Real,
                              groups::Vector{CompressedTypeGroup};
                              β::Float64,
                              M::Int,
                              ε::Float64 = 1e-10,
                              xatol::Float64 = 1e-10,
                              λatol::Float64 = 1e-10,
                              maxiter::Int = 200)

    λf = Float64(λ)

    sumL = 0.0
    sumG = 0.0
    zstars = Float64[]

    for g in groups
        z = zstar(λf;
                  p01 = g.p01, ρ = g.ρ, κ = g.κ, r = g.r, β = β,
                  ε = ε, xatol = xatol, λatol = λatol, maxiter = maxiter)
        push!(zstars, z)

        for (x, m) in zip(g.xs, g.mult)
            F = val(F_metric(x, z; p01 = g.p01, ρ = g.ρ, κ = g.κ, r = g.r, β = β, ε = ε))
            G = val(G_metric(x, z; p01 = g.p01, ρ = g.ρ, κ = g.κ,      β = β, ε = ε))

            sumL += m * (F - λf * G)
            sumG += m * G
        end
    end

    L = M * λf / (1 - β) + sumL
    s = M / (1 - β) - sumG

    return (L = L, subgrad = s, zstars = zstars)
end

# ----------------------------------------------------------
# Convexity-aware minimization by bisection on the subgradient
# ----------------------------------------------------------
"""
    lambda_star_vector_grouped_bisect(xvector; qvector, ρvector, κvector, rvector, β, M, ...)

Convexity-aware replacement for `lambda_star_vector`.

Method:
- compress repeated project types
- bracket λ* using the sign of a subgradient
- bisection on the monotone subgradient
- final comparison of λlo, λmid, λhi for robustness

Returns
    λstar, Lmin, info
"""
function lambda_star_vector_grouped_bisect(
    xvector;
    qvector,
    ρvector,
    κvector,
    rvector,
    β,
    M,
    λ0=0.0,
    step0=1.0,
    λtol=1e-6,
    max_expand=60,
    max_bisect_iter=80,
    ε=1e-10,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200,
    verbose=false
)
    N = length(xvector)

    # Validate before constructing the compressed representation
    N ≥ 1 ||
        throw(ArgumentError("xvector must be nonempty"))

    isfinite(β) && 0.0 ≤ β < 1.0 ||
        throw(ArgumentError("β must satisfy 0 ≤ β < 1"))

    M isa Integer && 0 ≤ M ≤ N ||
        throw(ArgumentError("M must be an integer satisfying 0 ≤ M ≤ N"))

    isfinite(λ0) && λ0 ≥ 0.0 ||
        throw(ArgumentError("λ0 must be finite and nonnegative"))

    isfinite(step0) && step0 > 0.0 ||
        throw(ArgumentError("step0 must be finite and strictly positive"))

    isfinite(λtol) && λtol > 0.0 ||
        throw(ArgumentError("λtol must be finite and strictly positive"))

    max_expand isa Integer && max_expand ≥ 1 ||
        throw(ArgumentError("max_expand must be a positive integer"))

    max_bisect_iter isa Integer && max_bisect_iter ≥ 1 ||
        throw(ArgumentError("max_bisect_iter must be a positive integer"))

    groups = compress_project_vectors(
        xvector;
        qvector=qvector,
        ρvector=ρvector,
        κvector=κvector,
        rvector=rvector
    )

    if verbose
        println("Compressed instance into ", length(groups), " type groups.")
        println("Total projects = ", N)
    end

    # Cache evaluations at λ-values used by the outer algorithm
    cache = Dict{Float64,NamedTuple}()

    function eval_cached(λ::Float64)
        get!(cache, λ) do
            eval_Lvector_grouped(
                λ,
                groups;
                β=Float64(β),
                M=Int(M),
                ε=ε,
                xatol=xatol,
                λatol=λatol,
                maxiter=maxiter
            )
        end
    end

    # Bracket the constrained minimizer over λ ≥ 0.
    fzero = eval_cached(0.0)
    szero = fzero.subgrad

    if szero ≥ 0.0
        info = (
            nit=0,
            λlo=0.0,
            λhi=0.0,
            candidates=[0.0],
            cache_size=length(cache),
            ngroups=length(groups),
            nprojects=length(xvector),
            bracket_subgrad_lo=szero,
            bracket_subgrad_hi=szero
        )

        return 0.0, fzero.L, info
    end

    λlo = 0.0
    slo = szero
    λhi = max(Float64(λ0), Float64(step0))

    found = false
    shi = NaN

    for _ in 1:max_expand
        shi = eval_cached(λhi).subgrad

        if shi ≥ 0.0
            found = true
            break
        end

        λlo = λhi
        slo = shi
        λhi *= 2.0
    end

    found ||
        error("Failed to bracket λ*: subgradient remained negative while expanding right.")

    bracket = (
        λlo=λlo,
        λhi=λhi,
        slo=slo,
        shi=shi
    )

    λlo = bracket.λlo
    λhi = bracket.λhi

    if verbose
        println("Initial bracket: [", λlo, ", ", λhi, "]")
        println("Initial subgradient signs: slo=", bracket.slo, ", shi=", bracket.shi)
    end

    # Bisection on subgradient
    nit = 0
    while (λhi - λlo > λtol) && (nit < max_bisect_iter)
        nit += 1
        λmid = (λlo + λhi) / 2
        smid = eval_cached(λmid).subgrad

        if verbose
            println(@sprintf("iter %3d | λlo = %.8g | λhi = %.8g | λmid = %.8g | s(mid) = %.8g",
                             nit, λlo, λhi, λmid, smid))
        end

        if smid <= 0.0
            λlo = λmid
        else
            λhi = λmid
        end
    end

    λmid = (λlo + λhi) / 2

    # Final robust comparison of candidates
    candidates = unique(sort([λlo, λmid, λhi]))
    vals = [eval_cached(λ).L for λ in candidates]
    jstar = argmin(vals)

    λstar = candidates[jstar]
    Lmin  = vals[jstar]

    info = (
        nit = nit,
        λlo = λlo,
        λhi = λhi,
        candidates = candidates,
        cache_size = length(cache),
        ngroups = length(groups),
        nprojects = length(xvector),
        bracket_subgrad_lo = eval_cached(λlo).subgrad,
        bracket_subgrad_hi = eval_cached(λhi).subgrad
    )

    return λstar, Lmin, info
end

lambda_star_vector_grouped_bisect

## New benchmark indices: Lagrangian, forced-capacity Lagrangian, and Liu--Weber--Zhang

The next cells implement and diagnose the three benchmark policies requested in the revision. The standard and forced-capacity Lagrangian policies use the same dual price and one-step advantage; only their capacity-selection rules differ. The Liu--Weber--Zhang benchmark uses the four-iteration version of their Algorithm 1. When its affine indifference equation is unsolvable, we use $r_n\kappa_n x_n$, the type-specific immediate expected reward, as the fallback index. Liu et al. write $r_nx_n$, which is ranking-equivalent in their setting because their observation-error probability is common across projects, but not in our heterogeneous-$\kappa_n$ setting.

In [48]:
# The companion file must be in the same directory as this notebook.
if !isdefined(Main, :SensingErrorBenchmarkIndices)
    include("benchmark_indices_rkappa_v4.jl")
end
using .SensingErrorBenchmarkIndices

# Connect the new benchmark code to the PCL routines defined above.
pcl_callbacks = notebook_callbacks(
    zstar_fn=zstar,
    F_metric_fn=F_metric,
    G_metric_fn=G_metric,
    f_metric_fn=f_metric2,
    g_metric_fn=g_metric2,
    epsilon=1e-10,
    xatol=1e-10,
    lambda_atol=1e-10,
    maxiter=200
)

benchmark_index_selfcheck()

(passed = true, n_lwz_points = 101, n_lwz_fallbacks = 0, max_indifference_residual = 1.1102230246251565e-16)

In [49]:
function build_representative_benchmark_diagnostic(;
    beta=0.99,
    N=200,
    capacity_ratio=0.10,
    prop_A=0.50,
    x_initial=0.50,
    nx=4001
)
    # Representative paper scenario: A_very_selfhealing_B_fragile.
    types = [
        BSFProjectType(
            p01=0.01, rho=0.90, kappa=0.70, r=1.0, beta=beta
        ),
        BSFProjectType(
            p01=0.10, rho=0.10, kappa=0.95, r=1.0, beta=beta
        )
    ]

    labels = ["type A", "type B"]
    n_A = round(Int, prop_A * N)
    n_B = N - n_A
    type_of = vcat(fill(1, n_A), fill(2, n_B))
    initial_beliefs = fill(Float64(x_initial), N)
    M = round(Int, capacity_ratio * N)

    plan = solve_lagrangian_plan(
        initial_beliefs,
        type_of,
        types,
        M,
        pcl_callbacks;
        lambda_tol=1e-8,
        subgradient_tol=1e-10,
        max_bisect_iter=100,
        value_is_certified=false
    )

    lwz_tables = build_lwz_tables(
        types;
        nx=nx,
        iterations=4,
        fallback=:type_myopic,
        labels=labels
    )

    lagrangian_tables = build_lagrangian_tables(
        plan,
        pcl_callbacks;
        nx=nx,
        labels=labels
    )

    reports = vcat(
        monotonicity_reports(lwz_tables; atol=1e-10, rtol=1e-10),
        monotonicity_reports(lagrangian_tables; atol=1e-10, rtol=1e-10)
    )

    return (
        types=types,
        labels=labels,
        type_of=type_of,
        initial_beliefs=initial_beliefs,
        M=M,
        plan=plan,
        lwz_tables=lwz_tables,
        lagrangian_tables=lagrangian_tables,
        reports=reports
    )
end

benchmark_diagnostic = build_representative_benchmark_diagnostic()

@show benchmark_diagnostic.plan.lambda_star
@show benchmark_diagnostic.plan.zstars
@show benchmark_diagnostic.plan.normalized_value

diagnostic_report = DataFrame(benchmark_diagnostic.reports)
select!(
    diagnostic_report,
    Not(:violation_indices)
)
diagnostic_report

benchmark_diagnostic.plan.lambda_star = 0.1899999992921947
benchmark_diagnostic.plan.zstars = [0.08031036041211337, 0.19999999913852662]
benchmark_diagnostic.plan.normalized_value = 0.03321175741991447


Row,name,type_id,label,ngrid,min_difference,min_secant_slope,n_violations,worst_x_left,worst_x_right,worst_value_left,worst_value_right,fallback_count
,Symbol,Int64,String,Int64,Float64,Float64,Int64,Float64,Float64,Float64,Float64,Int64
1,lwz,1,type A,4001,-0.942423,-3769.69,81,0.02525,0.0255,0.036551,-0.905872,0
2,lwz,2,type B,4001,0.000192872,0.771486,0,0.19975,0.2,0.189807,0.19,0
3,lagrangian,1,type A,4001,9.34973e-5,0.373989,0,0.09375,0.094,0.00541134,0.00550484,0
4,lagrangian,2,type B,4001,0.000237499,0.949997,0,0.99975,1.0,0.759763,0.76,0


In [50]:
# Diagnose the published four-iteration closure at the worst LWZ type-A step.
lwz_A_row = only(filter(
    row -> row.name == :lwz && row.type_id == 1,
    benchmark_diagnostic.reports
))

lwz_probe_x = lwz_A_row.worst_x_right
lwz_A_orbit = lwz_orbit_diagnostic(
    lwz_probe_x,
    benchmark_diagnostic.types[1];
    iterations=4,
    orbit_steps=80,
    max_period=16,
    cycle_atol=1e-10
)

@show lwz_probe_x

DataFrame(
    path=["reset", "no ACK", "passive"],
    closure_gap=[
        lwz_A_orbit.reset.published_closure_gap,
        lwz_A_orbit.no_ack.published_closure_gap,
        lwz_A_orbit.passive.published_closure_gap
    ],
    detected_period=[
        lwz_A_orbit.reset.detected_period,
        lwz_A_orbit.no_ack.detected_period,
        lwz_A_orbit.passive.detected_period
    ],
    cycle_values=[
        join(string.(lwz_A_orbit.reset.cycle_values), ", "),
        join(string.(lwz_A_orbit.no_ack.cycle_values), ", "),
        join(string.(lwz_A_orbit.passive.cycle_values), ", ")
    ],
    crossing_cycle=[
        join(string.(lwz_A_orbit.reset.crossing_cycle), ", "),
        join(string.(lwz_A_orbit.no_ack.crossing_cycle), ", "),
        join(string.(lwz_A_orbit.passive.crossing_cycle), ", ")
    ]
)

lwz_probe_x = 0.0255


Row,path,closure_gap,detected_period,cycle_values,crossing_cycle
,String,Float64,Int64,String,String
1,reset,0.028175,3,"0.01907264463228415, 0.0174768303019739, 0.017074280405047985","2, 1, 1"
2,no ACK,0.00159565,3,"0.017074280405047985, 0.01907264463228415, 0.0174768303019739","1, 2, 1"
3,passive,0.00159618,3,"0.017074280405047985, 0.01907264463228415, 0.0174768303019739","1, 2, 1"


In [51]:
benchmark_plots = plot_lwz_lagrangian_indices(
    benchmark_diagnostic.lwz_tables,
    benchmark_diagnostic.lagrangian_tables;
    atol=1e-10,
    rtol=1e-10
)

# Type A is black solid and type B is black dashed. Black triangles,
# if any, mark sampled downward steps. Save both plots as named PDFs.
lwz_pdf = "representative_lwz4_approximate_index.pdf"
lagrangian_pdf = "representative_lagrangian_priority.pdf"

PGFPlotsX.pgfsave(lwz_pdf, benchmark_plots.liu)
PGFPlotsX.pgfsave(lagrangian_pdf, benchmark_plots.lagrangian)

@show abspath(lwz_pdf)
@show abspath(lagrangian_pdf)

display(benchmark_plots.liu)
display(benchmark_plots.lagrangian)

LoadError: InterruptException:

In [52]:
@show isdefined(Main, :benchmark_diagnostic)
@show length(benchmark_diagnostic.lwz_tables)
@show length(benchmark_diagnostic.lagrangian_tables)

isdefined(Main, :benchmark_diagnostic) = true
length(benchmark_diagnostic.lwz_tables) = 2
length(benchmark_diagnostic.lagrangian_tables) = 2


2

## Seven-policy representative pilot

This block compares MP, myopic, LWZ4, standard Lagrangian, forced-capacity Lagrangian, round-robin, and random selection on the representative 50--50 two-type instance. It uses common ACK uniforms, deterministic project-label tie-breaking, fixed run-specific seeds, and reports capacity use, fallback counts, reproducibility, and timing.

Select the execution cell below and use **Run → Run All Above Selected Cell**, then run that cell. Do not run the obsolete development cells below this pilot section.


In [53]:
if !isdefined(Main, :SevenPolicyPilotRkappa)
    include("seven_policy_pilot_rkappa_v1.jl")
end
using .SevenPolicyPilotRkappa

mp_index_for_pilot = (x, p) -> mp_index(
    x;
    p01=p.p01, ρ=p.rho, κ=p.kappa, r=p.r, β=p.beta,
    ε=1e-10, gtol=0.0
)

seven_policy_pilot = run_representative_seven_policy_pilot(
    benchmark_diagnostic.types,
    benchmark_diagnostic.labels,
    benchmark_diagnostic.lwz_tables,
    benchmark_diagnostic.lagrangian_tables,
    mp_index_for_pilot;
    N=100, M=10, prop_A=0.50, x_initial=0.50,
    nx=4001, T=100, nruns=100, seed=20260802,
    threaded=true, verify_reproducibility=true
)


(policies = [:mp_index, :myopic, :lwz4, :lagrangian, :lagrangian_forced, :round_robin, :random], summary = NamedTuple[(policy = "mp_index", label = "MP index", mean = 0.018369279131928536, stderr = 0.00027395786897368936, ci_low = 0.01783233157545876, ci_high = 0.018906226688398312, nruns = 100), (policy = "myopic", label = "Myopic", mean = 0.01356146100846125, stderr = 0.00019085400009679194, ci_low = 0.013187394041966132, ci_high = 0.013935527974956367, nruns = 100), (policy = "lwz4", label = "LWZ4 approximate index", mean = 0.018204368869859953, stderr = 0.00027421619672890577, ci_low = 0.017666915000293746, ci_high = 0.01874182273942616, nruns = 100), (policy = "lagrangian", label = "Standard Lagrangian", mean = 0.016384754091900943, stderr = 0.0002661431157524943, ci_low = 0.01586312317029278, ci_high = 0.016906385013509106, nruns = 100), (policy = "lagrangian_forced", label = "Forced-capacity Lagrangian", mean = 0.01848910310464866, stderr = 0.0002775161645179708, ci_low = 0.0179

In [54]:
using Statistics
using Distributions
using DataFrames

# ------------------------------------------------------------
# 1. Paired policy-minus-MP comparisons
# ------------------------------------------------------------

mp_col = findfirst(==(:mp_index), seven_policy_pilot.policies)
zcrit = quantile(Normal(), 0.975)

paired_vs_mp = DataFrame(
    policy=String[],
    mean_difference=Float64[],
    stderr=Float64[],
    ci_low=Float64[],
    ci_high=Float64[]
)

for j in eachindex(seven_policy_pilot.policies)
    j == mp_col && continue

    differences =
        seven_policy_pilot.normalized_rewards[:, j] .-
        seven_policy_pilot.normalized_rewards[:, mp_col]

    μ = mean(differences)
    se = std(differences) / sqrt(length(differences))

    push!(
        paired_vs_mp,
        (
            policy=String(seven_policy_pilot.policies[j]),
            mean_difference=μ,
            stderr=se,
            ci_low=μ - zcrit * se,
            ci_high=μ + zcrit * se
        )
    )
end

paired_vs_mp

Row,policy,mean_difference,stderr,ci_low,ci_high
,String,Float64,Float64,Float64,Float64
1,myopic,-0.00480782,0.000330947,-0.00545646,-0.00415917
2,lwz4,-0.00016491,0.000166282,-0.000490817,0.000160997
3,lagrangian,-0.00198453,0.000335465,-0.00264202,-0.00132703
4,lagrangian_forced,0.000119824,0.000338274,-0.000543181,0.000782829
5,round_robin,-0.0111095,0.000282436,-0.0116631,-0.010556
6,random,-0.0112768,0.000291057,-0.0118473,-0.0107064


In [55]:
# ------------------------------------------------------------
# 2. Verify that MP and myopic priority tables differ internally
# ------------------------------------------------------------

mp_curve_check = DataFrame(
    type=String[],
    mp_at_half=Float64[],
    myopic_at_half=Float64[],
    max_abs_difference=Float64[],
    difference_location=Float64[],
    minimum_mp_increment=Float64[]
)

for k in eachindex(seven_policy_pilot.mp_tables)
    table = seven_policy_pilot.mp_tables[k]
    project_type = benchmark_diagnostic.types[k]

    myopic_values =
        project_type.r * project_type.kappa .* table.xgrid

    deviations = table.values .- myopic_values
    largest_index = argmax(abs.(deviations))
    half_index = argmin(abs.(table.xgrid .- 0.5))

    push!(
        mp_curve_check,
        (
            type=table.label,
            mp_at_half=table.values[half_index],
            myopic_at_half=myopic_values[half_index],
            max_abs_difference=abs(deviations[largest_index]),
            difference_location=table.xgrid[largest_index],
            minimum_mp_increment=minimum(diff(table.values))
        )
    )
end

mp_curve_check

Row,type,mp_at_half,myopic_at_half,max_abs_difference,difference_location,minimum_mp_increment
,String,Float64,Float64,Float64,Float64,Float64
1,type A,0.522908,0.35,0.24423,0.222,4.85878e-5
2,type B,0.475,0.475,0.00962507,0.11125,0.000192872


In [56]:
using Statistics
using Distributions
using DataFrames
using CSV

# Reuse the already validated priority tables.
type_of_horizon = vcat(
    fill(1, seven_policy_pilot.type_A_count),
    fill(2, seven_policy_pilot.type_B_count)
)

initial_beliefs_horizon =
    fill(seven_policy_pilot.x_initial, seven_policy_pilot.N)

# Reuse the existing T=100 result; run only the longer horizons.
horizon_results = Dict{Int,Any}(
    100 => seven_policy_pilot
)

for T_horizon in (300, 500, 700)
    horizon_results[T_horizon] = run_seven_policy_pilot(
        initial_beliefs_horizon,
        type_of_horizon,
        benchmark_diagnostic.types,
        seven_policy_pilot.M;
        mp_tables=seven_policy_pilot.mp_tables,
        lwz_tables=benchmark_diagnostic.lwz_tables,
        lagrangian_tables=benchmark_diagnostic.lagrangian_tables,
        T=T_horizon,
        nruns=100,
        seed=seven_policy_pilot.seed,
        threaded=true,
        verify_reproducibility=false
    )
end

horizon_summary = DataFrame(
    T=Int[],
    beta_tail=Float64[],
    policy=String[],
    label=String[],
    mean=Float64[],
    stderr=Float64[],
    paired_mean_vs_mp=Union{Missing,Float64}[],
    paired_ci_low=Union{Missing,Float64}[],
    paired_ci_high=Union{Missing,Float64}[]
)

for T_horizon in sort(collect(keys(horizon_results)))
    result = horizon_results[T_horizon]
    mp_column = findfirst(==(:mp_index), result.policies)
    tcrit = quantile(TDist(result.nruns - 1), 0.975)

    for j in eachindex(result.policies)
        summary_row = result.summary[j]

        if j == mp_column
            paired_mean = missing
            paired_low = missing
            paired_high = missing
        else
            differences =
                result.normalized_rewards[:, j] .-
                result.normalized_rewards[:, mp_column]

            paired_mean = mean(differences)
            paired_se = std(differences) / sqrt(length(differences))
            paired_low = paired_mean - tcrit * paired_se
            paired_high = paired_mean + tcrit * paired_se
        end

        push!(
            horizon_summary,
            (
                T=T_horizon,
                beta_tail=result.beta_tail,
                policy=String(result.policies[j]),
                label=summary_row.label,
                mean=summary_row.mean,
                stderr=summary_row.stderr,
                paired_mean_vs_mp=paired_mean,
                paired_ci_low=paired_low,
                paired_ci_high=paired_high
            )
        )
    end
end

horizon_csv = "seven_policy_horizon_sensitivity_v1.csv"
CSV.write(horizon_csv, horizon_summary)

@show abspath(horizon_csv)
horizon_summary

abspath(horizon_csv) = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\seven_policy_horizon_sensitivity_v1.csv"


Row,T,beta_tail,policy,label,mean,stderr,paired_mean_vs_mp,paired_ci_low,paired_ci_high
,Int64,Float64,String,String,Float64,Float64,Float64?,Float64?,Float64?
1,100,0.366032,mp_index,MP index,0.0183693,0.000273958,missing,missing,missing
2,100,0.366032,myopic,Myopic,0.0135615,0.000190854,-0.00480782,-0.00546449,-0.00415115
3,100,0.366032,lwz4,LWZ4 approximate index,0.0182044,0.000274216,-0.00016491,-0.00049485,0.00016503
4,100,0.366032,lagrangian,Standard Lagrangian,0.0163848,0.000266143,-0.00198453,-0.00265016,-0.00131889
5,100,0.366032,lagrangian_forced,Forced-capacity Lagrangian,0.0184891,0.000277516,0.000119824,-0.000551385,0.000791033
6,100,0.366032,round_robin,Round-robin,0.00725974,7.03938e-5,-0.0111095,-0.01167,-0.0105491
7,100,0.366032,random,Random,0.00709247,7.28525e-5,-0.0112768,-0.0118543,-0.0106993
8,300,0.0490409,mp_index,MP index,0.0253804,0.00031679,missing,missing,missing
9,300,0.0490409,myopic,Myopic,0.0172344,0.000190281,-0.00814607,-0.00885716,-0.00743497


In [57]:
using Statistics
using Distributions
using DataFrames

result700 = horizon_results[700]

# Runtime and capacity use
@show result700.elapsed_seconds
@show result700.seconds_per_replication

capacity_700 = DataFrame(result700.capacity)
display(capacity_700)

# Paired comparisons among the three leading policies
top_pairs = [
    (:mp_index, :lwz4),
    (:mp_index, :lagrangian_forced),
    (:lagrangian_forced, :lwz4)
]

tcrit700 = quantile(TDist(result700.nruns - 1), 0.975)

top3_pairwise = DataFrame(
    comparison=String[],
    mean_difference=Float64[],
    stderr=Float64[],
    ci_low=Float64[],
    ci_high=Float64[]
)

for (policy_a, policy_b) in top_pairs
    column_a = findfirst(==(policy_a), result700.policies)
    column_b = findfirst(==(policy_b), result700.policies)

    differences =
        result700.normalized_rewards[:, column_a] .-
        result700.normalized_rewards[:, column_b]

    mean_difference = mean(differences)
    standard_error = std(differences) / sqrt(length(differences))

    push!(
        top3_pairwise,
        (
            comparison="$(policy_a) minus $(policy_b)",
            mean_difference=mean_difference,
            stderr=standard_error,
            ci_low=mean_difference - tcrit700 * standard_error,
            ci_high=mean_difference + tcrit700 * standard_error
        )
    )
end

display(top3_pairwise)

result700.elapsed_seconds = 0.3878869
result700.seconds_per_replication = 0.0038788689999999996


Row,policy,label,min_active,max_active,mean_active,idle_slots,idle_slot_fraction,capacity
,String,String,Int64,Int64,Float64,Int64,Float64,Int64
1,mp_index,MP index,10,10,10.0,0,0.0,10
2,myopic,Myopic,10,10,10.0,0,0.0,10
3,lwz4,LWZ4 approximate index,10,10,10.0,0,0.0,10
4,lagrangian,Standard Lagrangian,0,10,6.63543,60512,0.864457,10
5,lagrangian_forced,Forced-capacity Lagrangian,10,10,10.0,0,0.0,10
6,round_robin,Round-robin,10,10,10.0,0,0.0,10
7,random,Random,10,10,10.0,0,0.0,10


Row,comparison,mean_difference,stderr,ci_low,ci_high
,String,Float64,Float64,Float64,Float64
1,mp_index minus lwz4,0.000120107,0.000212941,-0.000302414,0.000542628
2,mp_index minus lagrangian_forced,-8.0249e-5,0.00035348,-0.00078163,0.000621132
3,lagrangian_forced minus lwz4,0.000200356,0.000350661,-0.000495431,0.000896143


In [58]:
using Statistics
using Distributions
using DataFrames

# Recompute the dual object for the actual N=100, M=10 instance.
plan100 = solve_lagrangian_plan(
    initial_beliefs_horizon,
    type_of_horizon,
    benchmark_diagnostic.types,
    result700.M,
    pcl_callbacks;
    lambda_tol=1e-8,
    subgradient_tol=1e-10,
    max_bisect_iter=100,
    value_is_certified=false
)

plan200 = benchmark_diagnostic.plan

# Maximum possible normalized reward omitted beyond T=700.
# Each activation yields at most r_k, and there are at most M per period.
max_reward_per_activation =
    maximum(project_type.r for project_type in plan100.types)

tail_upper =
    result700.beta_tail *
    (result700.M / result700.N) *
    max_reward_per_activation

dual_check = DataFrame([(
    lambda_star_N100=plan100.lambda_star,
    lambda_star_N200=plan200.lambda_star,
    lambda_difference=
        plan100.lambda_star - plan200.lambda_star,
    threshold_dual_N100=plan100.normalized_value,
    threshold_dual_N200=plan200.normalized_value,
    normalized_value_difference=
        plan100.normalized_value - plan200.normalized_value,
    lambda_bracket_width=
        plan100.lambda_hi - plan100.lambda_lo,
    iterations=plan100.iterations,
    value_is_certified=plan100.value_is_certified,
    beta_tail=result700.beta_tail,
    normalized_tail_upper_bound=tail_upper
)])

tcrit700 = quantile(TDist(result700.nruns - 1), 0.975)
dual_value = plan100.normalized_value

dual_gap_rows = NamedTuple[]

for row in result700.summary
    policy_upper_95 = row.mean + tcrit700 * row.stderr

    # Conservative upper estimate for the infinite-horizon policy value:
    # upper endpoint for the truncated mean plus the maximum omitted tail.
    policy_upper_95_with_tail =
        policy_upper_95 + tail_upper

    push!(
        dual_gap_rows,
        (
            policy=row.policy,
            mean_T700=row.mean,
            threshold_dual=dual_value,
            point_gap=dual_value - row.mean,
            relative_point_gap=
                (dual_value - row.mean) / dual_value,
            upper_95_with_tail=policy_upper_95_with_tail,
            conservative_gap=
                dual_value - policy_upper_95_with_tail
        )
    )
end

dual_gaps_700 = DataFrame(dual_gap_rows)

display(dual_check)
display(dual_gaps_700)

Row,lambda_star_N100,lambda_star_N200,lambda_difference,threshold_dual_N100,threshold_dual_N200,normalized_value_difference,lambda_bracket_width,iterations,value_is_certified,beta_tail,normalized_tail_upper_bound
,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64,Bool,Float64,Float64
1,0.19,0.19,0.0,0.0332118,0.0332118,0.0,7.07805e-9,27,false,0.000880311,8.80311e-5


Row,policy,mean_T700,threshold_dual,point_gap,relative_point_gap,upper_95_with_tail,conservative_gap
,String,Float64,Float64,Float64,Float64,Float64,Float64
1,mp_index,0.026427,0.0332118,0.00678478,0.204289,0.0271468,0.006065
2,myopic,0.0177899,0.0332118,0.0154219,0.46435,0.0182568,0.014955
3,lwz4,0.0263069,0.0332118,0.00690489,0.207905,0.0269788,0.00623292
4,lagrangian,0.022434,0.0332118,0.0107778,0.324517,0.0231088,0.010103
5,lagrangian_forced,0.0265072,0.0332118,0.00670453,0.201872,0.0271702,0.00604154
6,round_robin,0.0104334,0.0332118,0.0227783,0.685852,0.010666,0.0225458
7,random,0.0102802,0.0332118,0.0229316,0.690466,0.0105236,0.0226882


In [59]:
using DataFrames

λ = plan100.lambda_star

bellman_threshold_check = DataFrame(
    type=String[],
    threshold=Float64[],
    x1=Float64[],
    x0=Float64[],
    regime=String[],
    m_at_threshold=Float64[],
    m_minus_lambda=Float64[],
    max_passive_advantage=Float64[],
    min_active_advantage=Float64[],
    max_bellman_violation=Float64[]
)

for (k, p) in enumerate(plan100.types)
    z = plan100.zstars[k]

    x0, x1, _ = fixed_points(
        p01=p.p01,
        ρ=p.rho,
        κ=p.kappa
    )

    regime =
        z <= x1 ? "z ≤ x¹ (analytical regime)" :
        z >= x0 ? "z ≥ x⁰ (analytical regime)" :
                  "x¹ < z < x⁰ (unresolved regime)"

    # Include the threshold itself in a dense uniform grid.
    xgrid = sort!(
        unique!(
            vcat(
                collect(range(0.0, 1.0; length=20_001)),
                [z]
            )
        )
    )

    # Q_1^{π_z}(x) - Q_0^{π_z}(x).
    advantages = [
        pcl_callbacks.f(x, z, p) -
        λ * pcl_callbacks.g(x, z, p)
        for x in xgrid
    ]

    passive_indices = findall(x -> x <= z, xgrid)
    active_indices = findall(x -> x > z, xgrid)

    max_passive =
        maximum(advantages[passive_indices])

    min_active =
        isempty(active_indices) ?
        Inf :
        minimum(advantages[active_indices])

    gzz = pcl_callbacks.g(z, z, p)
    m_at_z = pcl_callbacks.f(z, z, p) / gzz

    push!(
        bellman_threshold_check,
        (
            type="type $(k == 1 ? "A" : "B")",
            threshold=z,
            x1=x1,
            x0=x0,
            regime=regime,
            m_at_threshold=m_at_z,
            m_minus_lambda=m_at_z - λ,
            max_passive_advantage=max_passive,
            min_active_advantage=min_active,
            max_bellman_violation=max(
                0.0,
                max_passive,
                -min_active
            )
        )
    )
end

bellman_threshold_check

Row,type,threshold,x1,x0,regime,m_at_threshold,m_minus_lambda,max_passive_advantage,min_active_advantage,max_bellman_violation
,String,Float64,Float64,Float64,String,Float64,Float64,Float64,Float64,Float64
1,type A,0.0803104,0.013748,0.1,x¹ < z < x⁰ (unresolved regime),0.19,1.4235e-11,1.65602e-12,1.59689e-5,1.65602e-12
2,type B,0.2,0.100556,0.111111,z ≥ x⁰ (analytical regime),0.19,4.33466e-11,5.3389e-11,8.71789e-10,5.3389e-11


In [ ]:
pilot_summary = DataFrame(pilot_summary_rows(seven_policy_pilot))
pilot_capacity = DataFrame(pilot_capacity_rows(seven_policy_pilot))

pilot_diagnostics = DataFrame([(
    revision="2026-08-02-clean-v4-seven-policy-pilot",
    N=seven_policy_pilot.N,
    M=seven_policy_pilot.M,
    type_A_count=seven_policy_pilot.type_A_count,
    type_B_count=seven_policy_pilot.type_B_count,
    x_initial=seven_policy_pilot.x_initial,
    beta=seven_policy_pilot.beta,
    T=seven_policy_pilot.T,
    beta_tail=seven_policy_pilot.beta_tail,
    nruns=seven_policy_pilot.nruns,
    seed=seven_policy_pilot.seed,
    cpu_threads=Sys.CPU_THREADS,
    julia_threads=seven_policy_pilot.threads,
    mp_table_seconds=seven_policy_pilot.mp_table_seconds,
    simulation_seconds=seven_policy_pilot.elapsed_seconds,
    seconds_per_replication=seven_policy_pilot.seconds_per_replication,
    lambda_star=benchmark_diagnostic.plan.lambda_star,
    threshold_dual_normalized_value=
        benchmark_diagnostic.plan.normalized_value,
    value_is_certified=benchmark_diagnostic.plan.value_is_certified,
    lwz_fallback_count=seven_policy_pilot.lwz_fallback_count,
    lwz_fallback_by_type=
        join(seven_policy_pilot.lwz_fallback_by_type, ";"),
    tie_breaking_ok=seven_policy_pilot.tie_breaking_ok,
    reproducibility_ok=seven_policy_pilot.reproducibility_ok,
    all_rewards_finite=seven_policy_pilot.all_rewards_finite,
    all_priorities_finite=seven_policy_pilot.all_priorities_finite
)])

summary_csv = "seven_policy_pilot_summary_v1.csv"
capacity_csv = "seven_policy_pilot_capacity_v1.csv"
diagnostics_csv = "seven_policy_pilot_diagnostics_v1.csv"

CSV.write(summary_csv, pilot_summary)
CSV.write(capacity_csv, pilot_capacity)
CSV.write(diagnostics_csv, pilot_diagnostics)

@show abspath(summary_csv)
@show abspath(capacity_csv)
@show abspath(diagnostics_csv)

display(pilot_summary)
display(pilot_capacity)
pilot_diagnostics


In [60]:
using Statistics
using Distributions
using DataFrames
using CSV

@assert isdefined(Main, :plan100)
@assert plan100.value_is_certified == false

# ============================================================
# 1. Definitive representative simulation
# ============================================================

result700_final = run_seven_policy_pilot(
    initial_beliefs_horizon,
    type_of_horizon,
    benchmark_diagnostic.types,
    seven_policy_pilot.M;
    mp_tables=seven_policy_pilot.mp_tables,
    lwz_tables=benchmark_diagnostic.lwz_tables,
    lagrangian_tables=benchmark_diagnostic.lagrangian_tables,
    T=700,
    nruns=10_000,
    seed=20260803,
    threaded=true,
    verify_reproducibility=false
)

conditional_bound = plan100.normalized_value
lambda_star = plan100.lambda_star

tcrit = quantile(
    TDist(result700_final.nruns - 1),
    0.975
)

mp_column = findfirst(
    ==(:mp_index),
    result700_final.policies
)

# Rigorous upper bound on the omitted normalized reward after T=700.
max_reward_per_activation =
    maximum(project_type.r for project_type in plan100.types)

tail_upper =
    result700_final.beta_tail *
    (result700_final.M / result700_final.N) *
    max_reward_per_activation

# ============================================================
# 2. Main reporting table
# ============================================================

final_report = DataFrame(
    policy=String[],
    label=String[],
    mean_T700=Float64[],
    stderr=Float64[],
    ci_low=Float64[],
    ci_high=Float64[],
    paired_difference_vs_mp=Union{Missing,Float64}[],
    paired_ci_low=Union{Missing,Float64}[],
    paired_ci_high=Union{Missing,Float64}[],
    conditional_bound=Float64[],
    point_gap_to_bound=Float64[],
    relative_point_gap_to_bound=Float64[],
    upper_95_with_tail=Float64[],
    conservative_gap_to_bound=Float64[]
)

for j in eachindex(result700_final.policies)
    row = result700_final.summary[j]

    if j == mp_column
        paired_mean = missing
        paired_low = missing
        paired_high = missing
    else
        differences =
            result700_final.normalized_rewards[:, j] .-
            result700_final.normalized_rewards[:, mp_column]

        paired_mean = mean(differences)
        paired_se = std(differences) / sqrt(length(differences))
        paired_low = paired_mean - tcrit * paired_se
        paired_high = paired_mean + tcrit * paired_se
    end

    upper_95_with_tail =
        row.mean + tcrit * row.stderr + tail_upper

    push!(
        final_report,
        (
            policy=String(result700_final.policies[j]),
            label=row.label,
            mean_T700=row.mean,
            stderr=row.stderr,
            ci_low=row.mean - tcrit * row.stderr,
            ci_high=row.mean + tcrit * row.stderr,
            paired_difference_vs_mp=paired_mean,
            paired_ci_low=paired_low,
            paired_ci_high=paired_high,
            conditional_bound=conditional_bound,
            point_gap_to_bound=conditional_bound - row.mean,
            relative_point_gap_to_bound=
                (conditional_bound - row.mean) / conditional_bound,
            upper_95_with_tail=upper_95_with_tail,
            conservative_gap_to_bound=
                conditional_bound - upper_95_with_tail
        )
    )
end

# ============================================================
# 3. Pairwise comparisons among the leading policies
# ============================================================

top_pairs = [
    (:mp_index, :lwz4),
    (:mp_index, :lagrangian_forced),
    (:lagrangian_forced, :lwz4)
]

final_top3_pairwise = DataFrame(
    comparison=String[],
    mean_difference=Float64[],
    stderr=Float64[],
    ci_low=Float64[],
    ci_high=Float64[]
)

for (policy_a, policy_b) in top_pairs
    column_a = findfirst(==(policy_a), result700_final.policies)
    column_b = findfirst(==(policy_b), result700_final.policies)

    differences =
        result700_final.normalized_rewards[:, column_a] .-
        result700_final.normalized_rewards[:, column_b]

    μ = mean(differences)
    se = std(differences) / sqrt(length(differences))

    push!(
        final_top3_pairwise,
        (
            comparison="$(policy_a) minus $(policy_b)",
            mean_difference=μ,
            stderr=se,
            ci_low=μ - tcrit * se,
            ci_high=μ + tcrit * se
        )
    )
end

# ============================================================
# 4. Capacity and replication-level data
# ============================================================

final_capacity = DataFrame(result700_final.capacity)

final_rewards = DataFrame(
    replication=collect(1:result700_final.nruns)
)

for (j, policy) in enumerate(result700_final.policies)
    final_rewards[!, policy] =
        result700_final.normalized_rewards[:, j]
end

final_metadata = DataFrame([(
    N=result700_final.N,
    M=result700_final.M,
    T=result700_final.T,
    nruns=result700_final.nruns,
    beta=result700_final.beta,
    beta_tail=result700_final.beta_tail,
    seed=result700_final.seed,
    threaded=result700_final.threaded,
    threads=result700_final.threads,
    elapsed_seconds=result700_final.elapsed_seconds,
    seconds_per_replication=result700_final.seconds_per_replication,
    lambda_star=lambda_star,
    conditional_lagrangian_bound=conditional_bound,
    normalized_tail_upper_bound=tail_upper,
    bound_status="conditional on global PCL-indexability"
)])

# ============================================================
# 5. Save everything needed for later analysis
# ============================================================

CSV.write(
    "representative_T700_n10000_report_v1.csv",
    final_report
)

CSV.write(
    "representative_T700_n10000_top3_pairwise_v1.csv",
    final_top3_pairwise
)

CSV.write(
    "representative_T700_n10000_capacity_v1.csv",
    final_capacity
)

CSV.write(
    "representative_T700_n10000_rewards_v1.csv",
    final_rewards
)

CSV.write(
    "representative_T700_n10000_metadata_v1.csv",
    final_metadata
)

display(final_metadata)
display(final_report)
display(final_top3_pairwise)
display(final_capacity)

Row,N,M,T,nruns,beta,beta_tail,seed,threaded,threads,elapsed_seconds,seconds_per_replication,lambda_star,conditional_lagrangian_bound,normalized_tail_upper_bound,bound_status
,Int64,Int64,Int64,Int64,Float64,Float64,Int64,Bool,Int64,Float64,Float64,Float64,Float64,Float64,String
1,100,10,700,10000,0.99,0.000880311,20260803,true,12,39.8694,0.00398694,0.19,0.0332118,8.80311e-5,conditional on global PCL-indexability


Row,policy,label,mean_T700,stderr,ci_low,ci_high,paired_difference_vs_mp,paired_ci_low,paired_ci_high,conditional_bound,point_gap_to_bound,relative_point_gap_to_bound,upper_95_with_tail,conservative_gap_to_bound
,String,String,Float64,Float64,Float64,Float64,Float64?,Float64?,Float64?,Float64,Float64,Float64,Float64,Float64
1,mp_index,MP index,0.0263254,2.91173e-5,0.0262683,0.0263824,missing,missing,missing,0.0332118,0.00688639,0.207348,0.0264705,0.00674129
2,myopic,Myopic,0.0175314,2.04163e-5,0.0174914,0.0175715,-0.00879393,-0.00885801,-0.00872986,0.0332118,0.0156803,0.472132,0.0176595,0.0155523
3,lwz4,LWZ4 approximate index,0.0262858,2.92382e-5,0.0262285,0.0263431,-3.95831e-5,-8.43102e-5,5.14393e-6,0.0332118,0.00692598,0.20854,0.0264311,0.00678063
4,lagrangian,Standard Lagrangian,0.0221531,3.18035e-5,0.0220908,0.0222155,-0.00417225,-0.00424783,-0.00409667,0.0332118,0.0110586,0.332974,0.0223035,0.0109083
5,lagrangian_forced,Forced-capacity Lagrangian,0.0262443,2.93434e-5,0.0261868,0.0263018,-8.10813e-5,-0.000152258,-9.90425e-6,0.0332118,0.00696747,0.209789,0.0263898,0.00682192
6,round_robin,Round-robin,0.0103703,8.1222e-6,0.0103543,0.0103862,-0.0159551,-0.0160127,-0.0158975,0.0332118,0.0228415,0.687753,0.0104742,0.0227375
7,random,Random,0.0102599,8.80219e-6,0.0102426,0.0102771,-0.0160655,-0.0161242,-0.0160068,0.0332118,0.0229519,0.691077,0.0103652,0.0228466


Row,comparison,mean_difference,stderr,ci_low,ci_high
,String,Float64,Float64,Float64,Float64
1,mp_index minus lwz4,3.95831e-5,2.28176e-5,-5.14393e-6,8.43102e-5
2,mp_index minus lagrangian_forced,8.10813e-5,3.63111e-5,9.90425e-6,0.000152258
3,lagrangian_forced minus lwz4,-4.14982e-5,3.61445e-5,-0.000112349,2.93524e-5


Row,policy,label,min_active,max_active,mean_active,idle_slots,idle_slot_fraction,capacity
,String,String,Int64,Int64,Float64,Int64,Float64,Int64
1,mp_index,MP index,10,10,10.0,0,0.0,10
2,myopic,Myopic,10,10,10.0,0,0.0,10
3,lwz4,LWZ4 approximate index,10,10,10.0,0,0.0,10
4,lagrangian,Standard Lagrangian,0,10,6.61077,6071949,0.867421,10
5,lagrangian_forced,Forced-capacity Lagrangian,10,10,10.0,0,0.0,10
6,round_robin,Round-robin,10,10,10.0,0,0.0,10
7,random,Random,10,10,10.0,0,0.0,10


In [61]:
using DataFrames
using CSV

scaling_rows = NamedTuple[]

for N_test in (100, 200, 400, 800, 1600)
    M_test = round(Int, 0.10 * N_test)
    nA_test = N_test ÷ 2

    type_of_test = vcat(
        fill(1, nA_test),
        fill(2, N_test - nA_test)
    )

    initial_beliefs_test = fill(0.5, N_test)

    result_test = run_seven_policy_pilot(
        initial_beliefs_test,
        type_of_test,
        benchmark_diagnostic.types,
        M_test;
        mp_tables=seven_policy_pilot.mp_tables,
        lwz_tables=benchmark_diagnostic.lwz_tables,
        lagrangian_tables=benchmark_diagnostic.lagrangian_tables,
        T=700,
        nruns=100,
        seed=20260804,
        threaded=true,
        verify_reproducibility=false
    )

    forced_capacity_ok = all(
        row.policy == "lagrangian" ||
        (row.min_active == M_test && row.max_active == M_test)
        for row in result_test.capacity
    )

    push!(
        scaling_rows,
        (
            N=N_test,
            M=M_test,
            elapsed_seconds=result_test.elapsed_seconds,
            seconds_per_replication=result_test.seconds_per_replication,
            estimated_seconds_1000=
                1000 * result_test.seconds_per_replication,
            all_rewards_finite=result_test.all_rewards_finite,
            all_priorities_finite=result_test.all_priorities_finite,
            forced_capacity_ok=forced_capacity_ok
        )
    )
end

scaling_timing = DataFrame(scaling_rows)

CSV.write(
    "seven_policy_scaling_timing_T700_n100_v1.csv",
    scaling_timing
)

display(scaling_timing)

Row,N,M,elapsed_seconds,seconds_per_replication,estimated_seconds_1000,all_rewards_finite,all_priorities_finite,forced_capacity_ok
,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool
1,100,10,0.431451,0.00431451,4.31451,true,true,true
2,200,20,0.805308,0.00805308,8.05308,true,true,true
3,400,40,1.84438,0.0184438,18.4438,true,true,true
4,800,80,3.88321,0.0388321,38.8321,true,true,true
5,1600,160,7.82521,0.0782521,78.2521,true,true,true


In [62]:
include("five_policy_runner_rkappa_v1.jl")
using .FivePolicyRunnerRkappa

five_policy_equivalence = run_five_policy_pilot(
    initial_beliefs_horizon,
    type_of_horizon,
    benchmark_diagnostic.types,
    result700.M;
    mp_tables=seven_policy_pilot.mp_tables,
    lwz_tables=benchmark_diagnostic.lwz_tables,
    lagrangian_tables=benchmark_diagnostic.lagrangian_tables,
    T=result700.T,
    nruns=result700.nruns,
    seed=result700.seed,
    threaded=true,
    verify_reproducibility=true
)

five_policy_equivalence_check =
    verify_against_seven_policy_result(
        five_policy_equivalence,
        result700
    )

display(DataFrame([five_policy_equivalence_check]))
@show five_policy_equivalence.elapsed_seconds

Row,policy_order_ok,rewards_exact,active_totals_exact,idle_slots_exact,maximum_absolute_reward_difference,all_exact
,Bool,Bool,Bool,Bool,Float64,Bool
1,true,true,true,true,0.0,true


five_policy_equivalence.elapsed_seconds = 0.4168978


0.4168978

In [65]:
include("four_policy_runner_rkappa_v1.jl")
using .FourPolicyRunnerRkappa

four_policy_equivalence = run_four_policy_pilot(
    initial_beliefs_horizon,
    type_of_horizon,
    benchmark_diagnostic.types,
    result700.M;
    mp_tables=seven_policy_pilot.mp_tables,
    lwz_tables=benchmark_diagnostic.lwz_tables,
    lagrangian_tables=benchmark_diagnostic.lagrangian_tables,
    T=result700.T,
    nruns=result700.nruns,
    seed=result700.seed,
    threaded=true,
    verify_reproducibility=true
)

four_policy_equivalence_check =
    verify_against_prior_policy_result(
        four_policy_equivalence,
        result700
    )

display(DataFrame([four_policy_equivalence_check]))
@show four_policy_equivalence.elapsed_seconds

Row,policy_order_ok,rewards_exact,active_totals_exact,idle_slots_exact,maximum_absolute_reward_difference,all_exact
,Bool,Bool,Bool,Bool,Float64,Bool
1,true,true,true,true,0.0,true


four_policy_equivalence.elapsed_seconds = 0.3211747


0.3211747

In [66]:
include("four_policy_full_grid_checkpointed_v1.jl")
using .FourPolicyFullGridRkappa

display(four_policy_grid_manifest())

grid_s01_preflight = run_four_policy_grid_scenario(
    1;
    pcl_callbacks=pcl_callbacks,
    mp_index_fn=mp_index_for_pilot,
    N_values=[200],
    T=700,
    nruns=1_000,
    max_new_cases=1,
    threaded=true
)

Row,scenario_index,scenario,p01_type_A,rho_type_A,kappa_type_A,reward_type_A,p01_type_B,rho_type_B,kappa_type_B,reward_type_B,beta,cases_per_scenario
,Int64,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Int64
1,1,A_very_selfhealing_B_fragile,0.01,0.9,0.7,1.0,0.1,0.1,0.95,1.0,0.99,360
2,2,A_mild_selfhealing_B_fragile,0.03,0.8,0.7,1.0,0.1,0.1,0.95,1.0,0.99,360
3,3,A_balanced_B_fragile,0.05,0.7,0.7,1.0,0.1,0.1,0.95,1.0,0.99,360
4,4,A_harder_obs_B_fragile,0.02,0.85,0.55,1.0,0.1,0.1,0.95,1.0,0.99,360
5,5,A_very_selfhealing_B_very_fragile,0.01,0.9,0.7,1.0,0.15,0.05,0.95,1.0,0.99,360
6,6,A_mild_selfhealing_B_very_fragile,0.03,0.8,0.7,1.0,0.15,0.05,0.95,1.0,0.99,360
7,7,A_balanced_B_very_fragile,0.05,0.7,0.7,1.0,0.15,0.05,0.95,1.0,0.99,360
8,8,A_harder_obs_B_very_fragile,0.02,0.85,0.55,1.0,0.15,0.05,0.95,1.0,0.99,360
9,9,A_very_selfhealing_B_persistent_bad,0.01,0.9,0.7,1.0,0.08,0.2,0.95,1.0,0.99,360


Imported 72 validated cases from five_policy_grid_s01_N100_T700_n1000_v1.csv into four_policy_grid_s01_T700_n1000_v1.csv.
[73/360; requested 1/72] scenario=1, p_A=0.1, alpha=0.05, N=200, M=10, setup=0.53 s, simulation=5.47 s, elapsed=0.2 min
abspath(output_path) = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\four_policy_grid_s01_T700_n1000_v1.csv"


Row,scenario_index,completed_instances_full_scenario,expected_instances_full_scenario,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,total_plan_table_seconds,total_simulation_seconds,wall_minutes_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,total_lwz_fallback_count,minimum_conservative_gap_to_conditional_bound
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool,Int64,Float64
1,1,73,360,1,72,292,1,44.933,299.057,0.164467,true,true,true,0,0.00095437


(grid = 292×72 DataFrame
 Row │ case_id  scenario_index  scenario                      prop_index  p_A  ⋯
     │ Int64    Int64           String                        Int64       Floa ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │       1               1  A_very_selfhealing_B_fragile           1       ⋯
   2 │       1               1  A_very_selfhealing_B_fragile           1
   3 │       1               1  A_very_selfhealing_B_fragile           1
   4 │       1               1  A_very_selfhealing_B_fragile           1
   5 │       2               1  A_very_selfhealing_B_fragile           1       ⋯
   6 │       2               1  A_very_selfhealing_B_fragile           1
   7 │       2               1  A_very_selfhealing_B_fragile           1
   8 │       2               1  A_very_selfhealing_B_fragile           1
   9 │       6               1  A_very_selfhealing_B_fragile           1       ⋯
  10 │       6               1  A_very_selfhealing_

In [67]:
# complete checkpointed experiment:
for scenario_index in 1:12
    run_four_policy_grid_scenario(
        scenario_index;
        pcl_callbacks=pcl_callbacks,
        mp_index_fn=mp_index_for_pilot,
        N_values=FOUR_POLICY_GRID_N_VALUES,
        T=700,
        nruns=1_000,
        threaded=true
    )
end

[74/360; requested 74/360] scenario=1, p_A=0.1, alpha=0.05, N=400, M=20, setup=0.53 s, simulation=11.93 s, elapsed=0.2 min
[75/360; requested 75/360] scenario=1, p_A=0.1, alpha=0.05, N=800, M=40, setup=0.53 s, simulation=25.6 s, elapsed=0.7 min
[76/360; requested 76/360] scenario=1, p_A=0.1, alpha=0.05, N=1600, M=80, setup=0.54 s, simulation=54.7 s, elapsed=1.6 min
[77/360; requested 77/360] scenario=1, p_A=0.1, alpha=0.1, N=200, M=20, setup=0.56 s, simulation=5.97 s, elapsed=1.7 min
[78/360; requested 78/360] scenario=1, p_A=0.1, alpha=0.1, N=400, M=40, setup=0.58 s, simulation=13.46 s, elapsed=1.9 min
[79/360; requested 79/360] scenario=1, p_A=0.1, alpha=0.1, N=800, M=80, setup=0.57 s, simulation=26.56 s, elapsed=2.4 min
[80/360; requested 80/360] scenario=1, p_A=0.1, alpha=0.1, N=1600, M=160, setup=0.57 s, simulation=55.51 s, elapsed=3.3 min
[81/360; requested 81/360] scenario=1, p_A=0.1, alpha=0.15, N=200, M=30, setup=0.57 s, simulation=6.23 s, elapsed=3.4 min
[82/360; requested 82

Row,scenario_index,completed_instances_full_scenario,expected_instances_full_scenario,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,total_plan_table_seconds,total_simulation_seconds,wall_minutes_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,total_lwz_fallback_count,minimum_conservative_gap_to_conditional_bound
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool,Int64,Float64
1,2,360,360,360,360,1440,360,220.389,8419.56,144.139,true,true,true,0,0.00101142


[1/360; requested 1/360] scenario=3, p_A=0.1, alpha=0.05, N=100, M=5, setup=0.55 s, simulation=2.49 s, elapsed=0.1 min
[2/360; requested 2/360] scenario=3, p_A=0.1, alpha=0.05, N=200, M=10, setup=0.57 s, simulation=5.25 s, elapsed=0.2 min
[3/360; requested 3/360] scenario=3, p_A=0.1, alpha=0.05, N=400, M=20, setup=0.55 s, simulation=11.37 s, elapsed=0.4 min
[4/360; requested 4/360] scenario=3, p_A=0.1, alpha=0.05, N=800, M=40, setup=0.56 s, simulation=25.33 s, elapsed=0.8 min
[5/360; requested 5/360] scenario=3, p_A=0.1, alpha=0.05, N=1600, M=80, setup=0.55 s, simulation=52.98 s, elapsed=1.7 min
[6/360; requested 6/360] scenario=3, p_A=0.1, alpha=0.1, N=100, M=10, setup=0.65 s, simulation=2.66 s, elapsed=1.7 min
[7/360; requested 7/360] scenario=3, p_A=0.1, alpha=0.1, N=200, M=20, setup=0.65 s, simulation=5.88 s, elapsed=1.8 min
[8/360; requested 8/360] scenario=3, p_A=0.1, alpha=0.1, N=400, M=40, setup=0.65 s, simulation=13.12 s, elapsed=2.1 min
[9/360; requested 9/360] scenario=3, p_

Row,scenario_index,completed_instances_full_scenario,expected_instances_full_scenario,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,total_plan_table_seconds,total_simulation_seconds,wall_minutes_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,total_lwz_fallback_count,minimum_conservative_gap_to_conditional_bound
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool,Int64,Float64
1,3,360,360,360,360,1440,360,222.859,8291.11,142.029,true,true,true,0,0.00089346


[1/360; requested 1/360] scenario=4, p_A=0.1, alpha=0.05, N=100, M=5, setup=0.54 s, simulation=2.48 s, elapsed=0.1 min
[2/360; requested 2/360] scenario=4, p_A=0.1, alpha=0.05, N=200, M=10, setup=0.55 s, simulation=5.73 s, elapsed=0.2 min
[3/360; requested 3/360] scenario=4, p_A=0.1, alpha=0.05, N=400, M=20, setup=0.53 s, simulation=12.37 s, elapsed=0.4 min
[4/360; requested 4/360] scenario=4, p_A=0.1, alpha=0.05, N=800, M=40, setup=0.54 s, simulation=25.37 s, elapsed=0.8 min
[5/360; requested 5/360] scenario=4, p_A=0.1, alpha=0.05, N=1600, M=80, setup=0.53 s, simulation=53.79 s, elapsed=1.7 min
[6/360; requested 6/360] scenario=4, p_A=0.1, alpha=0.1, N=100, M=10, setup=0.57 s, simulation=2.74 s, elapsed=1.8 min
[7/360; requested 7/360] scenario=4, p_A=0.1, alpha=0.1, N=200, M=20, setup=0.58 s, simulation=5.98 s, elapsed=1.9 min
[8/360; requested 8/360] scenario=4, p_A=0.1, alpha=0.1, N=400, M=40, setup=0.58 s, simulation=12.94 s, elapsed=2.1 min
[9/360; requested 9/360] scenario=4, p_

Row,scenario_index,completed_instances_full_scenario,expected_instances_full_scenario,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,total_plan_table_seconds,total_simulation_seconds,wall_minutes_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,total_lwz_fallback_count,minimum_conservative_gap_to_conditional_bound
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool,Int64,Float64
1,4,360,360,360,360,1440,360,221.722,8401.09,143.847,true,true,true,0,0.000559666


... (13 outputs left out)

Row,scenario_index,completed_instances_full_scenario,expected_instances_full_scenario,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,total_plan_table_seconds,total_simulation_seconds,wall_minutes_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,total_lwz_fallback_count,minimum_conservative_gap_to_conditional_bound
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool,Int64,Float64
1,11,360,360,360,360,1440,360,221.989,8324.31,142.57,true,true,true,0,0.00109651


[1/360; requested 1/360] scenario=12, p_A=0.1, alpha=0.05, N=100, M=5, setup=0.54 s, simulation=2.73 s, elapsed=0.1 min
[2/360; requested 2/360] scenario=12, p_A=0.1, alpha=0.05, N=200, M=10, setup=0.53 s, simulation=5.49 s, elapsed=0.2 min
[3/360; requested 3/360] scenario=12, p_A=0.1, alpha=0.05, N=400, M=20, setup=0.54 s, simulation=11.98 s, elapsed=0.4 min
[4/360; requested 4/360] scenario=12, p_A=0.1, alpha=0.05, N=800, M=40, setup=0.55 s, simulation=25.3 s, elapsed=0.8 min
[5/360; requested 5/360] scenario=12, p_A=0.1, alpha=0.05, N=1600, M=80, setup=0.53 s, simulation=53.27 s, elapsed=1.7 min
[6/360; requested 6/360] scenario=12, p_A=0.1, alpha=0.1, N=100, M=10, setup=0.57 s, simulation=2.85 s, elapsed=1.8 min
[7/360; requested 7/360] scenario=12, p_A=0.1, alpha=0.1, N=200, M=20, setup=0.57 s, simulation=5.97 s, elapsed=1.9 min
[8/360; requested 8/360] scenario=12, p_A=0.1, alpha=0.1, N=400, M=40, setup=0.57 s, simulation=12.9 s, elapsed=2.1 min
[9/360; requested 9/360] scenario

Excessive output truncated after 524324 bytes.

Row,scenario_index,completed_instances_full_scenario,expected_instances_full_scenario,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,total_plan_table_seconds,total_simulation_seconds,wall_minutes_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,total_lwz_fallback_count,minimum_conservative_gap_to_conditional_bound
,Int64,Int64,Int64,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool,Int64,Float64
1,12,360,360,360,360,1440,360,220.462,8494.49,145.376,true,true,true,0,0.00114565


In [64]:
include("five_policy_smoke_block_s01_N100_v1.jl")

[1/72] p_A=0.1, alpha=0.05, M=5, setup=0.53 s, simulation=3.19 s, elapsed=0.1 min
[2/72] p_A=0.1, alpha=0.1, M=10, setup=0.56 s, simulation=3.48 s, elapsed=0.1 min
[3/72] p_A=0.1, alpha=0.15, M=15, setup=0.58 s, simulation=3.61 s, elapsed=0.2 min
[4/72] p_A=0.1, alpha=0.2, M=20, setup=0.59 s, simulation=3.55 s, elapsed=0.3 min
[5/72] p_A=0.1, alpha=0.25, M=25, setup=0.6 s, simulation=3.88 s, elapsed=0.4 min
[6/72] p_A=0.1, alpha=0.3, M=30, setup=0.62 s, simulation=3.81 s, elapsed=0.4 min
[7/72] p_A=0.1, alpha=0.4, M=40, setup=0.68 s, simulation=4.02 s, elapsed=0.5 min
[8/72] p_A=0.1, alpha=0.5, M=50, setup=0.7 s, simulation=3.92 s, elapsed=0.6 min
[9/72] p_A=0.2, alpha=0.05, M=5, setup=0.51 s, simulation=3.4 s, elapsed=0.7 min
[10/72] p_A=0.2, alpha=0.1, M=10, setup=0.59 s, simulation=3.8 s, elapsed=0.7 min
[11/72] p_A=0.2, alpha=0.15, M=15, setup=0.56 s, simulation=3.63 s, elapsed=0.8 min
[12/72] p_A=0.2, alpha=0.2, M=20, setup=0.59 s, simulation=3.72 s, elapsed=0.9 min
[13/72] p_A=0.

Row,completed_instances,expected_instances,policy_rows,expected_policy_rows,new_instances_this_run,total_plan_table_seconds,total_simulation_seconds,wall_minutes_this_run,all_rewards_finite,all_priorities_finite,forced_capacity_ok,total_lwz_fallback_count,minimum_conservative_gap_to_conditional_bound
,Int64,Int64,Int64,Int64,Int64,Float64,Float64,Float64,Bool,Bool,Bool,Int64,Float64
1,72,72,360,360,72,44.407,293.585,5.66137,true,true,true,0,0.00095437


"five_policy_grid_s01_N100_T700_n1000_v1.csv"

In [73]:
include("homogeneous_tie_grid_checkpointed_v1.jl")
using .HomogeneousTieGridRkappa

tie_preflight = run_homogeneous_tie_grid(
    mp_index_fn=mp_index_for_pilot,
    max_new_cases=1,
    threaded=true
)

[1/168] profile=A_very_selfhealing, alpha=0.05, N=100, M=5, simulation=1.39 s, elapsed=0.0 min, MP/myopic mismatches=0
abspath(output_path) = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\homogeneous_tie_grid_T700_n1000_v1.csv"


Row,completed_instances_full_grid,expected_instances_full_grid,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,all_mp_tables_nonnegative,all_mp_tables_nondecreasing,total_mp_fixed_action_mismatch_periods,all_mp_fixed_rewards_exact,wall_minutes_this_run
,Int64,Int64,Int64,Int64,Int64,Int64,Bool,Bool,Bool,Bool,Bool,Int64,Bool,Float64
1,1,168,1,168,4,1,true,true,true,true,true,0,true,0.0398167


(grid = 4×51 DataFrame
 Row │ policy         label                            mean_reward  stderr     ⋯
     │ String         String                           Float64      Float64    ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ belief_cyclic  Belief/myopic, cyclic ties         0.0158121  5.58176e-5 ⋯
   2 │ belief_fixed   Belief/myopic, fixed-label ties    0.015809   5.48536e-5
   3 │ belief_random  Belief/myopic, randomized ties     0.0158467  5.43705e-5
   4 │ mp_fixed       MP, fixed-label ties               0.015809   5.48536e-5
                                                              47 columns omitted, grid_check = 1×14 DataFrame
 Row │ completed_instances_full_grid  expected_instances_full_grid  completed_ ⋯
     │ Int64                          Int64                         Int64      ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │                             1                           

In [74]:
tie_full = run_homogeneous_tie_grid(
    mp_index_fn=mp_index_for_pilot,
    threaded=true
)

[2/168] profile=A_very_selfhealing, alpha=0.05, N=400, M=20, simulation=5.48 s, elapsed=0.1 min, MP/myopic mismatches=0
[3/168] profile=A_very_selfhealing, alpha=0.05, N=1600, M=80, simulation=24.21 s, elapsed=0.5 min, MP/myopic mismatches=0
[4/168] profile=A_very_selfhealing, alpha=0.1, N=100, M=10, simulation=1.35 s, elapsed=0.5 min, MP/myopic mismatches=0
[5/168] profile=A_very_selfhealing, alpha=0.1, N=400, M=40, simulation=5.75 s, elapsed=0.6 min, MP/myopic mismatches=0
[6/168] profile=A_very_selfhealing, alpha=0.1, N=1600, M=160, simulation=24.6 s, elapsed=1.0 min, MP/myopic mismatches=0
[7/168] profile=A_very_selfhealing, alpha=0.15, N=100, M=15, simulation=1.38 s, elapsed=1.1 min, MP/myopic mismatches=0
[8/168] profile=A_very_selfhealing, alpha=0.15, N=400, M=60, simulation=5.9 s, elapsed=1.2 min, MP/myopic mismatches=0
[9/168] profile=A_very_selfhealing, alpha=0.15, N=1600, M=240, simulation=25.33 s, elapsed=1.6 min, MP/myopic mismatches=0
[10/168] profile=A_very_selfhealing, 

Row,completed_instances_full_grid,expected_instances_full_grid,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,all_mp_tables_nonnegative,all_mp_tables_nondecreasing,total_mp_fixed_action_mismatch_periods,all_mp_fixed_rewards_exact,wall_minutes_this_run
,Int64,Int64,Int64,Int64,Int64,Int64,Bool,Bool,Bool,Bool,Bool,Int64,Bool,Float64
1,168,168,168,168,672,167,true,true,true,true,true,185279,false,31.3879


(grid = 672×51 DataFrame
 Row │ policy         label                            mean_reward  stderr     ⋯
     │ String         String                           Float64      Float64    ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ belief_cyclic  Belief/myopic, cyclic ties         0.0158121  5.58176e-5 ⋯
   2 │ belief_fixed   Belief/myopic, fixed-label ties    0.015809   5.48536e-5
   3 │ belief_random  Belief/myopic, randomized ties     0.0158467  5.43705e-5
   4 │ mp_fixed       MP, fixed-label ties               0.015809   5.48536e-5
   5 │ belief_cyclic  Belief/myopic, cyclic ties         0.0158608  2.70944e-5 ⋯
   6 │ belief_fixed   Belief/myopic, fixed-label ties    0.0158682  2.79091e-5
   7 │ belief_random  Belief/myopic, randomized ties     0.0158208  2.58805e-5
   8 │ mp_fixed       MP, fixed-label ties               0.0158682  2.79091e-5
   9 │ belief_cyclic  Belief/myopic, cyclic ties         0.0158681  1.34359e-5 ⋯
  10 │ belief_f

In [75]:
tie_full.grid_check

Row,completed_instances_full_grid,expected_instances_full_grid,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,all_mp_tables_nonnegative,all_mp_tables_nondecreasing,total_mp_fixed_action_mismatch_periods,all_mp_fixed_rewards_exact,wall_minutes_this_run
,Int64,Int64,Int64,Int64,Int64,Int64,Bool,Bool,Bool,Bool,Bool,Int64,Bool,Float64
1,168,168,168,168,672,167,true,true,true,true,true,185279,false,31.3879


In [76]:
for col in names(tie_full.grid_check)
    println(rpad(col, 48), tie_full.grid_check[1, col])
end

completed_instances_full_grid                   168
expected_instances_full_grid                    168
completed_instances_requested                   168
expected_instances_requested                    168
policy_rows                                     672
new_instances_this_run                          167
all_rewards_finite                              true
all_priorities_finite                           true
exact_capacity_ok                               true
all_mp_tables_nonnegative                       true
all_mp_tables_nondecreasing                     true
total_mp_fixed_action_mismatch_periods          185279
all_mp_fixed_rewards_exact                      false
wall_minutes_this_run                           31.387916664282482


In [77]:
include("homogeneous_tie_grid_lexicographic_random_v2.jl")
import .HomogeneousTieGridRkappaV2

tie_preflight_v2 =
    HomogeneousTieGridRkappaV2.run_homogeneous_tie_grid(
        mp_index_fn=mp_index_for_pilot,
        max_new_cases=1,
        threaded=true
    )

[1/168] profile=A_very_selfhealing, alpha=0.05, N=100, M=5, simulation=1.34 s, elapsed=0.0 min, MP/random-belief mismatches=0
abspath(output_path) = "C:\\Users\\JNINO\\Documents\\GitHub\\paperPCLsensingErrors\\clean_benchmark_v4\\homogeneous_tie_grid_T700_n1000_v2.csv"


Row,completed_instances_full_grid,expected_instances_full_grid,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,all_mp_tables_nonnegative,all_mp_tables_nondecreasing,total_mp_random_action_mismatch_periods,all_mp_random_rewards_exact,wall_minutes_this_run
,Int64,Int64,Int64,Int64,Int64,Int64,Bool,Bool,Bool,Bool,Bool,Int64,Bool,Float64
1,1,168,1,168,4,1,true,true,true,true,true,0,true,0.0360167


(grid = 4×51 DataFrame
 Row │ policy            label                              mean_reward  stder ⋯
     │ String            String                             Float64      Float ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ belief_cyclic     Belief/myopic, cyclic ties           0.0158121  5.581 ⋯
   2 │ belief_fixed      Belief/myopic, fixed-label ties      0.015809   5.485
   3 │ belief_random     Belief/myopic, randomized ties       0.0158467  5.437
   4 │ mp_belief_random  MP, belief secondary key and ran…    0.0158467  5.437
                                                              48 columns omitted, grid_check = 1×14 DataFrame
 Row │ completed_instances_full_grid  expected_instances_full_grid  completed_ ⋯
     │ Int64                          Int64                         Int64      ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │                             1                           

In [78]:
tie_full_v2 =
    HomogeneousTieGridRkappaV2.run_homogeneous_tie_grid(
        mp_index_fn=mp_index_for_pilot,
        threaded=true
    )

[2/168] profile=A_very_selfhealing, alpha=0.05, N=400, M=20, simulation=5.76 s, elapsed=0.1 min, MP/random-belief mismatches=0
[3/168] profile=A_very_selfhealing, alpha=0.05, N=1600, M=80, simulation=26.75 s, elapsed=0.5 min, MP/random-belief mismatches=0
[4/168] profile=A_very_selfhealing, alpha=0.1, N=100, M=10, simulation=1.39 s, elapsed=0.6 min, MP/random-belief mismatches=0
[5/168] profile=A_very_selfhealing, alpha=0.1, N=400, M=40, simulation=5.95 s, elapsed=0.7 min, MP/random-belief mismatches=0
[6/168] profile=A_very_selfhealing, alpha=0.1, N=1600, M=160, simulation=26.86 s, elapsed=1.1 min, MP/random-belief mismatches=0
[7/168] profile=A_very_selfhealing, alpha=0.15, N=100, M=15, simulation=1.49 s, elapsed=1.1 min, MP/random-belief mismatches=0
[8/168] profile=A_very_selfhealing, alpha=0.15, N=400, M=60, simulation=6.12 s, elapsed=1.2 min, MP/random-belief mismatches=0
[9/168] profile=A_very_selfhealing, alpha=0.15, N=1600, M=240, simulation=27.49 s, elapsed=1.7 min, MP/random

Row,completed_instances_full_grid,expected_instances_full_grid,completed_instances_requested,expected_instances_requested,policy_rows,new_instances_this_run,all_rewards_finite,all_priorities_finite,exact_capacity_ok,all_mp_tables_nonnegative,all_mp_tables_nondecreasing,total_mp_random_action_mismatch_periods,all_mp_random_rewards_exact,wall_minutes_this_run
,Int64,Int64,Int64,Int64,Int64,Int64,Bool,Bool,Bool,Bool,Bool,Int64,Bool,Float64
1,168,168,168,168,672,167,true,true,true,true,true,0,true,35.3951


(grid = 672×51 DataFrame
 Row │ policy            label                              mean_reward  stder ⋯
     │ String            String                             Float64      Float ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ belief_cyclic     Belief/myopic, cyclic ties           0.0158121  5.581 ⋯
   2 │ belief_fixed      Belief/myopic, fixed-label ties      0.015809   5.485
   3 │ belief_random     Belief/myopic, randomized ties       0.0158467  5.437
   4 │ mp_belief_random  MP, belief secondary key and ran…    0.0158467  5.437
   5 │ belief_cyclic     Belief/myopic, cyclic ties           0.0158608  2.709 ⋯
   6 │ belief_fixed      Belief/myopic, fixed-label ties      0.0158682  2.790
   7 │ belief_random     Belief/myopic, randomized ties       0.0158208  2.588
   8 │ mp_belief_random  MP, belief secondary key and ran…    0.0158208  2.588
   9 │ belief_cyclic     Belief/myopic, cyclic ties           0.0158681  1.343 ⋯
  10 │ belief_f

In [79]:
pwd()
isfile("pcli_boundary_stress_v1.jl")
@isdefined g_metric2
@isdefined fg_metric2

true

In [80]:
g_metric_stress(x, z; p01, rho, kappa, beta, epsilon) =
    g_metric2(
        x, z;
        p01=p01, ρ=rho, κ=kappa, β=beta, ε=epsilon
    )

fg_metric_stress(x, z; p01, rho, kappa, reward, beta, epsilon) =
    fg_metric2(
        x, z;
        p01=p01, ρ=rho, κ=kappa, r=reward, β=beta, ε=epsilon
    )

fg_metric_stress (generic function with 1 method)

In [81]:
include("pcli_boundary_stress_v1.jl")
import .PCLIBoundaryStressV1

pcli_boundary_preflight =
    PCLIBoundaryStressV1.run_pcli_boundary_stress(
        g_metric_fn=g_metric_stress,
        fg_metric_fn=fg_metric_stress,
        max_new_cases=1
    )

for col in names(pcli_boundary_preflight.check)
    println(rpad(col, 54), pcli_boundary_preflight.check[1, col])
end

[14:59:57] completed 1/320 boundary tuples
completed_parameter_tuples                            1
expected_parameter_tuples                             320
new_parameter_tuples_this_run                         1
coarse_grid_complete                                  false
all_pcli1_slacks_nonnegative                          true
all_pcli2_secant_slopes_nonnegative                   true
all_sampled_marginal_work_positive                    true
refined_pcli1_slacks_nonnegative                      missing
refined_pcli2_secant_slopes_nonnegative               missing
minimum_pcli1_slack                                   0.17155218769022518
minimum_pcli2_core_slope                              0.8191693303531585
minimum_pcli2_all_slope                               0.4999999997313323
refined_minimum_pcli1_slack                           NaN
refined_minimum_pcli2_core_slope                      NaN
refined_minimum_pcli2_all_slope                       NaN
coarse_pcli1_evaluations        

In [82]:
pcli_boundary_full =
    PCLIBoundaryStressV1.run_pcli_boundary_stress(
        g_metric_fn=g_metric_stress,
        fg_metric_fn=fg_metric_stress
    )

for col in names(pcli_boundary_full.check)
    println(rpad(col, 54), pcli_boundary_full.check[1, col])
end

[15:08:33] completed 25/320 boundary tuples
[15:08:37] completed 49/320 boundary tuples
[15:08:42] completed 73/320 boundary tuples
[15:08:46] completed 97/320 boundary tuples
[15:08:50] completed 121/320 boundary tuples
[15:09:00] completed 145/320 boundary tuples
[15:09:11] completed 169/320 boundary tuples
[15:09:23] completed 193/320 boundary tuples
[15:10:07] completed 217/320 boundary tuples
[15:10:58] completed 241/320 boundary tuples
[15:12:05] completed 265/320 boundary tuples
[15:12:42] completed 289/320 boundary tuples
[15:13:19] completed 313/320 boundary tuples
[15:13:25] completed 320/320 boundary tuples
completed_parameter_tuples                            320
expected_parameter_tuples                             320
new_parameter_tuples_this_run                         319
coarse_grid_complete                                  true
all_pcli1_slacks_nonnegative                          true
all_pcli2_secant_slopes_nonnegative                   true
all_sampled_marginal_wo

In [ ]:
# Example with three projects
qvector = [0.25, 0.10, 0.30]
ρvector = [0.60, 0.80, 0.65]
κvector = [0.80, 0.60, 0.70]
rvector = [1.00, 0.80, 1.20]

β       = 0.95
M       = 2
xvector = [0.50, 0.50, 0.50]

λstar_old, Lmin_old = lambda_star_vector(
    xvector;
    qvector=qvector,
    ρvector=ρvector,
    κvector=κvector,
    rvector=rvector,
    β=β,
    M=M
)

λstar_new, Lmin_new, info_new =
    lambda_star_vector_grouped_bisect(
        xvector;
        qvector=qvector,
        ρvector=ρvector,
        κvector=κvector,
        rvector=rvector,
        β=β,
        M=M,
        verbose=false
    )

@show λstar_old Lmin_old
@show λstar_new Lmin_new
@show info_new

In [ ]:
groups = compress_project_vectors(
    xvector;
    qvector=qvector,
    ρvector=ρvector,
    κvector=κvector,
    rvector=rvector
)

for λ in (0.0, 0.2, 0.8, 1.5)
    direct = Lvector(
        λ, xvector;
        qvector=qvector,
        ρvector=ρvector,
        κvector=κvector,
        rvector=rvector,
        β=β,
        M=M
    )

    grouped = eval_Lvector_grouped(
        λ, groups;
        β=β,
        M=M
    ).L

    @assert isapprox(direct, grouped; rtol=1e-9, atol=1e-9)
end

println("Grouped and direct evaluations agree.")

In [ ]:
# --- helper: snap consecutive values within tol to the same level (optional) ---
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    isempty(yy) && return yy

    anchor = yy[1]

    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(anchor) &&
           abs(yy[i] - anchor) ≤ tol
            yy[i] = anchor
        else
            anchor = yy[i]
        end
    end

    return yy
end

"""
Square axis plot on [0,1]×[0,1] with y-ticks only at 0 and 1, and no padding.

Unlike the step-plot version, this one plots the sampled curve directly.
If `smooth_curve=true`, PGFPlots is asked to draw a smooth interpolating curve.
"""
function plot_mp_index_vs_x_axis(; p01, ρ, κ, r, β, ε=1e-10,
                                 xgrid=collect(range(0.0, 1.0; length=2001)),
                                 xlabel=L"$x$",
                                 ylabel=L"$m(x)$",
                                 width="8cm", height="8cm",
                                 snap_tol_m=0.0,
                                 smooth_curve::Bool=false)

    x0_fp, x1_fp, _ = fixed_points(; p01=p01, ρ=ρ, κ=κ)

    items = [
        (0.0,             "\\ensuremath{0}"),
        (Float64(p01),    "\\ensuremath{p_{01}}"),
        (Float64(x1_fp),  "\\makebox[0pt][l]{\\ensuremath{x^{1}}}"),
        (Float64(x0_fp),  "\\ensuremath{x^{0}}"),
        (Float64(p01+ρ),  "\\ensuremath{p_{11}}"),
        (1.0,             "\\ensuremath{1}"),
    ]

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end
    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{p_{01}+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    # sample m(x) = f(x,x)/g(x,x)
    mvals = similar(xgrid, Float64)
    for (i, x) in enumerate(xgrid)
        fxx = Float64(f_metric2(x, x;
            p01=p01, ρ=ρ, κ=κ, r=r, β=β, ε=ε))

        gxx = Float64(g_metric2(x, x;
            p01=p01, ρ=ρ, κ=κ, β=β, ε=ε))
        mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
    end

    if snap_tol_m > 0
        mvals = _snap_consecutive(mvals; tol=snap_tol_m)
    end

    coords = Tuple{Float64,Float64}[
        (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
    ]

    curve_opts = Pair{Any,Any}[
        "line width" => "0.40pt",
        "black"      => nothing,
    ]
    if smooth_curve
        push!(curve_opts, "smooth" => nothing)
    else
        push!(curve_opts, "solid" => nothing)
    end

    return @pgf Axis(
        {
            width  => width,
            height => height,
            xlabel => xlabel,
            ylabel => ylabel,

            xmin   => 0,
            xmax   => 1,
            ymin   => 0,
            ymax   => 1,
            "axis equal image" => nothing,

            "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
            "tick align" => "outside",
            "major tick length" => "2pt",
            "minor tick length" => "1pt",

            "xtick" => xtick_str,
            "xticklabels" => xticklab_str,

            "ytick" => "{0,1}",
            "enlargelimits" => "false",

            "scaled y ticks" => "false",
            "unbounded coords" => "jump",
        },

        Plot(PGFPlotsX.Options(curve_opts...), Coordinates(coords)),
    )
end

function save_mp_index_vs_x_plot_pdf(outpdf::AbstractString;
                                     p01, ρ, κ, r, β,
                                     engine::AbstractString="pdflatex",
                                     quiet::Bool=true,
                                     keep_tex::Bool=true,
                                     kwargs...)
    ax = plot_mp_index_vs_x_axis(
        ; p01=p01, ρ=ρ, κ=κ, r=r, β=β, kwargs...
    )
    # remainder unchanged
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end
# Example:
# p01=0.25; ρ=0.6; κ=0.8; r=1.0; β=0.95
# save_mp_index_vs_x_plot_pdf("mp_index_vs_x.pdf"; p01=p01, ρ=ρ, κ=κ, r=r, β=β)

In [ ]:
# Parameters
p01 = 0.25
ρ   = 0.60
κ   = 0.80
β   = 0.95
r   = 1.00

save_mp_index_vs_x_plot_pdf(
    "mp_index_vs_x.pdf";
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    β=β,
    xgrid=collect(range(0.0, 1.0; length=4001))
)

In [ ]:
_fmt_beta(β; digits::Int=3) =
    Printf.format(Printf.Format("%.$(digits)f"), β)


function _sorted_betas_and_shades(
    betas::AbstractVector{<:Real};
    shade_min::Int=30,
    shade_max::Int=100
)
    βs = sort!(unique!(Float64.(collect(betas))))

    isempty(βs) &&
        throw(ArgumentError("betas must be nonempty"))

    all(β -> isfinite(β) && 0.0 ≤ β < 1.0, βs) ||
        throw(ArgumentError("every β must satisfy 0 ≤ β < 1"))

    0 ≤ shade_min ≤ shade_max ≤ 100 ||
        throw(ArgumentError(
            "shades must satisfy 0 ≤ shade_min ≤ shade_max ≤ 100"
        ))

    n = length(βs)

    shades = n == 1 ?
             [shade_max] :
             round.(Int, range(shade_min, shade_max; length=n))

    return βs, shades
end


"""
Plot the MP-index curves m_β(x) for several discount factors.

The horizontal range is [0,1] and the vertical range is [0,r].
By default, only the smallest and largest β values are included
in the legend.
"""
function plot_mp_index_vs_x_axis_betas(
    ;
    p01,
    ρ,
    κ,
    r,
    betas,
    ε=1e-10,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    xlabel=L"$x$",
    ylabel=L"$m(x)$",
    width="8cm",
    height="8cm",
    snap_tol_m=0.0,
    shade_min=30,
    shade_max=100,
    line_width="0.30pt",
    beta_digits=3,
    legend_extremes_only::Bool=true,
    smooth_curves::Bool=false
)
    rmax = Float64(r)

    isfinite(rmax) && rmax > 0.0 ||
        throw(ArgumentError("r must be finite and strictly positive"))

    x0_fp, x1_fp, _ = fixed_points(
        ; p01=p01, ρ=ρ, κ=κ
    )

    items = [
        (0.0,            "\\ensuremath{0}"),
        (Float64(p01),   "\\ensuremath{p_{01}}"),
        (Float64(x1_fp), "\\makebox[0pt][l]{\\ensuremath{x^{1}}}"),
        (Float64(x0_fp), "\\ensuremath{x^{0}}"),
        (Float64(p01+ρ), "\\ensuremath{p_{11}}"),
        (1.0,            "\\ensuremath{1}"),
    ]

    sort!(items; by=first)

    # Merge coincident tick positions
    merged = Tuple{Float64,String}[]

    for (u, label) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, label))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [label for (_, label) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"

    phantom = "\\vphantom{\\ensuremath{p_{01}+\\rho}}"
    xticklab_str =
        "{" * join([phantom * label for label in labels], ",") * "}"

    βs, shades = _sorted_betas_and_shades(
        betas;
        shade_min=shade_min,
        shade_max=shade_max
    )

    nβ = length(βs)
    elems = Any[]

    for (j, (βj, shade)) in enumerate(zip(βs, shades))
        mvals = Vector{Float64}(undef, length(xgrid))

        for (i, x) in enumerate(xgrid)
            fxx = Float64(
                f_metric2(
                    x, x;
                    p01=p01,
                    ρ=ρ,
                    κ=κ,
                    r=r,
                    β=βj,
                    ε=ε
                )
            )

            gxx = Float64(
                g_metric2(
                    x, x;
                    p01=p01,
                    ρ=ρ,
                    κ=κ,
                    β=βj,
                    ε=ε
                )
            )

            mvals[i] =
                isfinite(fxx) && isfinite(gxx) && gxx != 0.0 ?
                fxx / gxx :
                NaN
        end

        if snap_tol_m > 0
            mvals = _snap_consecutive(
                mvals;
                tol=snap_tol_m
            )
        end

        coords = Tuple{Float64,Float64}[
            (
                Float64(x),
                isfinite(y) ? Float64(y) : NaN
            )
            for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend =
            legend_extremes_only ?
            (nβ == 1 || j == 1 || j == nβ) :
            true

        plot_options = Pair{Any,Any}[
            "line width" => line_width,
            "color"      => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_options, "smooth" => nothing)
        else
            push!(plot_options, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_options, "forget plot" => nothing)
        end

        push!(
            elems,
            Plot(
                PGFPlotsX.Options(plot_options...),
                Coordinates(coords)
            )
        )

        if show_in_legend
            βlabel = _fmt_beta(
                βj;
                digits=beta_digits
            )

            push!(
                elems,
                LegendEntry("\$\\beta = $(βlabel)\$")
            )
        end
    end

    axis_options = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin" => 0,
        "xmax" => 1,
        "ymin" => 0,
        "ymax" => rmax,

        # Do not use "axis equal image": when r ≠ 1 it would
        # prevent the displayed frame from remaining square.

        "xticklabel style" =>
            "{text height=1.8ex, text depth=0.25ex}",
        "tick align"        => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick"       => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick"       => "{0,$(rmax)}",
        "yticklabels" => "{\\ensuremath{0},\\ensuremath{r}}",

        "enlargelimits"    => "false",
        "scaled y ticks"   => "false",
        "unbounded coords" => "jump",
        "legend pos"       => "north west",
    )

    return Axis(axis_options, elems...)
end


function save_mp_index_vs_x_betas_plot_pdf(
    outpdf::AbstractString;
    p01,
    ρ,
    κ,
    r,
    betas,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)
    ax = plot_mp_index_vs_x_axis_betas(
        ;
        p01=p01,
        ρ=ρ,
        κ=κ,
        r=r,
        betas=betas,
        kwargs...
    )

    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(
        texfile,
        pic;
        include_preamble=true
    )

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                output_buffer = IOBuffer()

                run(
                    pipeline(
                        cmd;
                        stdout=output_buffer,
                        stderr=output_buffer
                    )
                )
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            first_line = max(1, length(lines) - 80)
            tail = lines[first_line:end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    if pdffile != outpdf_abs
        cp(
            pdffile,
            outpdf_abs;
            force=true
        )
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# Parameters
p01 = 0.25
ρ   = 0.60
κ   = 0.80
r   = 1.00

betas = [0.10, 0.30, 0.50, 0.70, 0.90, 0.95, 0.999]

save_mp_index_vs_x_betas_plot_pdf(
    "mp_family_beta.pdf";
    p01=p01,
    ρ=ρ,
    κ=κ,
    r=r,
    betas=betas,
    xgrid=collect(range(0.0, 1.0; length=1001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true
)

In [ ]:
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

_fmt_kappa(κ; digits::Int=3) = Printf.format(Printf.Format("%.$(digits)f"), κ)

function _sorted_kappas_and_shades(kappas::AbstractVector{<:Real};
                                   shade_min::Int=30,
                                   shade_max::Int=100)
    κs = sort!(unique!(Float64.(collect(kappas))))
    n = length(κs)
    shades = n == 1 ? [shade_max] :
             round.(Int, collect(range(shade_min, shade_max; length=n)))
    return κs, shades
end

function plot_mp_index_vs_x_axis_kappas(; p01, ρ, β, r, kappas,
                                        ε=1e-10,
                                        xgrid=collect(range(0.0, 1.0; length=2001)),
                                        xlabel=L"$x$",
                                        ylabel=L"$m(x)$",
                                        width="8cm", height="8cm",
                                        snap_tol_m=0.0,
                                        shade_min=30,
                                        shade_max=100,
                                        line_width="0.30pt",
                                        kappa_digits=3,
                                        legend_extremes_only::Bool=true,
                                        kappa_ref=nothing,
                                        show_ref_fixed_points::Bool=false,
                                        smooth_curves::Bool=true)

    # x-axis ticks: only invariant locations by default
    items = Tuple{Float64,String}[
        (0.0,        "\\ensuremath{0}"),
        (float(p01),   "\\ensuremath{p_{01}}"),
        (float(p01+ρ), "\\ensuremath{p_{11}}"),
        (1.0,        "\\ensuremath{1}"),
    ]

    # optionally add x¹ and x⁰ for a chosen reference κ
    if show_ref_fixed_points
        κref = isnothing(kappa_ref) ? first(sort(Float64.(collect(kappas)))) : float(kappa_ref)
        x0_fp, x1_fp, _ = fixed_points(; p01=p01, ρ=ρ, κ=κref)
        push!(items, (float(x1_fp), "\\ensuremath{x^{1}}"))
        push!(items, (float(x0_fp), "\\ensuremath{x^{0}}"))
    end

    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks  = [u for (u, _) in merged]
    labels  = [lab for (_, lab) in merged]
    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{p01+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    κs, shades = _sorted_kappas_and_shades(kappas;
                                           shade_min=shade_min,
                                           shade_max=shade_max)
    nκ = length(κs)

    elems = Any[]

    # draw lowest κ first, highest κ last
    for (j, (κj, shade)) in enumerate(zip(κs, shades))
        mvals = similar(xgrid, Float64)

        for (i, x) in enumerate(xgrid)
            fxx = float(f_metric2(x, x; p01=p01, ρ=ρ, κ=κj, r=r, β=β, ε=ε))
            gxx = float(g_metric2(x, x; p01=p01, ρ=ρ, κ=κj,      β=β, ε=ε))
            mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
        end

        if snap_tol_m > 0
            mvals = _snap_consecutive(mvals; tol=snap_tol_m)
        end

        coords = Tuple{Float64,Float64}[
            (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend = legend_extremes_only ? (nκ == 1 || j == 1 || j == nκ) : true

        # slightly emphasize the largest κ curve
        lw = (j == nκ && nκ > 1) ? "0.45pt" : line_width

        plot_pairs = Pair{Any,Any}[
            "line width" => lw,
            "color" => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_pairs, "smooth" => nothing)
        else
            push!(plot_pairs, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_pairs, "forget plot" => nothing)
        end

        push!(elems, Plot(PGFPlotsX.Options(plot_pairs...), Coordinates(coords)))

        if show_in_legend
            push!(elems, LegendEntry("\$\\kappa = $(_fmt_kappa(κj; digits=kappa_digits))\$"))
        end
    end

    axis_opts = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin"   => 0,
        "xmax"   => 1,
        "ymin"   => 0,
        "ymax"   => 1,
        "axis equal image" => nothing,

        "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
        "tick align" => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick" => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick" => "{0,1}",
        "enlargelimits" => "false",

        "scaled y ticks" => "false",
        "unbounded coords" => "jump",
        "legend pos" => "north west",
    )

    return Axis(axis_opts, elems...)
end

function save_mp_index_vs_x_kappas_plot_pdf(outpdf::AbstractString;
                                            p01, ρ, β, r, kappas,
                                            engine::AbstractString="pdflatex",
                                            quiet::Bool=true,
                                            keep_tex::Bool=true,
                                            kwargs...)
    ax  = plot_mp_index_vs_x_axis_kappas(; p01=p01, ρ=ρ, β=β, r=r, kappas=kappas, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
p01  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

kappas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.70, 0.8, 0.90, 0.95, 0.999]

save_mp_index_vs_x_kappas_plot_pdf("mp_family_kappa.pdf";
    p01=p01, ρ=ρ, β=β, r=r,
    kappas=kappas,
    xgrid=collect(range(0.0, 1.0; length=1001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true)

In [ ]:
_fmt_rho(ρ; digits::Int=3) =
    Printf.format(Printf.Format("%.$(digits)f"), ρ)


function _sorted_rhos_and_shades(
    rhos::AbstractVector{<:Real};
    shade_min::Int=30,
    shade_max::Int=100
)
    ρs = sort!(unique!(Float64.(collect(rhos))))

    isempty(ρs) &&
        throw(ArgumentError("rhos must be nonempty"))

    all(ρ -> isfinite(ρ) && ρ > 0.0, ρs) ||
        throw(ArgumentError(
            "every rho must be finite and strictly positive"
        ))

    0 ≤ shade_min ≤ shade_max ≤ 100 ||
        throw(ArgumentError(
            "shades must satisfy 0 ≤ shade_min ≤ shade_max ≤ 100"
        ))

    n = length(ρs)

    shades = n == 1 ?
             [shade_max] :
             round.(Int, range(shade_min, shade_max; length=n))

    return ρs, shades
end


"""
Plot the MP-index curves m_ρ(x) for several autocorrelation parameters ρ.

The horizontal range is [0,1] and the vertical range is [0,r]. By default,
only the smallest and largest ρ values appear in the legend.

Since p₁₁ and the fixed points depend on ρ, they are omitted from the
horizontal ticks unless the corresponding reference options are enabled.
If no `rho_ref` is supplied, the smallest plotted ρ is used as the reference.
"""
function plot_mp_index_vs_x_axis_rhos(
    ;
    p01,
    κ,
    β,
    r,
    rhos,
    ε=1e-10,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    xlabel=L"$x$",
    ylabel=L"$m(x)$",
    width="8cm",
    height="8cm",
    snap_tol_m=0.0,
    shade_min=30,
    shade_max=100,
    line_width="0.30pt",
    rho_digits::Int=3,
    legend_extremes_only::Bool=true,
    rho_ref=nothing,
    show_ref_p11::Bool=false,
    show_ref_fixed_points::Bool=false,
    smooth_curves::Bool=false
)
    p01val = Float64(p01)
    κval   = Float64(κ)
    βval   = Float64(β)
    rmax   = Float64(r)

    isfinite(p01val) && 0.0 < p01val < 1.0 ||
        throw(ArgumentError("p01 must satisfy 0 < p01 < 1"))

    isfinite(κval) && 0.0 < κval ≤ 1.0 ||
        throw(ArgumentError("κ must satisfy 0 < κ ≤ 1"))

    isfinite(βval) && 0.0 ≤ βval < 1.0 ||
        throw(ArgumentError("β must satisfy 0 ≤ β < 1"))

    isfinite(rmax) && rmax > 0.0 ||
        throw(ArgumentError("r must be finite and strictly positive"))

    snap_tol_m ≥ 0.0 ||
        throw(ArgumentError("snap_tol_m must be nonnegative"))

    ρs, shades = _sorted_rhos_and_shades(
        rhos;
        shade_min=shade_min,
        shade_max=shade_max
    )

    all(ρj -> 0.0 < ρj < 1.0 - p01val, ρs) ||
        throw(ArgumentError(
            "every rho must satisfy 0 < rho < 1-p01"
        ))

    # Tick locations common to all rho values
    items = Tuple{Float64,String}[
        (0.0,      "\\ensuremath{0}"),
        (p01val,   "\\ensuremath{p_{01}}"),
        (1.0,      "\\ensuremath{1}"),
    ]

    # Add rho-dependent reference ticks only when requested
    if show_ref_p11 || show_ref_fixed_points
        ρref = isnothing(rho_ref) ? first(ρs) : Float64(rho_ref)

        isfinite(ρref) && 0.0 < ρref < 1.0 - p01val ||
            throw(ArgumentError(
                "rho_ref must satisfy 0 < rho_ref < 1-p01"
            ))

        if show_ref_p11
            push!(
                items,
                (
                    p01val + ρref,
                    "\\ensuremath{p_{11}}"
                )
            )
        end

        if show_ref_fixed_points
            x0_fp, x1_fp, _ = fixed_points(
                ;
                p01=p01val,
                ρ=ρref,
                κ=κval
            )

            push!(
                items,
                (
                    Float64(x1_fp),
                    "\\makebox[0pt][l]{\\ensuremath{x^{1}}}"
                )
            )

            push!(
                items,
                (
                    Float64(x0_fp),
                    "\\ensuremath{x^{0}}"
                )
            )
        end
    end

    sort!(items; by=first)

    # Merge coincident tick positions
    merged = Tuple{Float64,String}[]

    for (u, label) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, label))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [label for (_, label) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"

    phantom = "\\vphantom{\\ensuremath{p_{01}+\\rho}}"
    xticklab_str =
        "{" * join([phantom * label for label in labels], ",") * "}"

    nρ = length(ρs)
    elems = Any[]

    # Draw the smallest rho first and the largest rho last
    for (j, (ρj, shade)) in enumerate(zip(ρs, shades))
        mvals = Vector{Float64}(undef, length(xgrid))

        for (i, x) in enumerate(xgrid)
            fxx = Float64(
                f_metric2(
                    x, x;
                    p01=p01val,
                    ρ=ρj,
                    κ=κval,
                    r=rmax,
                    β=βval,
                    ε=ε
                )
            )

            gxx = Float64(
                g_metric2(
                    x, x;
                    p01=p01val,
                    ρ=ρj,
                    κ=κval,
                    β=βval,
                    ε=ε
                )
            )

            mvals[i] =
                if isfinite(fxx) && isfinite(gxx) && gxx != 0.0
                    fxx / gxx
                else
                    NaN
                end
        end

        if snap_tol_m > 0.0
            mvals = _snap_consecutive(
                mvals;
                tol=snap_tol_m
            )
        end

        coords = Tuple{Float64,Float64}[
            (
                Float64(x),
                isfinite(y) ? Float64(y) : NaN
            )
            for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend =
            legend_extremes_only ?
            (nρ == 1 || j == 1 || j == nρ) :
            true

        plot_options = Pair{Any,Any}[
            "line width" => line_width,
            "color"      => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_options, "smooth" => nothing)
        else
            push!(plot_options, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_options, "forget plot" => nothing)
        end

        push!(
            elems,
            Plot(
                PGFPlotsX.Options(plot_options...),
                Coordinates(coords)
            )
        )

        if show_in_legend
            ρlabel = _fmt_rho(
                ρj;
                digits=rho_digits
            )

            push!(
                elems,
                LegendEntry("\$\\rho = $(ρlabel)\$")
            )
        end
    end

    axis_options = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin" => 0,
        "xmax" => 1,
        "ymin" => 0,
        "ymax" => rmax,

        # Omitting "axis equal image" keeps the displayed frame square
        # when r differs from 1.

        "xticklabel style" =>
            "{text height=1.8ex, text depth=0.25ex}",
        "tick align"        => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick"       => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick"       => "{0,$(rmax)}",
        "yticklabels" => "{\\ensuremath{0},\\ensuremath{r}}",

        "enlargelimits"    => "false",
        "scaled y ticks"   => "false",
        "unbounded coords" => "jump",
        "legend pos"       => "north west",
    )

    return Axis(axis_options, elems...)
end


function save_mp_index_vs_x_rhos_plot_pdf(
    outpdf::AbstractString;
    p01,
    κ,
    β,
    r,
    rhos,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)
    ax = plot_mp_index_vs_x_axis_rhos(
        ;
        p01=p01,
        κ=κ,
        β=β,
        r=r,
        rhos=rhos,
        kwargs...
    )

    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(
        texfile,
        pic;
        include_preamble=true
    )

    cmd =
        `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                output_buffer = IOBuffer()

                run(
                    pipeline(
                        cmd;
                        stdout=output_buffer,
                        stderr=output_buffer
                    )
                )
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            first_line = max(1, length(lines) - 80)
            tail = lines[first_line:end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    if pdffile != outpdf_abs
        cp(
            pdffile,
            outpdf_abs;
            force=true
        )
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# Parameters
p01 = 0.25
κ   = 0.80
β   = 0.95
r   = 1.00

rhos = [
    0.001,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.749
]

save_mp_index_vs_x_rhos_plot_pdf(
    "mp_family_rho.pdf";
    p01=p01,
    κ=κ,
    β=β,
    r=r,
    rhos=rhos,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true
)

In [ ]:
function _snap_consecutive(y::AbstractVector{<:Real}; tol::Real=1e-8)
    yy = Float64.(y)
    @inbounds for i in 2:length(yy)
        if isfinite(yy[i]) && isfinite(yy[i-1]) && abs(yy[i] - yy[i-1]) ≤ tol
            yy[i] = yy[i-1]
        end
    end
    return yy
end

_fmt_q(p01; digits::Int=3) = Printf.format(Printf.Format("%.$(digits)f"), p01)

function _sorted_qs_and_shades(qs::AbstractVector{<:Real};
                               shade_min::Int=30,
                               shade_max::Int=100)
    qvals = sort!(unique!(Float64.(collect(qs))))
    n = length(qvals)
    shades = n == 1 ? [shade_max] :
             round.(Int, collect(range(shade_min, shade_max; length=n)))
    return qvals, shades
end

function plot_mp_index_vs_x_axis_qs(; qs, ρ, κ, β, r,
                                    ε=1e-10,
                                    xgrid=collect(range(0.0, 1.0; length=2001)),
                                    xlabel=L"$x$",
                                    ylabel=L"$m(x)$",
                                    width="8cm", height="8cm",
                                    snap_tol_m=0.0,
                                    shade_min=30,
                                    shade_max=100,
                                    line_width="0.30pt",
                                    q_digits=3,
                                    legend_extremes_only::Bool=true,
                                    q_ref=nothing,
                                    show_ref_p01::Bool=false,
                                    show_ref_p11::Bool=false,
                                    show_ref_fixed_points::Bool=false,
                                    smooth_curves::Bool=true)

    qvals, shades = _sorted_qs_and_shades(qs;
                                          shade_min=shade_min,
                                          shade_max=shade_max)

    # admissibility checks: 0 < p01 < 1-rho
    for qj in qvals
        if !(0.0 < qj < 1.0 - ρ)
            error("Each p01 must satisfy 0 < p01 < 1-rho. Got p01 = $(qj), rho = $(ρ).")
        end
    end

    # invariant ticks by default
    items = Tuple{Float64,String}[
        (0.0, "\\ensuremath{0}"),
        (1.0, "\\ensuremath{1}"),
    ]

    qref = isnothing(q_ref) ? first(qvals) : float(q_ref)
    if !(0.0 < qref < 1.0 - ρ)
        error("q_ref must satisfy 0 < q_ref < 1-rho. Got q_ref = $(qref), rho = $(ρ).")
    end

    if show_ref_p01
        push!(items, (float(qref), "\\ensuremath{p_{01}}"))
    end

    if show_ref_p11
        push!(items, (float(qref + ρ), "\\ensuremath{p_{11}}"))
    end

    if show_ref_fixed_points
        x0_fp, x1_fp, _ = fixed_points(; p01=qref, ρ=ρ, κ=κ)
        push!(items, (float(x1_fp), "\\ensuremath{x^{1}}"))
        push!(items, (float(x0_fp), "\\ensuremath{x^{0}}"))
    end

    sort!(items, by=first)

    merged = Tuple{Float64,String}[]
    for (u, lab) in items
        if isempty(merged) || abs(u - merged[end][1]) > 1e-12
            push!(merged, (u, lab))
        end
    end

    xticks = [u for (u, _) in merged]
    labels = [lab for (_, lab) in merged]

    xtick_str = "{" * join(string.(xticks), ",") * "}"
    phantom = "\\vphantom{\\ensuremath{p01+\\rho}}"
    xticklab_str = "{" * join([phantom * lab for lab in labels], ",") * "}"

    nq = length(qvals)
    elems = Any[]

    # draw smallest p01 first, largest p01 last
    for (j, (qj, shade)) in enumerate(zip(qvals, shades))
        mvals = similar(xgrid, Float64)

        for (i, x) in enumerate(xgrid)
            fxx = float(f_metric2(x, x; p01=qj, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
            gxx = float(g_metric2(x, x; p01=qj, ρ=ρ, κ=κ,      β=β, ε=ε))
            mvals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : NaN
        end

        if snap_tol_m > 0
            mvals = _snap_consecutive(mvals; tol=snap_tol_m)
        end

        coords = Tuple{Float64,Float64}[
            (float(x), isfinite(y) ? float(y) : NaN) for (x, y) in zip(xgrid, mvals)
        ]

        show_in_legend = legend_extremes_only ? (nq == 1 || j == 1 || j == nq) : true

        lw = (j == nq && nq > 1) ? "0.45pt" : line_width

        plot_pairs = Pair{Any,Any}[
            "line width" => lw,
            "color" => "black!$(shade)",
        ]

        if smooth_curves
            push!(plot_pairs, "smooth" => nothing)
        else
            push!(plot_pairs, "solid" => nothing)
        end

        if !show_in_legend
            push!(plot_pairs, "forget plot" => nothing)
        end

        push!(elems, Plot(PGFPlotsX.Options(plot_pairs...), Coordinates(coords)))

        if show_in_legend
            push!(elems, LegendEntry("\$p01 = $(_fmt_q(qj; digits=q_digits))\$"))
        end
    end

    axis_opts = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin"   => 0,
        "xmax"   => 1,
        "ymin"   => 0,
        "ymax"   => 1,
        "axis equal image" => nothing,

        "xticklabel style" => "{text height=1.8ex, text depth=0.25ex}",
        "tick align" => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick" => xtick_str,
        "xticklabels" => xticklab_str,

        "ytick" => "{0,1}",
        "enlargelimits" => "false",

        "scaled y ticks" => "false",
        "unbounded coords" => "jump",
        "legend pos" => "north west",
    )

    return Axis(axis_opts, elems...)
end

function save_mp_index_vs_x_qs_plot_pdf(outpdf::AbstractString;
                                        qs, ρ, κ, β, r,
                                        engine::AbstractString="pdflatex",
                                        quiet::Bool=true,
                                        keep_tex::Bool=true,
                                        kwargs...)
    ax  = plot_mp_index_vs_x_axis_qs(; qs=qs, ρ=ρ, κ=κ, β=β, r=r, kwargs...)
    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    PGFPlotsX.save(texfile, pic; include_preamble=true)

    cmd = `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                buf = IOBuffer()
                run(pipeline(cmd; stdout=buf, stderr=buf))
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)
        if isfile(logfile)
            lines = readlines(logfile)
            tail  = lines[max(1, length(lines)-80):end]
            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end
        rethrow()
    end

    if pdffile != outpdf_abs
        cp(pdffile, outpdf_abs; force=true)
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            f = joinpath(outdir, base * ext)
            isfile(f) && rm(f; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# parameters (example)
p01  = 0.25
ρ  = 0.6
κ  = 0.8
β  = 0.95
r  = 1.0

qs = [0.001, 0.10, 0.15, 0.20, 0.3, 0.35, 0.399]   # each must satisfy 0 < p01 < 1-rho

save_mp_index_vs_x_qs_plot_pdf("mp_family_q.pdf";
    qs=qs, ρ=ρ, κ=κ, β=β, r=r,
    xgrid=collect(range(0.0, 1.0; length=2001)),
    shade_min=35,
    shade_max=100,
    legend_extremes_only=true)

In [ ]:
# Assumes PGFPlotsX, LaTeXStrings, and the corrected
# _snap_consecutive helper were defined earlier.

"""
Plot the MP index m(x) as a function of p01, for fixed x.

By default:
- p01 is sampled over (p01eps, 1-rho-p01eps);
- the horizontal axis is [0, 1-rho];
- the vertical axis is [0, r];
- the sampled curve is drawn without PGFPlots smoothing.
"""
function plot_mp_index_vs_p01_axis(
    x;
    ρ,
    κ,
    r,
    β,
    ε=1e-10,
    p01grid=nothing,
    p01eps=1e-6,
    np01::Int=2001,
    xlabel=L"$p_{01}$",
    ylabel=L"$m(x)$",
    width="8cm",
    height="7cm",
    line_width="0.40pt",
    snap_tol_m=0.0,
    smooth_curve::Bool=false
)
    xval = Float64(x)
    ρval = Float64(ρ)
    κval = Float64(κ)
    βval = Float64(β)
    rmax = Float64(r)

    isfinite(xval) && 0.0 ≤ xval ≤ 1.0 ||
        throw(ArgumentError("x must satisfy 0 ≤ x ≤ 1"))

    isfinite(ρval) && 0.0 < ρval < 1.0 ||
        throw(ArgumentError("ρ must satisfy 0 < ρ < 1"))

    isfinite(κval) && 0.0 < κval ≤ 1.0 ||
        throw(ArgumentError("κ must satisfy 0 < κ ≤ 1"))

    isfinite(βval) && 0.0 ≤ βval < 1.0 ||
        throw(ArgumentError("β must satisfy 0 ≤ β < 1"))

    isfinite(rmax) && rmax > 0.0 ||
        throw(ArgumentError(
            "r must be finite and strictly positive"
        ))

    snap_tol_m ≥ 0.0 ||
        throw(ArgumentError(
            "snap_tol_m must be nonnegative"
        ))

    p01min_axis = 0.0
    p01max_axis = 1.0 - ρval

    if isnothing(p01grid)
        np01 ≥ 2 ||
            throw(ArgumentError("np01 must be at least 2"))

        p01epsval = Float64(p01eps)

        isfinite(p01epsval) && p01epsval > 0.0 ||
            throw(ArgumentError(
                "p01eps must be finite and strictly positive"
            ))

        p01lo = p01epsval
        p01hi = p01max_axis - p01epsval

        p01lo < p01hi ||
            throw(ArgumentError(
                "p01eps is too large relative to 1-ρ"
            ))

        p01s = collect(
            range(p01lo, p01hi; length=np01)
        )
    else
        p01s = sort!(
            unique!(
                Float64.(collect(p01grid))
            )
        )

        length(p01s) ≥ 2 ||
            throw(ArgumentError(
                "p01grid must contain at least two distinct values"
            ))

        all(
            p01 -> isfinite(p01) &&
                   0.0 < p01 < p01max_axis,
            p01s
        ) ||
            throw(ArgumentError(
                "every value in p01grid must satisfy " *
                "0 < p01 < 1-ρ"
            ))
    end

    # Sample m(x)=f(x,x)/g(x,x) as a function of p01.
    mvals = Vector{Float64}(undef, length(p01s))

    for (i, p01) in enumerate(p01s)
        fxx = Float64(
            f_metric2(
                xval,
                xval;
                p01=p01,
                ρ=ρval,
                κ=κval,
                r=rmax,
                β=βval,
                ε=ε
            )
        )

        gxx = Float64(
            g_metric2(
                xval,
                xval;
                p01=p01,
                ρ=ρval,
                κ=κval,
                β=βval,
                ε=ε
            )
        )

        mvals[i] =
            if isfinite(fxx) && isfinite(gxx) && gxx != 0.0
                fxx / gxx
            else
                NaN
            end
    end

    if snap_tol_m > 0.0
        mvals = _snap_consecutive(
            mvals;
            tol=snap_tol_m
        )
    end

    coords = Tuple{Float64,Float64}[
        (
            p01,
            isfinite(m) ? m : NaN
        )
        for (p01, m) in zip(p01s, mvals)
    ]

    curve_options = Pair{Any,Any}[
        "line width" => line_width,
    ]

    if smooth_curve
        push!(curve_options, "smooth" => nothing)
    else
        push!(curve_options, "solid" => nothing)
    end

    xtick_str =
        "{0,$(p01max_axis)}"

    phantom =
        "\\vphantom{\\ensuremath{1-\\rho}}"

    xticklabel_str =
        "{" *
        phantom * "\\ensuremath{0}," *
        phantom * "\\ensuremath{1-\\rho}" *
        "}"

    axis_options = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin" => p01min_axis,
        "xmax" => p01max_axis,
        "ymin" => 0,
        "ymax" => rmax,

        "xticklabel style" =>
            "{text height=1.8ex, text depth=0.25ex}",
        "tick align"        => "outside",
        "major tick length" => "2pt",
        "minor tick length" => "1pt",

        "xtick"       => xtick_str,
        "xticklabels" => xticklabel_str,

        "ytick"       => "{0,$(rmax)}",
        "yticklabels" =>
            "{\\ensuremath{0},\\ensuremath{r}}",

        "enlargelimits"    => "false",
        "scaled x ticks"   => "false",
        "scaled y ticks"   => "false",
        "unbounded coords" => "jump",
    )

    return Axis(
        axis_options,
        Plot(
            PGFPlotsX.Options(curve_options...),
            Coordinates(coords)
        )
    )
end


function save_mp_index_vs_p01_plot_pdf(
    outpdf::AbstractString,
    x;
    ρ,
    κ,
    r,
    β,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)
    ax = plot_mp_index_vs_p01_axis(
        x;
        ρ=ρ,
        κ=κ,
        r=r,
        β=β,
        kwargs...
    )

    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)
    outdir = dirname(outpdf_abs)
    base = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    isdir(outdir) || mkpath(outdir)

    PGFPlotsX.save(
        texfile,
        pic;
        include_preamble=true
    )

    cmd =
        `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                output_buffer = IOBuffer()

                run(
                    pipeline(
                        cmd;
                        stdout=output_buffer,
                        stderr=output_buffer
                    )
                )
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            first_line = max(1, length(lines) - 80)
            tail = lines[first_line:end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    isfile(pdffile) ||
        error("LaTeX completed without producing $(pdffile)")

    if pdffile != outpdf_abs
        cp(
            pdffile,
            outpdf_abs;
            force=true
        )
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end

In [ ]:
# Fixed parameters
ρ = 0.60
κ = 0.80
r = 1.00
β = 0.95

# Reference value used only to select the fixed state x
p01_ref = 0.25

x0_ref, x1_ref, _ = fixed_points(
    ;
    p01=p01_ref,
    ρ=ρ,
    κ=κ
)

# Select one fixed state
x = 0.5 * (x1_ref + x0_ref)

# Alternative choices:
# x = 0.5 * p01_ref
# x = 0.5 * (p01_ref + x1_ref)
# x = 0.5 * (x0_ref + p01_ref + ρ)
# x = 0.5 * (p01_ref + ρ + 1.0)

save_mp_index_vs_p01_plot_pdf(
    "mp_index_vs_p01.pdf",
    x;
    ρ=ρ,
    κ=κ,
    r=r,
    β=β
)

In [ ]:
"""
    theta_diff_at_x(x, z; p01, ρ, κ, β, ε=1e-10)

Compute the truncated approximation

    ΔΘ̃(x,z) = Θ̃(φ⁰(x),z) - Θ̃(φ¹(x),z).

The tolerance ε is divided equally between the two evaluations, so
the resulting truncation-error bound for the difference is at most ε.

Returns

    (ΔΘhat, Θ1hat, Θ0hat, phi1x, phi0x, info)
"""
function theta_diff_at_x(
    x,
    z;
    p01,
    ρ,
    κ,
    β,
    ε::Real=1e-10
)
    εval = Float64(ε)

    isfinite(εval) && εval > 0.0 ||
        throw(ArgumentError(
            "ε must be finite and strictly positive"
        ))

    εbranch = εval / 2.0

    phi1x = φ1(
        x;
        p01=p01,
        ρ=ρ,
        κ=κ
    )

    phi0x = φ0(
        x;
        p01=p01,
        ρ=ρ
    )

    Θ1hat, T1, tailΘ1 = tildeTheta(
        phi1x,
        z;
        p01=p01,
        ρ=ρ,
        κ=κ,
        β=β,
        ε=εbranch
    )

    Θ0hat, T0, tailΘ0 = tildeTheta(
        phi0x,
        z;
        p01=p01,
        ρ=ρ,
        κ=κ,
        β=β,
        ε=εbranch
    )

    ΔΘhat = Θ0hat - Θ1hat

    # Guaranteed truncation-error bound:
    # |ΔΘ̃ - ΔΘhat| ≤ tailΘ0 + tailΘ1.
    tailΔ = tailΘ0 + tailΘ1

    lowerΔ = ΔΘhat - tailΔ
    upperΔ = ΔΘhat + tailΔ

    certified_sign =
        if lowerΔ > 0.0
            :positive
        elseif upperΔ < 0.0
            :negative
        else
            :undetermined
        end

    info = (
        T1=T1,
        T0=T0,
        tailΘ1=tailΘ1,
        tailΘ0=tailΘ0,
        tailΔ=tailΔ,
        lowerΔ=lowerΔ,
        upperΔ=upperΔ,
        certified_sign=certified_sign
    )

    return (
        ΔΘhat,
        Θ1hat,
        Θ0hat,
        phi1x,
        phi0x,
        info
    )
end


# Example diagnostic
p01 = 0.25
ρ   = 0.60
κ   = 0.80
β   = 0.95
ε   = 1e-10

x0_fp, x1_fp, _ = fixed_points(
    ;
    p01=p01,
    ρ=ρ,
    κ=κ
)

z = 0.5 * (x1_fp + x0_fp)
x = z + 0.10

ΔΘhat, Θ1hat, Θ0hat, phi1x, phi0x, info =
    theta_diff_at_x(
        x,
        z;
        p01=p01,
        ρ=ρ,
        κ=κ,
        β=β,
        ε=ε
    )

@show phi0x phi1x
@show Θ0hat Θ1hat ΔΘhat
@show info.tailΔ info.lowerΔ info.upperΔ
@show info.certified_sign

In [ ]:
# Assumes PGFPlotsX, LaTeXStrings, and theta_diff_at_x
# have already been defined.

"""
    plot_theta_diff_vs_z(
        x;
        p01, ρ, κ, β,
        ε=1e-10,
        nz=201,
        ...
    )

Return an axis plotting

    ΔΘ̃(x,z)
      = Θ̃(φ⁰(x),z) - Θ̃(φ¹(x),z)

as a function of z ∈ [0,1].

The horizontal dashed line marks zero. No grid or title is
included by default.
"""
function plot_theta_diff_vs_z(
    x;
    p01,
    ρ,
    κ,
    β,
    ε::Real=1e-10,
    nz::Int=201,
    width::AbstractString="12cm",
    height::AbstractString="7cm",
    line_width::AbstractString="0.9pt",
    mark::AbstractString="none",
    xlabel=L"$z$",
    ylabel=L"$\widetilde{\Theta}(\phi^0(x),z)
              -\widetilde{\Theta}(\phi^1(x),z)$"
)
    xval   = Float64(x)
    p01val = Float64(p01)
    ρval   = Float64(ρ)
    κval   = Float64(κ)
    βval   = Float64(β)
    εval   = Float64(ε)

    isfinite(xval) && 0.0 ≤ xval ≤ 1.0 ||
        throw(ArgumentError(
            "x must satisfy 0 ≤ x ≤ 1"
        ))

    isfinite(p01val) && 0.0 < p01val < 1.0 ||
        throw(ArgumentError(
            "p01 must satisfy 0 < p01 < 1"
        ))

    isfinite(ρval) && 0.0 < ρval < 1.0 - p01val ||
        throw(ArgumentError(
            "ρ must satisfy 0 < ρ < 1-p01"
        ))

    isfinite(κval) && 0.0 < κval ≤ 1.0 ||
        throw(ArgumentError(
            "κ must satisfy 0 < κ ≤ 1"
        ))

    isfinite(βval) && 0.0 ≤ βval < 1.0 ||
        throw(ArgumentError(
            "β must satisfy 0 ≤ β < 1"
        ))

    isfinite(εval) && εval > 0.0 ||
        throw(ArgumentError(
            "ε must be finite and strictly positive"
        ))

    nz ≥ 2 ||
        throw(ArgumentError(
            "nz must be at least 2"
        ))

    zgrid = collect(
        range(0.0, 1.0; length=nz)
    )

    ΔΘvals = Vector{Float64}(undef, nz)

    for (i, z) in enumerate(zgrid)
        ΔΘhat, _, _, _, _, _ =
            theta_diff_at_x(
                xval,
                z;
                p01=p01val,
                ρ=ρval,
                κ=κval,
                β=βval,
                ε=εval
            )

        ΔΘvals[i] =
            isfinite(ΔΘhat) ? Float64(ΔΘhat) : NaN
    end

    coords = Tuple{Float64,Float64}[
        (z, ΔΘ)
        for (z, ΔΘ) in zip(zgrid, ΔΘvals)
    ]

    zero_line = Plot(
        PGFPlotsX.Options(
            "color"       => "black!40",
            "dashed"      => nothing,
            "line width"  => "0.35pt",
            "forget plot" => nothing
        ),
        Coordinates([
            (0.0, 0.0),
            (1.0, 0.0)
        ])
    )

    difference_curve = Plot(
        PGFPlotsX.Options(
            "mark"       => mark,
            "line width" => line_width
        ),
        Coordinates(coords)
    )

    axis_options = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,
        "xlabel" => xlabel,
        "ylabel" => ylabel,

        "xmin" => 0,
        "xmax" => 1,

        "tick align"        => "outside",
        "major tick length" => "2pt",

        "enlargelimits"    => "false",
        "scaled x ticks"   => "false",
        "scaled y ticks"   => "false",
        "unbounded coords" => "jump"
    )

    return Axis(
        axis_options,
        zero_line,
        difference_curve
    )
end


"""
    save_theta_diff_vs_z_pdf(
        outpdf,
        x;
        p01, ρ, κ, β,
        ...
    )

Plot ΔΘ̃(x,z) over z ∈ [0,1], save the standalone LaTeX
source, and compile it to PDF.

Returns the absolute path of the resulting PDF.
"""
function save_theta_diff_vs_z_pdf(
    outpdf::AbstractString,
    x;
    p01,
    ρ,
    κ,
    β,
    engine::AbstractString="pdflatex",
    quiet::Bool=true,
    keep_tex::Bool=true,
    kwargs...
)
    ax = plot_theta_diff_vs_z(
        x;
        p01=p01,
        ρ=ρ,
        κ=κ,
        β=β,
        kwargs...
    )

    pic = TikzPicture(ax)

    outpdf_abs = abspath(outpdf)

    lowercase(splitext(outpdf_abs)[2]) == ".pdf" ||
        throw(ArgumentError(
            "outpdf must have extension .pdf"
        ))

    outdir = dirname(outpdf_abs)
    base   = splitext(basename(outpdf_abs))[1]

    texfile = joinpath(outdir, base * ".tex")
    logfile = joinpath(outdir, base * ".log")
    pdffile = joinpath(outdir, base * ".pdf")

    isdir(outdir) || mkpath(outdir)

    PGFPlotsX.save(
        texfile,
        pic;
        include_preamble=true
    )

    cmd =
        `$(engine) -interaction=nonstopmode -halt-on-error $(basename(texfile))`

    try
        cd(outdir) do
            if quiet
                output_buffer = IOBuffer()

                run(
                    pipeline(
                        cmd;
                        stdout=output_buffer,
                        stderr=output_buffer
                    )
                )
            else
                run(cmd)
            end
        end
    catch
        println("\nLaTeX failed running: ", cmd)

        if isfile(logfile)
            lines = readlines(logfile)
            first_line = max(1, length(lines) - 80)
            tail = lines[first_line:end]

            println("\n---- last lines of $(logfile) ----")
            println(join(tail, "\n"))
            println("---- end log tail ----\n")
        end

        rethrow()
    end

    isfile(pdffile) ||
        error(
            "LaTeX completed without producing $(pdffile)"
        )

    if pdffile != outpdf_abs
        cp(
            pdffile,
            outpdf_abs;
            force=true
        )
    end

    if !keep_tex
        for ext in (".tex", ".aux", ".log")
            file = joinpath(outdir, base * ext)
            isfile(file) && rm(file; force=true)
        end
    end

    return outpdf_abs
end


# Example
p01 = 0.25
ρ   = 0.60
κ   = 0.80
β   = 0.25
x   = 0.60

save_theta_diff_vs_z_pdf(
    "theta_diff_vs_z.pdf",
    x;
    p01=p01,
    ρ=ρ,
    κ=κ,
    β=β,
    ε=1e-10,
    nz=3001
)

In [ ]:
# Assumes PGFPlotsX and the following functions were defined earlier:
#   φ0, φ1, varphi_t, fixed_points

# Julia vectors -> explicit TeX lists, avoiding PGFPlotsX tick bugs.
function tex_num_list(xs::AbstractVector{<:Real})
    isempty(xs) && return "{}"

    values = (
        string(round(Float64(x); sigdigits=15))
        for x in xs
    )

    return "{" * join(values, ",") * "}"
end

function tex_label_list(
    labels::AbstractVector{<:AbstractString}
)
    isempty(labels) && return "{}"
    return "{" * join(labels, ",") * "}"
end


"""
    plot_varphi_t_pgf(
        t, z;
        p01, ρ, κ,
        nx=2001,
        refine_iters=60,
        ...
    )

Plot

    x ↦ varphi_t(x,z)

over x ∈ [0,1].

The threshold convention is passive at equality:

    y ≤ z  ->  φ⁰(y),
    y > z  ->  φ¹(y).

Consequently, the iterate is represented as left-continuous at its
breakpoints: the filled circle gives the left-continuous value and the
open circle gives the right-hand limit.

Vertical jump lines are suppressed. Both axes show ticks at

    0, p01, x¹, x⁰, p01+ρ, 1,

together with z when it is distinct from those values.
"""
function plot_varphi_t_pgf(
    t::Integer,
    z;
    p01,
    ρ,
    κ,
    nx::Int=2001,
    refine_iters::Int=60,
    max_breaks::Int=100_000,
    width::AbstractString="10cm",
    height::AbstractString="10cm",
    curve_lw::AbstractString="0.35pt",
    jump_mark_size::AbstractString="0.9pt",
    diag_lw::AbstractString="0.25pt",
    jump_tol::Real=1e-12
)
    t ≥ 0 ||
        throw(ArgumentError(
            "t must be nonnegative"
        ))

    nx ≥ 3 ||
        throw(ArgumentError(
            "nx must be at least 3"
        ))

    refine_iters ≥ 1 ||
        throw(ArgumentError(
            "refine_iters must be positive"
        ))

    max_breaks ≥ 1 ||
        throw(ArgumentError(
            "max_breaks must be positive"
        ))

    zval   = Float64(z)
    p01val = Float64(p01)
    ρval   = Float64(ρ)
    κval   = Float64(κ)
    jtval  = Float64(jump_tol)

    isfinite(zval) && 0.0 ≤ zval ≤ 1.0 ||
        throw(ArgumentError(
            "z must satisfy 0 ≤ z ≤ 1"
        ))

    isfinite(p01val) && 0.0 < p01val < 1.0 ||
        throw(ArgumentError(
            "p01 must satisfy 0 < p01 < 1"
        ))

    isfinite(ρval) && 0.0 < ρval < 1.0 - p01val ||
        throw(ArgumentError(
            "ρ must satisfy 0 < ρ < 1-p01"
        ))

    isfinite(κval) && 0.0 < κval < 1.0 ||
        throw(ArgumentError(
            "κ must satisfy 0 < κ < 1"
        ))

    isfinite(jtval) && jtval ≥ 0.0 ||
        throw(ArgumentError(
            "jump_tol must be finite and nonnegative"
        ))

    @inline function step_map(y)
        if y > zval
            return φ1(
                y;
                p01=p01val,
                ρ=ρval,
                κ=κval
            )
        else
            return φ0(
                y;
                p01=p01val,
                ρ=ρval
            )
        end
    end

    # Earliest time at which the two initial states choose
    # different actions.
    function first_diff_step(xleft, xright)
        t == 0 && return nothing

        yleft  = Float64(xleft)
        yright = Float64(xright)

        for k in 0:(t - 1)
            aleft  = yleft > zval
            aright = yright > zval

            if aleft != aright
                return k
            end

            yleft  = Float64(step_map(yleft))
            yright = Float64(step_map(yright))
        end

        return nothing
    end

    # Locate the root of varphi_k(x,z)-z, maintaining
    #
    #     f(xbelow) ≤ 0 < f(xabove).
    function locate_break(xa, xb, k::Int)
        f(x) =
            Float64(
                varphi_t(
                    x,
                    zval,
                    k;
                    p01=p01val,
                    ρ=ρval,
                    κ=κval
                )
            ) - zval

        xbelow = Float64(xa)
        xabove = Float64(xb)

        fbelow = f(xbelow)
        fabove = f(xabove)

        if !(
            isfinite(fbelow) &&
            isfinite(fabove) &&
            fbelow ≤ 0.0 &&
            fabove > 0.0
        )
            error(
                "Could not bracket a threshold breakpoint " *
                "at iteration k=$k."
            )
        end

        for _ in 1:refine_iters
            xmid = (xbelow + xabove) / 2.0

            # Stop once floating-point bisection cannot refine further.
            if xmid == xbelow || xmid == xabove
                break
            end

            if f(xmid) ≤ 0.0
                xbelow = xmid
            else
                xabove = xmid
            end
        end

        xstar = (xbelow + xabove) / 2.0

        return xbelow, xabove, xstar
    end

    # Each entry is (x*, left limit, right limit).
    breaks = Tuple{Float64,Float64,Float64}[]

    # Recursion is needed because one initial interval may contain
    # several breakpoints generated at different iteration times.
    function collect_breaks!(xa, xb)
        xb > xa || return

        k = first_diff_step(xa, xb)
        k === nothing && return

        xbelow, xabove, xstar =
            locate_break(xa, xb, k)

        yleft = Float64(
            varphi_t(
                xbelow,
                zval,
                t;
                p01=p01val,
                ρ=ρval,
                κ=κval
            )
        )

        yright = Float64(
            varphi_t(
                xabove,
                zval,
                t;
                p01=p01val,
                ρ=ρval,
                κ=κval
            )
        )

        length(breaks) < max_breaks ||
            error(
                "More than $max_breaks breakpoints were detected. " *
                "Reduce t or increase max_breaks."
            )

        push!(
            breaks,
            (xstar, yleft, yright)
        )

        # Search on both sides using the certified one-sided
        # bisection points.
        xa < xbelow &&
            collect_breaks!(xa, xbelow)

        xabove < xb &&
            collect_breaks!(xabove, xb)

        return
    end

    if t ≥ 1
        collect_breaks!(0.0, 1.0)
        sort!(breaks; by=first)
    end

    # Internal breakpoints determine the continuous plotting pieces.
    internal_breaks = [
        b for b in breaks
        if 0.0 < b[1] < 1.0
    ]

    edges = Float64[
        0.0;
        first.(internal_breaks);
        1.0
    ]

    xline = Float64[]
    yline = Float64[]

    # A possible jump at x=0 needs separate treatment because
    # there is no positive-length piece on its left.
    zero_break_index =
        findfirst(b -> b[1] == 0.0, breaks)

    first_piece = true

    for s in 1:(length(edges) - 1)
        xa = edges[s]
        xb = edges[s + 1]

        xb > xa || continue

        # Allocate sampling points approximately in proportion
        # to the length of the continuous piece.
        nsegment = max(
            2,
            ceil(Int, (nx - 1) * (xb - xa)) + 1
        )

        xsegment = collect(
            range(xa, xb; length=nsegment)
        )

        ysegment = Float64[
            varphi_t(
                xv,
                zval,
                t;
                p01=p01val,
                ρ=ρval,
                κ=κval
            )
            for xv in xsegment
        ]

        # Replace numerical evaluations at breakpoints by the
        # appropriate one-sided limits.
        if s == 1
            if zero_break_index !== nothing
                ysegment[1] =
                    breaks[zero_break_index][3]
            end
        else
            ysegment[1] =
                internal_breaks[s - 1][3]
        end

        if s ≤ length(internal_breaks)
            ysegment[end] =
                internal_breaks[s][2]
        end

        if !first_piece
            push!(xline, NaN)
            push!(yline, NaN)
        end

        append!(xline, xsegment)
        append!(yline, ysegment)

        first_piece = false
    end

    # Jump markers: filled at the left-continuous value and
    # open at the right-hand limit.
    xclosed = Float64[]
    yclosed = Float64[]
    xopen   = Float64[]
    yopen   = Float64[]

    for (xstar, yleft, yright) in breaks
        if (
            isfinite(yleft) &&
            isfinite(yright) &&
            abs(yleft - yright) > jtval
        )
            push!(xclosed, xstar)
            push!(yclosed, yleft)

            push!(xopen, xstar)
            push!(yopen, yright)
        end
    end

    # Ticks on both axes.
    x0_fp, x1_fp, _ = fixed_points(
        ;
        p01=p01val,
        ρ=ρval,
        κ=κval
    )

    tick_tol = 1e-12
    tick_items = Tuple{Float64,String}[]

    function add_tick!(value, label)
        v = Float64(value)

        isfinite(v) ||
            error("Encountered a nonfinite tick position.")

        -tick_tol ≤ v ≤ 1.0 + tick_tol ||
            error(
                "Tick position $v lies outside [0,1]."
            )

        v = clamp(v, 0.0, 1.0)

        if all(
            abs(v - old_value) > tick_tol
            for (old_value, _) in tick_items
        )
            push!(tick_items, (v, label))
        end

        return
    end

    add_tick!(0.0,          raw"$0$")
    add_tick!(p01val,       raw"$p_{01}$")
    add_tick!(x1_fp,        raw"$x^1$")
    add_tick!(x0_fp,        raw"$x^0$")
    add_tick!(p01val + ρval, raw"$p_{01}+\rho$")
    add_tick!(1.0,          raw"$1$")

    # If z coincides with an existing distinguished point,
    # retain the existing label.
    add_tick!(zval, raw"$z$")

    sort!(tick_items; by=first)

    tick_positions = first.(tick_items)
    tick_labels    = last.(tick_items)

    tick_pos_tex = tex_num_list(tick_positions)
    tick_lab_tex = tex_label_list(tick_labels)

    diagonal = Plot(
        PGFPlotsX.Options(
            "no marks"   => nothing,
            "color"      => "gray",
            "dashed"     => nothing,
            "line width" => diag_lw
        ),
        Coordinates([
            (0.0, 0.0),
            (1.0, 1.0)
        ])
    )

    iterate_curve = Plot(
        PGFPlotsX.Options(
            "no marks"         => nothing,
            "color"            => "black",
            "line width"       => curve_lw,
            "unbounded coords" => "jump"
        ),
        Table(
            x=xline,
            y=yline
        )
    )

    axis_contents = Any[
        diagonal,
        iterate_curve
    ]

    if !isempty(xclosed)
        push!(
            axis_contents,
            Plot(
                PGFPlotsX.Options(
                    "only marks" => nothing,
                    "mark"       => "*",
                    "mark size"  => jump_mark_size,
                    "color"      => "black"
                ),
                Table(
                    x=xclosed,
                    y=yclosed
                )
            )
        )

        push!(
            axis_contents,
            Plot(
                PGFPlotsX.Options(
                    "only marks"   => nothing,
                    "mark"         => "o",
                    "mark size"    => jump_mark_size,
                    "color"        => "black",
                    "mark options" => "{fill=white}"
                ),
                Table(
                    x=xopen,
                    y=yopen
                )
            )
        )
    end

    axis_options = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,

        "xmin" => 0,
        "xmax" => 1,
        "ymin" => 0,
        "ymax" => 1,

        "axis equal image"  => nothing,
        "axis on top"       => "false",
        "clip marker paths" => "false",
        "grid"              => "none",

        "xlabel" => raw"$x$",
        "ylabel" => raw"$\varphi_t(x,z)$",

        "xtick"       => tick_pos_tex,
        "xticklabels" => tick_lab_tex,
        "ytick"       => tick_pos_tex,
        "yticklabels" => tick_lab_tex,

        "tick align"        => "outside",
        "major tick length" => "2pt",

        "scaled x ticks"   => "false",
        "scaled y ticks"   => "false",
        "enlargelimits"    => "false",
        "unbounded coords" => "jump"
    )

    axis = Axis(
        axis_options,
        axis_contents...
    )

    return TikzPicture(axis)
end

In [ ]:
# Illustrative iterate plot in the intermediate-threshold regime.

p01 = 0.25
ρ   = 0.60
κ   = 0.80

x0_fp, x1_fp, _ = fixed_points(
    ;
    p01=p01,
    ρ=ρ,
    κ=κ
)

# Midpoint of the intermediate regime x¹ < z < x⁰.
z_plot = 0.5 * (x1_fp + x0_fp)
t_plot = 100

fig = plot_varphi_t_pgf(
    t_plot,
    z_plot;
    p01=p01,
    ρ=ρ,
    κ=κ,
    nx=2501
)

display(fig)

In [ ]:
# Assumes the following were defined earlier:
#   φ0, φ1, fixed_points, tex_num_list, tex_label_list

"""
    plot_varphi_t_vs_z_pgf(
        t, x;
        p01, ρ, κ,
        refine_iters=60,
        max_breaks=100_000,
        ...
    )

Plot

    z ↦ varphi_t(x,z)

over z ∈ [0,1] for fixed x.

The threshold convention is

    y ≤ z  ->  φ⁰(y),
    y > z  ->  φ¹(y).

Hence the function is right-continuous in z. At each jump:

- the filled circle gives the value at the breakpoint;
- the white-filled circle gives the left-hand limit.

The returned `segments` record the threshold interval, action word,
and constant value on each piece.
"""
function plot_varphi_t_vs_z_pgf(
    t::Integer,
    x;
    p01,
    ρ,
    κ,
    refine_iters::Int=60,
    max_breaks::Int=100_000,
    width::AbstractString="10cm",
    height::AbstractString="10cm",
    seg_lw::AbstractString="0.50pt",
    diag_lw::AbstractString="0.25pt",
    mark_size::AbstractString="1.3pt",
    jump_tol::Real=1e-12
)
    t ≥ 0 ||
        throw(ArgumentError(
            "t must be nonnegative"
        ))

    refine_iters ≥ 1 ||
        throw(ArgumentError(
            "refine_iters must be positive"
        ))

    max_breaks ≥ 1 ||
        throw(ArgumentError(
            "max_breaks must be positive"
        ))

    xval   = Float64(x)
    p01val = Float64(p01)
    ρval   = Float64(ρ)
    κval   = Float64(κ)
    jtval  = Float64(jump_tol)

    isfinite(xval) && 0.0 ≤ xval ≤ 1.0 ||
        throw(ArgumentError(
            "x must satisfy 0 ≤ x ≤ 1"
        ))

    isfinite(p01val) && 0.0 < p01val < 1.0 ||
        throw(ArgumentError(
            "p01 must satisfy 0 < p01 < 1"
        ))

    isfinite(ρval) && 0.0 < ρval < 1.0 - p01val ||
        throw(ArgumentError(
            "ρ must satisfy 0 < ρ < 1-p01"
        ))

    isfinite(κval) && 0.0 < κval < 1.0 ||
        throw(ArgumentError(
            "κ must satisfy 0 < κ < 1"
        ))

    isfinite(jtval) && jtval ≥ 0.0 ||
        throw(ArgumentError(
            "jump_tol must be finite and nonnegative"
        ))

    """
    Return the length-t action word and the resulting belief at
    threshold z.
    """
    function word_and_value(z)
        zval = Float64(z)
        y = xval
        buffer = IOBuffer()

        for _ in 1:t
            if y > zval
                print(buffer, '1')

                y = Float64(
                    φ1(
                        y;
                        p01=p01val,
                        ρ=ρval,
                        κ=κval
                    )
                )
            else
                print(buffer, '0')

                y = Float64(
                    φ0(
                        y;
                        p01=p01val,
                        ρ=ρval
                    )
                )
            end
        end

        return String(take!(buffer)), y
    end

    """
    Starting at zleft, locate the first point to its right at which
    the action word differs from wleft.

    The action word is lexicographically nonincreasing as z increases:
    at the first differing action, the lower threshold chooses 1 and
    the higher threshold chooses 0. Thus the set on which the word
    equals wleft is an interval, making bisection valid.
    """
    function locate_next_change(zleft, wleft)
        zlo = Float64(zleft)
        zhi = 1.0

        word_hi, _ = word_and_value(zhi)

        word_hi != wleft ||
            error(
                "No itinerary change is bracketed to the right."
            )

        for _ in 1:refine_iters
            zmid = (zlo + zhi) / 2.0

            # Floating-point resolution has been exhausted.
            if zmid == zlo || zmid == zhi
                break
            end

            word_mid, _ = word_and_value(zmid)

            if word_mid == wleft
                zlo = zmid
            else
                zhi = zmid
            end
        end

        word_right, value_right =
            word_and_value(zhi)

        return zhi, word_right, value_right
    end

    segments = NamedTuple[]

    z_open = Float64[]
    y_open = Float64[]

    z_fill = Float64[]
    y_fill = Float64[]

    zleft = 0.0
    word_current, value_current =
        word_and_value(zleft)

    word_at_one, _ =
        word_and_value(1.0)

    while word_current != word_at_one
        length(segments) < max_breaks ||
            error(
                "More than $max_breaks itinerary changes were " *
                "detected. Increase max_breaks or reduce t."
            )

        zbreak, word_right, value_right =
            locate_next_change(
                zleft,
                word_current
            )

        push!(
            segments,
            (
                zL=zleft,
                zR=zbreak,
                word=word_current,
                value=value_current
            )
        )

        if abs(value_current - value_right) > jtval
            # Left-hand limit: open/white marker.
            push!(z_open, zbreak)
            push!(y_open, value_current)

            # Right-continuous value at the breakpoint.
            push!(z_fill, zbreak)
            push!(y_fill, value_right)
        end

        zleft        = zbreak
        word_current = word_right
        value_current = value_right
    end

    push!(
        segments,
        (
            zL=zleft,
            zR=1.0,
            word=word_current,
            value=value_current
        )
    )

    # Horizontal pieces with NaN separators to suppress vertical lines.
    xline = Float64[]
    yline = Float64[]

    for (i, segment) in enumerate(segments)
        if i > 1
            push!(xline, NaN)
            push!(yline, NaN)
        end

        push!(xline, segment.zL)
        push!(yline, segment.value)

        push!(xline, segment.zR)
        push!(yline, segment.value)
    end

    # Distinguished ticks on both axes.
    x0_fp, x1_fp, _ = fixed_points(
        ;
        p01=p01val,
        ρ=ρval,
        κ=κval
    )

    tick_tol = 1e-12
    tick_items = Tuple{Float64,String}[]

    function add_tick!(value, label)
        v = Float64(value)

        isfinite(v) ||
            error(
                "Encountered a nonfinite tick position."
            )

        -tick_tol ≤ v ≤ 1.0 + tick_tol ||
            error(
                "Tick position $v lies outside [0,1]."
            )

        v = clamp(v, 0.0, 1.0)

        if all(
            abs(v - old_value) > tick_tol
            for (old_value, _) in tick_items
        )
            push!(tick_items, (v, label))
        end

        return
    end

    add_tick!(0.0,           raw"\ensuremath{0}")
    add_tick!(p01val,        raw"\ensuremath{p_{01}}")
    add_tick!(x1_fp,         raw"\ensuremath{x^{1}}")
    add_tick!(x0_fp,         raw"\ensuremath{x^{0}}")
    add_tick!(p01val + ρval, raw"\ensuremath{p_{11}}")
    add_tick!(1.0,           raw"\ensuremath{1}")

    # If x coincides with a distinguished point, retain its existing label.
    add_tick!(xval, raw"\ensuremath{x}")

    sort!(tick_items; by=first)

    tick_positions = first.(tick_items)
    tick_labels    = last.(tick_items)

    tick_pos_tex = tex_num_list(tick_positions)
    tick_lab_tex = tex_label_list(tick_labels)

    diagonal = Plot(
        PGFPlotsX.Options(
            "no marks"   => nothing,
            "color"      => "gray",
            "dashed"     => nothing,
            "line width" => diag_lw
        ),
        Coordinates([
            (0.0, 0.0),
            (1.0, 1.0)
        ])
    )

    step_curve = Plot(
        PGFPlotsX.Options(
            "no marks"         => nothing,
            "color"            => "black",
            "line width"       => seg_lw,
            "unbounded coords" => "jump"
        ),
        Table(
            x=xline,
            y=yline
        )
    )

    axis_contents = Any[
        diagonal,
        step_curve
    ]

    if !isempty(z_fill)
        push!(
            axis_contents,
            Plot(
                PGFPlotsX.Options(
                    "only marks" => nothing,
                    "mark"       => "*",
                    "mark size"  => mark_size,
                    "color"      => "black"
                ),
                Table(
                    x=z_fill,
                    y=y_fill
                )
            )
        )

        # Draw open markers last so their white fill hides the line.
        push!(
            axis_contents,
            Plot(
                PGFPlotsX.Options(
                    "only marks"   => nothing,
                    "mark"         => "*",
                    "mark size"    => mark_size,
                    "color"        => "black",
                    "mark options" =>
                        "{fill=white,draw=black,line width=0.25pt}"
                ),
                Table(
                    x=z_open,
                    y=y_open
                )
            )
        )
    end

    axis_options = PGFPlotsX.Options(
        "width"  => width,
        "height" => height,

        "xmin" => 0,
        "xmax" => 1,
        "ymin" => 0,
        "ymax" => 1,

        "axis equal image"  => nothing,
        "axis on top"       => "false",
        "clip marker paths" => "false",
        "grid"              => "none",

        "xlabel" => raw"$z$",
        "ylabel" => raw"$\varphi_t(x,z)$",

        "xtick"       => tick_pos_tex,
        "xticklabels" => tick_lab_tex,
        "ytick"       => tick_pos_tex,
        "yticklabels" => tick_lab_tex,

        "xticklabel style" =>
            "{text height=1.6ex,text depth=0.3ex}",
        "yticklabel style" =>
            "{text height=1.6ex,text depth=0.3ex}",

        "tick align"        => "outside",
        "major tick length" => "2pt",

        "scaled x ticks"   => "false",
        "scaled y ticks"   => "false",
        "enlargelimits"    => "false",
        "unbounded coords" => "jump"
    )

    axis = Axis(
        axis_options,
        axis_contents...
    )

    return TikzPicture(axis), segments
end

In [ ]:
# Illustrative dependence on the threshold z.

p01 = 0.25
ρ   = 0.60
κ   = 0.80

x0_fp, x1_fp, _ = fixed_points(
    ;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x_plot = 0.5 * (x1_fp + x0_fp)
t_plot = 4

fig, segments = plot_varphi_t_vs_z_pgf(
    t_plot,
    x_plot;
    p01=p01,
    ρ=ρ,
    κ=κ,
    mark_size="1.2pt"
)

display(fig)

# Save the figure in the paperPCLsensingErrors folder.
pdf_filename = "varphi_t_vs_z_t4_xmid.pdf"
pdf_path = joinpath(pwd(), pdf_filename)

pgfsave(pdf_path, fig)

println("PDF saved as:")
println(pdf_path)

In [ ]:
# Assumes the revised function
#
#     plot_varphi_t_vs_z_pgf(...)
#
# and the helpers tex_num_list and tex_label_list were defined earlier.


"""
    axis_varphi_t_vs_z_pgf(t, x; p01, ρ, κ, ...)

Construct one Axis for

    z ↦ varphi_t(x,z).

The exact constant pieces are obtained from
`plot_varphi_t_vs_z_pgf`; no sampling grid is used.
"""
function axis_varphi_t_vs_z_pgf(
    t::Integer,
    x;
    p01,
    ρ,
    κ,
    refine_iters::Int=60,
    max_breaks::Int=100_000,
    width::AbstractString="8cm",
    height::AbstractString="8cm",
    seg_lw::AbstractString="0.50pt",
    diag_lw::AbstractString="0.25pt",
    mark_size::AbstractString="1.3pt",
    jump_tol::Real=1e-12,
    show_ylabel::Bool=true,
    show_yticklabels::Bool=true
)
    # Reuse the exact breakpoint and segment computation from the
    # preceding single-panel plotter.
    _, segments = plot_varphi_t_vs_z_pgf(
        t,
        x;
        p01=p01,
        ρ=ρ,
        κ=κ,
        refine_iters=refine_iters,
        max_breaks=max_breaks,
        width=width,
        height=height,
        seg_lw=seg_lw,
        diag_lw=diag_lw,
        mark_size=mark_size,
        jump_tol=jump_tol
    )

    # Horizontal pieces separated by NaNs, so no vertical jump
    # lines are drawn.
    xline = Float64[]
    yline = Float64[]

    for (i, segment) in enumerate(segments)
        if i > 1
            push!(xline, NaN)
            push!(yline, NaN)
        end

        push!(xline, segment.zL)
        push!(yline, segment.value)

        push!(xline, segment.zR)
        push!(yline, segment.value)
    end

    # Jump markers:
    #   open/white = left-hand limit;
    #   filled     = right-continuous value.
    z_open = Float64[]
    y_open = Float64[]

    z_fill = Float64[]
    y_fill = Float64[]

    jtval = Float64(jump_tol)

    for i in 1:(length(segments) - 1)
        zbreak = Float64(segments[i].zR)
        yleft  = Float64(segments[i].value)
        yright = Float64(segments[i + 1].value)

        if abs(yleft - yright) > jtval
            push!(z_open, zbreak)
            push!(y_open, yleft)

            push!(z_fill, zbreak)
            push!(y_fill, yright)
        end
    end

    # Distinguished ticks on both axes.
    p01val = Float64(p01)
    ρval   = Float64(ρ)
    κval   = Float64(κ)
    xval   = Float64(x)

    x0_fp, x1_fp, _ = fixed_points(
        ;
        p01=p01val,
        ρ=ρval,
        κ=κval
    )

    tick_tol = 1e-12
    tick_items = Tuple{Float64,String}[]

    function add_tick!(value, label)
        v = Float64(value)

        isfinite(v) ||
            error("Encountered a nonfinite tick position.")

        -tick_tol ≤ v ≤ 1.0 + tick_tol ||
            error("Tick position $v lies outside [0,1].")

        v = clamp(v, 0.0, 1.0)

        if all(
            abs(v - old_value) > tick_tol
            for (old_value, _) in tick_items
        )
            push!(tick_items, (v, label))
        end

        return
    end

    add_tick!(0.0,            raw"\ensuremath{0}")
    add_tick!(p01val,         raw"\ensuremath{p_{01}}")
    add_tick!(x1_fp,          raw"\ensuremath{x^{1}}")
    add_tick!(x0_fp,          raw"\ensuremath{x^{0}}")
    add_tick!(p01val + ρval,  raw"\ensuremath{p_{11}}")
    add_tick!(1.0,            raw"\ensuremath{1}")

    # If x coincides with a distinguished point, retain the
    # distinguished-point label.
    add_tick!(xval, raw"\ensuremath{x}")

    sort!(tick_items; by=first)

    tick_positions = first.(tick_items)
    tick_labels    = last.(tick_items)

    tick_pos_tex = tex_num_list(tick_positions)
    tick_lab_tex = tex_label_list(tick_labels)

    diagonal = Plot(
        PGFPlotsX.Options(
            "no marks"   => nothing,
            "color"      => "gray",
            "dashed"     => nothing,
            "line width" => diag_lw
        ),
        Coordinates([
            (0.0, 0.0),
            (1.0, 1.0)
        ])
    )

    step_curve = Plot(
        PGFPlotsX.Options(
            "no marks"         => nothing,
            "color"            => "black",
            "line width"       => seg_lw,
            "unbounded coords" => "jump"
        ),
        Table(
            x=xline,
            y=yline
        )
    )

    axis_contents = Any[
        diagonal,
        step_curve
    ]

    if !isempty(z_fill)
        push!(
            axis_contents,
            Plot(
                PGFPlotsX.Options(
                    "only marks" => nothing,
                    "mark"       => "*",
                    "mark size"  => mark_size,
                    "color"      => "black"
                ),
                Table(
                    x=z_fill,
                    y=y_fill
                )
            )
        )

        # Draw the white markers last so they cover the corresponding
        # horizontal segment.
        push!(
            axis_contents,
            Plot(
                PGFPlotsX.Options(
                    "only marks" => nothing,
                    "mark"       => "*",
                    "mark size"  => mark_size,
                    "color"      => "black",
                    "mark options" =>
                        "{fill=white,draw=black,line width=0.25pt}"
                ),
                Table(
                    x=z_open,
                    y=y_open
                )
            )
        )
    end

    title_tex = "\$t = $(t)\$"

    axis_options = PGFPlotsX.Options(
        "title"       => title_tex,
        "title style" =>
            "{at={(0.5,1.02)},anchor=south}",

        "width"  => width,
        "height" => height,

        "xmin" => 0,
        "xmax" => 1,
        "ymin" => 0,
        "ymax" => 1,

        "axis equal image"  => nothing,
        "axis on top"       => "false",
        "clip marker paths" => "false",
        "grid"              => "none",

        "xlabel" => raw"$z$",
        "ylabel" =>
            show_ylabel ? raw"$\varphi_t(x,z)$" : "",

        "xtick"       => tick_pos_tex,
        "xticklabels" => tick_lab_tex,
        "ytick"       => tick_pos_tex,
        "yticklabels" =>
            show_yticklabels ? tick_lab_tex : "{}",

        "xticklabel style" =>
            "{text height=1.6ex,text depth=0.3ex}",
        "yticklabel style" =>
            "{text height=1.6ex,text depth=0.3ex}",

        "tick align"        => "outside",
        "major tick length" => "2pt",

        "scaled x ticks"   => "false",
        "scaled y ticks"   => "false",
        "enlargelimits"    => "false",
        "unbounded coords" => "jump"
    )

    return Axis(
        axis_options,
        axis_contents...
    )
end


"""
    plot_varphi_t_vs_z_1x2_pgf(t1, t2, x; p01, ρ, κ, ...)

Construct a TikzPicture containing a 1×2 GroupPlot for two
iteration times. Only the left panel displays the y-axis title
and tick labels.
"""
function plot_varphi_t_vs_z_1x2_pgf(
    t1::Integer,
    t2::Integer,
    x;
    p01,
    ρ,
    κ,
    width_each::AbstractString="8cm",
    height_each::AbstractString="8cm",
    hsep::AbstractString="1.2cm",
    kwargs...
)
    left_axis = axis_varphi_t_vs_z_pgf(
        t1,
        x;
        p01=p01,
        ρ=ρ,
        κ=κ,
        width=width_each,
        height=height_each,
        show_ylabel=true,
        show_yticklabels=true,
        kwargs...
    )

    right_axis = axis_varphi_t_vs_z_pgf(
        t2,
        x;
        p01=p01,
        ρ=ρ,
        κ=κ,
        width=width_each,
        height=height_each,
        show_ylabel=false,
        show_yticklabels=false,
        kwargs...
    )

    group_plot = GroupPlot(
        PGFPlotsX.Options(
            "group style" =>
                "{group size={2 by 1},horizontal sep=$(hsep)}"
        ),
        left_axis,
        right_axis
    )

    return TikzPicture(group_plot)
end

In [ ]:
# Two-panel plot of z ↦ φ_t(x,z), for t = 1 and t = 2.

p01 = 0.25
ρ   = 0.60
κ   = 0.80

x0_fp, x1_fp, _ = fixed_points(
    ;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x_plot = 0.5 * (x1_fp + x0_fp)

fig = plot_varphi_t_vs_z_1x2_pgf(
    1,
    2,
    x_plot;
    p01=p01,
    ρ=ρ,
    κ=κ
)

pdf_filename = "varphi_t_vs_z_t1_t2_xmid.pdf"
pdf_path = joinpath(@__DIR__, pdf_filename)

PGFPlotsX.pgfsave(pdf_path, fig)

println("PDF saved as: ", pdf_path)

display(fig)

In [ ]:
# Enable the PGFPlots groupplots library required by GroupPlot.
groupplots_preamble = raw"\usepgfplotslibrary{groupplots}"

groupplots_preamble in PGFPlotsX.CUSTOM_PREAMBLE ||
    push!(
        PGFPlotsX.CUSTOM_PREAMBLE,
        groupplots_preamble
    )

# Prevent PGFPlotsX from opening an external PDF viewer.
PGFPlotsX.enable_interactive(false)

In [ ]:
# Two-panel plot of z ↦ φ_t(x,z), for t = 3 and t = 4.

fig = plot_varphi_t_vs_z_1x2_pgf(
    3,
    4,
    x_plot;
    p01=p01,
    ρ=ρ,
    κ=κ,
    mark_size="0.5pt",
    seg_lw="0.55pt"
)

pdf_filename = "varphi_t_vs_z_t3_t4_xmid.pdf"
pdf_path = joinpath(@__DIR__, pdf_filename)

PGFPlotsX.pgfsave(pdf_path, fig)

isfile(pdf_path) ||
    error("The PDF was not created: $pdf_path")

println("PDF saved as: ", pdf_path)

display(
    "application/pdf",
    read(pdf_path)
)

In [ ]:
# Convert a binary itinerary vector to a word such as "0101".
function itinerary_word(
    A::AbstractVector{<:Integer}
)
    all(a -> a == 0 || a == 1, A) ||
        throw(
            ArgumentError(
                "The itinerary must contain only 0 and 1."
            )
        )

    return join(
        a == 1 ? '1' : '0'
        for a in A
    )
end


# Validate a word supplied directly as a string.
function _validated_itinerary_word(
    w::AbstractString
)
    ws = String(w)

    all(c -> c == '0' || c == '1', ws) ||
        throw(
            ArgumentError(
                "The itinerary word must contain only '0' and '1'."
            )
        )

    return ws
end


_validated_itinerary_word(
    A::AbstractVector{<:Integer}
) = itinerary_word(A)


# Check whether b is a prefix of the infinite repetition
# of its first p symbols.
function _is_prefix_of_periodic(
    b::AbstractVector{UInt8},
    p::Int
)
    n = length(b)

    1 ≤ p ≤ n ||
        throw(
            ArgumentError(
                "The period length must lie between 1 and $n."
            )
        )

    @inbounds for k in 1:n
        if b[k] != b[mod1(k, p)]
            return false
        end
    end

    return true
end


# Convert an ASCII byte vector or view back to a String.
_bytes_to_string(
    b::AbstractVector{UInt8}
) = String(Vector{UInt8}(b))


"""
    eventual_periodic_decomp(
        w;
        min_full_repeats=2
    )

Find the earliest observed periodic-tail decomposition

    w = u ⋅ s^m ⋅ r,

where `r` is a prefix of `s`, possibly empty. Equivalently, the
suffix following `u` is a prefix of the infinite periodic word
`s^∞`.

The input `w` may be either:

- a binary string such as `"0010101"`;
- an integer vector containing only `0` and `1`.

Among admissible decompositions, the function chooses:

1. the smallest starting index of the periodic suffix;
2. the smallest compatible period length at that index.

At least `min_full_repeats` complete copies of `s` must occur in
the observed suffix.

Returns the named tuple

    (u, s, m, r, start_index, period_length).

Here, `start_index` is the one-based position at which the
periodic suffix begins.
"""
function eventual_periodic_decomp(
    w;
    min_full_repeats::Int=2
)
    min_full_repeats ≥ 1 ||
        throw(
            ArgumentError(
                "min_full_repeats must be positive."
            )
        )

    ws = _validated_itinerary_word(w)
    b  = codeunits(ws)
    n  = length(b)

    # Search first by earliest periodic-tail starting point.
    for i in 1:n
        suffix_length = n - i + 1

        # At least the requested number of full blocks must fit.
        max_period =
            suffix_length ÷ min_full_repeats

        max_period ≥ 1 ||
            continue

        suffix = @view b[i:n]

        # At the selected starting point, use the shortest period.
        for p in 1:max_period
            _is_prefix_of_periodic(suffix, p) ||
                continue

            m    = suffix_length ÷ p
            rlen = suffix_length % p

            u = i == 1 ?
                "" :
                _bytes_to_string(
                    @view b[1:(i - 1)]
                )

            s = _bytes_to_string(
                @view b[i:(i + p - 1)]
            )

            r = rlen == 0 ?
                "" :
                _bytes_to_string(
                    @view b[(n - rlen + 1):n]
                )

            return (
                u=u,
                s=s,
                m=m,
                r=r,
                start_index=i,
                period_length=p
            )
        end
    end

    # No sufficiently repeated periodic suffix was observed.
    return (
        u=ws,
        s="",
        m=0,
        r="",
        start_index=n + 1,
        period_length=0
    )
end


"""
    describe_eventual_periodic(
        w;
        min_full_repeats=2
    )

Return a human-readable description of the detected periodic
suffix.
"""
function describe_eventual_periodic(
    w;
    min_full_repeats::Int=2
)
    d = eventual_periodic_decomp(
        w;
        min_full_repeats=min_full_repeats
    )

    if d.m == 0
        return (
            "No repeating tail found with at least " *
            "$(min_full_repeats) full repeats."
        )
    end

    return (
        "u=\"$(d.u)\", tail = s^m r with " *
        "s=\"$(d.s)\", m=$(d.m), r=\"$(d.r)\" " *
        "(starts at index $(d.start_index))"
    )
end


"""
    format_eventual_periodic(
        w;
        min_full_repeats=2
    )

Format the decomposition as

    u (s)^m r.

If no sufficiently repeated tail is found, return the original
word.
"""
function format_eventual_periodic(
    w;
    min_full_repeats::Int=2
)
    d = eventual_periodic_decomp(
        w;
        min_full_repeats=min_full_repeats
    )

    d.m == 0 &&
        return _validated_itinerary_word(w)

    parts = String[]

    isempty(d.u) ||
        push!(parts, d.u)

    push!(
        parts,
        "($(d.s))^$(d.m)"
    )

    isempty(d.r) ||
        push!(parts, d.r)

    return join(parts, " ")
end

In [ ]:
# Assumes φ0(x; p01, ρ) and φ1(x; p01, ρ, κ)
# have already been defined.


# Validate a binary word.
function _validated_binary_word(
    w::AbstractString;
    allow_empty::Bool=true
)
    ws = String(w)

    if !allow_empty && isempty(ws)
        throw(
            ArgumentError(
                "The periodic word must be nonempty."
            )
        )
    end

    all(c -> c == '0' || c == '1', ws) ||
        throw(
            ArgumentError(
                "The word must contain only '0' and '1'."
            )
        )

    return ws
end


"""
    apply_symbol(x, sym; p01, ρ, κ)

Apply the belief map associated with one binary symbol:

- `'0'`: apply `φ0`;
- `'1'`: apply `φ1`.
"""
@inline function apply_symbol(
    x,
    sym::UInt8;
    p01,
    ρ,
    κ
)
    if sym == 0x30                    # ASCII '0'
        return φ0(
            x;
            p01=p01,
            ρ=ρ
        )
    elseif sym == 0x31               # ASCII '1'
        return φ1(
            x;
            p01=p01,
            ρ=ρ,
            κ=κ
        )
    end

    throw(
        ArgumentError(
            "The symbol must be '0' or '1'; received $(Char(sym))."
        )
    )
end


"""
    apply_word(x, w; p01, ρ, κ)

For a binary word

    w = w₁w₂⋯wₖ,

compute

    Φ_w(x) = φ^{wₖ} ∘ ⋯ ∘ φ^{w₂} ∘ φ^{w₁}(x).

Thus, the symbols of `w` are applied from left to right. The empty
word is allowed and represents the identity map.
"""
function apply_word(
    x,
    w::AbstractString;
    p01,
    ρ,
    κ
)
    ws = _validated_binary_word(
        w;
        allow_empty=true
    )

    y = x

    for sym in codeunits(ws)
        y = apply_symbol(
            y,
            sym;
            p01=p01,
            ρ=ρ,
            κ=κ
        )
    end

    return y
end


"""
    fixed_point_of_word(
        w,
        xinit;
        p01,
        ρ,
        κ,
        tol=1e-12,
        maxiter=10_000
    )

Use fixed-point iteration to find `y` satisfying

    y = Φ_w(y),

where `Φ_w` is the composition associated with the nonempty binary
word `w`.

Return `(y, iterations)`.
"""
function fixed_point_of_word(
    w::AbstractString,
    xinit;
    p01,
    ρ,
    κ,
    tol::Real=1e-12,
    maxiter::Int=10_000
)
    tol > 0 ||
        throw(
            ArgumentError(
                "tol must be positive."
            )
        )

    maxiter >= 1 ||
        throw(
            ArgumentError(
                "maxiter must be positive."
            )
        )

    ws = _validated_binary_word(
        w;
        allow_empty=false
    )

    y = xinit
    last_error = Inf

    for iteration in 1:maxiter
        ynew = apply_word(
            y,
            ws;
            p01=p01,
            ρ=ρ,
            κ=κ
        )

        last_error = abs(ynew - y)

        isfinite(last_error) ||
            error(
                "Fixed-point iteration generated a nonfinite value."
            )

        if last_error <= tol
            return ynew, iteration
        end

        y = ynew
    end

    error(
        "fixed_point_of_word did not converge within " *
        "$maxiter iterations; last increment = $last_error."
    )
end


"""
    limiting_cycle_points(
        xinit,
        z,
        w;
        p01,
        ρ,
        κ,
        tol=1e-12,
        maxiter=10_000,
        check=true
    )

Compute the limiting cycle associated with the repeating binary word
`w = w₁⋯wₖ`.

The returned vector `cycle` satisfies:

- `cycle[1]` is a fixed point of `Φ_w`;
- `cycle[i]` is the belief immediately before symbol `w[i]`;
- for `i < k`,

      cycle[i+1] = φ^{w[i]}(cycle[i]);

- applying `w[k]` to `cycle[k]` closes the cycle.

If `check=true`, verify both numerical closure and threshold
consistency. The threshold convention is

    action 1  ⇔  belief > z,

so equality with `z` selects action `0`.
"""
function limiting_cycle_points(
    xinit,
    z,
    w::AbstractString;
    p01,
    ρ,
    κ,
    tol::Real=1e-12,
    maxiter::Int=10_000,
    check::Bool=true
)
    ws = _validated_binary_word(
        w;
        allow_empty=false
    )

    y0, _ = fixed_point_of_word(
        ws,
        xinit;
        p01=p01,
        ρ=ρ,
        κ=κ,
        tol=tol,
        maxiter=maxiter
    )

    bytes = codeunits(ws)
    k     = length(bytes)

    cycle = Vector{typeof(y0)}(undef, k)
    cycle[1] = y0

    y = y0

    for i in 1:(k - 1)
        y = apply_symbol(
            y,
            bytes[i];
            p01=p01,
            ρ=ρ,
            κ=κ
        )

        cycle[i + 1] = y
    end

    # Applying the final symbol should return to the first point.
    yclose = apply_symbol(
        cycle[end],
        bytes[end];
        p01=p01,
        ρ=ρ,
        κ=κ
    )

    if check
        closure_tol = max(10 * tol, 1e-12)

        if !isapprox(
            yclose,
            y0;
            atol=closure_tol,
            rtol=0.0
        )
            @warn(
                "The limiting cycle did not close numerically.",
                first_point=y0,
                closing_point=yclose,
                closure_error=abs(yclose - y0)
            )
        end

        for i in eachindex(cycle)
            symbol_is_one = bytes[i] == 0x31
            threshold_selects_one = cycle[i] > z

            if symbol_is_one != threshold_selects_one
                @warn(
                    "Threshold consistency failed at phase $i.",
                    symbol=Char(bytes[i]),
                    value=cycle[i],
                    threshold=z
                )
            end
        end
    end

    return cycle
end

In [ ]:
##############################
# tests_words_cycles.jl
##############################

using Printf
using Test

# Assumes that φ0, φ1, apply_symbol, apply_word,
# fixed_point_of_word, and limiting_cycle_points
# have already been defined.


# -------------------------------------------------------------------
# Parameters and initial conditions
# -------------------------------------------------------------------

p01_test = 0.25
ρ_test   = 0.60
κ_test   = 0.80

x_test    = 0.329269626091
xinit_test = 0.10

@printf(
    "Parameters: p01=%.3f  ρ=%.3f  κ=%.3f\n\n",
    p01_test,
    ρ_test,
    κ_test
)


# -------------------------------------------------------------------
# 1. Tests of apply_symbol and apply_word
# -------------------------------------------------------------------

test_words = [
    "0",
    "1",
    "01",
    "10",
    "001",
    "100",
    "010",
    "011",
    "101",
    "110"
]

@printf(
    "=== apply_word smoke tests (x=%.12f) ===\n",
    x_test
)

for w in test_words
    y = apply_word(
        x_test,
        w;
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    )

    @printf(
        "apply_word(x, \"%s\") = %.12f\n",
        w,
        y
    )
end

println()

@testset "apply_symbol and apply_word" begin
    # Single-symbol words.
    @test isapprox(
        apply_word(
            x_test,
            "0";
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test
        ),
        φ0(
            x_test;
            p01=p01_test,
            ρ=ρ_test
        );
        atol=0.0,
        rtol=1e-14
    )

    @test isapprox(
        apply_word(
            x_test,
            "1";
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test
        ),
        φ1(
            x_test;
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test
        );
        atol=0.0,
        rtol=1e-14
    )

    # "10" means: first φ1, then φ0.
    y10 = apply_word(
        x_test,
        "10";
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    )

    y10_manual = φ0(
        φ1(
            x_test;
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test
        );
        p01=p01_test,
        ρ=ρ_test
    )

    @test isapprox(
        y10,
        y10_manual;
        atol=1e-12,
        rtol=0.0
    )

    # "01" means: first φ0, then φ1.
    y01 = apply_word(
        x_test,
        "01";
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    )

    y01_manual = φ1(
        φ0(
            x_test;
            p01=p01_test,
            ρ=ρ_test
        );
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    )

    @test isapprox(
        y01,
        y01_manual;
        atol=1e-12,
        rtol=0.0
    )

    # The empty word represents the identity map.
    @test apply_word(
        x_test,
        "";
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    ) == x_test

    # Invalid words and symbols must be rejected.
    @test_throws ArgumentError apply_word(
        x_test,
        "012";
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    )

    @test_throws ArgumentError apply_symbol(
        x_test,
        UInt8('2');
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    )
end


# -------------------------------------------------------------------
# 2. Fixed points of word maps
# -------------------------------------------------------------------

function word_fixed_point_residual(
    y,
    w;
    p01,
    ρ,
    κ
)
    return apply_word(
        y,
        w;
        p01=p01,
        ρ=ρ,
        κ=κ
    ) - y
end


@printf(
    "\n=== fixed_point_of_word tests (xinit=%.6f) ===\n",
    xinit_test
)

@testset "fixed_point_of_word residuals" begin
    for w in test_words
        y, iterations = fixed_point_of_word(
            w,
            xinit_test;
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test,
            tol=1e-12,
            maxiter=10_000
        )

        residual = word_fixed_point_residual(
            y,
            w;
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test
        )

        @printf(
            "w=\"%s\": y=%.12f  iterations=%d  residual=%.3e\n",
            w,
            y,
            iterations,
            residual
        )

        @test isfinite(y)
        @test 0.0 ≤ y ≤ 1.0
        @test abs(residual) ≤ 1e-9
        @test 1 ≤ iterations ≤ 10_000
    end

    @test_throws ArgumentError fixed_point_of_word(
        "",
        xinit_test;
        p01=p01_test,
        ρ=ρ_test,
        κ=κ_test
    )
end


# -------------------------------------------------------------------
# 3. Threshold consistent with a proposed limiting cycle
# -------------------------------------------------------------------

"""
    choose_consistent_threshold(cycle, w)

Find a threshold `z ∈ [0,1]` satisfying

    cycle[i] > z  ⇔  w[i] == '1'.

The threshold convention therefore assigns action `0` at equality.
Return `nothing` if no such threshold exists.
"""
function choose_consistent_threshold(
    cycle::AbstractVector{<:Real},
    w::AbstractString
)
    ws = _validated_binary_word(
        w;
        allow_empty=false
    )

    length(cycle) == length(codeunits(ws)) ||
        throw(
            DimensionMismatch(
                "The cycle and word must have the same length."
            )
        )

    all(y -> isfinite(y) && 0.0 ≤ y ≤ 1.0, cycle) ||
        throw(
            ArgumentError(
                "All cycle points must lie in [0,1]."
            )
        )

    bytes = codeunits(ws)

    zero_values = [
        cycle[i]
        for i in eachindex(cycle)
        if bytes[i] == UInt8('0')
    ]

    one_values = [
        cycle[i]
        for i in eachindex(cycle)
        if bytes[i] == UInt8('1')
    ]

    if isempty(one_values)
        # All symbols are zero. Equality is permitted for action 0.
        return maximum(zero_values)
    elseif isempty(zero_values)
        # All symbols are one, so z must be strictly below every point.
        min_one = minimum(one_values)
        return min_one > 0.0 ? min_one / 2 : nothing
    end

    max_zero = maximum(zero_values)
    min_one  = minimum(one_values)

    # Since action 0 is selected at equality, z=max_zero works
    # precisely when all action-1 points are strictly larger.
    return max_zero < min_one ? max_zero : nothing
end


@printf(
    "\n=== limiting_cycle_points tests ===\n"
)

@testset "limiting cycles and threshold consistency" begin
    for w in test_words
        # First compute the cycle without a threshold check.
        cycle = limiting_cycle_points(
            xinit_test,
            0.5,
            w;
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test,
            tol=1e-12,
            maxiter=10_000,
            check=false
        )

        z = choose_consistent_threshold(
            cycle,
            w
        )

        @printf(
            "w=\"%s\": cycle=%s\n",
            w,
            repr(cycle)
        )

        if z === nothing
            @printf(
                "         no threshold-consistent realization\n\n"
            )

            @test false
            continue
        end

        @printf(
            "         z=%.12f\n\n",
            z
        )

        # Exercise the internal checking branch.
        cycle_checked = limiting_cycle_points(
            xinit_test,
            z,
            w;
            p01=p01_test,
            ρ=ρ_test,
            κ=κ_test,
            tol=1e-12,
            maxiter=10_000,
            check=true
        )

        @test length(cycle_checked) == length(w)

        bytes = codeunits(w)
        k = length(bytes)

        # Every phase maps to the next phase, including closure.
        for i in 1:k
            next_i = mod1(i + 1, k)

            next_value = apply_symbol(
                cycle_checked[i],
                bytes[i];
                p01=p01_test,
                ρ=ρ_test,
                κ=κ_test
            )

            @test isapprox(
                next_value,
                cycle_checked[next_i];
                atol=1e-9,
                rtol=0.0
            )
        end

        # Explicitly test the threshold convention.
        for i in 1:k
            @test (
                cycle_checked[i] > z
            ) == (
                bytes[i] == UInt8('1')
            )
        end
    end
end

println("\nAll word-map and limiting-cycle tests passed.")

In [ ]:
# Assumes that the following functions have already been defined:
#
# - _validated_itinerary_word
# - eventual_periodic_decomp
# - limiting_cycle_points


# Format an already computed periodic decomposition, avoiding a
# second call to eventual_periodic_decomp.
function _format_periodic_decomposition(
    ws::AbstractString,
    d
)
    d.m == 0 &&
        return String(ws)

    parts = String[]

    isempty(d.u) ||
        push!(parts, d.u)

    push!(
        parts,
        "($(d.s))^$(d.m)"
    )

    isempty(d.r) ||
        push!(parts, d.r)

    return join(parts, " ")
end


"""
    limiting_cycle_from_itinerary(
        w,
        z;
        p01,
        ρ,
        κ,
        min_full_repeats=2,
        xinit=nothing,
        tol=1e-12,
        maxiter=10_000,
        check=true
    )

Given a finite binary itinerary `w`, detect an observed periodic-suffix
decomposition

    w = u ⋅ s^m ⋅ r,

where `r` is a prefix of `s`, possibly empty. If such a suffix is
detected, compute the limiting cycle associated with infinite repetition
of the block `s`.

The input `w` may be either:

- a binary string such as `"0010101"`;
- an integer vector containing only `0` and `1`.

If `xinit=nothing`, fixed-point iteration starts from

    p11 = p01 + ρ.

The threshold `z` does not determine the cycle. If `check=true`, it is
used to verify that the cycle is consistent with the convention

    action 1  ⇔  belief > z.

Return the named tuple

    (
        found,
        formatted,
        u,
        s,
        m,
        r,
        start_index,
        period_length,
        cycle
    ).

If no sufficiently repeated periodic suffix is found, return
`found=false` and an empty cycle.
"""
function limiting_cycle_from_itinerary(
    w,
    z;
    p01,
    ρ,
    κ,
    min_full_repeats::Int=2,
    xinit=nothing,
    tol::Real=1e-12,
    maxiter::Int=10_000,
    check::Bool=true
)
    ws = _validated_itinerary_word(w)

    decomposition = eventual_periodic_decomp(
        ws;
        min_full_repeats=min_full_repeats
    )

    formatted = _format_periodic_decomposition(
        ws,
        decomposition
    )

    if decomposition.m == 0
        return (
            found=false,
            formatted=formatted,
            u=decomposition.u,
            s="",
            m=0,
            r="",
            start_index=decomposition.start_index,
            period_length=0,
            cycle=Float64[]
        )
    end

    # Under the parameterization p11 = p01 + ρ, this is the
    # upper endpoint of the invariant belief interval.
    xseed = isnothing(xinit) ?
        p01 + ρ :
        xinit

    xseed isa Real ||
        throw(
            ArgumentError(
                "xinit must be a real number or nothing."
            )
        )

    isfinite(xseed) ||
        throw(
            ArgumentError(
                "The initial seed must be finite."
            )
        )

    0.0 ≤ xseed ≤ 1.0 ||
        throw(
            ArgumentError(
                "The initial seed must lie in [0,1]; received $xseed."
            )
        )

    cycle = limiting_cycle_points(
        xseed,
        z,
        decomposition.s;
        p01=p01,
        ρ=ρ,
        κ=κ,
        tol=tol,
        maxiter=maxiter,
        check=check
    )

    return (
        found=true,
        formatted=formatted,
        u=decomposition.u,
        s=decomposition.s,
        m=decomposition.m,
        r=decomposition.r,
        start_index=decomposition.start_index,
        period_length=decomposition.period_length,
        cycle=cycle
    )
end

In [ ]:
using Printf


"""
    print_cycle(cycle, s, z)

Print the limiting-cycle points associated with the nonempty binary
block `s`.

For each phase `i`, `cycle[i]` is the belief immediately before the
symbol `s[i]` is applied. The table reports:

- the proposed symbol `s[i]`;
- the cycle point `cycle[i]`;
- whether the point lies above the threshold;
- the action selected by the threshold rule;
- whether that action agrees with `s[i]`.

The threshold convention is

    action 1  ⇔  cycle[i] > z,

so equality with `z` selects action `0`.
"""
function print_cycle(
    cycle::AbstractVector{<:Real},
    s::AbstractString,
    z::Real
)
    ws = _validated_binary_word(
        s;
        allow_empty=false
    )

    bytes = codeunits(ws)
    k = length(bytes)

    length(cycle) == k ||
        throw(
            DimensionMismatch(
                "The cycle has length $(length(cycle)), " *
                "but the word has length $k."
            )
        )

    all(isfinite, cycle) ||
        throw(
            ArgumentError(
                "All cycle points must be finite."
            )
        )

    println("Limiting cycle for s=\"$ws\"")
    @printf("Threshold: z = %.12f\n\n", z)

    @printf(
        "%3s  %3s  %16s  %-6s  %-7s  %s\n",
        "i",
        "sym",
        "y_i",
        "region",
        "action",
        "consistent"
    )

    println(repeat("-", 63))

    for i in eachindex(cycle)
        yi = cycle[i]

        symbol_action = bytes[i] == UInt8('1') ? 1 : 0
        threshold_action = yi > z ? 1 : 0

        region = threshold_action == 1 ? "> z" : "≤ z"
        action = threshold_action == 1 ? "active" : "passive"

        consistent =
            symbol_action == threshold_action ?
            "yes" :
            "NO"

        @printf(
            "%3d  %3c  %16.12f  %-6s  %-7s  %s\n",
            i,
            Char(bytes[i]),
            yi,
            region,
            action,
            consistent
        )
    end

    return nothing
end

In [ ]:
"""
    cycle_one_fraction(out) -> (ones, cycle_length, fraction_string)

Given the output `out` from `limiting_cycle_from_itinerary`,
return:

- `ones`: number of `'1'` symbols in the repeating block `out.s`;
- `cycle_length`: length of `out.s`;
- `fraction_string`: the reduced fraction of ones, formatted as
  `"numerator/denominator"`.

For example, if `out.s == "0011"`, return

    (2, 4, "1/2")

If `out.found == false`, return

    (0, 0, "0/0")
"""
function cycle_one_fraction(out)
    hasproperty(out, :found) ||
        throw(
            ArgumentError(
                "The input must have a `found` field."
            )
        )

    found = getproperty(out, :found)

    found isa Bool ||
        throw(
            ArgumentError(
                "The `found` field must be Boolean."
            )
        )

    found ||
        return (0, 0, "0/0")

    hasproperty(out, :s) ||
        throw(
            ArgumentError(
                "A successful result must have an `s` field."
            )
        )

    s = _validated_binary_word(
        getproperty(out, :s);
        allow_empty=false
    )

    cycle_length = length(s)
    ones = count(==(UInt8('1')), codeunits(s))

    divisor = gcd(ones, cycle_length)

    numerator   = ones ÷ divisor
    denominator = cycle_length ÷ divisor

    fraction_string = string(
        numerator,
        "/",
        denominator
    )

    return (
        ones,
        cycle_length,
        fraction_string
    )
end

In [ ]:
using Printf


# -------------------------------------------------------------------
# Example parameters
# -------------------------------------------------------------------

p01 = 0.25
ρ   = 0.60
κ   = 0.80

x0_fp, x1_fp, _ = fixed_points(
    ;
    p01=p01,
    ρ=ρ,
    κ=κ
)

p11 = p01 + ρ

@printf(
    "p01=%.5f, x1=%.5f, x0=%.5f, p11=p01+ρ=%.5f\n",
    p01,
    x1_fp,
    x0_fp,
    p11
)


# -------------------------------------------------------------------
# Fixed points associated with selected binary words
# -------------------------------------------------------------------

# Since the word maps are contractions, the limiting fixed points do
# not depend on this seed. We use the upper endpoint p11.
xseed = p11

x01, _ = fixed_point_of_word(
    "01",
    xseed;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x10, _ = fixed_point_of_word(
    "10",
    xseed;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x011, _ = fixed_point_of_word(
    "011",
    xseed;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x101, _ = fixed_point_of_word(
    "101",
    xseed;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x010, _ = fixed_point_of_word(
    "010",
    xseed;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x100, _ = fixed_point_of_word(
    "100",
    xseed;
    p01=p01,
    ρ=ρ,
    κ=κ
)


# -------------------------------------------------------------------
# Threshold and initial belief
# -------------------------------------------------------------------

# For these parameters, x010 and x100 bound a threshold interval
# producing the cycle represented by a rotation of "010".
x010 < x100 ||
    error(
        "The selected parameters do not satisfy x010 < x100."
    )

z = (x010 + x100) / 2

# Start just below z, between z and its active image.
φ1z = φ1(
    z;
    p01=p01,
    ρ=ρ,
    κ=κ
)

x = 0.99 * z + 0.01 * φ1z

@printf(
    """
    x011=%.5f, x101=%.5f
    x01 =%.5f, x10 =%.5f
    x010=%.5f, x100=%.5f
    z   =%.5f, x   =%.5f
    """,
    x011,
    x101,
    x01,
    x10,
    x010,
    x100,
    z,
    x
)


# -------------------------------------------------------------------
# Finite skeleton orbit and itinerary
# -------------------------------------------------------------------

T = 20

X = skeleton_orbit(
    x,
    z,
    T;
    p01=p01,
    ρ=ρ,
    κ=κ
)

A = skeleton_itinerary(
    x,
    z,
    T;
    p01=p01,
    ρ=ρ,
    κ=κ
)

X2, A2 = skeleton_orbit_itinerary(
    x,
    z,
    T;
    p01=p01,
    ρ=ρ,
    κ=κ
)

w = skeleton_itinerary_string(
    x,
    z,
    T;
    p01=p01,
    ρ=ρ,
    κ=κ
)

# The separately computed outputs must agree.
@assert X == X2
@assert A == A2

println("\nSkeleton itinerary: ", w)

println(
    "Detected pattern: ",
    format_eventual_periodic(
        w;
        min_full_repeats=2
    )
)


# -------------------------------------------------------------------
# Detected limiting cycle
# -------------------------------------------------------------------

out = limiting_cycle_from_itinerary(
    w,
    z;
    p01=p01,
    ρ=ρ,
    κ=κ,
    min_full_repeats=2,
    xinit=xseed,
    tol=1e-12,
    maxiter=10_000,
    check=true
)

println("\nPeriodic decomposition: ", out.formatted)

if out.found
    println("Repeating block: s = \"", out.s, "\"")
    println(
        "Observed repetitions: ",
        out.m,
        "; period length: ",
        out.period_length
    )

    println()

    print_cycle(
        out.cycle,
        out.s,
        z
    )

    ones, cycle_length, fraction_string =
        cycle_one_fraction(out)

    println()
    println(
        "Ones in the symbolic cycle: ",
        ones,
        " out of ",
        cycle_length
    )
    println(
        "Limiting fraction of active actions: ",
        fraction_string
    )
else
    println(
        "No periodic tail with at least two full repetitions was detected."
    )
end


# -------------------------------------------------------------------
# Orbit table
# -------------------------------------------------------------------

println("\n t       X̃_t      Ã_t")
println("-------------------------")

for t in 0:T
    @printf(
        "%2d   %9.6f    %d\n",
        t,
        X[t + 1],
        A[t + 1]
    )
end

In [ ]:
using Printf


# -------------------------------------------------------------------
# Helpers for PGFPlotsX option lists
# -------------------------------------------------------------------

tex_num_list(xs::AbstractVector{<:Real}) =
    isempty(xs) ?
    "{}" :
    "{" *
    join(
        (@sprintf("%.15g", Float64(x)) for x in xs),
        ","
    ) *
    "}"


tex_label_list(labels::AbstractVector{<:AbstractString}) =
    isempty(labels) ?
    "{}" :
    "{" * join(labels, ",") * "}"


"""
    _refine_signature_boundary(
        zleft,
        zright,
        left_signature,
        signature_at;
        iterations=60
    )

Bisect an interval whose endpoint signatures differ.

The returned values satisfy, up to floating-point resolution,

    signature_at(left) == left_signature
    signature_at(right) != left_signature.

Thus, `left` and `right` are numerical representatives of the two
sides of the detected breakpoint. This is more reliable than evaluating
at `prevfloat(right)`.
"""
function _refine_signature_boundary(
    zleft::Real,
    zright::Real,
    left_signature,
    signature_at;
    iterations::Int=60
)
    iterations ≥ 1 ||
        throw(
            ArgumentError(
                "iterations must be positive."
            )
        )

    lo = Float64(zleft)
    hi = Float64(zright)

    lo < hi ||
        throw(
            ArgumentError(
                "The refinement interval must have positive length."
            )
        )

    signature_at(lo) == left_signature ||
        throw(
            ArgumentError(
                "The left endpoint does not have the supplied signature."
            )
        )

    signature_at(hi) != left_signature ||
        throw(
            ArgumentError(
                "The endpoint signatures do not differ."
            )
        )

    for _ in 1:iterations
        mid = lo + (hi - lo) / 2

        # Stop when no further floating-point refinement is possible.
        (mid == lo || mid == hi) &&
            break

        if signature_at(mid) == left_signature
            lo = mid
        else
            hi = mid
        end
    end

    return (
        left=lo,
        right=hi,
        right_signature=signature_at(hi)
    )
end


"""
    bifurcation_segments_pgf(
        zgrid;
        p01,
        ρ,
        κ,
        xinit=p01+ρ,
        T=600,
        min_full_repeats=2,
        mode=:limitcycle,
        ...
    )

Construct a PGFPlotsX bifurcation diagram over a strictly increasing
threshold grid `zgrid`.

Two modes are available.

## `mode=:limitcycle`

For each threshold, compute a finite skeleton itinerary, detect its
eventually periodic suffix, and extrapolate the detected repeating
block indefinitely. The resulting limiting-cycle points are plotted
as horizontal branches.

Each returned segment has fields

    (zL, zR, s, cycle, formatted).

## `mode=:iterate`

Plot the finite-time iterate

    z ↦ varphi_T(xinit,z).

For a fixed length-`T` itinerary, this map is constant in `z`; hence
the graph is represented by horizontal segments. At a discontinuity,
an open marker represents the left-hand value and a filled marker the
value selected at the breakpoint.

Each returned segment has fields

    (zL, zR, word, value).

The supplied grid must be sufficiently fine to expose the signature
changes of interest. Each detected change is refined by bisection, but
changes that occur and reverse entirely between two consecutive grid
points cannot be detected from their endpoints alone.

Return `(TikzPicture, segments)`.
"""
function bifurcation_segments_pgf(
    zgrid;
    p01,
    ρ,
    κ,
    xinit=p01 + ρ,
    T::Int=600,
    min_full_repeats::Int=2,
    width::AbstractString="14cm",
    height::AbstractString="9cm",
    seg_line_width::AbstractString="0.45pt",
    grid_line_width::AbstractString="0.10pt",
    endpoint_tick_width::AbstractString="0.20pt",
    show_break_lines::Bool=false,
    mode::Symbol=:limitcycle,
    square::Bool=true,
    mark_size::AbstractString="1.2pt",
    refine_iterations::Int=60
)
    # ---------------------------------------------------------------
    # Validation
    # ---------------------------------------------------------------

    mode in (:limitcycle, :iterate) ||
        throw(
            ArgumentError(
                "mode must be either :limitcycle or :iterate."
            )
        )

    T ≥ 0 ||
        throw(
            ArgumentError(
                "T must be nonnegative."
            )
        )

    min_full_repeats ≥ 2 ||
        throw(
            ArgumentError(
                "min_full_repeats must be at least 2."
            )
        )

    refine_iterations ≥ 1 ||
        throw(
            ArgumentError(
                "refine_iterations must be positive."
            )
        )

    xinit isa Real && isfinite(xinit) ||
        throw(
            ArgumentError(
                "xinit must be a finite real number."
            )
        )

    0.0 ≤ xinit ≤ 1.0 ||
        throw(
            ArgumentError(
                "xinit must lie in [0,1]."
            )
        )

    zs = Float64.(collect(zgrid))

    length(zs) ≥ 2 ||
        throw(
            ArgumentError(
                "zgrid must contain at least two points."
            )
        )

    all(isfinite, zs) ||
        throw(
            ArgumentError(
                "All threshold-grid values must be finite."
            )
        )

    all(diff(zs) .> 0.0) ||
        throw(
            ArgumentError(
                "zgrid must be strictly increasing."
            )
        )

    0.0 ≤ first(zs) &&
        last(zs) ≤ 1.0 ||
        throw(
            ArgumentError(
                "All threshold-grid values must lie in [0,1]."
            )
        )

    zmin = first(zs)
    zmax = last(zs)

    if square
        height = width
    end

    comparison_tolerance = 1e-12


    # ---------------------------------------------------------------
    # Labeled structural ticks
    # ---------------------------------------------------------------

    x0_tick, x1_tick, _ = fixed_points(
        ;
        p01=p01,
        ρ=ρ,
        κ=κ
    )

    p11 = p01 + ρ

    tick_candidates = [
        (0.0,     raw"$0$"),
        (p01,     raw"$p_{01}$"),
        (x1_tick, raw"$x^1$"),
        (x0_tick, raw"$x^0$"),
        (p11,     raw"$p_{01}+\rho$"),
        (1.0,     raw"$1$")
    ]

    xticks = Float64[]
    xticklabels = String[]

    for (tick, label) in tick_candidates
        if zmin - comparison_tolerance ≤
                tick ≤
                zmax + comparison_tolerance

            push!(xticks, Float64(tick))
            push!(xticklabels, String(label))
        end
    end

    # Sort and remove coincident structural ticks.
    permutation = sortperm(xticks)
    xticks = xticks[permutation]
    xticklabels = xticklabels[permutation]

    unique_ticks = Float64[]
    unique_labels = String[]

    for i in eachindex(xticks)
        if isempty(unique_ticks) ||
                abs(xticks[i] - unique_ticks[end]) >
                comparison_tolerance

            push!(unique_ticks, xticks[i])
            push!(unique_labels, xticklabels[i])
        end
    end

    xticks = unique_ticks
    xticklabels = unique_labels

    # Include xinit only if it does not coincide with a structural tick.
    if zmin - comparison_tolerance ≤
            xinit ≤
            zmax + comparison_tolerance &&
            all(
                abs(xinit - tick) >
                comparison_tolerance
                for tick in xticks
            )

        push!(xticks, Float64(xinit))
        push!(xticklabels, String(raw"$x$"))

        permutation = sortperm(xticks)
        xticks = xticks[permutation]
        xticklabels = xticklabels[permutation]
    end

    xtick_tex = tex_num_list(xticks)
    xticklabels_tex = tex_label_list(xticklabels)


    # ---------------------------------------------------------------
    # Common plotting helpers
    # ---------------------------------------------------------------

    function endpoint_data(segments)
        endpoints = sort(
            unique(
                vcat(
                    [segment.zL for segment in segments],
                    [segment.zR for segment in segments]
                )
            )
        )

        extra_ticks = [
            endpoint
            for endpoint in endpoints
            if all(
                abs(endpoint - tick) >
                comparison_tolerance
                for tick in xticks
            )
        ]

        extra_labels = fill(
            raw"\empty",
            length(extra_ticks)
        )

        breakpoints = [
            endpoint
            for endpoint in endpoints
            if zmin + comparison_tolerance <
                endpoint <
                zmax - comparison_tolerance
        ]

        return (
            endpoints=endpoints,
            extra_ticks=extra_ticks,
            extra_tick_tex=tex_num_list(extra_ticks),
            extra_label_tex=tex_label_list(extra_labels),
            breakpoints=breakpoints
        )
    end


    function break_line_coordinates(breakpoints)
        xbreak = Float64[]
        ybreak = Float64[]

        if show_break_lines
            for breakpoint in breakpoints
                push!(xbreak, breakpoint)
                push!(ybreak, 0.0)

                push!(xbreak, breakpoint)
                push!(ybreak, 1.0)

                push!(xbreak, NaN)
                push!(ybreak, NaN)
            end
        end

        return xbreak, ybreak
    end


    diagonal_plot = @pgf Plot(
        {
            "no marks",
            color="gray",
            style="dashed",
            "line width"="0.25pt"
        },
        Coordinates(
            [
                (zmin, zmin),
                (zmax, zmax)
            ]
        )
    )


    # ===============================================================
    # MODE 1: limiting-cycle bifurcation diagram
    # ===============================================================

    if mode == :limitcycle
        output_cache = Dict{Float64,Any}()

        function cycle_output_at(z)
            zvalue = Float64(z)

            return get!(output_cache, zvalue) do
                itinerary = skeleton_itinerary_string(
                    xinit,
                    zvalue,
                    T;
                    p01=p01,
                    ρ=ρ,
                    κ=κ
                )

                limiting_cycle_from_itinerary(
                    itinerary,
                    zvalue;
                    p01=p01,
                    ρ=ρ,
                    κ=κ,
                    min_full_repeats=min_full_repeats,
                    xinit=xinit,
                    check=false
                )
            end
        end

        cycle_signature_at(z) = begin
            output = cycle_output_at(z)
            output.found ? output.s : ""
        end

        grid_outputs = [
            cycle_output_at(z)
            for z in zs
        ]

        grid_signatures = [
            output.found ? output.s : ""
            for output in grid_outputs
        ]

        segments = NamedTuple[]

        zL = zs[1]

        current_output = grid_outputs[1]
        current_signature = grid_signatures[1]

        current_cycle =
            current_output.found ?
            current_output.cycle :
            Float64[]

        current_formatted = current_output.formatted

        for i in 2:length(zs)
            if grid_signatures[i] != current_signature
                boundary = _refine_signature_boundary(
                    zs[i - 1],
                    zs[i],
                    current_signature,
                    cycle_signature_at;
                    iterations=refine_iterations
                )

                zR = boundary.right

                push!(
                    segments,
                    (
                        zL=zL,
                        zR=zR,
                        s=current_signature,
                        cycle=current_cycle,
                        formatted=current_formatted
                    )
                )

                current_output = cycle_output_at(zR)
                current_signature =
                    current_output.found ?
                    current_output.s :
                    ""

                current_cycle =
                    current_output.found ?
                    current_output.cycle :
                    Float64[]

                current_formatted = current_output.formatted
                zL = zR
            end
        end

        push!(
            segments,
            (
                zL=zL,
                zR=zmax,
                s=current_signature,
                cycle=current_cycle,
                formatted=current_formatted
            )
        )

        # One horizontal branch for each point of each limiting cycle.
        xsline = Float64[]
        ysline = Float64[]

        for segment in segments
            for cycle_point in segment.cycle
                push!(xsline, segment.zL)
                push!(ysline, cycle_point)

                push!(xsline, segment.zR)
                push!(ysline, cycle_point)

                push!(xsline, NaN)
                push!(ysline, NaN)
            end
        end

        endpoints = endpoint_data(segments)

        xbreak, ybreak = break_line_coordinates(
            endpoints.breakpoints
        )

        break_plot = @pgf Plot(
            {
                "no marks",
                "unbounded coords"="jump",
                color="gray!55",
                style="dotted",
                "line width"="0.25pt"
            },
            Table(
                x=xbreak,
                y=ybreak
            )
        )

        main_plot = @pgf Plot(
            {
                "no marks",
                "unbounded coords"="jump",
                "line width"=seg_line_width,
                color="black"
            },
            Table(
                x=xsline,
                y=ysline
            )
        )

        axis = @pgf Axis(
            {
                xlabel=raw"$z$",
                ylabel=raw"limiting orbit points",

                width=width,
                height=height,
                "scale only axis",

                xmin=zmin,
                xmax=zmax,
                ymin=0,
                ymax=1,

                ytick="{0,1}",
                yticklabels=raw"{$0$,$1$}",

                xtick=xtick_tex,
                xticklabels=xticklabels_tex,

                "extra x ticks"=endpoints.extra_tick_tex,
                "extra x tick labels"=endpoints.extra_label_tex,

                "extra x tick style"=
                    "{tick style={black, line width=" *
                    endpoint_tick_width *
                    "}}",

                grid="none",
                xmajorgrids="true",
                ymajorgrids="true",
                xminorgrids="false",
                yminorgrids="false",

                "major grid style"=
                    "{draw=gray!40, line width=" *
                    grid_line_width *
                    "}",

                "axis on top"="false"
            },
            diagonal_plot,
            break_plot,
            main_plot
        )

        return TikzPicture(axis), segments
    end


    # ===============================================================
    # MODE 2: finite-time iterate z ↦ varphi_T(xinit,z)
    # ===============================================================

    word_cache = Dict{Float64,String}()
    value_cache = Dict{Float64,Float64}()

    # Computing varphi_T requires the T actions A_0,...,A_{T-1}.
    function word_at(z)
        zvalue = Float64(z)

        return get!(word_cache, zvalue) do
            if T == 0
                ""
            else
                skeleton_itinerary_string(
                    xinit,
                    zvalue,
                    T - 1;
                    p01=p01,
                    ρ=ρ,
                    κ=κ
                )
            end
        end
    end

    function value_at(z)
        zvalue = Float64(z)

        return get!(value_cache, zvalue) do
            Float64(
                varphi_t(
                    xinit,
                    zvalue,
                    T;
                    p01=p01,
                    ρ=ρ,
                    κ=κ
                )
            )
        end
    end

    grid_words = [
        word_at(z)
        for z in zs
    ]

    grid_values = [
        value_at(z)
        for z in zs
    ]

    segments = NamedTuple[]

    xsline = Float64[]
    ysline = Float64[]

    z_open = Float64[]
    y_open = Float64[]

    z_filled = Float64[]
    y_filled = Float64[]

    zL = zs[1]
    current_word = grid_words[1]
    current_value = grid_values[1]

    for i in 2:length(zs)
        if grid_words[i] != current_word
            boundary = _refine_signature_boundary(
                zs[i - 1],
                zs[i],
                current_word,
                word_at;
                iterations=refine_iterations
            )

            # Under the rule a=1 iff y>z, equality selects action 0.
            # Hence the breakpoint belongs to the right-hand regime.
            zbreak = boundary.right
            right_word = word_at(zbreak)
            right_value = value_at(zbreak)

            push!(
                segments,
                (
                    zL=zL,
                    zR=zbreak,
                    word=current_word,
                    value=current_value
                )
            )

            push!(xsline, zL)
            push!(ysline, current_value)

            push!(xsline, zbreak)
            push!(ysline, current_value)

            push!(xsline, NaN)
            push!(ysline, NaN)

            # Mark only actual jumps in the iterate value.
            if !isapprox(
                current_value,
                right_value;
                atol=1e-13,
                rtol=1e-12
            )
                # Open marker: left-hand value.
                push!(z_open, zbreak)
                push!(y_open, current_value)

                # Filled marker: value at the breakpoint.
                push!(z_filled, zbreak)
                push!(y_filled, right_value)
            end

            zL = zbreak
            current_word = right_word
            current_value = right_value
        end
    end

    push!(
        segments,
        (
            zL=zL,
            zR=zmax,
            word=current_word,
            value=current_value
        )
    )

    push!(xsline, zL)
    push!(ysline, current_value)

    push!(xsline, zmax)
    push!(ysline, current_value)

    endpoints = endpoint_data(segments)

    xbreak, ybreak = break_line_coordinates(
        endpoints.breakpoints
    )

    break_plot = @pgf Plot(
        {
            "no marks",
            "unbounded coords"="jump",
            color="gray!55",
            style="dotted",
            "line width"="0.25pt"
        },
        Table(
            x=xbreak,
            y=ybreak
        )
    )

    main_plot = @pgf Plot(
        {
            "no marks",
            "unbounded coords"="jump",
            "line width"=seg_line_width,
            color="black"
        },
        Table(
            x=xsline,
            y=ysline
        )
    )

    open_mark_plot = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black",
            "mark options"=
                "{fill=white, draw=black, line width=0.25pt}"
        },
        Table(
            x=z_open,
            y=y_open
        )
    )

    filled_mark_plot = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black"
        },
        Table(
            x=z_filled,
            y=y_filled
        )
    )

    axis = @pgf Axis(
        {
            xlabel=raw"$z$",
            ylabel=raw"$\varphi_T(x,z)$",

            width=width,
            height=height,
            "scale only axis",

            xmin=zmin,
            xmax=zmax,
            ymin=0,
            ymax=1,

            ytick="{0,1}",
            yticklabels=raw"{$0$,$1$}",

            xtick=xtick_tex,
            xticklabels=xticklabels_tex,

            "extra x ticks"=endpoints.extra_tick_tex,
            "extra x tick labels"=endpoints.extra_label_tex,

            "extra x tick style"=
                "{tick style={black, line width=" *
                endpoint_tick_width *
                "}}",

            grid="none",
            xmajorgrids="true",
            ymajorgrids="true",
            xminorgrids="false",
            yminorgrids="false",

            "major grid style"=
                "{draw=gray!40, line width=" *
                grid_line_width *
                "}",

            "axis on top"="false"
        },
        diagonal_plot,
        break_plot,
        main_plot,

        # Draw open markers first so a coincident filled marker remains
        # visible and correctly represents the breakpoint value.
        open_mark_plot,
        filled_mark_plot
    )

    return TikzPicture(axis), segments
end

In [ ]:
# -------------------------------------------------------------------
# Parameters and initial belief
# -------------------------------------------------------------------

p01 = 0.25
ρ   = 0.60
κ   = 0.80

x0_fp, x1_fp, _ = fixed_points(
    ;
    p01=p01,
    ρ=ρ,
    κ=κ
)

xinit = (x0_fp + x1_fp) / 2

@printf(
    "x0=%.12f, x1=%.12f, xinit=%.12f\n",
    x0_fp,
    x1_fp,
    xinit
)


# -------------------------------------------------------------------
# Bifurcation diagram
# -------------------------------------------------------------------

zgrid = range(
    0.0,
    1.0;
    length=1001
)

T = 1
mode = :iterate

fig, segments = bifurcation_segments_pgf(
    zgrid;
    p01=p01,
    ρ=ρ,
    κ=κ,
    xinit=xinit,
    T=T,
    mode=mode
)

PGFPlotsX.enable_interactive(false)

pdf_file = "bifurcation_T$(T)_$(string(mode)).pdf"

PGFPlotsX.pgfsave(
    pdf_file,
    fig
)

display(
    "application/pdf",
    read(pdf_file)
)


# -------------------------------------------------------------------
# Helper for finite binary words
# -------------------------------------------------------------------

"""
    finite_word_one_fraction(word)

Return the raw number of ones, the word length, and the reduced
fraction of ones in a nonempty finite binary word.
"""
function finite_word_one_fraction(
    word::AbstractString
)
    ws = _validated_binary_word(
        word;
        allow_empty=false
    )

    word_length = ncodeunits(ws)

    ones = count(
        ==(UInt8('1')),
        codeunits(ws)
    )

    divisor = gcd(
        ones,
        word_length
    )

    fraction_string = string(
        ones ÷ divisor,
        "/",
        word_length ÷ divisor
    )

    return (
        ones,
        word_length,
        fraction_string
    )
end


# -------------------------------------------------------------------
# Segment report
# -------------------------------------------------------------------

if mode == :limitcycle
    println("\nLimiting-cycle segments:")

    for (j, seg) in enumerate(segments)
        right_delimiter =
            j == lastindex(segments) ?
            "]" :
            ")"

        interval = @sprintf(
            "[%.12g, %.12g%s",
            seg.zL,
            seg.zR,
            right_delimiter
        )

        if isempty(seg.s)
            println(
                interval,
                "  no periodic tail detected",
                "  pattern: ",
                seg.formatted
            )

            continue
        end

        outseg = (
            found=true,
            s=seg.s
        )

        ones, period, fraction_string =
            cycle_one_fraction(outseg)

        println(
            interval,
            "  s=\"",
            seg.s,
            "\"",
            "  ones=",
            ones,
            "/",
            period,
            " (reduced ",
            fraction_string,
            ")",
            "  pattern: ",
            seg.formatted
        )
    end

elseif mode == :iterate
    println("\nFinite-itinerary subsegments:")

    println(
        "Each reported word contains A₀,…,A_T ",
        "and therefore has length ",
        T + 1,
        "."
    )

    # Each returned segment has constant:
    #
    #     A₀,...,A_{T-1}
    #
    # and a constant value X_T. The last action is
    #
    #     A_T = 1  iff  X_T > z.
    #
    # Therefore, split a segment at z=X_T whenever X_T lies
    # in the interior of that segment.

    for seg in segments
        ncodeunits(seg.word) == T ||
            error(
                "Expected a word of length $T, " *
                "but received \"$(seg.word)\"."
            )

        zL = seg.zL
        zR = seg.zR
        XT = seg.value

        # Active final action: z < X_T.
        active_right = min(
            zR,
            XT
        )

        if zL < active_right
            full_word = string(
                seg.word,
                '1'
            )

            ones, word_length, fraction_string =
                finite_word_one_fraction(full_word)

            @printf(
                "[%.12g, %.12g)  itinerary=\"%s\"  ones=%d/%d (reduced %s)  value=%.12f\n",
                zL,
                active_right,
                full_word,
                ones,
                word_length,
                fraction_string,
                XT
            )
        end

        # Passive final action: z ≥ X_T. Equality belongs here.
        passive_left = max(
            zL,
            XT
        )

        if passive_left < zR
            full_word = string(
                seg.word,
                '0'
            )

            ones, word_length, fraction_string =
                finite_word_one_fraction(full_word)

            @printf(
                "[%.12g, %.12g)  itinerary=\"%s\"  ones=%d/%d (reduced %s)  value=%.12f\n",
                passive_left,
                zR,
                full_word,
                ones,
                word_length,
                fraction_string,
                XT
            )
        end
    end

    # The preceding intervals are half-open. Report z=max(zgrid)
    # separately so the terminal threshold is covered exactly.

    zmax = Float64(last(zgrid))
    final_segment = last(segments)

    final_symbol =
        final_segment.value > zmax ?
        '1' :
        '0'

    final_word = string(
        final_segment.word,
        final_symbol
    )

    ones, word_length, fraction_string =
        finite_word_one_fraction(final_word)

    @printf(
        "{%.12g}             itinerary=\"%s\"  ones=%d/%d (reduced %s)  value=%.12f\n",
        zmax,
        final_word,
        ones,
        word_length,
        fraction_string,
        final_segment.value
    )

else
    error(
        "Unknown mode: $mode"
    )
end

In [ ]:
# -------------------------------------------------------------------
# Rate of ones in a finite binary word
# -------------------------------------------------------------------

"""
    ones_rate_from_word(word)

Return the reduced fraction and numerical rate of ones in a nonempty
finite binary word.

For example,

    ones_rate_from_word("010") == ("1/3", 1/3)
"""
function ones_rate_from_word(
    word::AbstractString
)
    ws = String(word)

    isempty(ws) &&
        throw(
            ArgumentError(
                "The binary word must be nonempty."
            )
        )

    all(
        byte ->
            byte == UInt8('0') ||
            byte == UInt8('1'),
        codeunits(ws)
    ) ||
        throw(
            ArgumentError(
                "The word must contain only '0' and '1'."
            )
        )

    word_length = ncodeunits(ws)

    ones = count(
        ==(UInt8('1')),
        codeunits(ws)
    )

    divisor = gcd(
        ones,
        word_length
    )

    fraction_string = string(
        ones ÷ divisor,
        "/",
        word_length ÷ divisor
    )

    return (
        fraction_string,
        ones / word_length
    )
end


"""
    bifurcation_ones_segments_pgf(
        zgrid;
        p01,
        ρ,
        κ,
        xinit=p01+ρ,
        T=1,
        ...
    )

Plot the rate of ones in the finite skeleton itinerary

    w_T(z) = Ã₀(z)⋯Ã_{T-1}(z),

whose length is exactly `T`.

Internally, this is obtained from

    skeleton_itinerary_string(xinit,z,T-1; ...).

Thus, when `T=1`,

    w₁(z) = "1"  if z < xinit,
    w₁(z) = "0"  if z ≥ xinit.

The plot uses the right-continuity convention induced by the rule

    Ã_t = 1  iff  X̃_t > z.

At a genuine rate discontinuity:

- an open marker shows the left-hand limit;
- a filled marker shows the value at the breakpoint.

The returned segments have fields

    (zL, zR, word, ones_str, ones_val).

Segments are distinguished by their complete itinerary words. Therefore,
two adjacent segments may have different words but the same rate of ones.

The initial grid must be sufficiently fine to expose the itinerary
changes of interest. Bisection refines detected transitions, but cannot
detect a change that occurs and reverses entirely between consecutive
grid points.

Return `(TikzPicture, segments)`.
"""
function bifurcation_ones_segments_pgf(
    zgrid;
    p01,
    ρ,
    κ,
    xinit=p01 + ρ,
    T::Int=1,
    width::AbstractString="10cm",
    height::AbstractString="10cm",
    seg_line_width::AbstractString="0.70pt",
    grid_line_width::AbstractString="0.08pt",
    endpoint_tick_width::AbstractString="0.20pt",
    mark_size::AbstractString="1.8pt",
    square::Bool=true,
    refine_breaks::Bool=true,
    refine_iters::Int=60
)
    # ---------------------------------------------------------------
    # Validation
    # ---------------------------------------------------------------

    T ≥ 1 ||
        throw(
            ArgumentError(
                "T is the itinerary length and must be at least 1."
            )
        )

    refine_iters ≥ 1 ||
        throw(
            ArgumentError(
                "refine_iters must be positive."
            )
        )

    xinit isa Real && isfinite(xinit) ||
        throw(
            ArgumentError(
                "xinit must be a finite real number."
            )
        )

    0.0 ≤ xinit ≤ 1.0 ||
        throw(
            ArgumentError(
                "xinit must lie in [0,1]."
            )
        )

    zs = Float64.(collect(zgrid))

    length(zs) ≥ 2 ||
        throw(
            ArgumentError(
                "zgrid must contain at least two points."
            )
        )

    all(isfinite, zs) ||
        throw(
            ArgumentError(
                "All threshold-grid values must be finite."
            )
        )

    all(diff(zs) .> 0.0) ||
        throw(
            ArgumentError(
                "zgrid must be strictly increasing."
            )
        )

    0.0 ≤ first(zs) &&
        last(zs) ≤ 1.0 ||
        throw(
            ArgumentError(
                "All threshold-grid values must lie in [0,1]."
            )
        )

    zmin = first(zs)
    zmax = last(zs)

    if square
        height = width
    end

    comparison_tolerance = 1e-12


    # ---------------------------------------------------------------
    # Itinerary evaluation and caching
    # ---------------------------------------------------------------

    word_cache = Dict{Float64,String}()

    function word_at(z)
        zvalue = Float64(z)

        return get!(word_cache, zvalue) do
            word = skeleton_itinerary_string(
                xinit,
                zvalue,
                T - 1;
                p01=p01,
                ρ=ρ,
                κ=κ
            )

            ncodeunits(word) == T ||
                error(
                    "Expected an itinerary of length $T, " *
                    "but received \"$(word)\"."
                )

            word
        end
    end


    # ---------------------------------------------------------------
    # Breakpoint refinement
    # ---------------------------------------------------------------

    """
    Refine an interval whose endpoint itinerary words differ.

    The returned `left` retains `left_word`, whereas `right` belongs
    to the regime immediately selected on the other side, up to
    floating-point resolution.
    """
    function refine_word_boundary(
        zleft::Real,
        zright::Real,
        left_word::AbstractString
    )
        lo = Float64(zleft)
        hi = Float64(zright)

        lo < hi ||
            throw(
                ArgumentError(
                    "A refinement interval must have positive length."
                )
            )

        word_at(lo) == left_word ||
            throw(
                ArgumentError(
                    "The left endpoint does not have the supplied word."
                )
            )

        word_at(hi) != left_word ||
            throw(
                ArgumentError(
                    "The endpoint itinerary words do not differ."
                )
            )

        for _ in 1:refine_iters
            mid = lo + (hi - lo) / 2

            # No further floating-point refinement is possible.
            (mid == lo || mid == hi) &&
                break

            if word_at(mid) == left_word
                lo = mid
            else
                hi = mid
            end
        end

        return (
            left=lo,
            right=hi,
            right_word=word_at(hi)
        )
    end


    grid_words = [
        word_at(z)
        for z in zs
    ]


    # ---------------------------------------------------------------
    # Construct word segments and rate-jump markers
    # ---------------------------------------------------------------

    segments = NamedTuple[]

    xsline = Float64[]
    ysline = Float64[]

    # Open markers represent left limits.
    z_open = Float64[]
    y_open = Float64[]

    # Filled markers represent values at breakpoints.
    z_filled = Float64[]
    y_filled = Float64[]

    zL = zmin
    current_word = first(grid_words)

    current_ones_string, current_ones_value =
        ones_rate_from_word(current_word)

    for i in 2:length(zs)
        if grid_words[i] != current_word
            boundary =
                if refine_breaks
                    refine_word_boundary(
                        zs[i - 1],
                        zs[i],
                        current_word
                    )
                else
                    (
                        left=zs[i - 1],
                        right=zs[i],
                        right_word=grid_words[i]
                    )
                end

            zbreak = boundary.right
            right_word = boundary.right_word

            # Under the assumed sufficiently fine grid, the regime
            # immediately to the right must be the one observed at zs[i].
            right_word == grid_words[i] ||
                throw(
                    ArgumentError(
                        "More than one itinerary transition may lie " *
                        "between zgrid[$(i-1)] and zgrid[$i]. " *
                        "Use a finer threshold grid."
                    )
                )

            right_ones_string, right_ones_value =
                ones_rate_from_word(right_word)

            push!(
                segments,
                (
                    zL=zL,
                    zR=zbreak,
                    word=current_word,
                    ones_str=current_ones_string,
                    ones_val=current_ones_value
                )
            )

            push!(xsline, zL)
            push!(ysline, current_ones_value)

            push!(xsline, zbreak)
            push!(ysline, current_ones_value)

            push!(xsline, NaN)
            push!(ysline, NaN)

            # Different words need not have different rates. Add
            # discontinuity markers only when the plotted rate changes.
            if current_ones_value != right_ones_value
                push!(z_open, zbreak)
                push!(y_open, current_ones_value)

                push!(z_filled, zbreak)
                push!(y_filled, right_ones_value)
            end

            zL = zbreak
            current_word = right_word
            current_ones_string = right_ones_string
            current_ones_value = right_ones_value
        end
    end

    # Final segment
    push!(
        segments,
        (
            zL=zL,
            zR=zmax,
            word=current_word,
            ones_str=current_ones_string,
            ones_val=current_ones_value
        )
    )

    push!(xsline, zL)
    push!(ysline, current_ones_value)

    push!(xsline, zmax)
    push!(ysline, current_ones_value)


    # ---------------------------------------------------------------
    # Structural x-axis ticks
    # ---------------------------------------------------------------

    x0_tick, x1_tick, _ = fixed_points(
        ;
        p01=p01,
        ρ=ρ,
        κ=κ
    )

    p11 = p01 + ρ

    tick_candidates = [
        (0.0,     raw"$0$"),
        (p01,     raw"$p_{01}$"),
        (x1_tick, raw"$x^1$"),
        (x0_tick, raw"$x^0$"),
        (p11,     raw"$p_{01}+\rho$"),
        (1.0,     raw"$1$")
    ]

    xticks = Float64[]
    xticklabels = String[]

    # Preserve the first label when structural ticks coincide.
    for (tick, label) in tick_candidates
        if zmin - comparison_tolerance ≤
                tick ≤
                zmax + comparison_tolerance &&
                all(
                    abs(tick - existing_tick) >
                    comparison_tolerance
                    for existing_tick in xticks
                )

            push!(xticks, Float64(tick))
            push!(xticklabels, String(label))
        end
    end

    # Include xinit unless it coincides with a structural tick.
    if zmin - comparison_tolerance ≤
            xinit ≤
            zmax + comparison_tolerance &&
            all(
                abs(xinit - tick) >
                comparison_tolerance
                for tick in xticks
            )

        push!(xticks, Float64(xinit))
        push!(xticklabels, String(raw"$x$"))
    end

    permutation = sortperm(xticks)

    xticks = xticks[permutation]
    xticklabels = xticklabels[permutation]

    xtick_tex = tex_num_list(xticks)
    xticklabels_tex = tex_label_list(xticklabels)


    # ---------------------------------------------------------------
    # Unlabelled ticks at all itinerary-segment endpoints
    # ---------------------------------------------------------------

    endpoints = sort(
        unique(
            vcat(
                [segment.zL for segment in segments],
                [segment.zR for segment in segments]
            )
        )
    )

    extra_xticks = [
        endpoint
        for endpoint in endpoints
        if all(
            abs(endpoint - tick) >
            comparison_tolerance
            for tick in xticks
        )
    ]

    extra_xticklabels = fill(
        raw"\empty",
        length(extra_xticks)
    )

    extra_xtick_tex = tex_num_list(extra_xticks)
    extra_xticklabels_tex =
        tex_label_list(extra_xticklabels)


    # ---------------------------------------------------------------
    # PGFPlotsX objects
    # ---------------------------------------------------------------

    rate_plot = @pgf Plot(
        {
            "no marks",
            "unbounded coords"="jump",
            "line width"=seg_line_width,
            color="black"
        },
        Table(
            x=xsline,
            y=ysline
        )
    )

    open_mark_plot = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black",
            "mark options"=
                "{fill=white, draw=black, line width=0.25pt}"
        },
        Table(
            x=z_open,
            y=y_open
        )
    )

    filled_mark_plot = @pgf Plot(
        {
            "only marks",
            mark="*",
            "mark size"=mark_size,
            color="black"
        },
        Table(
            x=z_filled,
            y=y_filled
        )
    )

    axis = @pgf Axis(
        {
            xlabel=raw"$z$",
            ylabel=raw"$T^{-1}\sum_{t=0}^{T-1}\widetilde A_t$",

            width=width,
            height=height,
            "scale only axis",

            xmin=zmin,
            xmax=zmax,
            ymin=0,
            ymax=1,

            ytick="{0,1}",
            yticklabels=raw"{$0$,$1$}",

            xtick=xtick_tex,
            xticklabels=xticklabels_tex,

            "extra x ticks"=extra_xtick_tex,
            "extra x tick labels"=extra_xticklabels_tex,

            "extra x tick style"=
                "{tick style={black, line width=" *
                endpoint_tick_width *
                "}}",

            grid="none",
            xmajorgrids="true",
            ymajorgrids="true",
            xminorgrids="false",
            yminorgrids="false",

            "major grid style"=
                "{draw=gray!40, line width=" *
                grid_line_width *
                "}",

            "axis on top"="false"
        },
        rate_plot,

        # Open markers are drawn first so that the filled value remains
        # visible if numerical values happen to coincide.
        open_mark_plot,
        filled_mark_plot
    )

    return TikzPicture(axis), segments
end

In [ ]:
using Printf
using PGFPlotsX


# -------------------------------------------------------------------
# Rate-of-ones bifurcation diagram
# -------------------------------------------------------------------

zgrid = range(
    0.0,
    1.0;
    length=2001
)

T_rate = 4

fig, segments_rate = bifurcation_ones_segments_pgf(
    zgrid;
    p01=p01,
    ρ=ρ,
    κ=κ,
    xinit=xinit,
    T=T_rate
)

PGFPlotsX.enable_interactive(false)

pdf_file_rate = "bifurcation_ones_T$(T_rate).pdf"

PGFPlotsX.pgfsave(
    pdf_file_rate,
    fig
)

display(
    "application/pdf",
    read(pdf_file_rate)
)


# -------------------------------------------------------------------
# Segment report
# -------------------------------------------------------------------

println(
    "\nRate-of-ones segments for itineraries of length ",
    T_rate,
    ":"
)

for (j, segment) in enumerate(segments_rate)
    ncodeunits(segment.word) == T_rate ||
        error(
            "Expected a word of length $T_rate, " *
            "but received \"$(segment.word)\"."
        )

    # All intervals are left-closed. The last one also includes z=1.
    right_delimiter =
        j == lastindex(segments_rate) ?
        "]" :
        ")"

    number_of_ones = count(
        ==(UInt8('1')),
        codeunits(segment.word)
    )

    @printf(
        "[%.12g, %.12g%s  word=\"%s\"  ones=%d/%d (reduced %s)  rate=%.12f\n",
        segment.zL,
        segment.zR,
        right_delimiter,
        segment.word,
        number_of_ones,
        T_rate,
        segment.ones_str,
        segment.ones_val
    )
end

In [ ]:
"""
    mword(α; tol=1e-12)

Return the shortest primitive binary word `w` such that its infinite
repetition satisfies

    (w^∞)_j = floor(α*(j+1)) - floor(α*j),    j ≥ 0.

This is the lower mechanical word of slope `α` and intercept zero,
represented by its primitive period.

For an exact rational input

    α = num//den,

the returned word has `den` symbols before primitive-period reduction.
Here `den` is the denominator of the rational slope and is unrelated
to the sensing-model parameter `p01`.

For a floating-point input, the function first replaces `α` by

    rationalize(Float64(α); tol=tol).

Consequently, for an inexact input, the returned word corresponds to
that rational approximation.

The output is a `String` containing only `'0'` and `'1'`.
"""
function mword(
    α::Real;
    tol::Real=1e-12
)
    # ---------------------------------------------------------------
    # Validate the input
    # ---------------------------------------------------------------

    isfinite(α) ||
        throw(
            ArgumentError(
                "α must be finite; received α=$α."
            )
        )

    0 ≤ α ≤ 1 ||
        throw(
            ArgumentError(
                "α must lie in [0,1]; received α=$α."
            )
        )

    isfinite(tol) && tol ≥ 0 ||
        throw(
            ArgumentError(
                "tol must be finite and nonnegative; received tol=$tol."
            )
        )


    # ---------------------------------------------------------------
    # Obtain an exact rational slope
    # ---------------------------------------------------------------

    slope =
        if α isa Rational
            α
        elseif α isa Integer
            α // one(α)
        else
            rationalize(
                Float64(α);
                tol=Float64(tol)
            )
        end

    num = numerator(slope)
    den_exact = denominator(slope)

    0 ≤ num ≤ den_exact ||
        throw(
            ArgumentError(
                "The rationalized slope must lie in [0,1]; " *
                "received slope=$slope."
            )
        )


    # ---------------------------------------------------------------
    # Constant mechanical words
    # ---------------------------------------------------------------

    if num == 0
        return "0"
    elseif num == den_exact
        return "1"
    end


    # ---------------------------------------------------------------
    # Convert the rational denominator into a feasible array length
    # ---------------------------------------------------------------

    den_exact ≤ typemax(Int) ||
        throw(
            ArgumentError(
                "The rational denominator $den_exact is too large " *
                "to use as a word length."
            )
        )

    den = Int(den_exact)


    # ---------------------------------------------------------------
    # Construct the length-den lower mechanical word
    #
    # Its jth symbol is
    #
    #   floor(num*j/den) - floor(num*(j-1)/den),
    #
    # for j=1,...,den.
    #
    # The accumulator formulation avoids products such as num*j.
    # ---------------------------------------------------------------

    bytes = Vector{UInt8}(
        undef,
        den
    )

    remainder = zero(num)
    gap = den_exact - num

    @inbounds for j in 1:den
        # remainder + num ≥ den_exact iff remainder ≥ den_exact-num.
        if remainder ≥ gap
            bytes[j] = UInt8('1')
            remainder -= gap
        else
            bytes[j] = UInt8('0')
            remainder += num
        end
    end

    return _primitive_period(
        String(bytes)
    )
end


# -------------------------------------------------------------------
# Internal helper: prefix function used to find the primitive period
# -------------------------------------------------------------------

function _prefix_function(
    bytes::AbstractVector{UInt8}
)
    n = length(bytes)
    prefix = zeros(Int, n)

    matched = 0

    @inbounds for i in 2:n
        while matched > 0 &&
                bytes[i] != bytes[matched + 1]

            matched = prefix[matched]
        end

        if bytes[i] == bytes[matched + 1]
            matched += 1
        end

        prefix[i] = matched
    end

    return prefix
end


# -------------------------------------------------------------------
# Internal helper: shortest block whose repetitions reproduce the word
# -------------------------------------------------------------------

function _primitive_period(
    word::AbstractString
)
    bytes = collect(
        codeunits(word)
    )

    n = length(bytes)

    n == 0 &&
        return ""

    prefix = _prefix_function(bytes)

    candidate_period =
        n - prefix[end]

    if candidate_period < n &&
            n % candidate_period == 0

        return String(
            bytes[1:candidate_period]
        )
    end

    return String(bytes)
end

In [ ]:
# -------------------------------------------------------------------
# Examples of lower mechanical words
# -------------------------------------------------------------------

α₁ = 3 // 5
α₂ = 2 // 3

# The midpoint is computed exactly:
#
#     ((3/5) + (2/3))/2 = 19/30.
#
α_mid = (α₁ + α₂) / 2

word₁ = mword(α₁)
word_mid = mword(α_mid)

println(
    "mword(",
    α₁,
    ") = \"",
    word₁,
    "\""
)

println(
    "mword(",
    α_mid,
    ") = \"",
    word_mid,
    "\""
)

# Sanity checks
@assert α_mid == 19 // 30
@assert word₁ == "01011"
@assert length(word₁) == denominator(α₁)
@assert count(==('1'), word₁) == numerator(α₁)
@assert length(word_mid) == denominator(α_mid)
@assert count(==('1'), word_mid) == numerator(α_mid)

In [ ]:
"""
    christoffel_conjugates_lstep(word)

Given a nonempty binary Christoffel word

    word = w₀w₁⋯w_{n-1},

let

    n = length(word),
    m = number of ones in word.

For `n > 1`, let `l` be the modular inverse of `m` modulo `n`:

    mod(l*m, n) == 1.

Return the conjugates in `l`-step order:

    u(0), u(l), u(2l), …, u((n-1)l),

where `u(k)` is the left rotation

    u(k) = w_k⋯w_{n-1}w₀⋯w_{k-1}.

For a one-letter word, the only rotation is the word itself, and the
convention `l = 0` is used.

The function checks that:

- `word` is nonempty and binary;
- `gcd(m,n) == 1`, as required for the modular inverse.

It assumes, but does not independently verify, that the supplied word has
the Christoffel structure.

Return

    conjugates, l, m, n

where `conjugates` is a `Vector{String}` of length `n`.
"""
function christoffel_conjugates_lstep(
    word::AbstractString
)
    word_string = String(word)
    bytes = collect(codeunits(word_string))
    n = length(bytes)

    n > 0 ||
        throw(
            ArgumentError(
                "The Christoffel word must be nonempty."
            )
        )

    all(
        byte ->
            byte == UInt8('0') ||
            byte == UInt8('1'),
        bytes
    ) ||
        throw(
            ArgumentError(
                "The word must contain only '0' and '1'."
            )
        )

    m = count(
        ==(UInt8('1')),
        bytes
    )

    gcd(m, n) == 1 ||
        throw(
            ArgumentError(
                "The number of ones m=$m and the word length n=$n " *
                "must be coprime; the modular inverse does not exist."
            )
        )

    # For n=1, there is only one rotation. For n>1, compute the
    # modular inverse of m modulo n.
    l =
        if n == 1
            0
        else
            invmod(m, n)
        end

    function rotate_left(k::Int)
        k = mod(k, n)

        k == 0 &&
            return word_string

        rotated_bytes = Vector{UInt8}(
            undef,
            n
        )

        tail_length = n - k

        copyto!(
            rotated_bytes,
            1,
            bytes,
            k + 1,
            tail_length
        )

        copyto!(
            rotated_bytes,
            tail_length + 1,
            bytes,
            1,
            k
        )

        return String(rotated_bytes)
    end

    conjugates = Vector{String}(
        undef,
        n
    )

    for j in 0:(n - 1)
        rotation_index = mod(
            j * l,
            n
        )

        conjugates[j + 1] =
            rotate_left(rotation_index)
    end

    return (
        conjugates,
        l,
        m,
        n
    )
end

In [ ]:
# -------------------------------------------------------------------
# Christoffel conjugates in modular-step order
# -------------------------------------------------------------------

α = 3 // 4
word = mword(α)

@assert word == "0111"

conjugates, step, ones, period =
    christoffel_conjugates_lstep(word)

println(
    "α=$α, word=\"$word\", ",
    "n=$period, m=$ones, l=$step"
)

println("Conjugates in l-step order:")

for (j, conjugate) in enumerate(conjugates)
    rotation_index = mod(
        (j - 1) * step,
        period
    )

    println(
        "u($rotation_index) = \"",
        conjugate,
        "\""
    )
end

# Sanity checks
@assert step == 3
@assert conjugates == [
    "0111",
    "1011",
    "1101",
    "1110"
]

In [ ]:
using Base.Threads
using LinearAlgebra


# Use Julia threads across parameter tuples, avoiding nested
# multithreading inside BLAS calls.
BLAS.set_num_threads(1)


# -------------------------------------------------------------------
# Endpoint-dense grid on [0,1]
# -------------------------------------------------------------------

"""
    cheb01(n)

Return `n` Chebyshev–Lobatto points mapped increasingly onto `[0,1]`.
"""
function cheb01(
    n::Integer
)
    n ≥ 2 ||
        throw(
            ArgumentError(
                "n must be at least 2; received n=$n."
            )
        )

    θgrid = range(
        0.0,
        π;
        length=Int(n)
    )

    grid = (
        1 .- cos.(θgrid)
    ) ./ 2

    # Guarantee exact endpoints.
    grid[begin] = 0.0
    grid[end] = 1.0

    return grid
end


# -------------------------------------------------------------------
# Discount-factor grid
# -------------------------------------------------------------------

"""
    beta_grid(; nmid=9, nnear=10, δmin=1e-3, δmax=0.1)

Construct a discount-factor grid consisting of:

- an evenly spaced grid from `0.1` to `0.9`;
- a logarithmically spaced grid of discount gaps
  `δ = 1-β` between `δmax` and `δmin`.

Duplicate discount factors are removed.
"""
function beta_grid(
    ;
    nmid::Integer=9,
    nnear::Integer=10,
    δmin::Real=1e-3,
    δmax::Real=0.1
)
    nmid ≥ 1 ||
        throw(
            ArgumentError(
                "nmid must be positive."
            )
        )

    nnear ≥ 1 ||
        throw(
            ArgumentError(
                "nnear must be positive."
            )
        )

    δmin_value = Float64(δmin)
    δmax_value = Float64(δmax)

    0.0 < δmin_value ≤ δmax_value < 1.0 ||
        throw(
            ArgumentError(
                "Require 0 < δmin ≤ δmax < 1."
            )
        )

    βmid = collect(
        range(
            0.1,
            0.9;
            length=Int(nmid)
        )
    )

    δnear = exp.(
        range(
            log(δmax_value),
            log(δmin_value);
            length=Int(nnear)
        )
    )

    βnear = 1 .- δnear

    return sort!(
        unique(
            vcat(
                βmid,
                βnear
            )
        )
    )
end


# -------------------------------------------------------------------
# Parameter tuples
# -------------------------------------------------------------------

"""
    make_params(p01_grid, α_grid, κ_grid, β_grid_values)

Construct parameter tuples using

    ρ = α*(1-p01).

Thus `α` is the normalized persistence parameter, while `p01` is the
model transition probability. No quantity called `q` is used here.
"""
function make_params(
    p01_grid,
    α_grid,
    κ_grid,
    β_grid_values
)
    p01_values = Float64.(collect(p01_grid))
    α_values = Float64.(collect(α_grid))
    κ_values = Float64.(collect(κ_grid))
    β_values = Float64.(collect(β_grid_values))

    all(p01 -> 0.0 < p01 < 1.0, p01_values) ||
        throw(
            ArgumentError(
                "Every p01 value must lie strictly between 0 and 1."
            )
        )

    all(α -> 0.0 ≤ α < 1.0, α_values) ||
        throw(
            ArgumentError(
                "Every α value must satisfy 0 ≤ α < 1."
            )
        )

    all(κ -> 0.0 < κ ≤ 1.0, κ_values) ||
        throw(
            ArgumentError(
                "Every κ value must satisfy 0 < κ ≤ 1."
            )
        )

    all(β -> 0.0 < β < 1.0, β_values) ||
        throw(
            ArgumentError(
                "Every β value must lie strictly between 0 and 1."
            )
        )

    return vec(
        [
            (
                p01=p01,
                ρ=α * (1.0 - p01),
                κ=κ,
                β=β
            )
            for p01 in p01_values,
                α in α_values,
                κ in κ_values,
                β in β_values
        ]
    )
end


# -------------------------------------------------------------------
# Scan for one parameter tuple
# -------------------------------------------------------------------

"""
    scan_one_tuple(; p01, ρ, κ, β, Nx=51, Nu=51, ...)

Numerically scan the PCLI1 slack

    g(x,z) - (1-β)

over

    x ∈ [0,1],
    z ∈ [x¹,x⁰].

The variable `u ∈ [0,1]` parameterizes the threshold as

    z = x¹ + u*(x⁰-x¹).

Return the minimum absolute and relative slack and the corresponding
grid point.
"""
function scan_one_tuple(
    ;
    p01::Real,
    ρ::Real,
    κ::Real,
    β::Real,
    Nx::Integer=51,
    Nu::Integer=51,
    ε::Real=1e-10,
    endpoint_dense::Bool=true
)
    Nx ≥ 2 ||
        throw(
            ArgumentError(
                "Nx must be at least 2."
            )
        )

    Nu ≥ 2 ||
        throw(
            ArgumentError(
                "Nu must be at least 2."
            )
        )

    ε > 0 && isfinite(ε) ||
        throw(
            ArgumentError(
                "ε must be finite and strictly positive."
            )
        )

    p01_value = Float64(p01)
    ρ_value = Float64(ρ)
    κ_value = Float64(κ)
    β_value = Float64(β)
    ε_value = Float64(ε)

    0.0 < p01_value < 1.0 ||
        throw(
            ArgumentError(
                "Require 0 < p01 < 1."
            )
        )

    0.0 ≤ ρ_value < 1.0 - p01_value ||
        throw(
            ArgumentError(
                "Require 0 ≤ ρ < 1-p01."
            )
        )

    0.0 < κ_value ≤ 1.0 ||
        throw(
            ArgumentError(
                "Require 0 < κ ≤ 1."
            )
        )

    0.0 < β_value < 1.0 ||
        throw(
            ArgumentError(
                "Require 0 < β < 1."
            )
        )

    x0, x1, _ = fixed_points(
        ;
        p01=p01_value,
        ρ=ρ_value,
        κ=κ_value
    )

    x0 = Float64(x0)
    x1 = Float64(x1)

    isfinite(x0) && isfinite(x1) ||
        error(
            "fixed_points returned nonfinite values: " *
            "x0=$x0 and x1=$x1."
        )

    x1 ≤ x0 ||
        error(
            "Expected x1 ≤ x0, but obtained x1=$x1 and x0=$x0."
        )

    xgrid =
        endpoint_dense ?
        cheb01(Nx) :
        collect(
            range(
                0.0,
                1.0;
                length=Int(Nx)
            )
        )

    ugrid =
        endpoint_dense ?
        cheb01(Nu) :
        collect(
            range(
                0.0,
                1.0;
                length=Int(Nu)
            )
        )

    zgrid = x1 .+ ugrid .* (x0 - x1)

    # Guarantee the exact fixed-point endpoints.
    zgrid[begin] = x1
    zgrid[end] = x0

    discount_gap = 1.0 - β_value

    min_slack = Inf
    xstar = NaN
    zstar = NaN

    for x in xgrid
        for z in zgrid
            g = Float64(
                g_metric2(
                    x,
                    z;
                    p01=p01_value,
                    ρ=ρ_value,
                    κ=κ_value,
                    β=β_value,
                    ε=ε_value
                )
            )

            isfinite(g) ||
                error(
                    "g_metric2 returned $g at " *
                    "(p01,ρ,κ,β,x,z)=" *
                    "($p01_value,$ρ_value,$κ_value,$β_value,$x,$z)."
                )

            slack = g - discount_gap

            if slack < min_slack
                min_slack = slack
                xstar = x
                zstar = z
            end
        end
    end

    return (
        min_slack=min_slack,
        rel_slack=min_slack / discount_gap,
        xstar=xstar,
        zstar=zstar,
        x0=x0,
        x1=x1
    )
end


# -------------------------------------------------------------------
# Threaded scan over parameter tuples
# -------------------------------------------------------------------

"""
    threaded_scan(params; ...)

Evaluate `scan_one_tuple` for every parameter tuple, distributing the
tuples dynamically across Julia threads. Results preserve the order of
`params`.
"""
function threaded_scan(
    params::AbstractVector;
    Nx::Integer=51,
    Nu::Integer=51,
    ε::Real=1e-10,
    endpoint_dense::Bool=true
)
    isempty(params) &&
        return NamedTuple[]

    parameter_values = collect(params)

    function evaluate(parameter)
        scan_result = scan_one_tuple(
            ;
            parameter...,
            Nx=Nx,
            Nu=Nu,
            ε=ε,
            endpoint_dense=endpoint_dense
        )

        return merge(
            parameter,
            scan_result
        )
    end

    # Evaluate one tuple first to infer a concrete result type.
    first_result = evaluate(
        parameter_values[1]
    )

    results = Vector{typeof(first_result)}(
        undef,
        length(parameter_values)
    )

    results[1] = first_result

    Threads.@threads :dynamic for i in 2:length(parameter_values)
        results[i] = evaluate(
            parameter_values[i]
        )
    end

    return results
end

In [ ]:
using Base.Threads
using LinearAlgebra
using Dates
using Printf
using DataFrames
using CSV


# Use Julia threads across parameter tuples, without nested BLAS threads.
BLAS.set_num_threads(1)


# -------------------------------------------------------------------
# Fixed discount-factor grid
# -------------------------------------------------------------------

function beta_grid_11()
    return Float64[
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.70,
        0.80,
        0.90,
        0.95,
        0.99
    ]
end


# -------------------------------------------------------------------
# One parameter tuple
# -------------------------------------------------------------------

"""
    scan_one_tuple_fg(; p01, ρ, κ, β, r=1.0, ...)

For one parameter tuple, scan

    x ∈ [0,1],
    z ∈ [x¹,x⁰],

and record the sampled minima of

    f(x,z) - (1-β),
    f(x,z),
    f(x,z)/(1-β),
    g(x,z) - f(x,z),
    g(x,z) - (1-β).

The variable `u ∈ [0,1]` parameterizes the threshold through

    z = x¹ + u*(x⁰-x¹).
"""
function scan_one_tuple_fg(
    ;
    p01::Real,
    ρ::Real,
    κ::Real,
    β::Real,
    r::Real=1.0,
    Nx::Integer=121,
    Nu::Integer=121,
    ε::Real=1e-8,
    endpoint_dense::Bool=true
)
    Nx ≥ 2 ||
        throw(ArgumentError("Nx must be at least 2."))

    Nu ≥ 2 ||
        throw(ArgumentError("Nu must be at least 2."))

    p01_value = Float64(p01)
    ρ_value   = Float64(ρ)
    κ_value   = Float64(κ)
    β_value   = Float64(β)
    r_value   = Float64(r)
    ε_value   = Float64(ε)

    all(
        isfinite,
        (
            p01_value,
            ρ_value,
            κ_value,
            β_value,
            r_value,
            ε_value
        )
    ) ||
        throw(
            ArgumentError(
                "All scalar parameters must be finite."
            )
        )

    0.0 < p01_value < 1.0 ||
        throw(ArgumentError("Require 0 < p01 < 1."))

    0.0 ≤ ρ_value < 1.0 - p01_value ||
        throw(ArgumentError("Require 0 ≤ ρ < 1-p01."))

    0.0 < κ_value ≤ 1.0 ||
        throw(ArgumentError("Require 0 < κ ≤ 1."))

    0.0 < β_value < 1.0 ||
        throw(ArgumentError("Require 0 < β < 1."))

    r_value > 0.0 ||
        throw(ArgumentError("Require r > 0."))

    ε_value > 0.0 ||
        throw(ArgumentError("Require ε > 0."))

    x0, x1, _ = fixed_points(
        ;
        p01=p01_value,
        ρ=ρ_value,
        κ=κ_value
    )

    x0 = Float64(x0)
    x1 = Float64(x1)

    isfinite(x0) && isfinite(x1) ||
        error(
            "fixed_points returned nonfinite values: " *
            "x0=$x0 and x1=$x1."
        )

    x1 ≤ x0 ||
        error(
            "Expected x1 ≤ x0, but obtained " *
            "x1=$x1 and x0=$x0."
        )

    xgrid =
        endpoint_dense ?
        cheb01(Nx) :
        collect(
            range(
                0.0,
                1.0;
                length=Int(Nx)
            )
        )

    ugrid =
        endpoint_dense ?
        cheb01(Nu) :
        collect(
            range(
                0.0,
                1.0;
                length=Int(Nu)
            )
        )

    zgrid = x1 .+ ugrid .* (x0 - x1)

    # Preserve the exact fixed-point endpoints.
    zgrid[begin] = x1
    zgrid[end] = x0

    discount_gap = 1.0 - β_value

    min_slack_f = Inf
    xstar_f = NaN
    zstar_f = NaN

    min_f = Inf
    xstar_f0 = NaN
    zstar_f0 = NaN

    min_ratio_f = Inf
    xstar_ratio_f = NaN
    zstar_ratio_f = NaN

    min_gap_g_minus_f = Inf
    xstar_gf = NaN
    zstar_gf = NaN

    min_slack_g = Inf
    xstar_g = NaN
    zstar_g = NaN

    for x in xgrid
        for z in zgrid
            f = Float64(
                f_metric2(
                    x,
                    z;
                    p01=p01_value,
                    ρ=ρ_value,
                    κ=κ_value,
                    r=r_value,
                    β=β_value,
                    ε=ε_value
                )
            )

            g = Float64(
                g_metric2(
                    x,
                    z;
                    p01=p01_value,
                    ρ=ρ_value,
                    κ=κ_value,
                    β=β_value,
                    ε=ε_value
                )
            )

            isfinite(f) && isfinite(g) ||
                error(
                    "A metric returned a nonfinite value at " *
                    "(p01,ρ,κ,β,x,z)=" *
                    "($p01_value,$ρ_value,$κ_value," *
                    "$β_value,$x,$z): f=$f, g=$g."
                )

            slack_f = f - discount_gap
            ratio_f = f / discount_gap
            gap_gf   = g - f
            slack_g  = g - discount_gap

            if slack_f < min_slack_f
                min_slack_f = slack_f
                xstar_f = x
                zstar_f = z
            end

            if f < min_f
                min_f = f
                xstar_f0 = x
                zstar_f0 = z
            end

            if ratio_f < min_ratio_f
                min_ratio_f = ratio_f
                xstar_ratio_f = x
                zstar_ratio_f = z
            end

            if gap_gf < min_gap_g_minus_f
                min_gap_g_minus_f = gap_gf
                xstar_gf = x
                zstar_gf = z
            end

            if slack_g < min_slack_g
                min_slack_g = slack_g
                xstar_g = x
                zstar_g = z
            end
        end
    end

    return (
        x0=x0,
        x1=x1,

        min_slack_f=min_slack_f,
        rel_slack_f=min_slack_f / discount_gap,
        xstar_f=xstar_f,
        zstar_f=zstar_f,

        min_f=min_f,
        xstar_f0=xstar_f0,
        zstar_f0=zstar_f0,

        min_ratio_f=min_ratio_f,
        xstar_ratio_f=xstar_ratio_f,
        zstar_ratio_f=zstar_ratio_f,

        min_gap_g_minus_f=min_gap_g_minus_f,
        xstar_gf=xstar_gf,
        zstar_gf=zstar_gf,

        min_slack_g=min_slack_g,
        rel_slack_g=min_slack_g / discount_gap,
        xstar_g=xstar_g,
        zstar_g=zstar_g
    )
end


# -------------------------------------------------------------------
# Threaded outer scan
# -------------------------------------------------------------------

function threaded_scan_fg(
    params::AbstractVector;
    r::Real=1.0,
    Nx::Integer=121,
    Nu::Integer=121,
    ε::Real=1e-8,
    endpoint_dense::Bool=true
)
    isempty(params) &&
        return NamedTuple[]

    parameter_values = collect(params)

    function evaluate(parameter)
        scan_result = scan_one_tuple_fg(
            ;
            parameter...,
            r=r,
            Nx=Nx,
            Nu=Nu,
            ε=ε,
            endpoint_dense=endpoint_dense
        )

        return merge(
            parameter,
            scan_result
        )
    end

    # Evaluate one tuple to determine the concrete result type.
    first_result = evaluate(parameter_values[1])

    results = Vector{typeof(first_result)}(
        undef,
        length(parameter_values)
    )

    results[1] = first_result

    Threads.@threads :dynamic for i in 2:length(parameter_values)
        results[i] = evaluate(
            parameter_values[i]
        )
    end

    return results
end


# -------------------------------------------------------------------
# Chunked experiment
# -------------------------------------------------------------------

function serious_run_fg_chunked(
    ;
    r::Real=1.0,
    Np01::Integer=14,
    Nα::Integer=14,
    Nκ::Integer=14,
    Nx::Integer=121,
    Nu::Integer=121,
    ε::Real=1e-8,
    endpoint_dense::Bool=true,
    chunk_size::Integer=512,
    p01min::Real=0.05,
    p01max::Real=0.95,
    αmin::Real=0.10,
    αmax::Real=0.90,
    κmin::Real=0.05,
    κmax::Real=0.95,
    outdir::AbstractString=
        "serious_run_fg_14x14x14x11_121x121_eps1e8"
)
    Np01 ≥ 1 ||
        throw(ArgumentError("Np01 must be positive."))

    Nα ≥ 1 ||
        throw(ArgumentError("Nα must be positive."))

    Nκ ≥ 1 ||
        throw(ArgumentError("Nκ must be positive."))

    Nx ≥ 2 ||
        throw(ArgumentError("Nx must be at least 2."))

    Nu ≥ 2 ||
        throw(ArgumentError("Nu must be at least 2."))

    chunk_size ≥ 1 ||
        throw(ArgumentError("chunk_size must be positive."))

    0.0 < p01min ≤ p01max < 1.0 ||
        throw(
            ArgumentError(
                "Require 0 < p01min ≤ p01max < 1."
            )
        )

    0.0 ≤ αmin ≤ αmax < 1.0 ||
        throw(
            ArgumentError(
                "Require 0 ≤ αmin ≤ αmax < 1."
            )
        )

    0.0 < κmin ≤ κmax ≤ 1.0 ||
        throw(
            ArgumentError(
                "Require 0 < κmin ≤ κmax ≤ 1."
            )
        )

    isempty(outdir) &&
        throw(
            ArgumentError(
                "outdir must be nonempty."
            )
        )

    println(
        "Threads.nthreads() = ",
        Threads.nthreads()
    )

    p01_grid = collect(
        range(
            Float64(p01min),
            Float64(p01max);
            length=Int(Np01)
        )
    )

    α_grid = collect(
        range(
            Float64(αmin),
            Float64(αmax);
            length=Int(Nα)
        )
    )

    κ_grid = collect(
        range(
            Float64(κmin),
            Float64(κmax);
            length=Int(Nκ)
        )
    )

    β_grid = beta_grid_11()

    # Reuse make_params from the preceding cell:
    #
    #     ρ = α*(1-p01).
    params = make_params(
        p01_grid,
        α_grid,
        κ_grid,
        β_grid
    )

    nparams = length(params)
    nevals_total = nparams * Int(Nx) * Int(Nu)
    nchunks = cld(nparams, Int(chunk_size))

    mkpath(outdir)

    CSV.write(
        joinpath(outdir, "beta_grid.csv"),
        DataFrame(β=β_grid)
    )

    CSV.write(
        joinpath(outdir, "p01_grid.csv"),
        DataFrame(p01=p01_grid)
    )

    CSV.write(
        joinpath(outdir, "alpha_grid.csv"),
        DataFrame(α=α_grid)
    )

    CSV.write(
        joinpath(outdir, "kappa_grid.csv"),
        DataFrame(κ=κ_grid)
    )

    open(
        joinpath(outdir, "run_info.txt"),
        "w"
    ) do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "r = ", r)
        println(io, "Np01 = ", Np01)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(β_grid))
        println(io, "Nx = ", Nx)
        println(io, "Nu = ", Nu)
        println(io, "ε = ", ε)
        println(io, "endpoint_dense = ", endpoint_dense)
        println(io, "chunk_size = ", chunk_size)
        println(
            io,
            "p01 range = [",
            p01min,
            ", ",
            p01max,
            "]"
        )
        println(
            io,
            "α range = [",
            αmin,
            ", ",
            αmax,
            "]"
        )
        println(
            io,
            "κ range = [",
            κmin,
            ", ",
            κmax,
            "]"
        )
        println(io, "parameter tuples = ", nparams)
        println(
            io,
            "total (x,z) evaluations = ",
            nevals_total
        )
        println(io, "sampled checks:")
        println(io, "  f(x,z) ≥ 1-β")
        println(io, "  f(x,z) ≥ 0")
        println(io, "  minimum f(x,z)/(1-β)")
        println(io, "  g(x,z) ≥ f(x,z)")
        println(io, "  g(x,z) ≥ 1-β")
    end

    println("parameter tuples = ", nparams)
    println("Nx = ", Nx, ", Nu = ", Nu)
    println(
        "total (x,z) evaluations = ",
        nevals_total
    )
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", β_grid)

    all_results = NamedTuple[]
    sizehint!(all_results, nparams)

    global_min_slack_f = Inf
    global_min_f = Inf
    global_min_ratio_f = Inf
    global_min_gap_gf = Inf
    global_min_slack_g = Inf

    t0_total = time()

    for chunk_index in 1:nchunks
        lo = (chunk_index - 1) * chunk_size + 1
        hi = min(chunk_index * chunk_size, nparams)

        params_chunk = params[lo:hi]

        GC.gc()

        t0_chunk = time()

        chunk_results = threaded_scan_fg(
            params_chunk;
            r=r,
            Nx=Nx,
            Nu=Nu,
            ε=ε,
            endpoint_dense=endpoint_dense
        )

        chunk_time = time() - t0_chunk

        df_chunk = DataFrame(chunk_results)

        CSV.write(
            joinpath(
                outdir,
                @sprintf(
                    "chunk_%03d.csv",
                    chunk_index
                )
            ),
            df_chunk
        )

        append!(
            all_results,
            chunk_results
        )

        global_min_slack_f = min(
            global_min_slack_f,
            minimum(df_chunk.min_slack_f)
        )

        global_min_f = min(
            global_min_f,
            minimum(df_chunk.min_f)
        )

        global_min_ratio_f = min(
            global_min_ratio_f,
            minimum(df_chunk.min_ratio_f)
        )

        global_min_gap_gf = min(
            global_min_gap_gf,
            minimum(df_chunk.min_gap_g_minus_f)
        )

        global_min_slack_g = min(
            global_min_slack_g,
            minimum(df_chunk.min_slack_g)
        )

        elapsed = time() - t0_total
        fraction_done = hi / nparams
        eta = elapsed * (1.0 - fraction_done) / fraction_done

        println(
            @sprintf(
                "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | min(f-(1-β)) = %.6e | min(f) = %.6e | min(f/(1-β)) = %.6e | min(g-f) = %.6e | min(g-(1-β)) = %.6e",
                Dates.format(now(), "HH:MM:SS"),
                chunk_index,
                nchunks,
                lo,
                hi,
                chunk_time,
                elapsed / 3600,
                eta / 3600,
                global_min_slack_f,
                global_min_f,
                global_min_ratio_f,
                global_min_gap_gf,
                global_min_slack_g
            )
        )
    end

    df_all = DataFrame(all_results)

    df_fslack = sort(
        df_all,
        :min_slack_f
    )

    CSV.write(
        joinpath(
            outdir,
            "all_results_sorted_by_min_slack_f.csv"
        ),
        df_fslack
    )

    df_f0 = sort(
        df_all,
        :min_f
    )

    CSV.write(
        joinpath(
            outdir,
            "all_results_sorted_by_min_f.csv"
        ),
        df_f0
    )

    CSV.write(
        joinpath(
            outdir,
            "worst_200_f_ge_0.csv"
        ),
        first(
            df_f0,
            min(200, nrow(df_f0))
        )
    )

    df_ratio_f = sort(
        df_all,
        :min_ratio_f
    )

    CSV.write(
        joinpath(
            outdir,
            "all_results_sorted_by_min_ratio_f.csv"
        ),
        df_ratio_f
    )

    CSV.write(
        joinpath(
            outdir,
            "worst_200_min_f_over_1minusbeta.csv"
        ),
        first(
            df_ratio_f,
            min(200, nrow(df_ratio_f))
        )
    )

    df_gf = sort(
        df_all,
        :min_gap_g_minus_f
    )

    CSV.write(
        joinpath(
            outdir,
            "all_results_sorted_by_min_gap_g_minus_f.csv"
        ),
        df_gf
    )

    CSV.write(
        joinpath(
            outdir,
            "worst_200_g_ge_f.csv"
        ),
        first(
            df_gf,
            min(200, nrow(df_gf))
        )
    )

    df_gslack = sort(
        df_all,
        :min_slack_g
    )

    CSV.write(
        joinpath(
            outdir,
            "all_results_sorted_by_min_slack_g.csv"
        ),
        df_gslack
    )

    total_time = time() - t0_total

    println("\nFinished.")
    println(
        "Total runtime = ",
        round(total_time; digits=1),
        " s = ",
        round(total_time / 3600; digits=3),
        " h"
    )

    println("\nWorst sampled case for f(x,z) ≥ 1-β:")
    println(
        df_fslack[
            1,
            [
                :p01,
                :ρ,
                :κ,
                :β,
                :xstar_f,
                :zstar_f,
                :min_slack_f,
                :rel_slack_f
            ]
        ]
    )

    println("\nWorst sampled case for f(x,z) ≥ 0:")
    println(
        df_f0[
            1,
            [
                :p01,
                :ρ,
                :κ,
                :β,
                :xstar_f0,
                :zstar_f0,
                :min_f
            ]
        ]
    )

    println("\nWorst sampled value of f(x,z)/(1-β):")
    println(
        df_ratio_f[
            1,
            [
                :p01,
                :ρ,
                :κ,
                :β,
                :xstar_ratio_f,
                :zstar_ratio_f,
                :min_ratio_f
            ]
        ]
    )

    println("\nWorst sampled case for g(x,z) ≥ f(x,z):")
    println(
        df_gf[
            1,
            [
                :p01,
                :ρ,
                :κ,
                :β,
                :xstar_gf,
                :zstar_gf,
                :min_gap_g_minus_f
            ]
        ]
    )

    println("\nWorst sampled case for g(x,z) ≥ 1-β:")
    println(
        df_gslack[
            1,
            [
                :p01,
                :ρ,
                :κ,
                :β,
                :xstar_g,
                :zstar_g,
                :min_slack_g,
                :rel_slack_g
            ]
        ]
    )

    return df_all
end

In [ ]:
deltaLstar_run = serious_run_DeltaLstar_chunked(
    r=1.0,

    # Outer parameter grid
    Np01=14,
    Nα=14,
    Nκ=14,

    # Coarse counterexample search
    Nx_screen=121,
    Nλ_screen=41,

    # Refine the worst coarse cases
    refine_top=200,
    Nx_refine=2403,
    Nλ_refine=81,

    # Numerical controls
    ε=1e-10,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200,
    λpad_frac=0.05,
    violation_tol=1e-8,

    chunk_size=512,

    outdir=
        "serious_run_DeltaLstar_14x14x14x11_" *
        "screen121x41_refine2403x81_eps1e10"
)

df_deltaLstar_screen  = deltaLstar_run.screen
df_deltaLstar_refined = deltaLstar_run.refined

In [ ]:
using CSV
using DataFrames

# -----------------------------------------------------------
# 1. Load the timed experiment results
# -----------------------------------------------------------

df_timed = CSV.read(
    "typed_scaling_experiment_binned_timed.csv",
    DataFrame
)

# -----------------------------------------------------------
# 2. Define and select the pathological instance
# -----------------------------------------------------------

scenario = (
    name = "A_mild_B_fragile",
    q_types = [0.03, 0.10],
    rho_types = [0.80, 0.10],
    kappa_types = [0.70, 0.95],
    r_types = [1.0, 1.0]
)

target_prop  = "0.8,0.2"
target_alpha = 0.10
target_N     = 100

matches = filter(df_timed) do r
    isequal(r.scenario, scenario.name) &&
    isequal(r.prop_type, target_prop) &&
    !ismissing(r.alphaM_target) &&
    isapprox(
        Float64(r.alphaM_target),
        target_alpha;
        atol=1e-12,
        rtol=0.0
    ) &&
    !ismissing(r.N) &&
    Int(r.N) == target_N
end

nrow(matches) == 1 ||
    error("Expected exactly one matching row; found $(nrow(matches)).")

row = matches[1, :]

N    = Int(row.N)
M    = Int(row.M)
seed = Int(row.seed)
β    = Float64(row.beta)

@show row.scenario row.prop_type row.alphaM_target
@show row.capacity_ratio N M seed β row.t_bound_case

# -----------------------------------------------------------
# 3. Rebuild the exact realized instance
# -----------------------------------------------------------

inst = build_typed_instance(
    N;
    prop_type=[0.8, 0.2],
    q_types=scenario.q_types,
    rho_types=scenario.rho_types,
    kappa_types=scenario.kappa_types,
    r_types=scenario.r_types,
    x_init_scheme=:linspace,
    xlo=0.0,
    xhi=1.0,
    shuffle_types=true,
    shuffle_x=true,
    seed=seed
)

# -----------------------------------------------------------
# 4. Plot Lvector versus λ
# -----------------------------------------------------------

outfile =
    "Lvector_vs_lambda_pathological_A_mild_B_fragile_prop_0p8_0p2_alpha_0p1_N100.pdf"

save_Lvector_plot_pdf(
    outfile,
    inst.x_vec;
    qvector=inst.q_vec,
    ρvector=inst.rho_vec,
    κvector=inst.kappa_vec,
    rvector=inst.r_vec,
    β=β,
    M=M
)

println("Saved plot to: ", outfile)

In [ ]:
using CSV
using DataFrames

df_timed = CSV.read("typed_scaling_experiment_binned_timed.csv", DataFrame)

row = only(filter(r ->
    r.scenario == "A_mild_B_fragile" &&
    r.prop_type == "0.8,0.2" &&
    isapprox(r.alphaM_target, 0.10; atol=1e-12) &&
    r.N == 100,
    df_timed
))

@show row.scenario row.prop_type row.alphaM_target row.capacity_ratio row.N row.M row.seed row.t_bound_case

scenario = (
    name = "A_mild_B_fragile",
    q_types = [0.03, 0.10],
    rho_types = [0.80, 0.10],
    kappa_types = [0.70, 0.95],
    r_types = [1.0, 1.0]
)

inst = build_typed_instance(
    Int(row.N);
    prop_type = [0.8, 0.2],
    q_types = scenario.q_types,
    rho_types = scenario.rho_types,
    kappa_types = scenario.kappa_types,
    r_types = scenario.r_types,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    seed = Int(row.seed)
)

# exact realized vectors
xvector = inst.x_vec
qvector = inst.q_vec
ρvector = inst.rho_vec
κvector = inst.kappa_vec
rvector = inst.r_vec
β = row.beta
M = Int(row.M)

@time λstar_old, Lmin_old = lambda_star_vector(
    xvector;
    qvector = qvector,
    ρvector = ρvector,
    κvector = κvector,
    rvector = rvector,
    β = β,
    M = M
)

@time λstar_new, Lmin_new, info_new = lambda_star_vector_grouped_bisect(
    xvector,
    qvector,
    ρvector,
    κvector,
    rvector,
    β,
    M;
    λtol = 1e-6,
    verbose = false
)

@show λstar_old Lmin_old
@show λstar_new Lmin_new
@show info_new

In [ ]:
qgrid0 = range(0.05, 0.95; length=3)
αgrid0 = range(0.10, 0.90; length=3)
κgrid0 = range(0.05, 0.95; length=3)
βgrid0 = beta_grid(nmid=3, nnear=3, δmin=1e-2, δmax=0.1)

params0 = make_params(qgrid0, αgrid0, κgrid0, βgrid0)

Nx0, Nu0 = 21, 21
nevals0 = length(params0) * Nx0 * Nu0

t0 = @elapsed res0 = threaded_scan(params0; Nx=Nx0, Nu=Nu0, ε=1e-8)

println("threads = ", Threads.nthreads())
println("pilot time = ", t0, " seconds")
println("eval rate = ", nevals0 / t0, " evaluations / second")

In [ ]:
include("pcli1_pilot.jl")

df_jup, stats_jup = pilot_benchmark(
    Nq=3, Nα=3, Nκ=3,
    Nx=21, Nu=21,
    ε=1e-8,
    repeats=3,
    nβmid=3, nβnear=4,
    δmin=1e-2, δmax=0.1,
    csvfile="pilot_jupyter_threads$(Threads.nthreads()).csv",
)

In [ ]:
include("pcli1_pilot.jl")

using Dates
using Printf
using DataFrames
using CSV

# Keep the same function name as before, so the rest of the code changes minimally.
function beta_grid_24()
    β = [
        0.10, 0.20, 0.30, 0.40, 0.50,
        0.60, 0.70, 0.80, 0.90, 0.95, 0.99
    ]
    @assert length(β) == 11
    return β
end

function serious_run_chunked(;
    Nq=14, Nα=14, Nκ=14,
    Nx=121, Nu=121,
    ε=1e-8,
    endpoint_dense=true,
    chunk_size=512,
    qmin=0.05, qmax=0.95,
    αmin=0.10, αmax=0.90,
    κmin=0.05, κmax=0.95,
    outdir="serious_run_14x14x14x11_121x121_eps1e8",
)

    println("Threads.nthreads() = ", Threads.nthreads())

    qgrid = collect(range(qmin, qmax; length=Nq))
    αgrid = collect(range(αmin, αmax; length=Nα))
    κgrid = collect(range(κmin, κmax; length=Nκ))
    βgrid = beta_grid_24()

    params = make_params(qgrid, αgrid, κgrid, βgrid)

    nparams = length(params)
    nevals_total = nparams * Nx * Nu
    nchunks = cld(nparams, chunk_size)

    mkpath(outdir)

    CSV.write(joinpath(outdir, "beta_grid.csv"), DataFrame(β=βgrid))
    CSV.write(joinpath(outdir, "q_grid.csv"), DataFrame(q=qgrid))
    CSV.write(joinpath(outdir, "alpha_grid.csv"), DataFrame(α=αgrid))
    CSV.write(joinpath(outdir, "kappa_grid.csv"), DataFrame(κ=κgrid))

    open(joinpath(outdir, "run_info.txt"), "w") do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "Nq = ", Nq)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(βgrid))
        println(io, "Nx = ", Nx)
        println(io, "Nu = ", Nu)
        println(io, "ε = ", ε)
        println(io, "endpoint_dense = ", endpoint_dense)
        println(io, "chunk_size = ", chunk_size)
        println(io, "q range = [", qmin, ", ", qmax, "]")
        println(io, "α range = [", αmin, ", ", αmax, "]")
        println(io, "κ range = [", κmin, ", ", κmax, "]")
        println(io, "parameter tuples = ", nparams)
        println(io, "total g_metric2 evaluations = ", nevals_total)
    end

    println("parameter tuples = ", nparams)
    println("Nx = ", Nx, ", Nu = ", Nu)
    println("total g_metric2 evaluations = ", nevals_total)
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", βgrid)

    all_results = NamedTuple[]
    global_worst = Inf

    t0_total = time()

    for c in 1:nchunks
        lo = (c - 1) * chunk_size + 1
        hi = min(c * chunk_size, nparams)
        params_chunk = params[lo:hi]

        GC.gc()
        t0_chunk = time()
        res = threaded_scan(params_chunk; Nx=Nx, Nu=Nu, ε=ε, endpoint_dense=endpoint_dense)
        t_chunk = time() - t0_chunk

        df_chunk = DataFrame(res)
        sort!(df_chunk, :min_slack)

        chunkfile = joinpath(outdir, @sprintf("chunk_%03d.csv", c))
        CSV.write(chunkfile, df_chunk)

        append!(all_results, res)

        if df_chunk.min_slack[1] < global_worst
            global_worst = df_chunk.min_slack[1]
        end

        elapsed = time() - t0_total
        frac_done = hi / nparams
        eta = elapsed * (1 - frac_done) / frac_done

        println(@sprintf(
            "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | current worst min_slack = %.6e",
            Dates.format(now(), "HH:MM:SS"), c, nchunks, lo, hi, t_chunk, elapsed / 3600, eta / 3600, global_worst
        ))
    end

    df_all = DataFrame(all_results)
    sort!(df_all, :min_slack)

    CSV.write(joinpath(outdir, "all_results_sorted.csv"), df_all)
    CSV.write(joinpath(outdir, "worst_200.csv"), first(df_all, min(200, nrow(df_all))))

    println("\nFinished.")
    println("Final worst min_slack = ", df_all.min_slack[1])
    println("Worst tuple:")
    println(df_all[1, [:q, :ρ, :κ, :β, :xstar, :zstar, :min_slack, :rel_slack]])

    return df_all
end

# Run the modified serious experiment
df_serious = serious_run_chunked(
    Nq=14, Nα=14, Nκ=14,
    Nx=121, Nu=121,
    ε=1e-8,
    endpoint_dense=true,
    chunk_size=512,
    outdir="serious_run_14x14x14x11_121x121_eps1e8"
)

In [ ]:
using Base.Threads
using LinearAlgebra
using Dates
using Printf
using CSV
using DataFrames

BLAS.set_num_threads(1)

function beta_grid_11()
    return [
        0.10, 0.20, 0.30, 0.40, 0.50,
        0.60, 0.70, 0.80, 0.90, 0.95, 0.99
    ]
end

function make_params(qgrid, αgrid, κgrid, βgrid)
    params = NamedTuple[]
    for q in qgrid, α in αgrid, κ in κgrid, β in βgrid
        ρ = α * (1 - q)
        push!(params, (q=q, ρ=ρ, κ=κ, β=β))
    end
    return params
end

function uniform_x_grid_with_padding(x1::Float64, x0::Float64;
                                     Nmid::Int=2001,
                                     Nside::Int=201,
                                     pad_frac::Float64=0.05)

    @assert 0.0 <= x1 <= x0 <= 1.0
    @assert Nmid >= 2
    @assert Nside >= 0

    gap = x0 - x1
    pad = pad_frac * gap

    xL = max(0.0, x1 - pad)
    xR = min(1.0, x0 + pad)

    left = if Nside > 0 && xL < x1
        collect(range(xL, x1; length=Nside + 1))[1:end-1]
    else
        Float64[]
    end

    mid = collect(range(x1, x0; length=Nmid))

    right = if Nside > 0 && x0 < xR
        collect(range(x0, xR; length=Nside + 1))[2:end]
    else
        Float64[]
    end

    xgrid = vcat(left, mid, right)

    i1 = length(left) + 1
    i0 = length(left) + length(mid)

    return xgrid, i1, i0, xL, xR
end

function finite_min_with_index(v::AbstractVector{<:Real})
    best = Inf
    ibest = 0
    @inbounds for i in eachindex(v)
        vi = v[i]
        if isfinite(vi) && vi < best
            best = vi
            ibest = i
        end
    end
    return best, ibest
end

function finite_maxabs_with_index(v::AbstractVector{<:Real})
    best = -Inf
    ibest = 0
    @inbounds for i in eachindex(v)
        vi = v[i]
        if isfinite(vi)
            avi = abs(vi)
            if avi > best
                best = avi
                ibest = i
            end
        end
    end
    return best, ibest
end

function scan_one_tuple_pcli2(; q, ρ, κ, β, r,
                              Nmid=2001,
                              Nside=201,
                              pad_frac=0.05,
                              ε=1e-8,
                              mono_tol=1e-12)

    x0v, x1v, _ = fixed_points(; q=q, ρ=ρ, κ=κ)

    xgrid, i1, i0, xL, xR = uniform_x_grid_with_padding(
        x1v, x0v; Nmid=Nmid, Nside=Nside, pad_frac=pad_frac
    )

    n = length(xgrid)
    fvals = Vector{Float64}(undef, n)
    gvals = Vector{Float64}(undef, n)
    mvals = Vector{Float64}(undef, n)

    @inbounds for i in eachindex(xgrid)
        x = xgrid[i]
        f = f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        g = g_metric2(x, x; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
        fvals[i] = f
        gvals[i] = g
        mvals[i] = f / g
    end

    n_nonfinite_f = count(x -> !isfinite(x), fvals)
    n_nonfinite_g = count(x -> !isfinite(x), gvals)
    n_nonpositive_g = count(x -> !(isfinite(x) && x > 0.0), gvals)
    n_nonfinite_m = count(x -> !isfinite(x), mvals)

    diffs = Vector{Float64}(undef, n - 1)
    @inbounds for i in 1:n-1
        a = mvals[i]
        b = mvals[i + 1]
        diffs[i] = (isfinite(a) && isfinite(b)) ? (b - a) : NaN
    end

    min_diff_all, j_all = finite_min_with_index(diffs)

    core_start = i1
    core_stop  = i0 - 1
    if core_start <= core_stop
        diffs_core = @view diffs[core_start:core_stop]
    else
        diffs_core = Float64[]
    end

    min_diff_core, j_core_local = isempty(diffs_core) ? (NaN, 0) : finite_min_with_index(diffs_core)
    j_core = j_core_local == 0 ? 0 : (core_start + j_core_local - 1)

    xdiff_all_left  = j_all == 0 ? NaN : xgrid[j_all]
    xdiff_all_right = j_all == 0 ? NaN : xgrid[j_all + 1]

    xdiff_core_left  = j_core == 0 ? NaN : xgrid[j_core]
    xdiff_core_right = j_core == 0 ? NaN : xgrid[j_core + 1]

    Δx1_left  = i1 > 1 ? mvals[i1]   - mvals[i1 - 1] : NaN
    Δx1_right = i1 < n ? mvals[i1+1] - mvals[i1]     : NaN
    Δx0_left  = i0 > 1 ? mvals[i0]   - mvals[i0 - 1] : NaN
    Δx0_right = i0 < n ? mvals[i0+1] - mvals[i0]     : NaN

    cont_x1_left  = i1 > 1 ? abs(mvals[i1]   - mvals[i1 - 1]) : NaN
    cont_x1_right = i1 < n ? abs(mvals[i1+1] - mvals[i1])     : NaN
    cont_x0_left  = i0 > 1 ? abs(mvals[i0]   - mvals[i0 - 1]) : NaN
    cont_x0_right = i0 < n ? abs(mvals[i0+1] - mvals[i0])     : NaN

    cont_vals = [cont_x1_left, cont_x1_right, cont_x0_left, cont_x0_right]
    cont_vals_finite = filter(isfinite, cont_vals)
    endpoint_cont_proxy = isempty(cont_vals_finite) ? NaN : maximum(cont_vals_finite)

    max_adj_jump, j_jump = finite_maxabs_with_index(diffs)
    xjump_left  = j_jump == 0 ? NaN : xgrid[j_jump]
    xjump_right = j_jump == 0 ? NaN : xgrid[j_jump + 1]

    mono_pass_all = isfinite(min_diff_all)  && min_diff_all  >= -mono_tol
    mono_pass_core = isfinite(min_diff_core) && min_diff_core >= -mono_tol

    endpoint_diffs = filter(isfinite, [Δx1_left, Δx1_right, Δx0_left, Δx0_right])
    mono_pass_endpoints = !isempty(endpoint_diffs) && minimum(endpoint_diffs) >= -mono_tol

    return (
        x1 = x1v,
        x0 = x0v,
        xL = xL,
        xR = xR,
        n_x = n,

        min_g_diag = minimum(gvals),
        n_nonfinite_f = n_nonfinite_f,
        n_nonfinite_g = n_nonfinite_g,
        n_nonpositive_g = n_nonpositive_g,
        n_nonfinite_m = n_nonfinite_m,

        min_diff_all = min_diff_all,
        xdiff_all_left = xdiff_all_left,
        xdiff_all_right = xdiff_all_right,

        min_diff_core = min_diff_core,
        xdiff_core_left = xdiff_core_left,
        xdiff_core_right = xdiff_core_right,

        Δx1_left = Δx1_left,
        Δx1_right = Δx1_right,
        Δx0_left = Δx0_left,
        Δx0_right = Δx0_right,

        cont_x1_left = cont_x1_left,
        cont_x1_right = cont_x1_right,
        cont_x0_left = cont_x0_left,
        cont_x0_right = cont_x0_right,
        endpoint_cont_proxy = endpoint_cont_proxy,

        max_adj_jump = max_adj_jump,
        xjump_left = xjump_left,
        xjump_right = xjump_right,

        mono_pass_all = mono_pass_all,
        mono_pass_core = mono_pass_core,
        mono_pass_endpoints = mono_pass_endpoints,
    )
end

function threaded_scan_pcli2(params; r, Nmid=2001, Nside=201, pad_frac=0.05,
                             ε=1e-8, mono_tol=1e-12)

    results = Vector{NamedTuple}(undef, length(params))

    Threads.@threads :dynamic for i in eachindex(params)
        p = params[i]
        s = scan_one_tuple_pcli2(; p..., r=r,
                                 Nmid=Nmid, Nside=Nside, pad_frac=pad_frac,
                                 ε=ε, mono_tol=mono_tol)
        results[i] = merge(p, s)
    end

    return results
end

function serious_run_pcli2_chunked(;
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nmid=2001,
    Nside=201,
    pad_frac=0.05,
    ε=1e-8,
    mono_tol=1e-12,
    chunk_size=512,
    qmin=0.05, qmax=0.95,
    αmin=0.10, αmax=0.90,
    κmin=0.05, κmax=0.95,
    outdir="serious_run_pcli2_14x14x14x11_uniformx"
)

    println("Threads.nthreads() = ", Threads.nthreads())

    qgrid = collect(range(qmin, qmax; length=Nq))
    αgrid = collect(range(αmin, αmax; length=Nα))
    κgrid = collect(range(κmin, κmax; length=Nκ))
    βgrid = beta_grid_11()

    params = make_params(qgrid, αgrid, κgrid, βgrid)

    nparams = length(params)
    n_x_total_per_tuple = Nmid + 2*Nside
    nevals_total = nparams * n_x_total_per_tuple
    nchunks = cld(nparams, chunk_size)

    mkpath(outdir)

    CSV.write(joinpath(outdir, "beta_grid.csv"), DataFrame(β=βgrid))
    CSV.write(joinpath(outdir, "q_grid.csv"), DataFrame(q=qgrid))
    CSV.write(joinpath(outdir, "alpha_grid.csv"), DataFrame(α=αgrid))
    CSV.write(joinpath(outdir, "kappa_grid.csv"), DataFrame(κ=κgrid))

    open(joinpath(outdir, "run_info.txt"), "w") do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "r = ", r)
        println(io, "Nq = ", Nq)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(βgrid))
        println(io, "Nmid = ", Nmid)
        println(io, "Nside = ", Nside)
        println(io, "pad_frac = ", pad_frac)
        println(io, "ε = ", ε)
        println(io, "mono_tol = ", mono_tol)
        println(io, "chunk_size = ", chunk_size)
        println(io, "parameter tuples = ", nparams)
        println(io, "x evaluations per tuple (nominal) = ", n_x_total_per_tuple)
        println(io, "total m(x) evaluations (nominal) = ", nevals_total)
    end

    println("parameter tuples = ", nparams)
    println("Nmid = ", Nmid, ", Nside = ", Nside, ", pad_frac = ", pad_frac)
    println("nominal x evaluations per tuple = ", n_x_total_per_tuple)
    println("nominal total evaluations = ", nevals_total)
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", βgrid)

    all_results = NamedTuple[]
    global_min_diff = Inf
    global_max_endpoint_cont = -Inf

    t0_total = time()

    for c in 1:nchunks
        lo = (c - 1) * chunk_size + 1
        hi = min(c * chunk_size, nparams)
        params_chunk = params[lo:hi]

        GC.gc()
        t0_chunk = time()
        res = threaded_scan_pcli2(params_chunk;
                                  r=r,
                                  Nmid=Nmid, Nside=Nside, pad_frac=pad_frac,
                                  ε=ε, mono_tol=mono_tol)
        t_chunk = time() - t0_chunk

        df_chunk = DataFrame(res)
        sort!(df_chunk, :min_diff_all)

        chunkfile = joinpath(outdir, @sprintf("chunk_%03d.csv", c))
        CSV.write(chunkfile, df_chunk)

        append!(all_results, res)

        if df_chunk.min_diff_all[1] < global_min_diff
            global_min_diff = df_chunk.min_diff_all[1]
        end

        epvals = filter(isfinite, collect(df_chunk.endpoint_cont_proxy))
        chunk_max_endpoint_cont = isempty(epvals) ? NaN : maximum(epvals)

        if isfinite(chunk_max_endpoint_cont) && chunk_max_endpoint_cont > global_max_endpoint_cont
            global_max_endpoint_cont = chunk_max_endpoint_cont
        end

        elapsed = time() - t0_total
        frac_done = hi / nparams
        eta = elapsed * (1 - frac_done) / frac_done

        println(@sprintf(
            "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | global min_diff_all = %.6e | global max endpoint proxy = %.6e",
            Dates.format(now(), "HH:MM:SS"), c, nchunks, lo, hi, t_chunk,
            elapsed / 3600, eta / 3600, global_min_diff, global_max_endpoint_cont
        ))
    end

    df_all = DataFrame(all_results)

    df_sorted_mono = sort(df_all, :min_diff_all)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_diff_all.csv"), df_sorted_mono)
    CSV.write(joinpath(outdir, "worst_200_monotonicity.csv"),
              first(df_sorted_mono, min(200, nrow(df_sorted_mono))))

    df_sorted_core = sort(df_all, :min_diff_core)
    CSV.write(joinpath(outdir, "all_results_sorted_by_min_diff_core.csv"), df_sorted_core)
    CSV.write(joinpath(outdir, "worst_200_core_monotonicity.csv"),
              first(df_sorted_core, min(200, nrow(df_sorted_core))))

    df_sorted_cont = sort(df_all, :endpoint_cont_proxy, rev=true)
    CSV.write(joinpath(outdir, "all_results_sorted_by_endpoint_cont_proxy_desc.csv"), df_sorted_cont)
    CSV.write(joinpath(outdir, "worst_200_endpoint_continuity_proxy.csv"),
              first(df_sorted_cont, min(200, nrow(df_sorted_cont))))

    println("\nFinished.")
    println("Worst monotonicity case over extended interval:")
    println(df_sorted_mono[1, [:q, :ρ, :κ, :β, :min_diff_all, :xdiff_all_left, :xdiff_all_right]])

    println("\nWorst monotonicity case on [x1, x0]:")
    println(df_sorted_core[1, [:q, :ρ, :κ, :β, :min_diff_core, :xdiff_core_left, :xdiff_core_right]])

    println("\nLargest endpoint continuity proxy:")
    println(df_sorted_cont[1, [:q, :ρ, :κ, :β, :endpoint_cont_proxy,
                               :cont_x1_left, :cont_x1_right, :cont_x0_left, :cont_x0_right]])

    return df_all
end

df_pcli2 = serious_run_pcli2_chunked(
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nmid=2001,
    Nside=201,
    pad_frac=0.05,
    ε=1e-8,
    mono_tol=1e-12,
    chunk_size=512,
    outdir="serious_run_pcli2_14x14x14x11_uniformx_Nmid2001_Nside201"
)

In [ ]:
using Base.Threads
using LinearAlgebra
using Dates
using Printf
using CSV
using DataFrames

BLAS.set_num_threads(1)

# ============================================================
# Same outer parameter grid as before
# ============================================================

function beta_grid_11()
    return [
        0.10, 0.20, 0.30, 0.40, 0.50,
        0.60, 0.70, 0.80, 0.90, 0.95, 0.99
    ]
end

function make_params(qgrid, αgrid, κgrid, βgrid)
    params = NamedTuple[]
    for q in qgrid, α in αgrid, κ in κgrid, β in βgrid
        ρ = α * (1 - q)
        push!(params, (q=q, ρ=ρ, κ=κ, β=β))
    end
    return params
end

# ============================================================
# Uniform x-grid on [0,1]
# Use Nx=2403 by default to match previous PCLI2 count
# ============================================================

uniform01_grid(Nx::Int=2403) = collect(range(0.0, 1.0; length=Nx))

# ============================================================
# Diagonal MP index on a grid
# ============================================================

function diag_index_grid(xgrid; q, ρ, κ, r, β, ε=1e-8)
    mvals = Vector{Float64}(undef, length(xgrid))
    @inbounds for i in eachindex(xgrid)
        x = xgrid[i]
        f = f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
        g = g_metric2(x, x; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
        mvals[i] = f / g
    end
    return mvals
end

# ============================================================
# Per-parameter lambda grid
#
# Built from sampled diagonal-index range on [0,1], padded a bit.
# ============================================================

function lambda_grid_from_diag(; q, ρ, κ, r, β,
                               Nλ::Int=41,
                               Nxλ::Int=401,
                               λpad_frac::Float64=0.05,
                               ε::Float64=1e-8)

    xgridλ = uniform01_grid(Nxλ)
    mvals = diag_index_grid(xgridλ; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)

    finite_m = filter(isfinite, mvals)
    @assert !isempty(finite_m) "No finite diagonal index values found."

    λmin0 = minimum(finite_m)
    λmax0 = maximum(finite_m)

    span = λmax0 - λmin0
    scale = max(abs(λmin0), abs(λmax0), 1.0)
    pad = span > 0 ? λpad_frac * span : λpad_frac * scale

    λmin = λmin0 - pad
    λmax = λmax0 + pad

    return collect(range(λmin, λmax; length=Nλ)), λmin0, λmax0
end

# ============================================================
# Delta L*(x, lambda)
# ============================================================

function DeltaLstar(x, λ; q, ρ, κ, r, β,
                    ε=1e-8,
                    xatol=1e-10,
                    λatol=1e-10,
                    maxiter=200)

    z = zstar(λ; q=q, ρ=ρ, κ=κ, r=r, β=β,
              ε=ε, xatol=xatol, λatol=λatol, maxiter=maxiter)

    f = f_metric2(x, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
    g = g_metric2(x, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)

    return f - λ * g
end

# ============================================================
# One parameter tuple:
# test monotonicity in x for a grid of lambdas
# ============================================================

function scan_one_tuple_DeltaLstar(; q, ρ, κ, β, r,
                                   Nx::Int=2403,
                                   Nλ::Int=41,
                                   Nxλ::Int=401,
                                   λpad_frac::Float64=0.05,
                                   ε::Float64=1e-8,
                                   xatol::Float64=1e-10,
                                   λatol::Float64=1e-10,
                                   maxiter::Int=200,
                                   mono_tol::Float64=1e-12)

    xgrid = uniform01_grid(Nx)

    λgrid, λmin_diag, λmax_diag = lambda_grid_from_diag(
        q=q, ρ=ρ, κ=κ, r=r, β=β,
        Nλ=Nλ, Nxλ=Nxλ, λpad_frac=λpad_frac, ε=ε
    )

    global_min_diff = Inf
    global_λstar = NaN
    global_zstar = NaN
    global_xleft = NaN
    global_xright = NaN

    n_nonfinite = 0
    n_viol_λ = 0

    for λ in λgrid
        z = zstar(λ; q=q, ρ=ρ, κ=κ, r=r, β=β,
                  ε=ε, xatol=xatol, λatol=λatol, maxiter=maxiter)

        yprev = begin
            f0 = f_metric2(xgrid[1], z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
            g0 = g_metric2(xgrid[1], z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
            f0 - λ * g0
        end

        λ_min_diff = Inf
        λ_has_violation = false

        @inbounds for i in 1:length(xgrid)-1
            xL = xgrid[i]
            xR = xgrid[i + 1]

            fR = f_metric2(xR, z; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε)
            gR = g_metric2(xR, z; q=q, ρ=ρ, κ=κ, β=β, ε=ε)
            yR = fR - λ * gR

            d = yR - yprev

            if !isfinite(d)
                n_nonfinite += 1
            else
                if d < λ_min_diff
                    λ_min_diff = d
                end
                if d < global_min_diff
                    global_min_diff = d
                    global_λstar = λ
                    global_zstar = z
                    global_xleft = xL
                    global_xright = xR
                end
                if d < -mono_tol
                    λ_has_violation = true
                end
            end

            yprev = yR
        end

        if λ_has_violation
            n_viol_λ += 1
        end
    end

    mono_pass = isfinite(global_min_diff) && global_min_diff >= -mono_tol

    return (
        λmin_diag = λmin_diag,
        λmax_diag = λmax_diag,
        λgrid_min = first(λgrid),
        λgrid_max = last(λgrid),
        Nλ = length(λgrid),

        min_diff = global_min_diff,
        λstar = global_λstar,
        zstar_at_λstar = global_zstar,
        xleft = global_xleft,
        xright = global_xright,

        n_nonfinite = n_nonfinite,
        n_viol_λ = n_viol_λ,
        mono_pass = mono_pass,
    )
end

# ============================================================
# Threaded outer scan
# ============================================================

function threaded_scan_DeltaLstar(params; r,
                                  Nx::Int=2403,
                                  Nλ::Int=41,
                                  Nxλ::Int=401,
                                  λpad_frac::Float64=0.05,
                                  ε::Float64=1e-8,
                                  xatol::Float64=1e-10,
                                  λatol::Float64=1e-10,
                                  maxiter::Int=200,
                                  mono_tol::Float64=1e-12)

    results = Vector{NamedTuple}(undef, length(params))

    Threads.@threads :dynamic for i in eachindex(params)
        p = params[i]
        s = scan_one_tuple_DeltaLstar(; p..., r=r,
                                      Nx=Nx, Nλ=Nλ, Nxλ=Nxλ,
                                      λpad_frac=λpad_frac,
                                      ε=ε, xatol=xatol, λatol=λatol,
                                      maxiter=maxiter, mono_tol=mono_tol)
        results[i] = merge(p, s)
    end

    return results
end

# ============================================================
# Chunked run over the same parameter grid as before
# ============================================================

function serious_run_DeltaLstar_chunked(;
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nx=2403,              # same number of x-points as previous PCLI2 test
    Nλ=41,                # start moderate; increase later if needed
    Nxλ=401,              # grid size used to build the lambda range
    λpad_frac=0.05,
    ε=1e-8,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200,
    mono_tol=1e-12,
    chunk_size=256,       # smaller chunk_size is safer here
    qmin=0.05, qmax=0.95,
    αmin=0.10, αmax=0.90,
    κmin=0.05, κmax=0.95,
    outdir="serious_run_DeltaLstar_14x14x14x11_Nx2403"
)

    println("Threads.nthreads() = ", Threads.nthreads())

    qgrid = collect(range(qmin, qmax; length=Nq))
    αgrid = collect(range(αmin, αmax; length=Nα))
    κgrid = collect(range(κmin, κmax; length=Nκ))
    βgrid = beta_grid_11()

    params = make_params(qgrid, αgrid, κgrid, βgrid)

    nparams = length(params)
    nominal_total_evals = nparams * Nx * Nλ
    nchunks = cld(nparams, chunk_size)

    mkpath(outdir)

    CSV.write(joinpath(outdir, "beta_grid.csv"), DataFrame(β=βgrid))
    CSV.write(joinpath(outdir, "q_grid.csv"), DataFrame(q=qgrid))
    CSV.write(joinpath(outdir, "alpha_grid.csv"), DataFrame(α=αgrid))
    CSV.write(joinpath(outdir, "kappa_grid.csv"), DataFrame(κ=κgrid))

    open(joinpath(outdir, "run_info.txt"), "w") do io
        println(io, "Threads.nthreads() = ", Threads.nthreads())
        println(io, "r = ", r)
        println(io, "Nq = ", Nq)
        println(io, "Nα = ", Nα)
        println(io, "Nκ = ", Nκ)
        println(io, "Nβ = ", length(βgrid))
        println(io, "Nx = ", Nx)
        println(io, "Nλ = ", Nλ)
        println(io, "Nxλ = ", Nxλ)
        println(io, "λpad_frac = ", λpad_frac)
        println(io, "ε = ", ε)
        println(io, "xatol = ", xatol)
        println(io, "λatol = ", λatol)
        println(io, "maxiter = ", maxiter)
        println(io, "mono_tol = ", mono_tol)
        println(io, "chunk_size = ", chunk_size)
        println(io, "parameter tuples = ", nparams)
        println(io, "nominal total ΔL* evaluations = ", nominal_total_evals)
    end

    println("parameter tuples = ", nparams)
    println("Nx = ", Nx, ", Nλ = ", Nλ, ", Nxλ = ", Nxλ)
    println("nominal total ΔL* evaluations = ", nominal_total_evals)
    println("number of chunks = ", nchunks)
    println("output directory = ", outdir)
    println("β grid = ", βgrid)

    all_results = NamedTuple[]
    global_min_diff = Inf

    t0_total = time()

    for c in 1:nchunks
        lo = (c - 1) * chunk_size + 1
        hi = min(c * chunk_size, nparams)
        params_chunk = params[lo:hi]

        GC.gc()
        t0_chunk = time()
        res = threaded_scan_DeltaLstar(params_chunk;
                                       r=r,
                                       Nx=Nx, Nλ=Nλ, Nxλ=Nxλ,
                                       λpad_frac=λpad_frac,
                                       ε=ε, xatol=xatol, λatol=λatol,
                                       maxiter=maxiter, mono_tol=mono_tol)
        t_chunk = time() - t0_chunk

        df_chunk = DataFrame(res)
        sort!(df_chunk, :min_diff)

        chunkfile = joinpath(outdir, @sprintf("chunk_%03d.csv", c))
        CSV.write(chunkfile, df_chunk)

        append!(all_results, res)

        if df_chunk.min_diff[1] < global_min_diff
            global_min_diff = df_chunk.min_diff[1]
        end

        elapsed = time() - t0_total
        frac_done = hi / nparams
        eta = elapsed * (1 - frac_done) / frac_done

        println(@sprintf(
            "[%s] chunk %3d/%3d | tuples %6d:%6d | chunk %.1f s | elapsed %.2f h | ETA %.2f h | global min_diff = %.6e",
            Dates.format(now(), "HH:MM:SS"), c, nchunks, lo, hi, t_chunk,
            elapsed / 3600, eta / 3600, global_min_diff
        ))
    end

    df_all = DataFrame(all_results)
    sort!(df_all, :min_diff)

    CSV.write(joinpath(outdir, "all_results_sorted_by_min_diff.csv"), df_all)
    CSV.write(joinpath(outdir, "worst_200.csv"),
              first(df_all, min(200, nrow(df_all))))

    println("\nFinished.")
    println("Worst case:")
    println(df_all[1, [:q, :ρ, :κ, :β, :min_diff, :λstar, :zstar_at_λstar, :xleft, :xright, :n_viol_λ]])

    return df_all
end

In [ ]:
df_DL = serious_run_DeltaLstar_chunked(
    r=1.0,
    Nq=14, Nα=14, Nκ=14,
    Nx=2403,
    Nλ=41,
    Nxλ=401,
    λpad_frac=0.05,
    ε=1e-8,
    xatol=1e-10,
    λatol=1e-10,
    maxiter=200,
    mono_tol=1e-12,
    chunk_size=256,
    outdir="serious_run_DeltaLstar_14x14x14x11_Nx2403_Nλ41"
)

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads

"""
    simulate_multi_project_policies(x_vec, q_vec, rho_vec, kappa_vec, r_vec, beta, M;
                                    T=100, nruns=1000,
                                    conf_level=0.95,
                                    seed=nothing, ε=1e-10)

Monte Carlo estimation of the approximate long-run average discounted reward
per project for the current ACK/NACK sensing model under four policies:

  1. Random policy,
  2. Myopic policy,
  3. Round-robin policy,
  4. MP-index policy,

together with the Lagrangian upper bound obtained from

    λstar, Lmin = lambda_star_vector(...)

The initial beliefs are fixed at X_i(0)=x_vec[i] for all runs.
"""
function simulate_multi_project_policies(x_vec::AbstractVector{<:Real},
                                         q_vec::AbstractVector{<:Real},
                                         rho_vec::AbstractVector{<:Real},
                                         kappa_vec::AbstractVector{<:Real},
                                         r_vec::AbstractVector{<:Real},
                                         beta::Float64,
                                         M::Integer;
                                         T::Int = 100,
                                         nruns::Int = 1_000,
                                         conf_level::Float64 = 0.95,
                                         seed::Union{Int,Nothing} = nothing,
                                         ε::Float64 = 1e-10)

    N = length(x_vec)
    @assert length(q_vec)     == N "q_vec must have the same length as x_vec"
    @assert length(rho_vec)   == N "rho_vec must have the same length as x_vec"
    @assert length(kappa_vec) == N "kappa_vec must have the same length as x_vec"
    @assert length(r_vec)     == N "r_vec must have the same length as x_vec"
    @assert 0.0 < beta < 1.0  "beta must be in (0,1)"
    @assert 1 <= M <= N       "M must satisfy 1 ≤ M ≤ N"
    @assert T > 0             "T must be positive"
    @assert nruns > 0         "nruns must be positive"

    x_vec_f     = Float64.(x_vec)
    q_vec_f     = Float64.(q_vec)
    rho_vec_f   = Float64.(rho_vec)
    kappa_vec_f = Float64.(kappa_vec)
    r_vec_f     = Float64.(r_vec)

    for i in 1:N
        @assert 0.0 <= x_vec_f[i] <= 1.0 "Each x_i must satisfy 0 ≤ x_i ≤ 1"
        @assert 0.0 <= q_vec_f[i] < 1.0 "Each q_i must satisfy 0 ≤ q_i < 1"
        @assert 0.0 <= rho_vec_f[i] < 1.0 - q_vec_f[i] "Each rho_i must satisfy 0 ≤ rho_i < 1-q_i"
        @assert 0.0 <= kappa_vec_f[i] <= 1.0 "Each kappa_i must satisfy 0 ≤ kappa_i ≤ 1"
        @assert r_vec_f[i] >= 0.0 "Each r_i must satisfy r_i ≥ 0"
    end

    # ----------------------------------------------------------
    # Lagrangian upper bound (computed once)
    # ----------------------------------------------------------
    λstar, Lmin = lambda_star_vector(
        x_vec_f;
        qvector = q_vec_f,
        ρvector = rho_vec_f,
        κvector = kappa_vec_f,
        rvector = r_vec_f,
        β = beta,
        M = M
    )

    β_powers = [beta^t for t in 0:(T-1)]

    total_random     = zeros(Float64, nruns)
    total_myopic     = zeros(Float64, nruns)
    total_roundrobin = zeros(Float64, nruns)
    total_mpindex    = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function phi0(x, q, rho)
        q + rho * x
    end

    @inline function phi1(x, q, rho, kappa)
        denom = 1.0 - kappa * x
        if denom <= 0.0
            return q + rho
        end
        post_nack = (1.0 - kappa) * x / denom
        return q + rho * post_nack
    end

    @inline function mp_index_current(x, q, rho, kappa, r, beta, ε)
        fxx = float(f_metric2(x, x; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=rho, κ=kappa,      β=beta, ε=ε))
        return (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        # Same initial beliefs in every run
        X0 = copy(x_vec_f)

        X_rand = copy(X0)
        X_myop = copy(X0)
        X_rr   = copy(X0)
        X_mp   = copy(X0)

        rr_start = 1

        R_rand = 0.0
        R_myop = 0.0
        R_rr   = 0.0
        R_mp   = 0.0

        for t in 1:T
            disc = β_powers[t]

            # ----------------------------------------------------------
            # 1. Random policy
            # ----------------------------------------------------------
            active_rand = randperm(rng, N)[1:M]
            active_rand_mask = falses(N)
            active_rand_mask[active_rand] .= true

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_rand[i]

                if active_rand_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rand[i] = q + ρ
                    else
                        X_rand[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_rand[i] = phi0(xi, q, ρ)
                end
            end
            R_rand += disc * reward_t

            # ----------------------------------------------------------
            # 2. Myopic policy
            # ----------------------------------------------------------
            myopic_value = similar(X_myop)
            for i in 1:N
                myopic_value[i] = r_vec_f[i] * kappa_vec_f[i] * X_myop[i]
            end

            idx_candidates = findall(i -> myopic_value[i] > 0.0, 1:N)

            active_myop_mask = falses(N)
            if !isempty(idx_candidates)
                if length(idx_candidates) <= M
                    active_myop_mask[idx_candidates] .= true
                else
                    vals = myopic_value[idx_candidates]
                    local_idx = partialsortperm(vals, rev=true, 1:M)
                    chosen = idx_candidates[local_idx]
                    active_myop_mask[chosen] .= true
                end
            end

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_myop[i]

                if active_myop_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_myop[i] = q + ρ
                    else
                        X_myop[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_myop[i] = phi0(xi, q, ρ)
                end
            end
            R_myop += disc * reward_t

            # ----------------------------------------------------------
            # 3. Round-robin policy
            # ----------------------------------------------------------
            active_rr_mask = falses(N)
            for k in 0:(M-1)
                i = ((rr_start - 1 + k) % N) + 1
                active_rr_mask[i] = true
            end
            rr_start = ((rr_start - 1 + M) % N) + 1

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_rr[i]

                if active_rr_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rr[i] = q + ρ
                    else
                        X_rr[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_rr[i] = phi0(xi, q, ρ)
                end
            end
            R_rr += disc * reward_t

            # ----------------------------------------------------------
            # 4. MP-index policy
            # ----------------------------------------------------------
            mpvals = similar(X_mp)
            for i in 1:N
                mpvals[i] = mp_index_current(X_mp[i],
                                             q_vec_f[i], rho_vec_f[i], kappa_vec_f[i],
                                             r_vec_f[i], beta, ε)
            end

            idx_candidates = findall(i -> mpvals[i] >= 0.0, 1:N)

            active_mp_mask = falses(N)
            if !isempty(idx_candidates)
                if length(idx_candidates) <= M
                    active_mp_mask[idx_candidates] .= true
                else
                    vals = mpvals[idx_candidates]
                    local_idx = partialsortperm(vals, rev=true, 1:M)
                    chosen = idx_candidates[local_idx]
                    active_mp_mask[chosen] .= true
                end
            end

            reward_t = 0.0
            for i in 1:N
                q  = q_vec_f[i]
                ρ  = rho_vec_f[i]
                κ  = kappa_vec_f[i]
                r  = r_vec_f[i]
                xi = X_mp[i]

                if active_mp_mask[i]
                    p_ack = κ * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_mp[i] = q + ρ
                    else
                        X_mp[i] = phi1(xi, q, ρ, κ)
                    end
                else
                    X_mp[i] = phi0(xi, q, ρ)
                end
            end
            R_mp += disc * reward_t
        end

        total_random[run_idx]     = R_rand
        total_myopic[run_idx]     = R_myop
        total_roundrobin[run_idx] = R_rr
        total_mpindex[run_idx]    = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        @threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    # Normalize to approximate long-run average discounted reward per project
    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (mean = μ,
                stderr = se,
                ci_low = ci_low,
                ci_high = ci_high,
                nruns = length(samples))
    end

    results = Dict{String,NamedTuple}()

    results["random"]      = summarize(total_random)
    results["myopic"]      = summarize(total_myopic)
    results["round_robin"] = summarize(total_roundrobin)
    results["mp_index"]    = summarize(total_mpindex)

    # Deterministic Lagrangian upper bound, normalized the same way
    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    return results
end

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads
using LinearAlgebra

BLAS.set_num_threads(1)

struct MPIndexLookupUniform
    xlo::Float64
    xhi::Float64
    h::Float64
    vals::Vector{Float64}
    mode::Symbol   # :leftbin, :nearest, :linear
end

"""
    build_mp_index_lookup(; q, ρ, κ, r, β, ε=1e-10,
                             ngrid=2001, xlo=0.0, xhi=1.0,
                             mode=:linear)

Precompute m(x)=f(x,x)/g(x,x) on a uniform grid of x-values.
"""
function build_mp_index_lookup(; q, ρ, κ, r, β,
                                 ε::Float64 = 1e-10,
                                 ngrid::Int = 2001,
                                 xlo::Float64 = 0.0,
                                 xhi::Float64 = 1.0,
                                 mode::Symbol = :linear)

    @assert ngrid >= 2
    @assert xlo < xhi
    @assert mode in (:leftbin, :nearest, :linear)

    xgrid = collect(range(xlo, xhi; length=ngrid))
    vals = Vector{Float64}(undef, ngrid)

    for i in eachindex(xgrid)
        x = xgrid[i]
        fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
        vals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    return MPIndexLookupUniform(xlo, xhi, (xhi - xlo) / (ngrid - 1), vals, mode)
end

"""
    mp_index_lookup(L, x)

Approximate m(x) from a precomputed uniform lookup table.
"""
@inline function mp_index_lookup(L::MPIndexLookupUniform, x::Real)
    xf = clamp(Float64(x), L.xlo, L.xhi)
    vals = L.vals
    n = length(vals)

    if xf <= L.xlo
        return vals[1]
    elseif xf >= L.xhi
        return vals[end]
    end

    t = (xf - L.xlo) / L.h
    i = floor(Int, t) + 1
    i = clamp(i, 1, n - 1)

    if L.mode === :leftbin
        return vals[i]
    elseif L.mode === :nearest
        α = t - floor(t)
        return α < 0.5 ? vals[i] : vals[i + 1]
    else
        α = t - floor(t)
        return (1 - α) * vals[i] + α * vals[i + 1]
    end
end

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads
using Printf
using LinearAlgebra
BLAS.set_num_threads(1)

# ============================================================
# Fast simulator for identical projects
#
# Policies simulated exactly:
#   1. random
#   2. myopic
#   3. MP-index
#
# plus the deterministic Lagrangian upper bound
# ============================================================

"""
    simulate_identical_project_policies_fast(
        N, M, x0, q, rho, kappa, r, beta;
        T=100, nruns=1000, conf_level=0.95,
        seed=nothing, ε=1e-10
    )

Fast Monte Carlo simulator for the identical-project setting.

All N projects have:
- common initial belief x0,
- common parameters (q, rho, kappa, r).

The state of each policy is represented by a dictionary
    counts[x] = number of projects currently with belief x.

This exploits exchangeability and caches MP index values by belief x.
"""


using Random
using Statistics
using Distributions
using Base.Threads
using LinearAlgebra

BLAS.set_num_threads(1)

function simulate_identical_project_policies_fast(
    N::Integer,
    M::Integer,
    x0::Real,
    q::Real,
    rho::Real,
    kappa::Real,
    r::Real,
    beta::Float64;
    T::Int = 100,
    nruns::Int = 1_000,
    conf_level::Float64 = 0.95,
    seed::Union{Int,Nothing} = nothing,
    ε::Float64 = 1e-10
)

    @assert N >= 1               "N must be positive"
    @assert 1 <= M <= N          "M must satisfy 1 ≤ M ≤ N"
    @assert 0.0 <= x0 <= 1.0     "x0 must satisfy 0 ≤ x0 ≤ 1"
    @assert 0.0 <= q < 1.0       "q must satisfy 0 ≤ q < 1"
    @assert 0.0 <= rho < 1.0 - q "rho must satisfy 0 ≤ rho < 1-q"
    @assert 0.0 <= kappa <= 1.0  "kappa must satisfy 0 ≤ kappa ≤ 1"
    @assert r >= 0.0             "r must satisfy r ≥ 0"
    @assert 0.0 < beta < 1.0     "beta must be in (0,1)"
    @assert T > 0                "T must be positive"
    @assert nruns > 0            "nruns must be positive"

    x0f    = Float64(x0)
    qf     = Float64(q)
    rhof   = Float64(rho)
    kappaf = Float64(kappa)
    rf     = Float64(r)

    β_powers = [beta^t for t in 0:(T-1)]
    x_ack = qf + rhof

    # Lagrangian upper bound (computed once)
    x_vec     = fill(x0f, N)
    q_vec     = fill(qf, N)
    rho_vec   = fill(rhof, N)
    kappa_vec = fill(kappaf, N)
    r_vec     = fill(rf, N)

    λstar, Lmin = lambda_star_vector(
        x_vec;
        qvector = q_vec,
        ρvector = rho_vec,
        κvector = kappa_vec,
        rvector = r_vec,
        β = beta,
        M = M
    )

    total_random  = zeros(Float64, nruns)
    total_myopic  = zeros(Float64, nruns)
    total_mpindex = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function addcount!(d::Dict{Float64,Int}, x::Float64, n::Int)
        n <= 0 && return nothing
        d[x] = get(d, x, 0) + n
        return nothing
    end

    @inline function get_transitions!(cache::Dict{Float64,NTuple{3,Float64}}, x::Float64)
        get!(cache, x) do
            p_ack = kappaf * x
            x_pass = qf + rhof * x

            denom = 1.0 - kappaf * x
            x_nack = if denom <= 0.0
                qf + rhof
            else
                qf + rhof * ((1.0 - kappaf) * x / denom)
            end

            (p_ack, x_pass, x_nack)
        end
    end

    @inline function get_mpindex!(cache::Dict{Float64,Float64}, x::Float64)
        get!(cache, x) do
            fxx = float(f_metric2(x, x; q=qf, ρ=rhof, κ=kappaf, r=rf, β=beta, ε=ε))
            gxx = float(g_metric2(x, x; q=qf, ρ=rhof, κ=kappaf,      β=beta, ε=ε))
            (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
        end
    end

    function apply_active_counts!(
        rng::AbstractRNG,
        pairs::Vector{Pair{Float64,Int}},
        active_counts::Vector{Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}}
    )
        new_counts = Dict{Float64,Int}()
        total_acks = 0

        @inbounds for j in eachindex(pairs)
            x = first(pairs[j])
            n = last(pairs[j])
            a = active_counts[j]
            p = n - a

            p_ack, x_pass, x_nack = get_transitions!(trans_cache, x)

            if a > 0
                n_ack = rand(rng, Binomial(a, p_ack))
                n_nack = a - n_ack
                total_acks += n_ack

                addcount!(new_counts, x_ack, n_ack)
                addcount!(new_counts, x_nack, n_nack)
            end

            if p > 0
                addcount!(new_counts, x_pass, p)
            end
        end

        return new_counts, rf * total_acks
    end

    function step_random!(
        rng::AbstractRNG,
        counts::Dict{Float64,Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}}
    )
        pairs = collect(counts)
        J = length(pairs)
        active_counts = zeros(Int, J)

        rem_pop = N
        rem_draw = M

        if J == 1
            active_counts[1] = M
        else
            for j in 1:(J-1)
                n = last(pairs[j])

                a = if rem_draw == 0
                    0
                elseif rem_draw == rem_pop
                    n
                else
                    rand(rng, Hypergeometric(n, rem_pop - n, rem_draw))
                end

                active_counts[j] = a
                rem_draw -= a
                rem_pop  -= n
            end
            active_counts[J] = rem_draw
        end

        return apply_active_counts!(rng, pairs, active_counts, trans_cache)
    end

    function step_myopic!(
        rng::AbstractRNG,
        counts::Dict{Float64,Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}}
    )
        pairs = collect(counts)
        sort!(pairs; by = p -> first(p), rev = true)

        active_counts = zeros(Int, length(pairs))
        rem = M

        @inbounds for j in eachindex(pairs)
            rem == 0 && break
            x = first(pairs[j])
            n = last(pairs[j])

            if x > 0.0
                a = min(n, rem)
                active_counts[j] = a
                rem -= a
            else
                break
            end
        end

        return apply_active_counts!(rng, pairs, active_counts, trans_cache)
    end

    function step_mpindex!(
        rng::AbstractRNG,
        counts::Dict{Float64,Int},
        trans_cache::Dict{Float64,NTuple{3,Float64}},
        mp_cache::Dict{Float64,Float64}
    )
        pairs = collect(counts)
        J = length(pairs)

        mpvals = Vector{Float64}(undef, J)
        @inbounds for j in 1:J
            mpvals[j] = get_mpindex!(mp_cache, first(pairs[j]))
        end

        order = sortperm(mpvals; rev = true)

        active_counts = zeros(Int, J)
        rem = M

        @inbounds for idx in order
            rem == 0 && break
            v = mpvals[idx]
            if v >= 0.0
                n = last(pairs[idx])
                a = min(n, rem)
                active_counts[idx] = a
                rem -= a
            else
                break
            end
        end

        return apply_active_counts!(rng, pairs, active_counts, trans_cache)
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        # Local caches for this run
        trans_cache = Dict{Float64,NTuple{3,Float64}}()
        mp_cache    = Dict{Float64,Float64}()

        counts_rand = Dict{Float64,Int}(x0f => N)
        counts_myop = Dict{Float64,Int}(x0f => N)
        counts_mp   = Dict{Float64,Int}(x0f => N)

        R_rand = 0.0
        R_myop = 0.0
        R_mp   = 0.0

        for t in 1:T
            disc = β_powers[t]

            counts_rand, rew = step_random!(rng, counts_rand, trans_cache)
            R_rand += disc * rew

            counts_myop, rew = step_myopic!(rng, counts_myop, trans_cache)
            R_myop += disc * rew

            counts_mp, rew = step_mpindex!(rng, counts_mp, trans_cache, mp_cache)
            R_mp += disc * rew
        end

        total_random[run_idx]  = R_rand
        total_myopic[run_idx]  = R_myop
        total_mpindex[run_idx] = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        Threads.@threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (
            mean = μ,
            stderr = se,
            ci_low = ci_low,
            ci_high = ci_high,
            nruns = length(samples)
        )
    end

    results = Dict{String,NamedTuple}()

    results["random"]   = summarize(total_random)
    results["myopic"]   = summarize(total_myopic)
    results["mp_index"] = summarize(total_mpindex)

    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    return results
end

# ============================================================
# Scaling experiment driver
# ============================================================

"""
    run_identical_scaling_experiment(; ...)

Runs the identical-project asymptotic-optimality experiment for a list of N values,
with M scaled in fixed proportion to N.
"""
function run_identical_scaling_experiment(;
    N_values = [50, 100, 200, 400, 800],
    αcap = 0.10,
    beta = 0.99,
    x0 = 0.50,
    q = 0.01,
    rho = 0.90,
    kappa = 0.70,
    r = 1.0,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 2001,
    mp_lookup_mode = :linear
)
    # Build once and reuse
    mp_lookup = build_mp_index_lookup(
        q = q, ρ = rho, κ = kappa, r = r, β = beta,
        ε = ε,
        ngrid = mp_lookup_ngrid,
        mode = mp_lookup_mode
    )

    rows = NamedTuple[]

    for (j, N) in enumerate(N_values)
        M = max(1, min(N, round(Int, αcap * N)))

        println("Running N = $N, M = $M ...")

        results = simulate_identical_project_policies_fast(
            N, M, x0, q, rho, kappa, r, beta;
            T = T_policy,
            nruns = nruns_policy,
            conf_level = conf_level,
            seed = seed + j,
            ε = ε,
            mp_lookup = mp_lookup
        )

        rand_stats = results["random"]
        myop_stats = results["myopic"]
        mp_stats   = results["mp_index"]
        ub_stats   = results["lagrangian_bound"]

        ub = ub_stats.mean
        mp = mp_stats.mean

        abs_gap = ub - mp
        rel_gap = ub != 0.0 ? abs_gap / ub : NaN

        push!(rows, (
            N = N,
            M = M,
            capacity_ratio = M / N,
            lambda_star = ub_stats.lambda_star,
            raw_Lmin = ub_stats.raw_Lmin,
            lagrangian_bound = ub,
            random_mean = rand_stats.mean,
            myopic_mean = myop_stats.mean,
            mp_index_mean = mp_stats.mean,
            mp_index_ci_low = mp_stats.ci_low,
            mp_index_ci_high = mp_stats.ci_high,
            abs_gap_mp_to_bound = abs_gap,
            rel_gap_mp_to_bound = rel_gap
        ))
    end

    return rows
end

In [ ]:
using DataFrames
using CSV
using Printf
using PGFPlotsX

function scaling_rows_to_dataframe(rows)
    df = DataFrame(rows)
    sort!(df, :N)

    df.mp_index_ci_halfwidth =
        (df.mp_index_ci_high .- df.mp_index_ci_low) ./ 2

    df.abs_gap_ci_low  =
        df.lagrangian_bound .- df.mp_index_ci_high
    df.abs_gap_ci_high =
        df.lagrangian_bound .- df.mp_index_ci_low

    df.rel_gap_ci_low  =
        df.abs_gap_ci_low ./ df.lagrangian_bound
    df.rel_gap_ci_high =
        df.abs_gap_ci_high ./ df.lagrangian_bound

    # New scaled-gap columns
    df.N_abs_gap_mp_to_bound      = df.N .* df.abs_gap_mp_to_bound
    df.sqrtN_abs_gap_mp_to_bound  = sqrt.(Float64.(df.N)) .* df.abs_gap_mp_to_bound

    df.N_abs_gap_ci_low           = df.N .* df.abs_gap_ci_low
    df.N_abs_gap_ci_high          = df.N .* df.abs_gap_ci_high

    df.sqrtN_abs_gap_ci_low       = sqrt.(Float64.(df.N)) .* df.abs_gap_ci_low
    df.sqrtN_abs_gap_ci_high      = sqrt.(Float64.(df.N)) .* df.abs_gap_ci_high

    return df
end

function print_scaling_summary(df::DataFrame; conf_level::Float64=0.95)
    ci_pct = Int(round(conf_level * 100))

    println("Identical-project scaling experiment")
    println()

    for row in eachrow(df)
        println("--------------------------------------------------")
        println("N = ", row.N, ", M = ", row.M,
                " (M/N = ", @sprintf("%.4f", row.capacity_ratio), ")")
        println("lambda_star        = ", row.lambda_star)
        println("lagrangian bound   = ", row.lagrangian_bound)
        println("random mean        = ", row.random_mean)
        println("myopic mean        = ", row.myopic_mean)
        println("MP-index mean      = ", row.mp_index_mean)
        println("MP-index ", ci_pct, "% CI = [",
                row.mp_index_ci_low, ", ", row.mp_index_ci_high, "]")
        println("abs gap to bound   = ", row.abs_gap_mp_to_bound)
        println("rel gap to bound   = ", row.rel_gap_mp_to_bound)
        println()
    end
end

function save_scaling_results(df::DataFrame; csvfile::AbstractString="identical_scaling_results.csv")
    CSV.write(csvfile, df)
    println("Saved results to: ", csvfile)
    return csvfile
end

# -----------------------------------------------------------
# PGFPlotsX helpers
# -----------------------------------------------------------

_pgf_xmode(xscale) = xscale == :log10 ? "log" : "linear"

function _legend_style(; pos="north east")
    return (
        "legend pos" => pos,
        "legend cell align" => "left",
        "font" => raw"\small",
    )
end

function scaling_performance_axis(df::DataFrame;
                                  xscale=:identity,
                                  width="11cm",
                                  height="7cm",
                                  legend_pos="south east")

    N = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"normalized reward per project",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "black",
                mark = "square*",
                "thick"
            },
            Table(x = N, y = df.lagrangian_bound)
        ),
        LegendEntry("Lagrangian bound"),

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = N, y = df.mp_index_mean)
        ),
        LegendEntry("MP-index"),
        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = N, y = df.mp_index_ci_low)
        ),
        LegendEntry("MP-index CI bounds"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = N, y = df.mp_index_ci_high)
        ),

        Plot(
            {
                color = "red",
                mark = "triangle*",
                "thick"
            },
            Table(x = N, y = df.myopic_mean)
        ),
        LegendEntry("Myopic"),

        Plot(
            {
                color = "gray",
                mark = "o",
                "thick"
            },
            Table(x = N, y = df.random_mean)
        ),
        LegendEntry("Random")
    )

    return ax
end

function scaling_abs_gap_axis(df::DataFrame;
                              xscale=:identity,
                              width="11cm",
                              height="7cm",
                              legend_pos="north east")

    N = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"absolute gap to bound",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = N, y = df.abs_gap_mp_to_bound)
        ),
        LegendEntry("Absolute gap"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = N, y = df.abs_gap_ci_low)
        ),
        LegendEntry("Gap CI bounds"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = N, y = df.abs_gap_ci_high)
        )
    )

    return ax
end

function scaling_rel_gap_axis(df::DataFrame;
                              xscale=:identity,
                              width="11cm",
                              height="7cm",
                              legend_pos="north east")

    N = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"relative gap to bound",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = N, y = df.rel_gap_mp_to_bound)
        ),
        LegendEntry("Relative gap"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = N, y = df.rel_gap_ci_low)
        ),
        LegendEntry("Gap CI bounds"),
        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = N, y = df.rel_gap_ci_high)
        )
    )

    return ax
end


function save_scaling_pgfplots(df::DataFrame;
                               prefix::AbstractString="identical_scaling",
                               xscale=:identity,
                               width="11cm",
                               height="7cm")

    ax_perf  = scaling_performance_axis(df; xscale=xscale, width=width, height=height)
    ax_abs   = scaling_abs_gap_axis(df; xscale=xscale, width=width, height=height)
    ax_rel   = scaling_rel_gap_axis(df; xscale=xscale, width=width, height=height)
    ax_Ngap  = scaling_N_gap_axis(df; xscale=xscale, width=width, height=height)
    ax_sNgap = scaling_sqrtN_gap_axis(df; xscale=xscale, width=width, height=height)

    PGFPlotsX.pgfsave(prefix * "_performance.tex", ax_perf)
    PGFPlotsX.pgfsave(prefix * "_abs_gap.tex", ax_abs)
    PGFPlotsX.pgfsave(prefix * "_rel_gap.tex", ax_rel)
    PGFPlotsX.pgfsave(prefix * "_N_gap.tex", ax_Ngap)
    PGFPlotsX.pgfsave(prefix * "_sqrtN_gap.tex", ax_sNgap)

    return (
        performance = ax_perf,
        abs_gap = ax_abs,
        rel_gap = ax_rel,
        N_gap = ax_Ngap,
        sqrtN_gap = ax_sNgap
    )
end

function scaling_N_gap_axis(df::DataFrame;
                            xscale=:identity,
                            width="11cm",
                            height="7cm",
                            legend_pos="north east")

    Nvals = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"$N\,(\mathrm{bound}-\mathrm{MP})$",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = Nvals, y = df.N_abs_gap_mp_to_bound)
        ),
        LegendEntry(raw"$N(\mathrm{bound}-\mathrm{MP})$"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = Nvals, y = df.N_abs_gap_ci_low)
        ),
        LegendEntry("CI bounds"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = Nvals, y = df.N_abs_gap_ci_high)
        )
    )

    return ax
end

function scaling_sqrtN_gap_axis(df::DataFrame;
                                xscale=:identity,
                                width="11cm",
                                height="7cm",
                                legend_pos="north east")

    Nvals = Float64.(df.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"$\sqrt{N}\,(\mathrm{bound}-\mathrm{MP})$",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot(
            {
                color = "blue",
                mark = "*",
                "thick"
            },
            Table(x = Nvals, y = df.sqrtN_abs_gap_mp_to_bound)
        ),
        LegendEntry(raw"$\sqrt{N}(\mathrm{bound}-\mathrm{MP})$"),

        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin"
            },
            Table(x = Nvals, y = df.sqrtN_abs_gap_ci_low)
        ),
        LegendEntry("CI bounds"),
        Plot(
            {
                color = "blue",
                style = "dashed",
                "thin",
                "forget plot"
            },
            Table(x = Nvals, y = df.sqrtN_abs_gap_ci_high)
        )
    )

    return ax
end

In [ ]:
rows = run_identical_scaling_experiment(
    N_values = [50, 100, 200, 400],
    αcap = 0.10,
    beta = 0.99,
    x0 = 0.50,
    q = 0.01,
    rho = 0.90,
    kappa = 0.70,
    r = 1.0,
    T_policy = 200,
    nruns_policy = 1000,
    seed = 1234,
    mp_lookup_ngrid = 1001,   # start here
    mp_lookup_mode = :linear
)

df = scaling_rows_to_dataframe(rows)

print_scaling_summary(df)
save_scaling_results(df; csvfile="identical_scaling_results.csv")

axs = save_scaling_pgfplots(
    df;
    prefix = "identical_scaling",
    xscale = :log10,
    width = "12cm",
    height = "7cm"
)

# Optional: inspect the most relevant columns in tabular form
select(df,
    :N, :M, :capacity_ratio,
    :lagrangian_bound, :mp_index_mean,
    :abs_gap_mp_to_bound, :rel_gap_mp_to_bound,
    :N_abs_gap_mp_to_bound, :sqrtN_abs_gap_mp_to_bound
)

# In Jupyter, display the PGFPlotsX axes directly:
axs.performance
axs.abs_gap
axs.rel_gap
axs.N_gap
axs.sqrtN_gap

In [ ]:
println("Threads.nthreads()      = ", Threads.nthreads())
println("Threads.nthreads(:default) = ", Threads.nthreads(:default))
println("Threads.nthreads(:interactive) = ", Threads.nthreads(:interactive))
println("Threads.maxthreadid()   = ", Threads.maxthreadid())
println("Threads.threadid()      = ", Threads.threadid())

In [ ]:
using Random
using Statistics
using Distributions
using Base.Threads
using LinearAlgebra
using DataFrames
using CSV
using Printf

BLAS.set_num_threads(1)

# -----------------------------------------------------------
# Uniform MP-index lookup table
# -----------------------------------------------------------

struct MPIndexLookupUniform
    xlo::Float64
    xhi::Float64
    h::Float64
    vals::Vector{Float64}
    mode::Symbol   # :leftbin, :nearest, :linear
end

"""
    build_mp_index_lookup(; q, ρ, κ, r, β, ε=1e-10,
                             ngrid=2001, xlo=0.0, xhi=1.0,
                             mode=:linear)

Precompute m(x)=f(x,x)/g(x,x) on a uniform x-grid.
"""
function build_mp_index_lookup(; q, ρ, κ, r, β,
                                 ε::Float64 = 1e-10,
                                 ngrid::Int = 2001,
                                 xlo::Float64 = 0.0,
                                 xhi::Float64 = 1.0,
                                 mode::Symbol = :linear)

    @assert ngrid >= 2
    @assert xlo < xhi
    @assert mode in (:leftbin, :nearest, :linear)

    xgrid = collect(range(xlo, xhi; length=ngrid))
    vals = Vector{Float64}(undef, ngrid)

    for i in eachindex(xgrid)
        x = xgrid[i]
        fxx = float(f_metric2(x, x; q=q, ρ=ρ, κ=κ, r=r, β=β, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=ρ, κ=κ,      β=β, ε=ε))
        vals[i] = (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    return MPIndexLookupUniform(xlo, xhi, (xhi - xlo) / (ngrid - 1), vals, mode)
end

"""
    mp_index_lookup(L, x)

Approximate m(x) from a precomputed uniform lookup table.
"""
@inline function mp_index_lookup(L::MPIndexLookupUniform, x::Real)
    xf = clamp(Float64(x), L.xlo, L.xhi)
    vals = L.vals
    n = length(vals)

    if xf <= L.xlo
        return vals[1]
    elseif xf >= L.xhi
        return vals[end]
    end

    t = (xf - L.xlo) / L.h
    i = floor(Int, t) + 1
    i = clamp(i, 1, n - 1)

    if L.mode === :leftbin
        return vals[i]
    elseif L.mode === :nearest
        α = t - floor(t)
        return α < 0.5 ? vals[i] : vals[i + 1]
    else
        α = t - floor(t)
        return (1 - α) * vals[i] + α * vals[i + 1]
    end
end

"""
    build_type_mp_lookups(q_types, rho_types, kappa_types, r_types, beta; ...)

Build one MP-index lookup table per project type.
"""
function build_type_mp_lookups(q_types::AbstractVector,
                               rho_types::AbstractVector,
                               kappa_types::AbstractVector,
                               r_types::AbstractVector,
                               beta::Real;
                               ε::Float64 = 1e-10,
                               ngrid::Int = 2001,
                               xlo::Float64 = 0.0,
                               xhi::Float64 = 1.0,
                               mode::Symbol = :linear)

    K = length(q_types)
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K

    return [
        build_mp_index_lookup(
            q = Float64(q_types[k]),
            ρ = Float64(rho_types[k]),
            κ = Float64(kappa_types[k]),
            r = Float64(r_types[k]),
            β = Float64(beta),
            ε = ε,
            ngrid = ngrid,
            xlo = xlo,
            xhi = xhi,
            mode = mode
        )
        for k in 1:K
    ]
end

In [ ]:
# -----------------------------------------------------------
# Structured K-type instance builder
# -----------------------------------------------------------

"""
    build_typed_instance(N; prop_type, q_types, rho_types, kappa_types, r_types,
                            x_init_scheme=:linspace, x0=0.5,
                            xlo=0.0, xhi=1.0,
                            shuffle_types=true, shuffle_x=false, seed=nothing)

Build one structured instance with fixed type proportions.

Returns a NamedTuple with:
- x_vec
- type_id_vec
- q_vec, rho_vec, kappa_vec, r_vec
- n_types
"""
function build_typed_instance(N::Integer;
                              prop_type::AbstractVector,
                              q_types::AbstractVector,
                              rho_types::AbstractVector,
                              kappa_types::AbstractVector,
                              r_types::AbstractVector,
                              x_init_scheme::Symbol = :linspace,
                              x0::Real = 0.5,
                              xlo::Real = 0.0,
                              xhi::Real = 1.0,
                              shuffle_types::Bool = true,
                              shuffle_x::Bool = false,
                              seed::Union{Int,Nothing} = nothing)

    @assert N >= 1
    K = length(prop_type)
    @assert length(q_types)     == K
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K
    @assert isapprox(sum(prop_type), 1.0; atol=1e-12)

    n_types = round.(Int, N .* Float64.(prop_type))
    n_types[end] += N - sum(n_types)

    type_id_vec = Vector{Int}(undef, N)
    idx = 1
    for k in 1:K
        for _ in 1:n_types[k]
            type_id_vec[idx] = k
            idx += 1
        end
    end

    rng = seed === nothing ? Random.default_rng() : MersenneTwister(seed)

    if shuffle_types
        shuffle!(rng, type_id_vec)
    end

    x_vec = if x_init_scheme === :linspace
        collect(range(Float64(xlo), Float64(xhi); length=N))
    elseif x_init_scheme === :constant
        fill(Float64(x0), N)
    else
        error("Unsupported x_init_scheme = $x_init_scheme. Use :linspace or :constant.")
    end

    if shuffle_x
        shuffle!(rng, x_vec)
    end

    q_vec     = [Float64(q_types[type_id_vec[i]])     for i in 1:N]
    rho_vec   = [Float64(rho_types[type_id_vec[i]])   for i in 1:N]
    kappa_vec = [Float64(kappa_types[type_id_vec[i]]) for i in 1:N]
    r_vec     = [Float64(r_types[type_id_vec[i]])     for i in 1:N]

    return (
        x_vec = x_vec,
        type_id_vec = type_id_vec,
        q_vec = q_vec,
        rho_vec = rho_vec,
        kappa_vec = kappa_vec,
        r_vec = r_vec,
        n_types = n_types
    )
end

In [ ]:
# -----------------------------------------------------------
# Helper for top-M selection
# -----------------------------------------------------------

@inline function choose_topM_mask!(mask::BitVector,
                                   vals::AbstractVector{<:Real},
                                   M::Int;
                                   positive::Bool = false,
                                   nonnegative::Bool = false)
    fill!(mask, false)
    m = min(M, length(vals))
    ord = partialsortperm(vals, 1:m; rev=true)

    for idx in ord
        v = vals[idx]
        if positive
            v > 0.0 || break
        elseif nonnegative
            v >= 0.0 || break
        end
        mask[idx] = true
    end
    return nothing
end

# -----------------------------------------------------------
# Multi-project simulator with optional lookup MP-index
# -----------------------------------------------------------

"""
    simulate_multi_project_policies_lookup(
        x_vec, type_id_vec, q_types, rho_types, kappa_types, r_types, beta, M;
        T=100, nruns=1000, conf_level=0.95,
        seed=nothing, ε=1e-10, mp_lookups=nothing
    )

Monte Carlo estimation of approximate long-run average discounted reward per project
under four policies:

  1. Random
  2. Myopic
  3. Round-robin
  4. MP-index

with the MP-index policy using precomputed lookup tables if `mp_lookups !== nothing`.

Also computes the normalized Lagrangian upper bound via
    λstar, Lmin = lambda_star_vector(...)
"""
function simulate_multi_project_policies_lookup(
    x_vec::AbstractVector{<:Real},
    type_id_vec::AbstractVector{<:Integer},
    q_types::AbstractVector{<:Real},
    rho_types::AbstractVector{<:Real},
    kappa_types::AbstractVector{<:Real},
    r_types::AbstractVector{<:Real},
    beta::Float64,
    M::Integer;
    T::Int = 100,
    nruns::Int = 1_000,
    conf_level::Float64 = 0.95,
    seed::Union{Int,Nothing} = nothing,
    ε::Float64 = 1e-10,
    mp_lookups::Union{Nothing,AbstractVector{<:MPIndexLookupUniform}} = nothing
)

    N = length(x_vec)
    @assert length(type_id_vec) == N
    K = length(q_types)
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K
    @assert 0.0 < beta < 1.0
    @assert 1 <= M <= N
    @assert T > 0
    @assert nruns > 0

    if mp_lookups !== nothing
        @assert length(mp_lookups) == K
    end

    x_vec_f = Float64.(x_vec)

    q_types_f     = Float64.(q_types)
    rho_types_f   = Float64.(rho_types)
    kappa_types_f = Float64.(kappa_types)
    r_types_f     = Float64.(r_types)

    type_id = Int.(type_id_vec)

    for i in 1:N
        @assert 1 <= type_id[i] <= K
        @assert 0.0 <= x_vec_f[i] <= 1.0
    end

    for k in 1:K
        @assert 0.0 <= q_types_f[k] < 1.0
        @assert 0.0 <= rho_types_f[k] < 1.0 - q_types_f[k]
        @assert 0.0 <= kappa_types_f[k] <= 1.0
        @assert r_types_f[k] >= 0.0
    end

    q_vec     = [q_types_f[type_id[i]]     for i in 1:N]
    rho_vec   = [rho_types_f[type_id[i]]   for i in 1:N]
    kappa_vec = [kappa_types_f[type_id[i]] for i in 1:N]
    r_vec     = [r_types_f[type_id[i]]     for i in 1:N]

    λstar, Lmin = lambda_star_vector(
        x_vec_f;
        qvector = q_vec,
        ρvector = rho_vec,
        κvector = kappa_vec,
        rvector = r_vec,
        β = beta,
        M = M
    )

    β_powers = [beta^t for t in 0:(T-1)]

    total_random     = zeros(Float64, nruns)
    total_myopic     = zeros(Float64, nruns)
    total_roundrobin = zeros(Float64, nruns)
    total_mpindex    = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function phi0(x, q, rho)
        q + rho * x
    end

    @inline function phi1(x, q, rho, kappa)
        denom = 1.0 - kappa * x
        if denom <= 0.0
            return q + rho
        end
        post_nack = (1.0 - kappa) * x / denom
        return q + rho * post_nack
    end

    @inline function mp_index_current_exact(x, q, rho, kappa, r, beta, ε)
        fxx = float(f_metric2(x, x; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=rho, κ=kappa,      β=beta, ε=ε))
        return (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        X_rand = copy(x_vec_f)
        X_myop = copy(x_vec_f)
        X_rr   = copy(x_vec_f)
        X_mp   = copy(x_vec_f)

        rr_start = 1

        R_rand = 0.0
        R_myop = 0.0
        R_rr   = 0.0
        R_mp   = 0.0

        # reusable work arrays
        myopic_value = zeros(Float64, N)
        mpvals       = zeros(Float64, N)

        active_rand_mask = falses(N)
        active_myop_mask = falses(N)
        active_rr_mask   = falses(N)
        active_mp_mask   = falses(N)

        perm = collect(1:N)

        for t in 1:T
            disc = β_powers[t]

            # ------------------------------------------------------
            # 1. Random policy
            # ------------------------------------------------------
            fill!(active_rand_mask, false)
            shuffle!(rng, perm)
            active_rand_mask[perm[1:M]] .= true

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rand[i]

                if active_rand_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rand[i] = q + rho
                    else
                        X_rand[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rand[i] = phi0(xi, q, rho)
                end
            end
            R_rand += disc * reward_t

            # ------------------------------------------------------
            # 2. Myopic policy
            # ------------------------------------------------------
            for i in 1:N
                myopic_value[i] = r_vec[i] * kappa_vec[i] * X_myop[i]
            end
            choose_topM_mask!(active_myop_mask, myopic_value, M; positive=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_myop[i]

                if active_myop_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_myop[i] = q + rho
                    else
                        X_myop[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_myop[i] = phi0(xi, q, rho)
                end
            end
            R_myop += disc * reward_t

            # ------------------------------------------------------
            # 3. Round-robin policy
            # ------------------------------------------------------
            fill!(active_rr_mask, false)
            for k in 0:(M-1)
                i = ((rr_start - 1 + k) % N) + 1
                active_rr_mask[i] = true
            end
            rr_start = ((rr_start - 1 + M) % N) + 1

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rr[i]

                if active_rr_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rr[i] = q + rho
                    else
                        X_rr[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rr[i] = phi0(xi, q, rho)
                end
            end
            R_rr += disc * reward_t

            # ------------------------------------------------------
            # 4. MP-index policy (lookup-based if provided)
            # ------------------------------------------------------
            for i in 1:N
                k = type_id[i]
                xi = X_mp[i]
                mpvals[i] = mp_lookups === nothing ?
                    mp_index_current_exact(xi,
                                           q_types_f[k], rho_types_f[k], kappa_types_f[k],
                                           r_types_f[k], beta, ε) :
                    mp_index_lookup(mp_lookups[k], xi)
            end
            choose_topM_mask!(active_mp_mask, mpvals, M; nonnegative=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_mp[i]

                if active_mp_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_mp[i] = q + rho
                    else
                        X_mp[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_mp[i] = phi0(xi, q, rho)
                end
            end
            R_mp += disc * reward_t
        end

        total_random[run_idx]     = R_rand
        total_myopic[run_idx]     = R_myop
        total_roundrobin[run_idx] = R_rr
        total_mpindex[run_idx]    = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        Threads.@threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (mean = μ,
                stderr = se,
                ci_low = ci_low,
                ci_high = ci_high,
                nruns = length(samples))
    end

    results = Dict{String,NamedTuple}()
    results["random"]          = summarize(total_random)
    results["myopic"]          = summarize(total_myopic)
    results["round_robin"]     = summarize(total_roundrobin)
    results["mp_index"]        = summarize(total_mpindex)

    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    return results
end

In [ ]:
# -----------------------------------------------------------
# Driver for scaling experiments on a grid of parameter instances
# -----------------------------------------------------------

"""
    run_typed_scaling_experiment(; scenario_grid, prop_grid, alphaM_values, N_values, ...)

Runs the two-type / K-type benchmark experiment over a grid of:
- parameter scenarios,
- type proportions,
- capacity ratios alphaM = M/N,
- system sizes N.

Each scenario should be a NamedTuple with fields:
- name
- q_types
- rho_types
- kappa_types
- r_types
"""
function run_typed_scaling_experiment(;
    scenario_grid,
    prop_grid,
    alphaM_values,
    N_values,
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    x0 = 0.5,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true
)

    rows = NamedTuple[]
    counter = 0

    for (s_idx, scen) in enumerate(scenario_grid)
        name = hasproperty(scen, :name) ? String(scen.name) : "scenario_$s_idx"

        q_types     = Float64.(scen.q_types)
        rho_types   = Float64.(scen.rho_types)
        kappa_types = Float64.(scen.kappa_types)
        r_types     = Float64.(scen.r_types)

        mp_lookups = build_type_mp_lookups(
            q_types, rho_types, kappa_types, r_types, beta;
            ε = ε,
            ngrid = mp_lookup_ngrid,
            mode = mp_lookup_mode
        )

        for prop_type in prop_grid
            prop_type_f = Float64.(prop_type)
            K = length(prop_type_f)

            for alphaM in alphaM_values
                for N in N_values
                    counter += 1
                    M = max(1, min(N, round(Int, alphaM * N)))

                    instance_seed = base_seed + counter

                    inst = build_typed_instance(
                        N;
                        prop_type = prop_type_f,
                        q_types = q_types,
                        rho_types = rho_types,
                        kappa_types = kappa_types,
                        r_types = r_types,
                        x_init_scheme = x_init_scheme,
                        x0 = x0,
                        xlo = xlo,
                        xhi = xhi,
                        shuffle_types = shuffle_types,
                        shuffle_x = shuffle_x,
                        seed = instance_seed
                    )

                    println("Running ",
                            name,
                            " | prop = ", prop_type_f,
                            " | αM = ", alphaM,
                            " | N = ", N,
                            " | M = ", M)

                    results = simulate_multi_project_policies_lookup(
                        inst.x_vec,
                        inst.type_id_vec,
                        q_types,
                        rho_types,
                        kappa_types,
                        r_types,
                        beta,
                        M;
                        T = T_policy,
                        nruns = nruns_policy,
                        conf_level = conf_level,
                        seed = instance_seed,
                        ε = ε,
                        mp_lookups = mp_lookups
                    )

                    rand_stats = results["random"]
                    myop_stats = results["myopic"]
                    rr_stats   = results["round_robin"]
                    mp_stats   = results["mp_index"]
                    ub_stats   = results["lagrangian_bound"]

                    ub = ub_stats.mean
                    rand_mean = rand_stats.mean
                    myop_mean = myop_stats.mean
                    rr_mean   = rr_stats.mean
                    mp_mean   = mp_stats.mean

                    abs_gap_mp = ub - mp_mean
                    rel_gap_mp = ub != 0.0 ? abs_gap_mp / ub : NaN

                    abs_gap_myop = ub - myop_mean
                    abs_gap_rr   = ub - rr_mean
                    abs_gap_rand = ub - rand_mean

                    sqrtN = sqrt(Float64(N))

                    row = (
                        scenario = name,
                        N = N,
                        M = M,
                        capacity_ratio = M / N,
                        beta = beta,

                        K = K,
                        prop_type = join(string.(round.(prop_type_f; digits=4)), ","),

                        q_types = join(string.(q_types), ","),
                        rho_types = join(string.(rho_types), ","),
                        kappa_types = join(string.(kappa_types), ","),
                        r_types = join(string.(r_types), ","),

                        lambda_star = ub_stats.lambda_star,
                        raw_Lmin = ub_stats.raw_Lmin,
                        lagrangian_bound = ub,

                        random_mean = rand_mean,
                        random_ci_low = rand_stats.ci_low,
                        random_ci_high = rand_stats.ci_high,

                        myopic_mean = myop_mean,
                        myopic_ci_low = myop_stats.ci_low,
                        myopic_ci_high = myop_stats.ci_high,

                        round_robin_mean = rr_mean,
                        round_robin_ci_low = rr_stats.ci_low,
                        round_robin_ci_high = rr_stats.ci_high,

                        mp_index_mean = mp_mean,
                        mp_index_ci_low = mp_stats.ci_low,
                        mp_index_ci_high = mp_stats.ci_high,

                        abs_gap_mp_to_bound = abs_gap_mp,
                        rel_gap_mp_to_bound = rel_gap_mp,
                        N_abs_gap_mp_to_bound = N * abs_gap_mp,
                        sqrtN_abs_gap_mp_to_bound = sqrtN * abs_gap_mp,

                        abs_gap_myopic_to_bound = abs_gap_myop,
                        abs_gap_roundrobin_to_bound = abs_gap_rr,
                        abs_gap_random_to_bound = abs_gap_rand,

                        mp_lookup_ngrid = mp_lookup_ngrid,
                        mp_lookup_mode = String(mp_lookup_mode),
                        x_init_scheme = String(x_init_scheme),
                        seed = instance_seed
                    )

                    # add actual counts by type as separate columns when K is small
                    if K >= 1
                        row = merge(row, (n_type1 = inst.n_types[1],))
                    end
                    if K >= 2
                        row = merge(row, (n_type2 = inst.n_types[2],))
                    end

                    push!(rows, row)
                end
            end
        end
    end

    return rows
end

In [ ]:
function typed_scaling_rows_to_dataframe(rows)
    df = DataFrame(rows)
    sort!(df, [:scenario, :prop_type, :capacity_ratio, :N])
    return df
end

In [ ]:
# -----------------------------------------------------------
# Example scenario grid
# -----------------------------------------------------------

scenario_grid = [
    (
        name = "A_selfhealing_B_fragile",
        q_types = [0.01, 0.10],
        rho_types = [0.90, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    ),

    (
        name = "A_mild_B_fragile",
        q_types = [0.03, 0.10],
        rho_types = [0.80, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    )
]

prop_grid = [
    [0.20, 0.80],
    [0.40, 0.60],
    [0.60, 0.40],
    [0.80, 0.20]
]

alphaM_values = [0.05, 0.10, 0.20]
N_values = [50, 100, 200, 400]

rows = run_typed_scaling_experiment(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = alphaM_values,
    N_values = N_values,
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true
)

df = typed_scaling_rows_to_dataframe(rows)

CSV.write("typed_scaling_experiment.csv", df)

first(df, 10)

In [ ]:
using CSV
using DataFrames
using Statistics
using Printf
using PGFPlotsX

# -----------------------------------------------------------
# 1) Load and basic checks
# -----------------------------------------------------------

csvfile = "typed_scaling_experiment.csv"
df = CSV.read(csvfile, DataFrame)

function check_typed_experiment_completeness(df::DataFrame)
    ns = length(unique(df.scenario))
    np = length(unique(df.prop_type))
    na = length(unique(df.capacity_ratio))
    nN = length(unique(df.N))
    expected = ns * np * na * nN
    actual = nrow(df)

    println("Rows in CSV          = ", actual)
    println("Unique scenarios     = ", ns)
    println("Unique prop_type     = ", np)
    println("Unique capacity_ratio= ", na)
    println("Unique N             = ", nN)
    println("Expected rows        = ", expected)
    println("Complete?            = ", actual == expected)

    return (actual=actual, expected=expected, complete=(actual == expected))
end

chk = check_typed_experiment_completeness(df)

# -----------------------------------------------------------
# 2) Add derived columns
# -----------------------------------------------------------

function enrich_typed_results!(df::DataFrame)
    # CI half-widths
    df.mp_ci_halfwidth      = (df.mp_index_ci_high .- df.mp_index_ci_low) ./ 2
    df.myopic_ci_halfwidth  = (df.myopic_ci_high .- df.myopic_ci_low) ./ 2
    df.rr_ci_halfwidth      = (df.round_robin_ci_high .- df.round_robin_ci_low) ./ 2
    df.random_ci_halfwidth  = (df.random_ci_high .- df.random_ci_low) ./ 2

    # Scaled gaps for other benchmarks too
    df.rel_gap_myopic_to_bound      = df.abs_gap_myopic_to_bound ./ df.lagrangian_bound
    df.rel_gap_roundrobin_to_bound  = df.abs_gap_roundrobin_to_bound ./ df.lagrangian_bound
    df.rel_gap_random_to_bound      = df.abs_gap_random_to_bound ./ df.lagrangian_bound

    df.N_abs_gap_myopic_to_bound     = df.N .* df.abs_gap_myopic_to_bound
    df.N_abs_gap_roundrobin_to_bound = df.N .* df.abs_gap_roundrobin_to_bound
    df.N_abs_gap_random_to_bound     = df.N .* df.abs_gap_random_to_bound

    df.sqrtN_abs_gap_myopic_to_bound     = sqrt.(Float64.(df.N)) .* df.abs_gap_myopic_to_bound
    df.sqrtN_abs_gap_roundrobin_to_bound = sqrt.(Float64.(df.N)) .* df.abs_gap_roundrobin_to_bound
    df.sqrtN_abs_gap_random_to_bound     = sqrt.(Float64.(df.N)) .* df.abs_gap_random_to_bound

    # Best simulated benchmark in each row
    best_name = Vector{String}(undef, nrow(df))
    best_mean = Vector{Float64}(undef, nrow(df))

    for i in 1:nrow(df)
        vals = [
            ("random",      df.random_mean[i]),
            ("myopic",      df.myopic_mean[i]),
            ("round_robin", df.round_robin_mean[i]),
            ("mp_index",    df.mp_index_mean[i]),
        ]
        j = argmax(last.(vals))
        best_name[i] = vals[j][1]
        best_mean[i] = vals[j][2]
    end

    df.best_policy = best_name
    df.best_policy_mean = best_mean

    sort!(df, [:scenario, :prop_type, :capacity_ratio, :N])
    return df
end

enrich_typed_results!(df)

# -----------------------------------------------------------
# 3) Quick textual summaries
# -----------------------------------------------------------

function quick_summary_tables(df::DataFrame)
    println("\n=== Overall averages by scenario ===")
    scen = combine(groupby(df, :scenario),
        :abs_gap_mp_to_bound => mean => :mean_abs_gap_mp,
        :rel_gap_mp_to_bound => mean => :mean_rel_gap_mp,
        :mp_index_mean       => mean => :mean_mp,
        :myopic_mean         => mean => :mean_myopic,
        :round_robin_mean    => mean => :mean_rr,
        :random_mean         => mean => :mean_random
    )
    display(scen)

    println("\n=== How often each policy is best ===")
    besttab = combine(groupby(df, [:scenario, :best_policy]), nrow => :count)
    display(besttab)

    println("\n=== Worst MP gaps (largest abs_gap_mp_to_bound) ===")
    display(first(sort(df, :abs_gap_mp_to_bound, rev=true), 12))

    println("\n=== Best MP cases (smallest abs_gap_mp_to_bound) ===")
    display(first(sort(df, :abs_gap_mp_to_bound), 12))

    return nothing
end

quick_summary_tables(df)

# -----------------------------------------------------------
# 4) Slice-specific summary table
# -----------------------------------------------------------

function slice_table(df::DataFrame; scenario, prop_type, capacity_ratio)
    d = filter(row ->
        row.scenario == scenario &&
        row.prop_type == prop_type &&
        row.capacity_ratio == capacity_ratio,
        df
    )
    sort!(d, :N)
    return select(d,
        :scenario, :prop_type, :capacity_ratio, :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
        :myopic_mean, :round_robin_mean, :random_mean,
        :abs_gap_mp_to_bound, :rel_gap_mp_to_bound,
        :N_abs_gap_mp_to_bound, :sqrtN_abs_gap_mp_to_bound,
        :best_policy
    )
end

# Example:
# display(slice_table(df; scenario="A_selfhealing_B_fragile", prop_type="0.4,0.6", capacity_ratio=0.1))

# -----------------------------------------------------------
# 5) PGFPlotsX helpers
# -----------------------------------------------------------

_pgf_xmode(xscale) = xscale == :log10 ? "log" : "linear"

# NOTE:
# Do not use helper tuples expanded inside @pgf option braces.
# Inline the options directly; that is more robust with PGFPlotsX.

function performance_axis_typed(d::DataFrame;
                                xscale=:identity,
                                width="11cm",
                                height="7cm",
                                legend_pos="south east")

    N = Float64.(d.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"normalized reward per project",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot({color="black", mark="square*", "thick"},
             Table(x=N, y=d.lagrangian_bound)),
        LegendEntry("Lagrangian bound"),

        Plot({color="blue", mark="*", "thick"},
             Table(x=N, y=d.mp_index_mean)),
        LegendEntry("MP-index"),

        Plot({color="blue", style="dashed", "thin"},
             Table(x=N, y=d.mp_index_ci_low)),
        LegendEntry("MP-index CI bounds"),

        Plot({color="blue", style="dashed", "thin", "forget plot"},
             Table(x=N, y=d.mp_index_ci_high)),

        Plot({color="red", mark="triangle*", "thick"},
             Table(x=N, y=d.myopic_mean)),
        LegendEntry("Myopic"),

        Plot({color="gray", mark="diamond*", "thick"},
             Table(x=N, y=d.round_robin_mean)),
        LegendEntry("Round-robin"),

        Plot({color="black!50", mark="o", "thick"},
             Table(x=N, y=d.random_mean)),
        LegendEntry("Random")
    )

    return ax
end

function abs_gap_axis_typed(d::DataFrame;
                            xscale=:identity,
                            width="11cm",
                            height="7cm",
                            legend_pos="north east")

    N = Float64.(d.N)

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = raw"absolute gap to bound",
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot({color="blue", mark="*", "thick"},
             Table(x=N, y=d.abs_gap_mp_to_bound)),
        LegendEntry("MP-index"),

        Plot({color="red", mark="triangle*", "thick"},
             Table(x=N, y=d.abs_gap_myopic_to_bound)),
        LegendEntry("Myopic"),

        Plot({color="gray", mark="diamond*", "thick"},
             Table(x=N, y=d.abs_gap_roundrobin_to_bound)),
        LegendEntry("Round-robin"),

        Plot({color="black!50", mark="o", "thick"},
             Table(x=N, y=d.abs_gap_random_to_bound)),
        LegendEntry("Random")
    )

    return ax
end

function scaled_gap_axis_typed(d::DataFrame;
                               scale=:N,
                               xscale=:identity,
                               width="11cm",
                               height="7cm",
                               legend_pos="north east")

    N = Float64.(d.N)

    if scale == :N
        y_mp = d.N_abs_gap_mp_to_bound
        y_my = d.N_abs_gap_myopic_to_bound
        y_rr = d.N_abs_gap_roundrobin_to_bound
        y_ra = d.N_abs_gap_random_to_bound
        ylabel = raw"$N\,(\mathrm{bound}-\mathrm{policy})$"
    elseif scale == :sqrtN
        y_mp = d.sqrtN_abs_gap_mp_to_bound
        y_my = d.sqrtN_abs_gap_myopic_to_bound
        y_rr = d.sqrtN_abs_gap_roundrobin_to_bound
        y_ra = d.sqrtN_abs_gap_random_to_bound
        ylabel = raw"$\sqrt{N}\,(\mathrm{bound}-\mathrm{policy})$"
    else
        error("scale must be :N or :sqrtN")
    end

    ax = @pgf Axis(
        {
            width = width,
            height = height,
            xlabel = raw"$N$",
            ylabel = ylabel,
            xmode = _pgf_xmode(xscale),
            ymajorgrids,
            xmajorgrids,
            "legend pos" = legend_pos,
            "legend cell align" = "left",
            "font" = raw"\small",
            "mark size" = "2.2pt"
        },

        Plot({color="blue", mark="*", "thick"},
             Table(x=N, y=y_mp)),
        LegendEntry("MP-index"),

        Plot({color="red", mark="triangle*", "thick"},
             Table(x=N, y=y_my)),
        LegendEntry("Myopic"),

        Plot({color="gray", mark="diamond*", "thick"},
             Table(x=N, y=y_rr)),
        LegendEntry("Round-robin"),

        Plot({color="black!50", mark="o", "thick"},
             Table(x=N, y=y_ra)),
        LegendEntry("Random")
    )

    return ax
end

# -----------------------------------------------------------
# 6) Build and save plots for one slice
# -----------------------------------------------------------

function save_typed_slice_plots(df::DataFrame;
                                scenario,
                                prop_type,
                                capacity_ratio,
                                prefix="typed_slice",
                                xscale=:log10,
                                width="12cm",
                                height="7cm")

    d = filter(row ->
        row.scenario == scenario &&
        row.prop_type == prop_type &&
        row.capacity_ratio == capacity_ratio,
        df
    )
    sort!(d, :N)

    @assert nrow(d) > 0 "No rows found for the requested slice."

    ax_perf  = performance_axis_typed(d; xscale=xscale, width=width, height=height)
    ax_abs   = abs_gap_axis_typed(d; xscale=xscale, width=width, height=height)
    ax_Ngap  = scaled_gap_axis_typed(d; scale=:N, xscale=xscale, width=width, height=height)
    ax_sNgap = scaled_gap_axis_typed(d; scale=:sqrtN, xscale=xscale, width=width, height=height)

    base = string(prefix,
                  "_", replace(scenario, " " => "_"),
                  "_prop_", replace(prop_type, "," => "-"),
                  "_cap_", capacity_ratio)

    PGFPlotsX.pgfsave(base * "_performance.tex", ax_perf)
    PGFPlotsX.pgfsave(base * "_abs_gap.tex", ax_abs)
    PGFPlotsX.pgfsave(base * "_N_gap.tex", ax_Ngap)
    PGFPlotsX.pgfsave(base * "_sqrtN_gap.tex", ax_sNgap)

    return (performance=ax_perf, abs_gap=ax_abs, N_gap=ax_Ngap, sqrtN_gap=ax_sNgap, slice=d)
end

# Example:
# axs = save_typed_slice_plots(df;
#     scenario="A_selfhealing_B_fragile",
#     prop_type="0.4,0.6",
#     capacity_ratio=0.1,
#     prefix="typed_scaling")
# axs.performance
# axs.abs_gap
# axs.N_gap
# axs.sqrtN_gap

# -----------------------------------------------------------
# 7) Save a summary CSV aggregated over N
# -----------------------------------------------------------

function aggregated_summary_by_slice(df::DataFrame)
    agg = combine(groupby(df, [:scenario, :prop_type, :capacity_ratio]),
        :abs_gap_mp_to_bound => mean => :mean_abs_gap_mp,
        :rel_gap_mp_to_bound => mean => :mean_rel_gap_mp,
        :N_abs_gap_mp_to_bound => mean => :mean_N_gap_mp,
        :sqrtN_abs_gap_mp_to_bound => mean => :mean_sqrtN_gap_mp,
        :mp_index_mean => mean => :mean_mp,
        :myopic_mean => mean => :mean_myopic,
        :round_robin_mean => mean => :mean_round_robin,
        :random_mean => mean => :mean_random
    )
    sort!(agg, [:scenario, :prop_type, :capacity_ratio])
    return agg
end

agg = aggregated_summary_by_slice(df)
CSV.write("typed_scaling_experiment_aggregated.csv", agg)

# -----------------------------------------------------------
# 8) Handy commands to inspect the results
# -----------------------------------------------------------

println("\nRows in df = ", nrow(df))
println("Scenarios = ", unique(df.scenario))
println("Proportions = ", unique(df.prop_type))
println("Capacity ratios = ", unique(df.capacity_ratio))
println("N values = ", unique(df.N))

println("\nTop 12 smallest MP gaps:")
display(first(sort(df, :abs_gap_mp_to_bound), 12))

println("\nTop 12 largest MP gaps:")
display(first(sort(df, :abs_gap_mp_to_bound, rev=true), 12))

select(df,
    :scenario, :prop_type, :capacity_ratio, :N, :M,
    :random_mean, :random_ci_low, :random_ci_high,
    :myopic_mean, :myopic_ci_low, :myopic_ci_high,
    :round_robin_mean, :round_robin_ci_low, :round_robin_ci_high,
    :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high
)

In [ ]:
using DataFrames
using Statistics
using Printf

# ----------------------------------------------------------
# 0) Ensure the needed derived columns exist
# ----------------------------------------------------------

"""
    ensure_gap_and_improvement_columns!(df)

Add, if missing:

- relative gaps to Lagrangian bound for each policy
- absolute MP-minus-myopic improvement
- relative MP-minus-myopic improvement:
      (J_mp - J_myopic) / J_myopic
"""
function ensure_gap_and_improvement_columns!(df::DataFrame)
    # Relative gaps to bound
    if :rel_gap_mp_to_bound ∉ names(df)
        df.rel_gap_mp_to_bound = df.abs_gap_mp_to_bound ./ df.lagrangian_bound
    end
    if :rel_gap_myopic_to_bound ∉ names(df)
        df.rel_gap_myopic_to_bound = df.abs_gap_myopic_to_bound ./ df.lagrangian_bound
    end
    if :rel_gap_roundrobin_to_bound ∉ names(df)
        df.rel_gap_roundrobin_to_bound = df.abs_gap_roundrobin_to_bound ./ df.lagrangian_bound
    end
    if :rel_gap_random_to_bound ∉ names(df)
        df.rel_gap_random_to_bound = df.abs_gap_random_to_bound ./ df.lagrangian_bound
    end

    # MP vs myopic improvement
    if :abs_improvement_mp_vs_myopic ∉ names(df)
        df.abs_improvement_mp_vs_myopic = df.mp_index_mean .- df.myopic_mean
    end
    if :rel_improvement_mp_vs_myopic ∉ names(df)
        df.rel_improvement_mp_vs_myopic =
            ifelse.(df.myopic_mean .!= 0.0,
                    (df.mp_index_mean .- df.myopic_mean) ./ df.myopic_mean,
                    NaN)
    end

    return df
end

ensure_gap_and_improvement_columns!(df)

# ----------------------------------------------------------
# 1) Range of relative gaps w.r.t. Lagrangian bound for each policy
# ----------------------------------------------------------

"""
    relative_gap_ranges(df)

Return a DataFrame with min/max relative gap to the Lagrangian bound
for each policy.
"""
function relative_gap_ranges(df::DataFrame)
    return DataFrame(
        policy = ["mp_index", "myopic", "round_robin", "random"],
        min_rel_gap = [
            minimum(skipmissing(df.rel_gap_mp_to_bound)),
            minimum(skipmissing(df.rel_gap_myopic_to_bound)),
            minimum(skipmissing(df.rel_gap_roundrobin_to_bound)),
            minimum(skipmissing(df.rel_gap_random_to_bound))
        ],
        max_rel_gap = [
            maximum(skipmissing(df.rel_gap_mp_to_bound)),
            maximum(skipmissing(df.rel_gap_myopic_to_bound)),
            maximum(skipmissing(df.rel_gap_roundrobin_to_bound)),
            maximum(skipmissing(df.rel_gap_random_to_bound))
        ]
    )
end

gap_range_df = relative_gap_ranges(df)
println("\n=== Range of relative gaps to Lagrangian bound for each policy ===")
display(gap_range_df)

# ----------------------------------------------------------
# 2) Best/worst instances for MP and myopic relative gaps
#    ("best" = smallest relative gap to bound; "worst" = largest)
# ----------------------------------------------------------

"""
    best_worst_gap_instances(df, col)

Return NamedTuple(best=..., worst=...) where:
- best  = row with smallest value in column `col`
- worst = row with largest  value in column `col`
"""
function best_worst_gap_instances(df::DataFrame, col::Symbol)
    d = filter(row -> isfinite(row[col]), df)
    d_best  = first(sort(d, col), 1)
    d_worst = first(sort(d, col, rev=true), 1)

    keepcols = [
        :scenario, :prop_type, :capacity_ratio, :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :myopic_mean, :round_robin_mean, :random_mean,
        col
    ]

    return (
        best = select(d_best, keepcols...),
        worst = select(d_worst, keepcols...)
    )
end

mp_gap_extremes = best_worst_gap_instances(df, :rel_gap_mp_to_bound)
my_gap_extremes = best_worst_gap_instances(df, :rel_gap_myopic_to_bound)

println("\n=== Best instance for MP relative gap (smallest gap) ===")
display(mp_gap_extremes.best)

println("\n=== Worst instance for MP relative gap (largest gap) ===")
display(mp_gap_extremes.worst)

println("\n=== Best instance for myopic relative gap (smallest gap) ===")
display(my_gap_extremes.best)

println("\n=== Worst instance for myopic relative gap (largest gap) ===")
display(my_gap_extremes.worst)

# ----------------------------------------------------------
# 3) Range of relative performance improvement of MP vs myopic
#    relative improvement = (J_mp - J_myopic)/J_myopic
# ----------------------------------------------------------

"""
    relative_improvement_range(df)

Return min/max of relative MP-vs-myopic improvement.
"""
function relative_improvement_range(df::DataFrame)
    vals = filter(isfinite, df.rel_improvement_mp_vs_myopic)
    return (
        min_rel_improvement = minimum(vals),
        max_rel_improvement = maximum(vals)
    )
end

imp_range = relative_improvement_range(df)

println("\n=== Range of relative performance improvement of MP vs myopic ===")
println("min relative improvement = ", imp_range.min_rel_improvement,
        "  (", 100 * imp_range.min_rel_improvement, "%)")
println("max relative improvement = ", imp_range.max_rel_improvement,
        "  (", 100 * imp_range.max_rel_improvement, "%)")

# ----------------------------------------------------------
# 4) Instances where MP-vs-myopic relative improvement is smallest/largest
# ----------------------------------------------------------

"""
    best_worst_improvement_instances(df)

Return NamedTuple(best=..., worst=...) where:
- best  = row with largest relative MP-vs-myopic improvement
- worst = row with smallest relative MP-vs-myopic improvement
"""
function best_worst_improvement_instances(df::DataFrame)
    d = filter(row -> isfinite(row.rel_improvement_mp_vs_myopic), df)

    d_best  = first(sort(d, :rel_improvement_mp_vs_myopic, rev=true), 1)
    d_worst = first(sort(d, :rel_improvement_mp_vs_myopic), 1)

    keepcols = [
        :scenario, :prop_type, :capacity_ratio, :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :myopic_mean,
        :abs_improvement_mp_vs_myopic,
        :rel_improvement_mp_vs_myopic,
        :rel_gap_mp_to_bound,
        :rel_gap_myopic_to_bound
    ]

    return (
        largest = select(d_best, keepcols...),
        smallest = select(d_worst, keepcols...)
    )
end

imp_extremes = best_worst_improvement_instances(df)

println("\n=== Instance with largest relative performance improvement of MP over myopic ===")
display(imp_extremes.largest)

println("\n=== Instance with smallest relative performance improvement of MP over myopic ===")
display(imp_extremes.smallest)

# ----------------------------------------------------------
# 5) Optional: concise printable summary
# ----------------------------------------------------------

function print_gap_and_improvement_summary(df::DataFrame)
    ensure_gap_and_improvement_columns!(df)

    gr = relative_gap_ranges(df)
    mp_ext = best_worst_gap_instances(df, :rel_gap_mp_to_bound)
    my_ext = best_worst_gap_instances(df, :rel_gap_myopic_to_bound)
    ir = relative_improvement_range(df)
    ie = best_worst_improvement_instances(df)

    println("\n==================================================")
    println("Relative gaps to Lagrangian bound: ranges")
    println("==================================================")
    display(gr)

    println("\n==================================================")
    println("MP relative gap: best (smallest)")
    println("==================================================")
    display(mp_ext.best)

    println("\n==================================================")
    println("MP relative gap: worst (largest)")
    println("==================================================")
    display(mp_ext.worst)

    println("\n==================================================")
    println("Myopic relative gap: best (smallest)")
    println("==================================================")
    display(my_ext.best)

    println("\n==================================================")
    println("Myopic relative gap: worst (largest)")
    println("==================================================")
    display(my_ext.worst)

    println("\n==================================================")
    println("Relative performance improvement of MP over myopic")
    println("==================================================")
    println("Min relative improvement = ",
            @sprintf("%.6f", ir.min_rel_improvement),
            " = ", @sprintf("%.2f", 100 * ir.min_rel_improvement), "%")
    println("Max relative improvement = ",
            @sprintf("%.6f", ir.max_rel_improvement),
            " = ", @sprintf("%.2f", 100 * ir.max_rel_improvement), "%")

    println("\n==================================================")
    println("Largest relative MP-vs-myopic improvement")
    println("==================================================")
    display(ie.largest)

    println("\n==================================================")
    println("Smallest relative MP-vs-myopic improvement")
    println("==================================================")
    display(ie.smallest)

    return nothing
end

# Run the full summary if desired:
# print_gap_and_improvement_summary(df)

gap_range_df = relative_gap_ranges(df)
mp_gap_extremes = best_worst_gap_instances(df, :rel_gap_mp_to_bound)
my_gap_extremes = best_worst_gap_instances(df, :rel_gap_myopic_to_bound)
imp_range = relative_improvement_range(df)
imp_extremes = best_worst_improvement_instances(df)

display(gap_range_df)
display(mp_gap_extremes.best)
display(mp_gap_extremes.worst)
display(my_gap_extremes.best)
display(my_gap_extremes.worst)
println(imp_range)
display(imp_extremes.largest)
display(imp_extremes.smallest)

In [ ]:
using DataFrames
using Statistics

"""
    add_ci_dominance_flags!(df)

Add Boolean columns indicating whether the MP-index CI lies entirely above
the CI of each benchmark policy, row by row.
"""
function add_ci_dominance_flags!(df::DataFrame)
    df.mp_ci_above_myopic =
        df.mp_index_ci_low .> df.myopic_ci_high

    df.mp_ci_above_round_robin =
        df.mp_index_ci_low .> df.round_robin_ci_high

    df.mp_ci_above_random =
        df.mp_index_ci_low .> df.random_ci_high

    # stricter: MP CI above all three benchmark CIs simultaneously
    df.mp_ci_above_all =
        df.mp_ci_above_myopic .&
        df.mp_ci_above_round_robin .&
        df.mp_ci_above_random

    return df
end

add_ci_dominance_flags!(df)

ci_summary_scenario = combine(groupby(df, :scenario),
    :mp_ci_above_myopic      => sum => :count_mp_above_myopic,
    :mp_ci_above_round_robin => sum => :count_mp_above_round_robin,
    :mp_ci_above_random      => sum => :count_mp_above_random,
    :mp_ci_above_all         => sum => :count_mp_above_all,
    nrow => :total_cases
)

display(ci_summary_scenario)

ci_summary_slice = combine(groupby(df, [:scenario, :prop_type, :capacity_ratio]),
    :mp_ci_above_myopic      => sum => :count_mp_above_myopic,
    :mp_ci_above_round_robin => sum => :count_mp_above_round_robin,
    :mp_ci_above_random      => sum => :count_mp_above_random,
    :mp_ci_above_all         => sum => :count_mp_above_all,
    nrow => :total_cases
)

sort!(ci_summary_slice, [:scenario, :prop_type, :capacity_ratio])
display(ci_summary_slice)

rows_unclear_vs_myopic = filter(row ->
    row.best_policy == "mp_index" && !row.mp_ci_above_myopic,
    df
)

display(select(rows_unclear_vs_myopic,
    :scenario, :prop_type, :capacity_ratio, :N, :M,
    :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
    :myopic_mean, :myopic_ci_low, :myopic_ci_high
))

rows_clear_mp = filter(:mp_ci_above_all => identity, df)

display(select(rows_clear_mp,
    :scenario, :prop_type, :capacity_ratio, :N, :M,
    :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
    :myopic_mean, :myopic_ci_low, :myopic_ci_high,
    :round_robin_mean, :round_robin_ci_low, :round_robin_ci_high,
    :random_mean, :random_ci_low, :random_ci_high
))




In [ ]:
using Statistics
using Printf
using DataFrames
using CSV
using PGFPlotsX

# -----------------------------------------------------------
# 1) Helpers
# -----------------------------------------------------------

function _sanitize_filename_piece(s)
    s = string(s)
    s = replace(s, " " => "_")
    s = replace(s, "," => "-")
    s = replace(s, "." => "p")
    s = replace(s, "[" => "")
    s = replace(s, "]" => "")
    s = replace(s, "(" => "")
    s = replace(s, ")" => "")
    return s
end

function _filter_slice(df::DataFrame; scenario, prop_type, capacity_ratio, atol=1e-12)
    d = filter(row ->
        row.scenario == scenario &&
        row.prop_type == prop_type &&
        isapprox(row.capacity_ratio, capacity_ratio; atol=atol, rtol=0.0),
        df
    )
    sort!(d, :N)
    return d
end

"""
    loglog_slope(x, y)

Estimate the slope in the regression
    log(y) ≈ a + b log(x)
using ordinary least squares on the positive finite entries.
If y ≈ C N^{-p}, then slope ≈ -p.
"""
function loglog_slope(x::AbstractVector, y::AbstractVector)
    keep = [isfinite(xi) && isfinite(yi) && xi > 0 && yi > 0 for (xi, yi) in zip(x, y)]
    xx = log.(Float64.(x[keep]))
    yy = log.(Float64.(y[keep]))

    if length(xx) < 2
        return NaN
    end

    xbar = mean(xx)
    ybar = mean(yy)
    num = sum((xx .- xbar) .* (yy .- ybar))
    den = sum((xx .- xbar).^2)

    return den > 0 ? num / den : NaN
end

# -----------------------------------------------------------
# 2) Slice table for one fixed (scenario, prop_type, capacity_ratio)
# -----------------------------------------------------------

function typed_slice_table(df::DataFrame; scenario, prop_type, capacity_ratio)
    d = _filter_slice(df; scenario=scenario, prop_type=prop_type, capacity_ratio=capacity_ratio)

    @assert nrow(d) > 0 "No rows found for the requested slice."

    return select(d,
        :scenario, :prop_type, :capacity_ratio,
        :N, :M,
        :lagrangian_bound,
        :mp_index_mean, :mp_index_ci_low, :mp_index_ci_high,
        :myopic_mean, :round_robin_mean, :random_mean,
        :abs_gap_mp_to_bound, :rel_gap_mp_to_bound,
        :N_abs_gap_mp_to_bound, :sqrtN_abs_gap_mp_to_bound,
        :abs_gap_myopic_to_bound,
        :abs_gap_roundrobin_to_bound,
        :abs_gap_random_to_bound,
        :best_policy
    )
end

# Example:
# display(typed_slice_table(df;
#     scenario="A_selfhealing_B_fragile",
#     prop_type="0.4,0.6",
#     capacity_ratio=0.1))

# -----------------------------------------------------------
# 3) One-line summary for one slice
# -----------------------------------------------------------

function typed_slice_summary(df::DataFrame; scenario, prop_type, capacity_ratio)
    d = _filter_slice(df; scenario=scenario, prop_type=prop_type, capacity_ratio=capacity_ratio)
    @assert nrow(d) > 0 "No rows found for the requested slice."

    Nvals = Float64.(d.N)
    gap   = Float64.(d.abs_gap_mp_to_bound)
    Ngap  = Float64.(d.N_abs_gap_mp_to_bound)
    sNgap = Float64.(d.sqrtN_abs_gap_mp_to_bound)

    return (
        scenario = scenario,
        prop_type = prop_type,
        capacity_ratio = capacity_ratio,
        npoints = nrow(d),
        N_min = minimum(d.N),
        N_max = maximum(d.N),

        gap_first = gap[1],
        gap_last = gap[end],
        gap_last_over_first = gap[end] / gap[1],

        Ngap_first = Ngap[1],
        Ngap_last = Ngap[end],
        Ngap_last_over_first = Ngap[end] / Ngap[1],

        sqrtNgap_first = sNgap[1],
        sqrtNgap_last = sNgap[end],
        sqrtNgap_last_over_first = sNgap[end] / sNgap[1],

        loglog_slope_gap = loglog_slope(Nvals, gap),
        loglog_slope_Ngap = loglog_slope(Nvals, Ngap),
        loglog_slope_sqrtNgap = loglog_slope(Nvals, sNgap),

        mp_last = d.mp_index_mean[end],
        bound_last = d.lagrangian_bound[end],
        myopic_last = d.myopic_mean[end],
        rr_last = d.round_robin_mean[end],
        random_last = d.random_mean[end],
    )
end

# -----------------------------------------------------------
# 4) Summary table over all slices
# -----------------------------------------------------------

function build_all_slice_summaries(df::DataFrame)
    rows = NamedTuple[]

    scenarios = unique(df.scenario)
    props = unique(df.prop_type)
    caps = unique(df.capacity_ratio)

    for scen in scenarios, prop in props, cap in caps
        d = _filter_slice(df; scenario=scen, prop_type=prop, capacity_ratio=cap)
        if nrow(d) > 0
            push!(rows, typed_slice_summary(df;
                scenario=scen,
                prop_type=prop,
                capacity_ratio=cap
            ))
        end
    end

    out = DataFrame(rows)
    sort!(out, [:scenario, :prop_type, :capacity_ratio])
    return out
end

slice_summary_df = build_all_slice_summaries(df)
CSV.write("typed_scaling_slice_summary.csv", slice_summary_df)

println("\n=== Slice-by-slice summary ===")
display(slice_summary_df)

println("\n=== Slices with smallest final MP gap ===")
display(first(sort(slice_summary_df, :gap_last), 12))

println("\n=== Slices with largest final MP gap ===")
display(first(sort(slice_summary_df, :gap_last, rev=true), 12))

println("\n=== Slices whose raw MP gap seems to decay fastest (most negative log-log slope) ===")
display(first(sort(slice_summary_df, :loglog_slope_gap), 12))

# -----------------------------------------------------------
# 5) Print a readable summary for one slice
# -----------------------------------------------------------

function print_typed_slice_summary(df::DataFrame; scenario, prop_type, capacity_ratio)
    s = typed_slice_summary(df; scenario=scenario, prop_type=prop_type, capacity_ratio=capacity_ratio)

    println("--------------------------------------------------")
    println("scenario       = ", s.scenario)
    println("prop_type      = ", s.prop_type)
    println("capacity_ratio = ", s.capacity_ratio)
    println("N range        = [", s.N_min, ", ", s.N_max, "]")
    println("gap first/last = ", s.gap_first, " / ", s.gap_last,
            "   (last/first = ", s.gap_last_over_first, ")")
    println("N*gap first/last = ", s.Ngap_first, " / ", s.Ngap_last,
            "   (last/first = ", s.Ngap_last_over_first, ")")
    println("sqrt(N)*gap first/last = ", s.sqrtNgap_first, " / ", s.sqrtNgap_last,
            "   (last/first = ", s.sqrtNgap_last_over_first, ")")
    println("log-log slope of gap          = ", s.loglog_slope_gap)
    println("log-log slope of N*gap        = ", s.loglog_slope_Ngap)
    println("log-log slope of sqrt(N)*gap  = ", s.loglog_slope_sqrtNgap)
    println("last-N performances:")
    println("  bound      = ", s.bound_last)
    println("  MP-index   = ", s.mp_last)
    println("  myopic     = ", s.myopic_last)
    println("  roundrobin = ", s.rr_last)
    println("  random     = ", s.random_last)
    println("--------------------------------------------------")
    return nothing
end

# Example:
# print_typed_slice_summary(df;
#     scenario="A_selfhealing_B_fragile",
#     prop_type="0.4,0.6",
#     capacity_ratio=0.1)

# -----------------------------------------------------------
# 6) Save plots for all slices
#    (uses your existing save_typed_slice_plots function)
# -----------------------------------------------------------

function save_all_typed_slice_plots(df::DataFrame;
                                    outdir="typed_slice_plots",
                                    xscale=:log10,
                                    width="12cm",
                                    height="7cm")

    mkpath(outdir)

    scenarios = unique(df.scenario)
    props = unique(df.prop_type)
    caps = unique(df.capacity_ratio)

    saved = NamedTuple[]

    for scen in scenarios, prop in props, cap in caps
        d = _filter_slice(df; scenario=scen, prop_type=prop, capacity_ratio=cap)
        if nrow(d) == 0
            continue
        end

        prefix = joinpath(
            outdir,
            "slice_" *
            _sanitize_filename_piece(scen) *
            "_prop_" * _sanitize_filename_piece(prop) *
            "_cap_" * _sanitize_filename_piece(cap)
        )

        axs = save_typed_slice_plots(df;
            scenario = scen,
            prop_type = prop,
            capacity_ratio = cap,
            prefix = prefix,
            xscale = xscale,
            width = width,
            height = height
        )

        push!(saved, (
            scenario = scen,
            prop_type = prop,
            capacity_ratio = cap,
            prefix = prefix
        ))
    end

    saved_df = DataFrame(saved)
    CSV.write(joinpath(outdir, "saved_plot_prefixes.csv"), saved_df)
    return saved_df
end

# Uncomment to save all slice plots:
# saved_plot_df = save_all_typed_slice_plots(df;
#     outdir="typed_slice_plots",
#     xscale=:log10,
#     width="12cm",
#     height="7cm")

In [ ]:
@inline function choose_topM_mask!(mask::BitVector,
                                   vals::AbstractVector{<:Real},
                                   M::Int;
                                   positive::Bool = false,
                                   nonnegative::Bool = false)
    fill!(mask, false)
    m = min(M, length(vals))
    ord = partialsortperm(vals, 1:m; rev = true)

    for idx in ord
        v = vals[idx]
        if positive
            v > 0.0 || break
        elseif nonnegative
            v >= 0.0 || break
        end
        mask[idx] = true
    end
    return nothing
end
    
function simulate_multi_project_policies_lookup_newbound(
    x_vec::AbstractVector{<:Real},
    type_id_vec::AbstractVector{<:Integer},
    q_types::AbstractVector{<:Real},
    rho_types::AbstractVector{<:Real},
    kappa_types::AbstractVector{<:Real},
    r_types::AbstractVector{<:Real},
    beta::Float64,
    M::Integer;
    T::Int = 100,
    nruns::Int = 1_000,
    conf_level::Float64 = 0.95,
    seed::Union{Int,Nothing} = nothing,
    ε::Float64 = 1e-10,
    mp_lookups::Union{Nothing,AbstractVector{<:MPIndexLookupUniform}} = nothing,
    bound_λ0::Float64 = 0.0,
    bound_step0::Float64 = 1.0,
    bound_λtol::Float64 = 1e-6,
    bound_max_bisect_iter::Int = 80,
    bound_xatol::Float64 = 1e-10,
    bound_λatol::Float64 = 1e-10,
    bound_maxiter::Int = 200,
    verbose_bound::Bool = false
)

    N = length(x_vec)
    @assert length(type_id_vec) == N
    K = length(q_types)
    @assert length(rho_types)   == K
    @assert length(kappa_types) == K
    @assert length(r_types)     == K
    @assert 0.0 < beta < 1.0
    @assert 1 <= M <= N
    @assert T > 0
    @assert nruns > 0

    if mp_lookups !== nothing
        @assert length(mp_lookups) == K
    end

    x_vec_f = Float64.(x_vec)

    q_types_f     = Float64.(q_types)
    rho_types_f   = Float64.(rho_types)
    kappa_types_f = Float64.(kappa_types)
    r_types_f     = Float64.(r_types)

    type_id = Int.(type_id_vec)

    for i in 1:N
        @assert 1 <= type_id[i] <= K
        @assert 0.0 <= x_vec_f[i] <= 1.0
    end

    for k in 1:K
        @assert 0.0 <= q_types_f[k] < 1.0
        @assert 0.0 <= rho_types_f[k] < 1.0 - q_types_f[k]
        @assert 0.0 <= kappa_types_f[k] <= 1.0
        @assert r_types_f[k] >= 0.0
    end

    q_vec     = [q_types_f[type_id[i]]     for i in 1:N]
    rho_vec   = [rho_types_f[type_id[i]]   for i in 1:N]
    kappa_vec = [kappa_types_f[type_id[i]] for i in 1:N]
    r_vec     = [r_types_f[type_id[i]]     for i in 1:N]

    λstar, Lmin, bound_info = lambda_star_vector_grouped_bisect(
        x_vec_f,
        q_vec,
        rho_vec,
        kappa_vec,
        r_vec,
        beta,
        M;
        λ0 = bound_λ0,
        step0 = bound_step0,
        λtol = bound_λtol,
        max_bisect_iter = bound_max_bisect_iter,
        ε = ε,
        xatol = bound_xatol,
        λatol = bound_λatol,
        maxiter = bound_maxiter,
        verbose = verbose_bound
    )

    β_powers = [beta^t for t in 0:(T-1)]

    total_random     = zeros(Float64, nruns)
    total_myopic     = zeros(Float64, nruns)
    total_roundrobin = zeros(Float64, nruns)
    total_mpindex    = zeros(Float64, nruns)

    seedgen = seed === nothing ? Random.default_rng() : MersenneTwister(seed)
    run_seeds = rand(seedgen, UInt64, nruns)

    @inline function phi0(x, q, rho)
        q + rho * x
    end

    @inline function phi1(x, q, rho, kappa)
        denom = 1.0 - kappa * x
        if denom <= 0.0
            return q + rho
        end
        post_nack = (1.0 - kappa) * x / denom
        return q + rho * post_nack
    end

    @inline function mp_index_current_exact(x, q, rho, kappa, r, beta, ε)
        fxx = float(f_metric2(x, x; q=q, ρ=rho, κ=kappa, r=r, β=beta, ε=ε))
        gxx = float(g_metric2(x, x; q=q, ρ=rho, κ=kappa,      β=beta, ε=ε))
        return (isfinite(fxx) && isfinite(gxx) && gxx != 0.0) ? (fxx / gxx) : -Inf
    end

    function simulate_one_run!(run_idx::Int)
        rng = MersenneTwister(run_seeds[run_idx])

        X_rand = copy(x_vec_f)
        X_myop = copy(x_vec_f)
        X_rr   = copy(x_vec_f)
        X_mp   = copy(x_vec_f)

        rr_start = 1

        R_rand = 0.0
        R_myop = 0.0
        R_rr   = 0.0
        R_mp   = 0.0

        myopic_value = zeros(Float64, N)
        mpvals       = zeros(Float64, N)

        active_rand_mask = falses(N)
        active_myop_mask = falses(N)
        active_rr_mask   = falses(N)
        active_mp_mask   = falses(N)

        perm = collect(1:N)

        for t in 1:T
            disc = β_powers[t]

            fill!(active_rand_mask, false)
            shuffle!(rng, perm)
            active_rand_mask[perm[1:M]] .= true

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rand[i]

                if active_rand_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rand[i] = q + rho
                    else
                        X_rand[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rand[i] = phi0(xi, q, rho)
                end
            end
            R_rand += disc * reward_t

            for i in 1:N
                myopic_value[i] = r_vec[i] * kappa_vec[i] * X_myop[i]
            end
            choose_topM_mask!(active_myop_mask, myopic_value, M; positive=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_myop[i]

                if active_myop_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_myop[i] = q + rho
                    else
                        X_myop[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_myop[i] = phi0(xi, q, rho)
                end
            end
            R_myop += disc * reward_t

            fill!(active_rr_mask, false)
            for k in 0:(M-1)
                i = ((rr_start - 1 + k) % N) + 1
                active_rr_mask[i] = true
            end
            rr_start = ((rr_start - 1 + M) % N) + 1

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_rr[i]

                if active_rr_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_rr[i] = q + rho
                    else
                        X_rr[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_rr[i] = phi0(xi, q, rho)
                end
            end
            R_rr += disc * reward_t

            for i in 1:N
                k = type_id[i]
                xi = X_mp[i]
                mpvals[i] = mp_lookups === nothing ?
                    mp_index_current_exact(xi,
                                           q_types_f[k], rho_types_f[k], kappa_types_f[k],
                                           r_types_f[k], beta, ε) :
                    mp_index_lookup(mp_lookups[k], xi)
            end
            choose_topM_mask!(active_mp_mask, mpvals, M; nonnegative=true)

            reward_t = 0.0
            for i in 1:N
                q  = q_vec[i]
                rho = rho_vec[i]
                kappa = kappa_vec[i]
                r  = r_vec[i]
                xi = X_mp[i]

                if active_mp_mask[i]
                    p_ack = kappa * xi
                    if rand(rng) < p_ack
                        reward_t += r
                        X_mp[i] = q + rho
                    else
                        X_mp[i] = phi1(xi, q, rho, kappa)
                    end
                else
                    X_mp[i] = phi0(xi, q, rho)
                end
            end
            R_mp += disc * reward_t
        end

        total_random[run_idx]     = R_rand
        total_myopic[run_idx]     = R_myop
        total_roundrobin[run_idx] = R_rr
        total_mpindex[run_idx]    = R_mp
        return nothing
    end

    if Threads.nthreads() > 1
        Threads.@threads for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    else
        for run_idx in 1:nruns
            simulate_one_run!(run_idx)
        end
    end

    norm_factor = (1.0 - beta) / N

    function summarize(raw_samples::Vector{Float64})
        samples = norm_factor .* raw_samples
        μ  = mean(samples)
        s  = std(samples)
        se = s / sqrt(length(samples))
        z  = quantile(Normal(), 0.5 + conf_level / 2)
        ci_low  = μ - z * se
        ci_high = μ + z * se
        return (mean = μ,
                stderr = se,
                ci_low = ci_low,
                ci_high = ci_high,
                nruns = length(samples))
    end

    results = Dict{String,NamedTuple}()
    results["random"]          = summarize(total_random)
    results["myopic"]          = summarize(total_myopic)
    results["round_robin"]     = summarize(total_roundrobin)
    results["mp_index"]        = summarize(total_mpindex)

    lagrangian_bound = norm_factor * Lmin
    results["lagrangian_bound"] = (
        lambda_star = λstar,
        raw_Lmin = Lmin,
        mean = lagrangian_bound,
        stderr = 0.0,
        ci_low = lagrangian_bound,
        ci_high = lagrangian_bound,
        nruns = 1
    )

    results["bound_info"] = bound_info

    return results
end

In [ ]:
# ----------------------------------------------------------
# Exact typed scaling experiment with new bound solver
# ----------------------------------------------------------

function run_typed_scaling_experiment_newbound(;
    scenario_grid,
    prop_grid,
    alphaM_values,
    N_values,
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    x0 = 0.5,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    show_progress = true,
    # new bound-solver controls
    bound_λ0::Float64 = 0.0,
    bound_step0::Float64 = 1.0,
    bound_λtol::Float64 = 1e-6,
    bound_max_bisect_iter::Int = 80,
    bound_xatol::Float64 = 1e-10,
    bound_λatol::Float64 = 1e-10,
    bound_maxiter::Int = 200,
    verbose_bound::Bool = false
)

    rows = NamedTuple[]
    counter = 0
    total_cases = length(scenario_grid) * length(prop_grid) * length(alphaM_values) * length(N_values)

    t0 = time()

    for (s_idx, scen) in enumerate(scenario_grid)
        name = hasproperty(scen, :name) ? String(scen.name) : "scenario_$s_idx"

        q_types     = Float64.(scen.q_types)
        rho_types   = Float64.(scen.rho_types)
        kappa_types = Float64.(scen.kappa_types)
        r_types     = Float64.(scen.r_types)

        # Build MP-index lookup tables once per scenario
        mp_lookups = build_type_mp_lookups(
            q_types, rho_types, kappa_types, r_types, beta;
            ε = ε,
            ngrid = mp_lookup_ngrid,
            mode = mp_lookup_mode
        )

        for prop_type in prop_grid
            prop_type_f = Float64.(prop_type)
            K = length(prop_type_f)

            for alphaM in alphaM_values
                for N in N_values
                    counter += 1
                    M = max(1, min(N, round(Int, alphaM * N)))
                    capacity_ratio = M / N

                    instance_seed = base_seed + counter

                    inst = build_typed_instance(
                        N;
                        prop_type = prop_type_f,
                        q_types = q_types,
                        rho_types = rho_types,
                        kappa_types = kappa_types,
                        r_types = r_types,
                        x_init_scheme = x_init_scheme,
                        x0 = x0,
                        xlo = xlo,
                        xhi = xhi,
                        shuffle_types = shuffle_types,
                        shuffle_x = shuffle_x,
                        seed = instance_seed
                    )

                    if show_progress
                        elapsed = time() - t0
                        avg_time = elapsed / max(counter - 1, 1)
                        eta = counter == 1 ? NaN : avg_time * (total_cases - counter)

                        println("[$counter/$total_cases] ",
                                name,
                                " | prop = ", prop_type_f,
                                " | αM_target = ", alphaM,
                                " | realized M/N = ", @sprintf("%.4f", capacity_ratio),
                                " | N = ", N,
                                " | M = ", M,
                                " | elapsed = ", _fmt_seconds(elapsed),
                                " | ETA = ", counter == 1 ? "?" : _fmt_seconds(eta))
                    end

                    results = simulate_multi_project_policies_lookup_newbound(
                        inst.x_vec,
                        inst.type_id_vec,
                        q_types,
                        rho_types,
                        kappa_types,
                        r_types,
                        beta,
                        M;
                        T = T_policy,
                        nruns = nruns_policy,
                        conf_level = conf_level,
                        seed = instance_seed,
                        ε = ε,
                        mp_lookups = mp_lookups,
                        bound_λ0 = bound_λ0,
                        bound_step0 = bound_step0,
                        bound_λtol = bound_λtol,
                        bound_max_bisect_iter = bound_max_bisect_iter,
                        bound_xatol = bound_xatol,
                        bound_λatol = bound_λatol,
                        bound_maxiter = bound_maxiter,
                        verbose_bound = verbose_bound
                    )

                    rand_stats = results["random"]
                    myop_stats = results["myopic"]
                    rr_stats   = results["round_robin"]
                    mp_stats   = results["mp_index"]
                    ub_stats   = results["lagrangian_bound"]
                    binfo      = results["bound_info"]

                    ub = ub_stats.mean
                    rand_mean = rand_stats.mean
                    myop_mean = myop_stats.mean
                    rr_mean   = rr_stats.mean
                    mp_mean   = mp_stats.mean

                    abs_gap_mp = ub - mp_mean
                    rel_gap_mp = ub != 0.0 ? abs_gap_mp / ub : NaN

                    abs_gap_myop = ub - myop_mean
                    abs_gap_rr   = ub - rr_mean
                    abs_gap_rand = ub - rand_mean

                    sqrtN = sqrt(Float64(N))

                    row = (
                        scenario = name,
                        N = N,
                        M = M,
                        alphaM_target = Float64(alphaM),
                        capacity_ratio = M / N,
                        beta = beta,

                        K = K,
                        prop_type = join(string.(round.(prop_type_f; digits=4)), ","),

                        q_types = join(string.(q_types), ","),
                        rho_types = join(string.(rho_types), ","),
                        kappa_types = join(string.(kappa_types), ","),
                        r_types = join(string.(r_types), ","),

                        lambda_star = ub_stats.lambda_star,
                        raw_Lmin = ub_stats.raw_Lmin,
                        lagrangian_bound = ub,

                        random_mean = rand_mean,
                        random_ci_low = rand_stats.ci_low,
                        random_ci_high = rand_stats.ci_high,

                        myopic_mean = myop_mean,
                        myopic_ci_low = myop_stats.ci_low,
                        myopic_ci_high = myop_stats.ci_high,

                        round_robin_mean = rr_mean,
                        round_robin_ci_low = rr_stats.ci_low,
                        round_robin_ci_high = rr_stats.ci_high,

                        mp_index_mean = mp_mean,
                        mp_index_ci_low = mp_stats.ci_low,
                        mp_index_ci_high = mp_stats.ci_high,

                        abs_gap_mp_to_bound = abs_gap_mp,
                        rel_gap_mp_to_bound = rel_gap_mp,
                        N_abs_gap_mp_to_bound = N * abs_gap_mp,
                        sqrtN_abs_gap_mp_to_bound = sqrtN * abs_gap_mp,

                        abs_gap_myopic_to_bound = abs_gap_myop,
                        abs_gap_roundrobin_to_bound = abs_gap_rr,
                        abs_gap_random_to_bound = abs_gap_rand,

                        mp_lookup_ngrid = mp_lookup_ngrid,
                        mp_lookup_mode = String(mp_lookup_mode),
                        x_init_scheme = String(x_init_scheme),
                        seed = instance_seed,

                        bound_nit = binfo.nit,
                        bound_cache_size = binfo.cache_size,
                        bound_ngroups = binfo.ngroups,
                        bound_bracket_subgrad_lo = binfo.bracket_subgrad_lo,
                        bound_bracket_subgrad_hi = binfo.bracket_subgrad_hi
                    )

                    if K >= 1
                        row = merge(row, (n_type1 = inst.n_types[1],))
                    end
                    if K >= 2
                        row = merge(row, (n_type2 = inst.n_types[2],))
                    end

                    push!(rows, row)
                end
            end
        end
    end

    total_time = time() - t0
    println("\nCompleted ", total_cases, " cases in ", _fmt_seconds(total_time), ".")

    return rows
end

In [ ]:
rows = run_typed_scaling_experiment_newbound(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = [0.05, 0.10, 0.20],
    N_values = [50, 100, 200, 400],
    beta = 0.99,
    T_policy = 200,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    show_progress = true,
    bound_λtol = 1e-6,
    bound_max_bisect_iter = 80,
    verbose_bound = false
)

df = DataFrame(rows)
sort!(df, [:scenario, :prop_type, :alphaM_target, :N])

CSV.write("typed_scaling_experiment_newbound.csv", df)

In [ ]:
# -----------------------------------------------------------
# Serious experimental grid for the paper
# All projects start at x0 = 0.5
# -----------------------------------------------------------

# Four type-A profiles ("more self-healing")
A_types = [
    (name = "A_very_selfhealing", q = 0.01, rho = 0.90, kappa = 0.70, r = 1.0),
    (name = "A_mild_selfhealing", q = 0.03, rho = 0.80, kappa = 0.70, r = 1.0),
    (name = "A_balanced",         q = 0.05, rho = 0.70, kappa = 0.70, r = 1.0),
    (name = "A_harder_obs",       q = 0.02, rho = 0.85, kappa = 0.55, r = 1.0)
]

# Three type-B profiles ("more fragile")
B_types = [
    (name = "B_fragile",          q = 0.10, rho = 0.10, kappa = 0.95, r = 1.0),
    (name = "B_very_fragile",     q = 0.15, rho = 0.05, kappa = 0.95, r = 1.0),
    (name = "B_persistent_bad",   q = 0.08, rho = 0.20, kappa = 0.95, r = 1.0)
]

# Cross A-types and B-types to form 12 scenarios
scenario_grid = vec([
    (
        name = "$(A.name)_$(B.name)",
        q_types = [A.q, B.q],
        rho_types = [A.rho, B.rho],
        kappa_types = [A.kappa, B.kappa],
        r_types = [A.r, B.r]
    )
    for A in A_types, B in B_types
])

# Type-A proportions
prop_grid = [[p, 1 - p] for p in 0.1:0.1:0.9]

# Capacity ratios: all multiples of 0.05
alphaM_values = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

# N values: multiples of 20 so alphaM * N is always an integer
N_values = [100, 200, 400, 800, 1600]

ncases = length(scenario_grid) * length(prop_grid) * length(alphaM_values) * length(N_values)
println("Total number of cases = ", ncases)

rows = run_typed_scaling_experiment_newbound(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = alphaM_values,
    N_values = N_values,
    beta = 0.99,
    T_policy = 300,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    mp_lookup_ngrid = 1001,   # keep the exact-style lookup resolution you already validated
    mp_lookup_mode = :linear,
    x_init_scheme = :constant,
    x0 = 0.5,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = false,        # irrelevant when x is constant, but explicit
    show_progress = true,
    bound_λtol = 1e-6,
    bound_max_bisect_iter = 80,
    verbose_bound = false
)

df = DataFrame(rows)
sort!(df, [:scenario, :prop_type, :alphaM_target, :N])

CSV.write("typed_scaling_experiment_paper_run.csv", df)

In [ ]:
combine(groupby(df, :scenario),
    :t_bound_case => mean => :mean_t_bound,
    :t_init_counts_case => mean => :mean_t_init,
    :t_mc_case => mean => :mean_t_mc,
    :t_case_total => mean => :mean_t_case_total
)

select(df,
    :scenario, :prop_type, :alphaM_target, :capacity_ratio, :N, :M,
    :t_bound_case, :t_init_counts_case, :t_mc_case, :t_case_total
)

In [ ]:
scenario_grid = [
    (
        name = "A_selfhealing_B_fragile",
        q_types = [0.01, 0.10],
        rho_types = [0.90, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    ),
    (
        name = "A_mild_B_fragile",
        q_types = [0.03, 0.10],
        rho_types = [0.80, 0.10],
        kappa_types = [0.70, 0.95],
        r_types = [1.0, 1.0]
    )
]

prop_grid = [
    [0.20, 0.80],
    [0.40, 0.60],
    [0.60, 0.40],
    [0.80, 0.20]
]

alphaM_values = [0.05, 0.10, 0.20]
N_values = [50, 100, 200, 400]

rows = run_typed_scaling_experiment_binned(
    scenario_grid = scenario_grid,
    prop_grid = prop_grid,
    alphaM_values = alphaM_values,
    N_values = N_values,
    beta = 0.99,
    T_policy = 300,
    nruns_policy = 1000,
    conf_level = 0.95,
    base_seed = 1234,
    ε = 1e-10,
    state_ngrid = 101,
    mp_lookup_ngrid = 201,
    mp_lookup_mode = :linear,
    x_init_scheme = :linspace,
    xlo = 0.0,
    xhi = 1.0,
    shuffle_types = true,
    shuffle_x = true,
    show_progress = true
)

df = typed_binned_rows_to_dataframe(rows)
CSV.write("typed_scaling_experiment_binned.csv", df)

first(df, 10)

In [ ]:
using PGFPlotsX

"""
    plot_indices_two_types(p_types, q_types, r_types, beta; filename="")

For K = 2 project types with parameters (p_k, q_k, r_k), k = 1,2, and discount
factor `beta`, plot their health-based MP indices m_k(x) and the common myopic
index r x as functions of the belief state x ∈ [0,1], without adjusting
for intervention costs.

Assumes a function

    mpindex(x::Float64, p::Float64, q::Float64, r::Float64, beta::Float64) -> Float64

is defined and returns the MP index m(x) for a single project.
"""
function plot_indices_two_types(p_types::AbstractVector{<:Real},
                                q_types::AbstractVector{<:Real},
                                r_types::AbstractVector{<:Real},
                                beta::Float64;
                                filename::AbstractString = "")

    @assert length(p_types) == 2 "p_types must have length 2"
    @assert length(q_types) == 2 "q_types must have length 2"
    @assert length(r_types) == 2 "r_types must have length 2"
    @assert 0.0 < beta < 1.0 "beta must be in (0,1)"
    @assert r_types[1] == r_types[2] "this plot assumes r_1 = r_2 (common myopic index)"

    # x-grid on [0,1]
    xs = collect(range(0.0, 1.0; length = 501))

    # Preallocate arrays for indices
    mA_mp    = similar(xs)  # MP index for type A (type 1)
    mB_mp    = similar(xs)  # MP index for type B (type 2)
    m_myopic = similar(xs)  # common myopic index r x

    pA, pB = float.(p_types)
    qA, qB = float.(q_types)
    r      = float(r_types[1])

    # Compute indices for each x
    for (i, x) in enumerate(xs)
        m_myopic[i] = r * x
        mA_mp[i]    = mpindex(x, pA, qA, r, beta)
        mB_mp[i]    = mpindex(x, pB, qB, r, beta)
    end

    gp = @pgf GroupPlot(
        {
            group_style = {
                group_size = "2 by 1",
                horizontal_sep = "2.5cm",
            },
        },

        # Left panel: x in [0,1]
        Axis(
            {
                xmin = 0,
                xmax = 1,
                ymin = 0,
                axis_lines = "left",
                xlabel = "x",
                ylabel = "Index value",
                legend_pos = "north west",
            },
            Plot(
                {thin, color = "black"},
                Table(x = xs, y = mA_mp),
            ),
            Plot(
                {thin, "dotted", color = "black"},
                Table(x = xs, y = mB_mp),
            ),
            Plot(
                {thin, "dashed", color = "black"},
                Table(x = xs, y = m_myopic),
            ),
            Legend([
                "Type A MP",
                "Type B MP",
                "Myopic",
            ]),
        ),

        # Right panel: zoom x in [0,0.5]
        Axis(
            {
                xmin = 0,
                xmax = 0.5,
                ymin = 0,
                axis_lines = "left",
                xlabel = "x",
                ylabel = "Index value",
            },
            Plot(
                {thin, color = "black"},
                Table(x = xs, y = mA_mp),
            ),
            Plot(
                {thin, "dotted", color = "black"},
                Table(x = xs, y = mB_mp),
            ),
            Plot(
                {thin, "dashed", color = "black"},
                Table(x = xs, y = m_myopic),
            ),
        ),
    )

    if filename != ""
        PGFPlotsX.save(filename, gp)  # e.g. "indices_two_types.pdf"
    end

    return gp
end

In [ ]:
# Type 1 (self-healing-ish), Type 2 (fragile-ish)
p_types = [0.10, 0.35]
q_types = [0.05, 0.01]
r_types = [1.0, 1.0]
beta    = 0.99

ax = plot_indices_two_types(p_types, q_types, r_types, beta;
                            filename = "indices_two_types.pdf")

In [ ]:
using Random
using CSV
using DataFrames
using Dates

# helper: format seconds nicely
function _fmt_seconds(s::Real)
    s = max(0.0, s)
    si = round(Int, s)
    h, r = divrem(si, 3600)
    m, sec = divrem(r, 60)
    h > 0 ? "$(h)h $(m)m $(sec)s" :
    m > 0 ? "$(m)m $(sec)s" : "$(sec)s"
end

function run_two_type_grid_to_csv(filename::AbstractString;
                                  p_grid::AbstractVector,
                                  q_grid::AbstractVector,
                                  beta::Float64,
                                  N::Int,
                                  prop_type_A_list::AbstractVector,
                                  M_ratios::AbstractVector,
                                  T_policy::Int,
                                  nruns_policy::Int,
                                  conf_level::Float64 = 0.95,
                                  delta_rho::Float64 = 0.05,
                                  base_seed::Int = 1234,
                                  # NEW:
                                  show_progress::Bool = true,
                                  update_every::Int = 25,                 # print every N completed cases
                                  min_seconds_between_updates::Real = 2.0 # and/or at least every this many seconds
                                  )

    @assert 0.0 < beta < 1.0 "beta must be in (0,1)"

    # Filter feasible (p,q) pairs (no p+q too close to 1)
    feasible_pairs = [(p,q) for p in p_grid, q in q_grid
                      if p > 0 && q > 0 && p + q <= 1 - delta_rho]

    if isempty(feasible_pairs)
        error("No feasible (p,q) pairs satisfy p>0, q>0, p+q ≤ 1-delta_rho. Check p_grid, q_grid, delta_rho.")
    end

    # Precompute valid ordered type-pairs so we can compute an accurate total for ETA
    type_pairs = [(pA,qA,pB,qB) for (pA,qA) in feasible_pairs, (pB,qB) in feasible_pairs
                  if (pA < pB && qA > qB)]

    if isempty(type_pairs)
        error("No (A,B) type pairs satisfy pA < pB and qA > qB under the feasible_pairs set.")
    end

    total_cases = length(type_pairs) * length(prop_type_A_list) * length(M_ratios)
    done_cases  = 0
    t0 = time()
    t_last = t0

    rows = Vector{NamedTuple}()
    case_idx = 0

    for (pA, qA, pB, qB) in type_pairs
        # Type-level parameters (we keep r_types = 1.0)
        p_types = [pA, pB]
        q_types = [qA, qB]
        r_types = [1.0, 1.0]

        for prop_type_A in prop_type_A_list
            @assert 0.0 < prop_type_A < 1.0
            nA = round(Int, prop_type_A * N)
            nA = max(1, min(nA, N-1))  # ensure at least 1 of each type
            nB = N - nA

            # Build project-level vectors
            p_vec = vcat(fill(pA, nA), fill(pB, nB))
            q_vec = vcat(fill(qA, nA), fill(qB, nB))
            r_vec = ones(Float64, N)

            case_idx += 1
            case_seed = base_seed + case_idx

            for ratio in M_ratios
                M = max(1, round(Int, ratio * N))
                M = min(M, N)

                # 1) Lagrangian bound (cost-free case: c_vec ≡ 0)
                λ_star, L_star = lambda_star_vector(p_vec, q_vec, r_vec, beta, M;
                                                    tol = 1e-6,
                                                    max_iters = 60,
                                                    return_diagnostics = false)

                # Normalize by (1-β)/N
                L_norm = (1 - beta) * L_star / N

                # 2) Simulate policies (MP, myopic, random, round-robin),
                # using same seed for common randomness
                res = simulate_multi_project_policies(p_vec, q_vec, r_vec, beta, M;
                                                      T = T_policy,
                                                      nruns = nruns_policy,
                                                      conf_level = conf_level,
                                                      seed = case_seed)

                J_mp     = res["mp_index"].mean
                J_myopic = res["myopic"].mean
                J_rand   = haskey(res, "random")      ? res["random"].mean      : NaN
                J_rr     = haskey(res, "round_robin") ? res["round_robin"].mean : NaN

                push!(rows, (
                    pA = pA,
                    qA = qA,
                    pB = pB,
                    qB = qB,
                    beta = beta,
                    N = N,
                    prop_type_A = prop_type_A,
                    nA = nA,
                    nB = nB,
                    M = M,
                    M_ratio = ratio,
                    T = T_policy,
                    nruns = nruns_policy,
                    L_norm = L_norm,
                    J_mp = J_mp,
                    J_myopic = J_myopic,
                    J_random = J_rand,
                    J_roundrobin = J_rr,
                ))

                # ---- NEW: progress / ETA ----
                done_cases += 1
                if show_progress
                    nowt = time()
                    if done_cases == 1 || (done_cases % update_every == 0) || (nowt - t_last ≥ min_seconds_between_updates)
                        elapsed = nowt - t0
                        rate = done_cases / max(elapsed, 1e-9)          # cases/sec
                        eta  = (total_cases - done_cases) / max(rate, 1e-9)

                        pct = 100 * done_cases / total_cases
                        finish_time = Dates.now() + Dates.Second(round(Int, eta))

                        println("Progress: $(round(pct; digits=1))%  ($done_cases/$total_cases)  ",
                                "elapsed=", _fmt_seconds(elapsed), "  ",
                                "ETA≈", _fmt_seconds(eta), "  ",
                                "finish≈", Dates.format(finish_time, "yyyy-mm-dd HH:MM:SS"))
                        flush(stdout)
                        t_last = nowt
                    end
                end
            end
        end
    end

    df = DataFrame(rows)
    CSV.write(filename, df)
    return df
end

In [ ]:
using CSV
using DataFrames
using Statistics

"""
    analyze_two_type_grid(filename; print_summary=true)

Read the CSV produced by `run_two_type_grid_to_csv`, compute:

  - gaps to the Lagrangian bound for each policy: gap_mp, gap_myopic, gap_random, gap_rr
  - gap ratio gap_ratio_mp_over_myopic = gap_mp / gap_myopic
  - reward ratio reward_ratio_mp_over_myopic = J_mp / J_myopic

Return:

  - df::DataFrame with added columns gap_*, gap_ratio_*, reward_ratio_*
  - stats::Dict{String,Any} with summary statistics for each gap and ratio.

If `print_summary=true`, print a tidy multi-line summary.
"""
function analyze_two_type_grid(filename::AbstractString; print_summary::Bool = true)
    df = CSV.read(filename, DataFrame)

    # Avoid division by zero in gaps
    valid_L = df.L_norm .> 0
    if !all(valid_L)
        @warn "Some rows have non-positive L_norm; gaps will be NaN there."
    end

    # Compute gaps to the bound (L_norm - J)/L_norm
    df.gap_mp     = similar(df.L_norm)
    df.gap_myopic = similar(df.L_norm)
    df.gap_random = similar(df.L_norm)
    df.gap_rr     = similar(df.L_norm)

    df.gap_mp[valid_L]     .= (df.L_norm[valid_L] .- df.J_mp[valid_L])       ./ df.L_norm[valid_L]
    df.gap_myopic[valid_L] .= (df.L_norm[valid_L] .- df.J_myopic[valid_L])   ./ df.L_norm[valid_L]
    df.gap_random[valid_L] .= (df.L_norm[valid_L] .- df.J_random[valid_L])   ./ df.L_norm[valid_L]
    df.gap_rr[valid_L]     .= (df.L_norm[valid_L] .- df.J_roundrobin[valid_L])./ df.L_norm[valid_L]

    # Initialize gaps for invalid rows to NaN
    df.gap_mp[.!valid_L]     .= NaN
    df.gap_myopic[.!valid_L] .= NaN
    df.gap_random[.!valid_L] .= NaN
    df.gap_rr[.!valid_L]     .= NaN

    # ------------------------------------------------------------------
    # Gap ratio: gap_ratio_mp_over_myopic = gap_mp / gap_myopic
    # Reward ratio: reward_ratio_mp_over_myopic = J_mp / J_myopic
    # ------------------------------------------------------------------
    n = nrow(df)
    df.gap_ratio_mp_over_myopic    = fill(NaN, n)
    df.reward_ratio_mp_over_myopic = fill(NaN, n)

    eps = 1e-8

    for i in 1:n
        gm = df.gap_mp[i]
        gy = df.gap_myopic[i]
        Jm = df.J_mp[i]
        Jy = df.J_myopic[i]

        # gap ratio: only if both gaps finite and |gap_myopic| not tiny
        if isfinite(gm) && isfinite(gy) && abs(gy) > eps
            df.gap_ratio_mp_over_myopic[i] = gm / gy
        end

        # reward ratio: only if J_myopic not tiny and both finite
        if isfinite(Jm) && isfinite(Jy) && abs(Jy) > eps
            df.reward_ratio_mp_over_myopic[i] = Jm / Jy
        end
    end

    # ------------------------------------------------------------------
    # Summary statistics
    # ------------------------------------------------------------------
    stats = Dict{String,Any}()

    # Helper to compute stats for a given column name
    function col_stats(colname::String)
        v = df[!, colname]
        v = v[.!isnan.(v) .& .!isinf.(v)]
        if isempty(v)
            return "no data"
        else
            return (
                mean   = mean(v),
                std    = std(v),
                min    = minimum(v),
                q25    = quantile(v, 0.25),
                median = quantile(v, 0.50),
                q75    = quantile(v, 0.75),
                max    = maximum(v),
            )
        end
    end

    # Gaps
    for col in ("gap_mp", "gap_myopic", "gap_random", "gap_rr")
        stats[col] = col_stats(col)
    end

    # Ratios
    stats["gap_ratio_mp_over_myopic"]    = col_stats("gap_ratio_mp_over_myopic")
    stats["reward_ratio_mp_over_myopic"] = col_stats("reward_ratio_mp_over_myopic")

    # ------------------------------------------------------------------
    # Pretty printing
    # ------------------------------------------------------------------
    if print_summary
        fmt(x) = lpad(string(round(x; digits = 4)), 8)

        println("Summary of gaps to Lagrangian bound (per-policy):")
        println(rpad("metric", 18), "  mean     std      min      q25      med      q75      max")
        println(repeat("-", 90))
        for name in ("gap_mp", "gap_myopic", "gap_random", "gap_rr")
            st = stats[name]
            if st === "no data"
                println(rpad(name, 18), "  (no data)")
            else
                line = string(
                    rpad(name, 18), "  ",
                    fmt(st.mean),   " ",
                    fmt(st.std),    " ",
                    fmt(st.min),    " ",
                    fmt(st.q25),    " ",
                    fmt(st.median), " ",
                    fmt(st.q75),    " ",
                    fmt(st.max)
                )
                println(line)
            end
        end

        println()
        println("Summary of ratios (MP vs Myopic):")
        println(rpad("metric", 28), "  mean     std      min      q25      med      q75      max")
        println(repeat("-", 100))

        for name in ("gap_ratio_mp_over_myopic", "reward_ratio_mp_over_myopic")
            st = stats[name]
            if st === "no data"
                println(rpad(name, 28), "  (no data)")
            else
                line = string(
                    rpad(name, 28), "  ",
                    fmt(st.mean),   " ",
                    fmt(st.std),    " ",
                    fmt(st.min),    " ",
                    fmt(st.q25),    " ",
                    fmt(st.median), " ",
                    fmt(st.q75),    " ",
                    fmt(st.max)
                )
                println(line)
            end
        end
    end

    return df, stats
end

"""
    mp_vs_myopic_cases(df;
                       gap_ratio_thresh=0.7,
                       reward_ratio_thresh=1.03,
                       print_summary=true)

Given a DataFrame `df` (as returned by `analyze_two_type_grid`) with columns

  - gap_ratio_mp_over_myopic = gap_mp / gap_myopic
  - reward_ratio_mp_over_myopic = J_mp / J_myopic,

identify:

  1. all cases where gap_ratio_mp_over_myopic < gap_ratio_thresh
     (MP closer to the bound than myopic, by at least that factor);
  2. all cases where reward_ratio_mp_over_myopic > reward_ratio_thresh
     (MP reward at least that multiplicative factor above myopic).

Returns `(df_gap_better, df_reward_better)` as DataFrames.

If `print_summary=true`, prints counts and a few representative rows for each set,
with constant columns and some “uninteresting” ones dropped (beta, M, N, nA).
"""
function mp_vs_myopic_cases(df::DataFrame;
                            gap_ratio_thresh::Float64 = 0.7,
                            reward_ratio_thresh::Float64 = 1.03,
                            print_summary::Bool = true)

    gap_ratio    = df.gap_ratio_mp_over_myopic
    reward_ratio = df.reward_ratio_mp_over_myopic

    valid_gap    = .!isnan.(gap_ratio) .& .!isinf.(gap_ratio)
    valid_reward = .!isnan.(reward_ratio) .& .!isinf.(reward_ratio)

    # 1. MP clearly closer to bound than myopic
    gap_better_mask = valid_gap .& (gap_ratio .< gap_ratio_thresh)

    # 2. MP clearly better in reward than myopic
    reward_better_mask = valid_reward .& (reward_ratio .> reward_ratio_thresh)

    df_gap_better     = df[gap_better_mask, :]
    df_reward_better  = df[reward_better_mask, :]

    # Helper: drop non-varying columns and some explicitly unwanted ones
    function reduce_columns(df_sub::DataFrame)
        if nrow(df_sub) == 0
            return df_sub
        end
        # Always hide these if present
        skip = Set(Symbol.(["beta", "M", "N", "nA"]))
        keep = Symbol[]
        for name in names(df_sub)
            col_sym = Symbol(name)
            if col_sym in skip
                continue
            end
            # keep column only if it varies within this subset
            if length(unique(df_sub[!, name])) > 1
                push!(keep, col_sym)
            end
        end
        if isempty(keep)
            return df_sub
        else
            return select(df_sub, keep; copycols=false)
        end
    end

    if print_summary
        println("\nMP vs Myopic comparison:")
        println("  MP closer to bound than myopic (gap_ratio < $(gap_ratio_thresh)): ",
                nrow(df_gap_better))
        println("  MP higher reward than myopic (reward_ratio > $(reward_ratio_thresh)): ",
                nrow(df_reward_better))

        if nrow(df_gap_better) > 0
            println("\nExamples where MP has much smaller gap than myopic:")
            df_g_red = reduce_columns(df_gap_better)
            show(first(df_g_red, min(5, nrow(df_g_red))))
            println()
        end

        if nrow(df_reward_better) > 0
            println("\nExamples where MP has significantly higher reward than myopic:")
            df_r_red = reduce_columns(df_reward_better)
            show(first(df_r_red, min(5, nrow(df_r_red))))
            println()
        end
    end

    # df_mp_gap_better and df_mp_reward_better returned by mp_vs_myopic_cases
    show(first(df_mp_gap_better, 5); allcols=true)
    show(first(df_mp_reward_better, 5); allcols=true)

    return df_gap_better, df_reward_better
end

In [ ]:
p_grid = [0.01, 0.05, 0.10, 0.20, 0.30, 0.35]
q_grid = [0.01, 0.05, 0.10, 0.20, 0.30, 0.35]

df = run_two_type_grid_to_csv("two_type_grid_results.csv";
                              p_grid = collect(p_grid),
                              q_grid = collect(q_grid),
                              beta = 0.99,
                              N = 100,
                              prop_type_A_list = [0.1, 0.3, 0.5, 0.7, 0.9],      # 10%, 30%, 50% A
                              M_ratios = [0.05, 0.1, 0.2, 0.3, 0.4, 0.5],        # capacities
                              T_policy = 300,
                              nruns_policy = 1000,
                              delta_rho = 0.05,
                              base_seed = 1234)

In [ ]:
df_res, stats = analyze_two_type_grid("two_type_grid_results.csv"; print_summary=true)

df_mp_gap_better, df_mp_reward_better =
    mp_vs_myopic_cases(df_res;
                       gap_ratio_thresh=0.7,
                       reward_ratio_thresh=1.03,
                       print_summary=true)

In [ ]:
using CSV
using DataFrames
using Statistics

# ----------------------------------------------------------
# 1. Load the CSV produced by run_two_type_grid_to_csv
# ----------------------------------------------------------
df = CSV.read("two_type_grid_1000.csv", DataFrame)

println("Loaded two_type_grid_1000.csv with ", nrow(df), " rows.")
println("Columns: ", names(df))

# Expected relevant columns from the function you gave:
# :pA, :qA, :pB, :qB,
# :beta, :N,
# :prop_type_A, :nA, :nB,
# :M, :M_ratio,
# :T, :nruns,
# :L_norm,
# :J_mp, :J_myopic, :J_random, :J_roundrobin

# ----------------------------------------------------------
# 2. Compute relative Lagrangian gaps (in percent)
#    gap_pi = 100 * (L_norm - J_pi) / L_norm
# ----------------------------------------------------------

# Protect against L_norm == 0 just in case (should not happen here)
function rel_lagrangian_gap(L_norm, J)
    if isnan(J) || L_norm ≤ 0
        return NaN
    else
        return 100 * (L_norm - J) / L_norm
    end
end

df.gap_mp        = map(rel_lagrangian_gap, df.L_norm, df.J_mp)
df.gap_myopic    = map(rel_lagrangian_gap, df.L_norm, df.J_myopic)
df.gap_random    = map(rel_lagrangian_gap, df.L_norm, df.J_random)
df.gap_roundrobin = map(rel_lagrangian_gap, df.L_norm, df.J_roundrobin)

# ----------------------------------------------------------
# 3. Ratio r = gap_myopic / gap_mp, and bad-myopic flag r ≥ 1.3
# ----------------------------------------------------------

function safe_ratio(num, den)
    if isnan(num) || isnan(den) || den == 0
        return NaN
    else
        return num / den
    end
end

df.ratio_myopic_mp = map(safe_ratio, df.gap_myopic, df.gap_mp)

threshold_ratio = 1.3  # myopic gap at least 30% larger than MP gap
df.is_bad_myopic = df.ratio_myopic_mp .>= threshold_ratio

n_total = nrow(df)
n_bad   = count(df.is_bad_myopic)

println("\nTotal grid points: ", n_total)
println("Bad myopic points (ratio ≥ ", threshold_ratio, "): ",
        n_bad, " (", round(100 * n_bad / n_total, digits = 1), "%)")

# ----------------------------------------------------------
# 4. Look at the worst myopic cases
# ----------------------------------------------------------
bad_df = filter(:is_bad_myopic => ==(true), df)
bad_df_sorted = sort(bad_df, :ratio_myopic_mp, rev = true)

println("\nTop 10 worst myopic / MP ratios:")
first(bad_df_sorted, 10) |> println

# This prints rows with:
# pA, qA, pB, qB, prop_type_A, M_ratio, gap_mp, gap_myopic, ratio_myopic_mp, ...

# ----------------------------------------------------------
# 5. Qualitative summaries: where do bad cases concentrate?
#    (a) by capacity ratio M_ratio
# ----------------------------------------------------------

println("\nShare of bad myopic cases by M_ratio:")
by_M = combine(groupby(df, :M_ratio)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; M_ratio = first(sdf.M_ratio),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_M)

# (b) by prop_type_A
println("\nShare of bad myopic cases by prop_type_A:")
by_rhoA = combine(groupby(df, :prop_type_A)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; prop_type_A = first(sdf.prop_type_A),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_rhoA)

# (c) by pA and qA (type A parameters)
println("\nShare of bad myopic cases by pA:")
by_pA = combine(groupby(df, :pA)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; pA = first(sdf.pA),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_pA)

println("\nShare of bad myopic cases by qA:")
by_qA = combine(groupby(df, :qA)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; qA = first(sdf.qA),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_qA)

# (d) by pB and qB (type B parameters)
println("\nShare of bad myopic cases by pB:")
by_pB = combine(groupby(df, :pB)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; pB = first(sdf.pB),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_pB)

println("\nShare of bad myopic cases by qB:")
by_qB = combine(groupby(df, :qB)) do sdf
    n = nrow(sdf)
    n_bad = count(sdf.is_bad_myopic)
    (; qB = first(sdf.qB),
       n_total = n,
       n_bad = n_bad,
       frac_bad = n_bad / n)
end
println(by_qB)

In [ ]:
using CSV
using DataFrames
using Statistics
using Printf

# ----------------------------------------------------------
# 1. Load results (if not already loaded)
# ----------------------------------------------------------
# If you already have `df` from before, you can skip this line.
# df = CSV.read("two_type_grid_results.csv", DataFrame)
df = CSV.read("two_type_grid_1000.csv", DataFrame)

println("Loaded two_type_grid_results.csv with ", nrow(df), " rows.")

# Expected columns: :pA, :qA, :pB, :qB, :prop_type_A, :M_ratio, :J_mp, :J_myopic, ...

# ----------------------------------------------------------
# 2. Compute reward ratio R = J_mp / J_myopic
# ----------------------------------------------------------

function safe_ratio(num, den)
    if isnan(num) || isnan(den) || den == 0
        return NaN
    else
        return num / den
    end
end

df.ratio_mp_myopic = map(safe_ratio, df.J_mp, df.J_myopic)

# Drop NaNs if any (should not happen in your current setup, but just in case)
ratio_vals = filter(!isnan, df.ratio_mp_myopic)

# ----------------------------------------------------------
# 3. Summary statistics for J_mp / J_myopic
# ----------------------------------------------------------

ratio_mean = mean(ratio_vals)
ratio_std  = std(ratio_vals)
ratio_min  = minimum(ratio_vals)
q25, q50, q75 = quantile(ratio_vals, (0.25, 0.50, 0.75))
ratio_max  = maximum(ratio_vals)

println("\nSummary of reward ratio J_mp / J_myopic:")
@printf("mean   = %.4f\n", ratio_mean)
@printf("std    = %.4f\n", ratio_std)
@printf("min    = %.4f\n", ratio_min)
@printf("q25    = %.4f\n", q25)
@printf("median = %.4f\n", q50)
@printf("q75    = %.4f\n", q75)
@printf("max    = %.4f\n", ratio_max)

# If you want a small DataFrame that you can use to format a LaTeX table:
summary_df = DataFrame(
    metric = ["J_mp / J_myopic"],
    mean   = [ratio_mean],
    std    = [ratio_std],
    min    = [ratio_min],
    q25    = [q25],
    median = [q50],
    q75    = [q75],
    max    = [ratio_max],
)

println("\nSummary DataFrame:")
println(summary_df)

# ----------------------------------------------------------
# 4. Top 10 worst cases (largest J_mp / J_myopic)
# ----------------------------------------------------------

df_nonan = filter(:ratio_mp_myopic => x -> !isnan(x), df)
df_sorted = sort(df_nonan, :ratio_mp_myopic, rev = true)

worst10 = first(df_sorted, 10)

# Keep only the requested columns
worst10_small = select(
    worst10,
    :pA, :qA, :pB, :qB, :prop_type_A, :M_ratio, :ratio_mp_myopic
)

println("\nTop 10 instances by reward ratio J_mp / J_myopic:")
println(worst10_small)

# ----------------------------------------------------------
# 5. (Optional) Print worst 10 in LaTeX-friendly format
# ----------------------------------------------------------

println("\nLaTeX rows for worst 10 (pA, qA, pB, qB, rho_A, M/N, ratio):")
for row in eachrow(worst10_small)
    @printf("%.2f & %.2f & %.2f & %.2f & %.2f & %.2f & %.3f \\\\\n",
            row.pA, row.qA, row.pB, row.qB,
            row.prop_type_A, row.M_ratio, row.ratio_mp_myopic)
end

In [ ]:
using CSV
using DataFrames
using Statistics
using Printf

# -----------------------------
# Load
# -----------------------------
df = CSV.read("two_type_grid_1000.csv", DataFrame)
@info "Loaded" nrow(df) names(df)

# -----------------------------
# Helpers
# -----------------------------
# Relative Lagrangian gap in percent: 100*(L - J)/L
rel_gap(L, J) = (isnan(J) || L <= 0) ? NaN : 100 * (L - J) / L

safe_ratio(num, den) = (isnan(num) || isnan(den) || den == 0) ? NaN : num / den

# Summary statistics function (matches your table columns)
function summarize_vec(v::AbstractVector{<:Real})
    vv = collect(skipmissing(v))
    vv = filter(!isnan, vv)
    return (
        mean   = mean(vv),
        std    = std(vv),
        min    = minimum(vv),
        q25    = quantile(vv, 0.25),
        median = quantile(vv, 0.50),
        q75    = quantile(vv, 0.75),
        max    = maximum(vv),
    )
end

# Nice LaTeX row printer
function latex_row(name::AbstractString, s; digits=2)
    @printf "%s & %.*f & %.*f & %.*f & %.*f & %.*f & %.*f & %.*f \\\\\n" name digits s.mean digits s.std digits s.min digits s.q25 digits s.median digits s.q75 digits s.max
end

# -----------------------------
# Compute gaps + ratio
# -----------------------------
df.gap_mp         = map(rel_gap, df.L_norm, df.J_mp)
df.gap_myopic     = map(rel_gap, df.L_norm, df.J_myopic)
df.gap_random     = map(rel_gap, df.L_norm, df.J_random)
df.gap_roundrobin = map(rel_gap, df.L_norm, df.J_roundrobin)

df.ratio_myopic_mp = map(safe_ratio, df.gap_myopic, df.gap_mp)

# -----------------------------
# TABLE A (Main paper): gap summary for 4 policies
# -----------------------------
summ_mp     = summarize_vec(df.gap_mp)
summ_myopic = summarize_vec(df.gap_myopic)
summ_rand   = summarize_vec(df.gap_random)
summ_rr     = summarize_vec(df.gap_roundrobin)

gap_summary = DataFrame(
    Policy = ["MP index", "Myopic index", "Random", "Round-robin"],
    mean   = [summ_mp.mean, summ_myopic.mean, summ_rand.mean, summ_rr.mean],
    std    = [summ_mp.std,  summ_myopic.std,  summ_rand.std,  summ_rr.std],
    min    = [summ_mp.min,  summ_myopic.min,  summ_rand.min,  summ_rr.min],
    q25    = [summ_mp.q25,  summ_myopic.q25,  summ_rand.q25,  summ_rr.q25],
    median = [summ_mp.median, summ_myopic.median, summ_rand.median, summ_rr.median],
    q75    = [summ_mp.q75,  summ_myopic.q75,  summ_rand.q75,  summ_rr.q75],
    max    = [summ_mp.max,  summ_myopic.max,  summ_rand.max,  summ_rr.max],
)

println("\n=== TABLE A: Summary of relative Lagrangian gaps (percent) ===")
println(gap_summary)

println("\nLaTeX rows for tab:gaps-summary (mean/std/min/q25/median/q75/max):")
latex_row("MP index",      summ_mp;     digits=2)
latex_row("Myopic index",  summ_myopic; digits=2)
latex_row("Random",        summ_rand;   digits=2)
latex_row("Round-robin",   summ_rr;     digits=2)

# -----------------------------
# TABLE C (Main paper, optional): reward ratio summary J_mp / J_myopic
# -----------------------------
df.ratio_J_mp_myopic = df.J_mp ./ df.J_myopic
summ_ratio = summarize_vec(df.ratio_J_mp_myopic)

ratio_summary = DataFrame(
    metric = ["J_mp / J_myopic"],
    mean   = [summ_ratio.mean],
    std    = [summ_ratio.std],
    min    = [summ_ratio.min],
    q25    = [summ_ratio.q25],
    median = [summ_ratio.median],
    q75    = [summ_ratio.q75],
    max    = [summ_ratio.max],
)

println("\n=== TABLE C: Summary of reward ratio J_mp / J_myopic ===")
println(ratio_summary)

# -----------------------------
# TABLE B (Supplement): top-10 worst myopic/MP gap ratios
# -----------------------------
worst10 = sort(df, :ratio_myopic_mp, rev=true)[1:10, [
    :pA, :qA, :pB, :qB, :prop_type_A, :M_ratio,
    :gap_mp, :gap_myopic, :gap_random, :gap_roundrobin,
    :ratio_myopic_mp
]]

println("\n=== TABLE B: Top 10 by ratio gap_myopic / gap_mp (supplement) ===")
println(worst10)

println("\nLaTeX rows for Table S1 (pA,qA,pB,qB,rhoA,M/N,gap_MP,gap_myopic,gap_random,gap_rr,ratio):")
for r in eachrow(worst10)
    @printf("%.2f & %.2f & %.2f & %.2f & %.2f & %.2f & %.3f & %.3f & %.3f & %.3f & %.3f \\\\\n",
            r.pA, r.qA, r.pB, r.qB, r.prop_type_A, r.M_ratio,
            r.gap_mp, r.gap_myopic, r.gap_random, r.gap_roundrobin,
            r.ratio_myopic_mp)
end
